In [ ]:
import os
import sys
import time
import json
import warnings
from pathlib import Path
from datetime import datetime
import urllib.request
import zipfile
import tempfile
import shutil
import hashlib
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import torch
import monai
import ants
import SimpleITK as sitk
import nibabel as nib
from scipy import ndimage
from skimage import measure, morphology, exposure
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
import seaborn as sns
from huggingface_hub import hf_hub_download


import os
import shutil 
import tempfile
from pathlib import Path
import json
import numpy as np
import nibabel as nib
from datetime import datetime
import urllib.request
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy import ndimage
import warnings


print(f"PyTorch version: {torch.__version__}")
print(f"MONAI version: {monai.__version__}")
print(f"ANTsPy version: {ants.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


# ── Configuration globale du sujet ───────────────────────────────────────
# CHANGER UNIQUEMENT ICI POUR UTILISER UN AUTRE SUJET
SUBJECT_RAW     = "A00043299"
SUBJECT_ID      = f"{SUBJECT_RAW}_NFB3_T1w"
SUBJECT_BIDS_ID = SUBJECT_ID
GS_BASE         = Path("datasets/NFBS/NFBS_Dataset/NFBS_Dataset")
if not GS_BASE.exists():
    GS_BASE     = Path("gold_standart/NFBS_Dataset/NFBS_Dataset")
T1_RAW_PATH     = GS_BASE / SUBJECT_RAW / f"sub-{SUBJECT_RAW}_ses-NFB3_T1w.nii.gz"
GS_MASK_PATH    = GS_BASE / SUBJECT_RAW / f"sub-{SUBJECT_RAW}_ses-NFB3_T1w_brainmask.nii.gz"
DERIVATIVES_DIR = Path("outputs/multisubject_eval") / SUBJECT_RAW
ANAT_DIR        = DERIVATIVES_DIR / "anat"
ANAT_DIR.mkdir(parents=True, exist_ok=True)
BIDS_WORK_DIR   = Path("sam_neuroadapt_v2_real_atlases/results/bids_dataset")
BIDS_WORK_DIR.mkdir(parents=True, exist_ok=True)
# Aliases pour compatibilité avec les cellules suivantes
subject_id      = SUBJECT_ID
subject_bids_id = SUBJECT_BIDS_ID
derivatives_dir = DERIVATIVES_DIR
anat_dir        = ANAT_DIR
print(f"Sujet actif : {SUBJECT_RAW}  |  T1={T1_RAW_PATH.name}  |  Sortie={ANAT_DIR}")
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
import os
import numpy as np
import nibabel as nib
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import ndimage
import warnings
from nilearn.image import resample_img
import pandas as pd

# Suppression des warnings non-critiques
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Configuration des Chemins ---
BASE_DIR = Path('./sam_neuroadapt_v2_real_atlases')
DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
ATLASES_DIR = BASE_DIR / 'atlases'
RESULTS_DIR = BASE_DIR / 'results'

# Vérifier que les répertoires existent
for dir_path in [DATA_DIR, MODELS_DIR, ATLASES_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Configuration des répertoires terminée")

# --- Fonction pour inspecter les atlas ---
def inspect_atlas(filepath, title=None, display_slices=True, figure_size=(12, 6)):
    
    try:
        img = nib.load(filepath)
        data = img.get_fdata()
        zooms = img.header.get_zooms()
        unique_vals = np.unique(data)
        nonzero_voxels = np.sum(data > 0)
        total_voxels = data.size
        
        # Informations générales
        info = {
            "Dimensions": data.shape,
            "Résolution (mm)": zooms[:3],
            "Valeurs uniques": unique_vals if len(unique_vals) < 10 else f"{len(unique_vals)} valeurs - min:{np.min(unique_vals)}, max:{np.max(unique_vals)}",
            "Voxels non-zéro": f"{nonzero_voxels} ({nonzero_voxels/total_voxels:.2%} du volume)"
        }
        
        # Affichage des informations
        title_display = title or Path(filepath).name
        print(f"\n--- Informations sur {title_display} ---")
        for key, value in info.items():
            print(f"{key}: {value}")
        
        # Affichage des coupes si demandé
        if display_slices and nonzero_voxels > 0:
            # Trouver des coupes contenant des données
            z_sums = np.sum(data, axis=(0, 1))
            y_sums = np.sum(data, axis=(0, 2))
            x_sums = np.sum(data, axis=(1, 2))
            
            if np.any(z_sums > 0) and np.any(y_sums > 0) and np.any(x_sums > 0):
                z_slice = np.argmax(z_sums)
                y_slice = np.argmax(y_sums)
                x_slice = np.argmax(x_sums)
                
                plt.figure(figsize=figure_size)
                
                # Coupe axiale (z)
                plt.subplot(1, 3, 1)
                plt.imshow(data[:, :, z_slice].T, cmap='hot', interpolation='nearest')
                plt.title(f"Axial (z={z_slice})")
                plt.colorbar(fraction=0.046, pad=0.04)
                plt.axis('off')
                
                # Coupe coronale (y)
                plt.subplot(1, 3, 2)
                plt.imshow(data[:, y_slice, :].T, cmap='hot', interpolation='nearest')
                plt.title(f"Coronal (y={y_slice})")
                plt.colorbar(fraction=0.046, pad=0.04)
                plt.axis('off')
                
                # Coupe sagittale (x)
                plt.subplot(1, 3, 3)
                plt.imshow(data[x_slice, :, :].T, cmap='hot', interpolation='nearest')
                plt.title(f"Sagittal (x={x_slice})")
                plt.colorbar(fraction=0.046, pad=0.04)
                plt.axis('off')
                
                plt.suptitle(f"Coupes de {title_display}")
                plt.tight_layout()
                plt.show()
    except Exception as e:
        print(f"Erreur lors de l'inspection de {filepath}: {e}")

# --- Vérification des atlas déjà téléchargés ---
print("\n--- Vérification des atlas disponibles dans", ATLASES_DIR, "---")

atlas_paths = {
    "MNI_T1_TEMPLATE": ATLASES_DIR / "MNI152_T1_2mm.nii.gz",
    "MNI_T1_TEMPLATE_TPLT": ATLASES_DIR / "MNI152NLin2009cAsym_T1w_2mm.nii.gz",
    "MNI_BRAIN_MASK": ATLASES_DIR / "MNI152_T1_2mm_brain_mask.nii.gz",
    "MNI_BRAIN_MASK_TPLT": ATLASES_DIR / "MNI152NLin2009cAsym_brainmask_2mm.nii.gz",
}

# Vérifier les atlas existants
available_atlas = {}
for name, path in atlas_paths.items():
    if path.exists():
        print(f"✓ {name} trouvé: {path}")
        available_atlas[name] = path
    else:
        print(f"⚠️ {name} non trouvé: {path}")

# --- Détection des masques de structures anatomiques ---
print("\n--- Détection des masques anatomiques ---")

# Définir les chemins des masques que nous allons utiliser
structure_paths = {
    "BRAIN_MASK": None,
}

# Choisir le masque cérébral parmi ceux disponibles
if "MNI_BRAIN_MASK_TPLT" in available_atlas:
    structure_paths["BRAIN_MASK"] = available_atlas["MNI_BRAIN_MASK_TPLT"]
    print(f"✓ Masque cérébral: MNI152NLin2009cAsym (template)")
elif "MNI_BRAIN_MASK" in available_atlas:
    structure_paths["BRAIN_MASK"] = available_atlas["MNI_BRAIN_MASK"]
    print(f"✓ Masque cérébral: MNI152_T1 standard")
else:
    print("⚠️ Aucun masque cérébral standard trouvé")

# --- Inspection des atlas sélectionnés ---
print("\n--- Inspection des atlas sélectionnés ---")

# Inspection du masque cérébral
if structure_paths["BRAIN_MASK"]:
    inspect_atlas(structure_paths["BRAIN_MASK"], "Masque cérébral MNI")

# --- Exemple d'image T1 pour testing ---
print("\n--- Vérification de l'image T1 d'exemple ---")
example_t1_path = T1_RAW_PATH  # sujet défini dans cell 0

if example_t1_path.exists():
    print(f"✓ Image T1 d'exemple trouvée: {example_t1_path}")
    # Visualiser un aperçu
    try:
        t1_img = nib.load(example_t1_path)
        t1_data = t1_img.get_fdata()
        
        plt.figure(figsize=(12, 4))
        z_idx = t1_data.shape[2] // 2
        plt.imshow(t1_data[:, :, z_idx].T, cmap='gray')
        plt.title(f"T1 Example (coupe axiale z={z_idx})")
        plt.colorbar(fraction=0.046, pad=0.04)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"⚠️ Erreur lors de la visualisation de l'image T1: {e}")
else:
    print(f"⚠️ Image T1 d'exemple non trouvée: {example_t1_path}")

# --- Configuration pour config.py ---
print("\n--- Configuration à ajouter à config.py ---")

config_paths = {}

# Masque cérébral
if structure_paths["BRAIN_MASK"]:
    config_paths["brain_mask"] = structure_paths["BRAIN_MASK"]

print("ATLAS_CONFIG = {")
for key, path in config_paths.items():
    print(f"    '{key}': Path('{path}'),")
print("}")

print("\nConfiguration des atlas pour SAM-NeuroAdapt v2 terminée!")

### Audit Installation

In [ ]:
# ── Fix B1 : résolution du conflit de chemin SAM2 ──────────────────────
# Le dossier sam2/ (repo) shadowe le package Python. On ajoute le repo
# à sys.path pour que 'import sam2' trouve sam2/sam2/ (le vrai package).
import sys as _sys
from pathlib import Path as _Path
_sam2_repo = (_Path('.') / 'sam2').resolve()
if _sam2_repo.exists() and str(_sam2_repo) not in _sys.path:
    _sys.path.insert(0, str(_sam2_repo))
del _sam2_repo, _Path, _sys
# ─────────────────────────────────────────────────────────────────────────

errors = []


if not torch.__version__.startswith('2.'):
    errors.append(f"PyTorch version should be ≥2.0, got {torch.__version__}")
if not int(monai.__version__.split('.')[0]) >= 1:
    errors.append(f"MONAI version should be ≥1.0, got {monai.__version__}")
if not torch.cuda.is_available():
    errors.append("CUDA not available - this notebook requires GPU acceleration")


try:
    import ants
    print(f"✓ ANTsPy version: {ants.__version__}")
except ImportError:
    errors.append("ANTsPy not found - needed for atlas registration")


files_to_check = {
    "MNI Brain Mask": ATLASES_DIR / "MNI152NLin2009cAsym_brainmask_2mm.nii.gz", 
    "T1 Example": T1_RAW_PATH
}

for name, path in files_to_check.items():
    if not path.exists():
        errors.append(f"{name} file not found at {path}")
    elif path.stat().st_size < 10000:  
        errors.append(f"{name} file at {path} is too small, may be corrupted")


atlas_to_check = {
    "Hippocampus AAL3": ATLASES_DIR / "AAL3_Hippocampus_2mm.nii.gz",
    "Entorhinal Cortex AAL3": ATLASES_DIR / "AAL3_EntorhinalCortex_2mm.nii.gz",
    "Harvard-Oxford Subcortical": ATLASES_DIR / "HarvardOxford-sub-maxprob-thr25-2mm.nii.gz"
}

atlas_ok = {}
for name, path in atlas_to_check.items():
    if path.exists():
        atlas_ok[name] = path
    else:
        print(f"⚠️ AVERTISSEMENT: {name} atlas non trouvé à {path}")


try:
    from preprocessing.config import ATLAS_CONFIG
    atlas_config_present = True
    print(f"{chr(10)}Configuration ATLAS_CONFIG trouvée")
    print(f"✓ ATLAS_CONFIG trouvé dans config.py avec {len(ATLAS_CONFIG)} entrées")
except Exception as e:
    atlas_config_present = False
    ATLAS_CONFIG = {}
    print(f"ℹ️ ATLAS_CONFIG non trouvé ou non importable — non bloquant pour la publi ({e})")


if errors:
    for error in errors:
        print(f"❌ ERROR: {error}")
    raise AssertionError("Installation or download errors detected")
else:
    print("✅ All installations and downloads verified!")
    
    
    print("\nFichiers principaux :")
    for name, path in files_to_check.items():
        print(f"- {name}: {path} ({path.stat().st_size / (1024*1024):.2f} MB)")
    
    
    print("\nAtlas disponibles :")
    for name, path in atlas_ok.items():
        atlas_img = nib.load(path)
        unique_labels = len(np.unique(atlas_img.get_fdata()))
        print(f"- {name}: {path.name} ({path.stat().st_size / (1024*1024):.2f} MB, {unique_labels} labels)")

    
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    
    t1_img = nib.load(files_to_check["T1 Example"])
    t1_data = t1_img.get_fdata()
    slice_idx = t1_data.shape[2] // 2
    axes[0].imshow(t1_data[:, :, slice_idx].T, cmap='gray')
    axes[0].set_title(f"T1 Example (slice {slice_idx})")
    axes[0].axis('off')
    
    
    mni_mask = nib.load(files_to_check["MNI Brain Mask"])
    mni_data = mni_mask.get_fdata()
    slice_idx = mni_data.shape[2] // 2
    axes[1].imshow(mni_data[:, :, slice_idx].T, cmap='gray')
    axes[1].set_title(f"MNI Brain Mask (slice {slice_idx})")
    axes[1].axis('off')
    
    
    if "Hippocampus AAL3" in atlas_ok:
        hippo_img = nib.load(atlas_ok["Hippocampus AAL3"])
        hippo_data = hippo_img.get_fdata()
        
        z_sums = np.sum(hippo_data, axis=(0, 1))
        slice_idx = np.argmax(z_sums) if np.any(z_sums > 0) else hippo_data.shape[2] // 2
        axes[2].imshow(hippo_data[:, :, slice_idx].T, cmap='hot')
        axes[2].set_title(f"Hippocampus Atlas (slice {slice_idx})")
        axes[2].axis('off')
    else:
        axes[2].text(0.5, 0.5, "Atlas hippocampe non disponible", 
                     horizontalalignment='center', verticalalignment='center')
        axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

    
    if atlas_config_present:
        print("\nConfiguration ATLAS_CONFIG:")
        for key, path in ATLAS_CONFIG.items():
            print(f"- {key}: {path}")
    
    print("\nLe système est prêt pour l'extraction du cerveau avec guidage par atlas!")

### Conversion DICOM → BIDS

In [ ]:
import os
import tempfile
import shutil
from pathlib import Path
import json
import numpy as np
import nibabel as nib
from datetime import datetime
import matplotlib.pyplot as plt


def dicom_to_nifti(dicom_dir, output_nifti_path):
        
    try:
        
        import pydicom
        
        
        dicom_files = sorted([str(f) for f in Path(dicom_dir).glob('*.dcm')])
        if not dicom_files:
            dicom_files = sorted([str(f) for f in Path(dicom_dir).rglob('*.dcm')])
        
        if not dicom_files:
            raise ValueError(f"Aucun fichier DICOM trouvé dans {dicom_dir}")
        
        
        print(f"Lecture de {len(dicom_files)} fichiers DICOM...")
        slices = [pydicom.dcmread(dcm) for dcm in dicom_files]
        
        
        try:
            slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
        except (AttributeError, KeyError):
            
            slices.sort(key=lambda x: int(getattr(x, 'InstanceNumber', 0)))
        
        
        img_shape = list(slices[0].pixel_array.shape)
        img_shape.append(len(slices))
        
        img_data = np.zeros(img_shape, dtype=slices[0].pixel_array.dtype)
        
        
        for i, s in enumerate(slices):
            img_data[:, :, i] = s.pixel_array
        
        
        first_slice = slices[0]
        
        
        if hasattr(first_slice, 'RescaleSlope') and hasattr(first_slice, 'RescaleIntercept'):
            slope = float(first_slice.RescaleSlope)
            intercept = float(first_slice.RescaleIntercept)
            img_data = img_data.astype(np.float32) * slope + intercept
        
        
        affine = np.eye(4)
        
        
        try:
            
            pixel_spacing = first_slice.PixelSpacing
            slice_thickness = float(first_slice.SliceThickness)
            
            
            image_orientation = first_slice.ImageOrientationPatient
            orientation_x = np.array(image_orientation[:3])
            orientation_y = np.array(image_orientation[3:])
            orientation_z = np.cross(orientation_x, orientation_y)
            
            position = np.array(first_slice.ImagePositionPatient)
            
            
            affine[:3, 0] = orientation_x * pixel_spacing[0]
            affine[:3, 1] = orientation_y * pixel_spacing[1]
            affine[:3, 2] = orientation_z * slice_thickness
            affine[:3, 3] = position
        except (AttributeError, KeyError):
            
            print("Informations de positionnement incomplètes, utilisation d'une affine standard")
            affine[0, 0] = affine[1, 1] = affine[2, 2] = 1.0
        
        
        img = nib.Nifti1Image(img_data, affine)
        
        
        img.header.set_xyzt_units(2)  
        
        
        output_nifti_path = Path(output_nifti_path)
        output_nifti_path.parent.mkdir(parents=True, exist_ok=True)
        nib.save(img, output_nifti_path)
        
        print(f"Conversion DICOM → NIfTI réussie: {output_nifti_path}")
        return output_nifti_path
        
    except ImportError:
        
        try:
            from nipype.interfaces.dcm2nii import Dcm2niix
            print("pydicom non disponible, utilisation de dcm2niix via nipype...")
            
            converter = Dcm2niix()
            converter.inputs.source_dir = str(dicom_dir)
            output_dir = Path(output_nifti_path).parent
            converter.inputs.output_dir = str(output_dir)
            converter.inputs.compress = 'y'
            converter.inputs.bids_format = False
            result = converter.run()
            
            
            nii_files = list(output_dir.glob("*.nii*"))
            if nii_files:
                
                shutil.copy(nii_files[0], output_nifti_path)
                
                for f in nii_files:
                    f.unlink()
                
                print(f"Conversion DICOM → NIfTI réussie avec dcm2niix: {output_nifti_path}")
                return output_nifti_path
            else:
                raise ValueError("dcm2niix n'a pas généré de fichier NIfTI")
        
        except Exception as e:
            print(f"Échec de la conversion DICOM → NIfTI avec dcm2niix: {e}")
            raise


def create_bids_structure(subject_id, session_id=None, nifti_path=None, output_dir=None, reorient=True):

    output_dir = Path(output_dir) if output_dir else Path('data/bids')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    
    dataset_desc = {
        "Name": "SAM-NeuroAdapt v2 Dataset",
        "BIDSVersion": "1.6.0",
        "DatasetType": "raw",
        "Authors": ["SAM-NeuroAdapt Team"],
        "HowToAcknowledge": "Please cite SAM-NeuroAdapt v2"
    }
    
    with open(output_dir / "dataset_description.json", "w") as f:
        json.dump(dataset_desc, f, indent=4)
    
    
    if session_id:
        subject_dir = output_dir / f"sub-{subject_id}" / f"ses-{session_id}" / "anat"
        bids_basename = f"sub-{subject_id}_ses-{session_id}_T1w"
    else:
        subject_dir = output_dir / f"sub-{subject_id}" / "anat"
        bids_basename = f"sub-{subject_id}"
    
    subject_dir.mkdir(parents=True, exist_ok=True)
    bids_nifti_path = subject_dir / f"{bids_basename}.nii.gz"
    
    
    if nifti_path:
        nifti_path = Path(nifti_path)
        if nifti_path.exists():
            
            img = nib.load(nifti_path)
            
            # if reorient:
            #     img = reorient_to_ras(img)
                
            nib.save(img, bids_nifti_path)
            
            metadata = {
                "Manufacturer": "Unknown",
                "ManufacturersModelName": "Unknown",
                "MagneticFieldStrength": 3,
                "ProtocolName": "T1w",
                "ScanningSequence": "GR",
                "SequenceVariant": "SP",
                "ScanOptions": "IR",
                "ImageType": ["ORIGINAL", "PRIMARY", "M", "ND", "NORM"],
                "SeriesDescription": "T1-weighted structural MRI",
                "AcquisitionDateTime": datetime.now().isoformat(),
            }
            
            with open(subject_dir / f"{bids_basename}.json", "w") as f:
                json.dump(metadata, f, indent=4)
    
    return output_dir


def verify_bids_for_neuro_adapt(bids_dir, subject_id, show_visualization=True):
 
    bids_dir = Path(bids_dir)
    
    
    required_files = [
        "dataset_description.json",
        f"sub-{subject_id}/anat/sub-{subject_id}.nii.gz",
        f"sub-{subject_id}/anat/sub-{subject_id}.json"
    ]
    
    missing_files = []
    for file_path in required_files:
        if not (bids_dir / file_path).exists():
            missing_files.append(file_path)
    
    
    t1_path = bids_dir / f"sub-{subject_id}/anat/sub-{subject_id}.nii.gz"
    if t1_path.exists():
        try:
            t1_img = nib.load(t1_path)
            t1_data = t1_img.get_fdata()
            
            
            
            if t1_data.ndim != 3:
                missing_files.append(f"{t1_path.name} (n'est pas une image 3D)")
            
            
            resolution = t1_img.header.get_zooms()[:3]
            if any(res > 2.0 for res in resolution):
                print(f"⚠️ AVERTISSEMENT: Résolution de {t1_path.name} ({resolution} mm) > 2mm recommandée")
                
            
            if np.max(t1_data) == 0:
                missing_files.append(f"{t1_path.name} (intensité nulle)")
                
            
            try:
                orientation = ''.join(nib.orientations.aff2axcodes(t1_img.affine))
                if orientation != 'RAS' and orientation != 'LAS':
                    print(f"⚠️ AVERTISSEMENT: Orientation de {t1_path.name} est {orientation}, RAS ou LAS recommandée")
            except Exception:
                print("⚠️ AVERTISSEMENT: Impossible de déterminer l'orientation")
                
            
            if show_visualization:
                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                
                
                slice_z = t1_data.shape[2] // 2
                slice_y = t1_data.shape[1] // 2
                slice_x = t1_data.shape[0] // 2
                
                axes[0].imshow(t1_data[slice_x, :, :].T, cmap='gray', aspect='auto')
                axes[0].set_title(f"Sagittal (x={slice_x})")
                axes[0].axis('off')
                
                axes[1].imshow(t1_data[:, slice_y, :].T, cmap='gray', aspect='auto')
                axes[1].set_title(f"Coronal (y={slice_y})")
                axes[1].axis('off')
                
                axes[2].imshow(t1_data[:, :, slice_z].T, cmap='gray', aspect='auto')
                axes[2].set_title(f"Axial (z={slice_z})")
                axes[2].axis('off')
                
                plt.suptitle(f"T1w du sujet {subject_id}", fontsize=16)
                plt.tight_layout()
                plt.show()
                
        except Exception as e:
            missing_files.append(f"{t1_path.name} (erreur: {e})")
    
    
    if missing_files:
        print("❌ Structure BIDS invalide pour SAM-NeuroAdapt v2")
        for file_path in missing_files:
            print(f"  - Fichier manquant ou invalide: {file_path}")
        return False
    else:
        print("✅ Structure BIDS valide pour SAM-NeuroAdapt v2")
        
        
        print("\nArborescence BIDS :")
        for root, dirs, files in os.walk(bids_dir):
            level = root.replace(str(bids_dir), '').count(os.sep)
            indent = '│   ' * level
            print(f"{indent}├── {os.path.basename(root)}/")
            for f in sorted(files):
                if f.startswith('.'):  
                    continue
                fsize = os.path.getsize(os.path.join(root, f)) / 1024  
                size_str = f"({fsize:.1f} KB)" if fsize < 1024 else f"({fsize/1024:.1f} MB)"
                print(f"{indent}│   ├── {f} {size_str}")
                
        return True
    
def reorient_to_ras(nifti_img):
    
    print("Réorientation de l'image vers RAS (Right-Anterior-Superior)...")
    try:
        
        img_ras = nib.as_closest_canonical(nifti_img)
        
        
        orientation = ''.join(nib.orientations.aff2axcodes(img_ras.affine))
        print(f"Orientation après réorientation: {orientation}")
        
        return img_ras
    except Exception as e:
        print(f"⚠️ Erreur lors de la réorientation: {e}")
        print("Utilisation de l'image originale.")
        return nifti_img

if __name__ == "__main__":
    
    print("Module de conversion DICOM vers BIDS chargé pour SAM-NeuroAdapt v2")

In [ ]:
"""
Préparation des données pour le pipeline SAM-NeuroAdapt v2 :
- Conversion d'images au format BIDS
- Validation de la structure pour le traitement par le pipeline
"""

# Utiliser l'image T1 d'exemple déjà téléchargée
t1_example_path = T1_RAW_PATH  # défini dans cell 0

# Identifiant du sujet
subject_id = SUBJECT_ID  # défini dans cell 0

# Création du répertoire de sortie BIDS
bids_dir = BIDS_WORK_DIR if 'BIDS_WORK_DIR' in globals() else RESULTS_DIR / "bids_dataset"

print(f"Étape 1: Conversion BIDS pour {subject_id}")

# Vérifier si l'image T1 existe
if not t1_example_path.exists():
    print(f"❌ Erreur: Image T1 d'exemple non trouvée à {t1_example_path}")
    
    # Option: Créer une image synthétique si nécessaire
    print("Création d'une image T1 synthétique simple...")
    synthetic_t1 = np.zeros((91, 109, 91), dtype=np.float32)  # Dimensions MNI standard 2mm
    x, y, z = np.indices(synthetic_t1.shape)
    center = (synthetic_t1.shape[0]//2, synthetic_t1.shape[1]//2, synthetic_t1.shape[2]//2)
    r = np.sqrt((x - center[0])**2 + (y - center[1])**2 + (z - center[2])**2)
    
    # Générer un gradient radial pour simuler l'intensité du cerveau
    synthetic_t1 = np.clip(100 - r/3, 0, 100).astype(np.float32)
    
    # Ajouter un peu de bruit pour plus de réalisme
    synthetic_t1 += np.random.normal(0, 5, synthetic_t1.shape)
    
    # Sauvegarder l'image synthétique
    t1_example_path = DATA_DIR / "synthetic_T1.nii.gz"
    
    # Créer une transformation affine basée sur MNI 2mm
    affine = np.eye(4)
    affine[0, 0] = affine[1, 1] = affine[2, 2] = 2.0  # Voxels de 2mm
    
    nib_img = nib.Nifti1Image(synthetic_t1, affine)
    nib.save(nib_img, t1_example_path)
    print(f"✅ Image T1 synthétique créée: {t1_example_path}")

# Création de la structure BIDS
bids_dir = create_bids_structure(
    subject_id=subject_id.replace("sub-", ""),  # Enlever le préfixe "sub-" s'il est présent
    nifti_path=t1_example_path,
    output_dir=bids_dir
)

print(f"✅ Structure BIDS créée dans {bids_dir}")

# Vérification de la structure BIDS pour SAM-NeuroAdapt v2
is_valid = verify_bids_for_neuro_adapt(
    bids_dir=bids_dir,
    subject_id=subject_id.replace("sub-", ""),
    show_visualization=True
)

if is_valid:
    print("\n=== Prêt pour l'extraction du cerveau ===")
    
    # Chemin complet pour l'extraction du cerveau
    t1_input_path = bids_dir / f"sub-{subject_id.replace('sub-', '')}" / "anat" / f"sub-{subject_id.replace('sub-', '')}.nii.gz"
    
    # Définir un chemin pour les dérivés de sortie
    bids_derivatives_dir = bids_dir / "derivatives" / "sam-neuroadapt"
    bids_derivatives_dir.mkdir(parents=True, exist_ok=True)
    
    output_brain_path = ANAT_DIR / f"sub-{subject_id.replace('sub-', '')}_desc-brain_T1w.nii.gz"
    output_brain_path.parent.mkdir(parents=True, exist_ok=True)
    
    # NOUVEAU: Recaler le masque cérébral MNI vers l'espace du sujet
    brain_mask_path = None
    
    # Vérifier si des masques cérébraux sont disponibles
    if 'structure_paths' in globals() and "BRAIN_MASK" in structure_paths and structure_paths["BRAIN_MASK"] is not None:
        print("\nRecalage du masque cérébral MNI vers l'espace du sujet...")
        try:
            import ants
            
            # Charger les images
            fixed_img = ants.image_read(str(t1_input_path))
            moving_img = ants.image_read(str(structure_paths["BRAIN_MASK"]))
            
            # Recalage affine rapide
            reg = ants.registration(
                fixed=fixed_img,
                moving=moving_img,
                type_of_transform='SyNRA',
                verbose=False
            )
            
            # Sauvegarder le masque recalé
            brain_mask_path = ANAT_DIR / f"sub-{subject_id.replace('sub-', '')}_desc-mni_mask.nii.gz"
            brain_mask_path.parent.mkdir(parents=True, exist_ok=True)
            ants.image_write(reg['warpedmovout'], str(brain_mask_path))
            
            print(f"✅ Masque cérébral recalé sauvegardé: {brain_mask_path}")
        except Exception as e:
            print(f"⚠️ Erreur lors du recalage du masque: {e}")
            brain_mask_path = None
    
    # Modification de la commande pour le prétraitement avec masque
    print("\nCommande à exécuter pour le prétraitement avec masque cérébral:")
    if brain_mask_path is not None:
        print(f"n4_corrected_path, n4_stats = n4_bias_correction(")
        print(f"    input_path='{t1_input_path}',")
        print(f"    output_path='{ANAT_DIR / f'sub-{subject_id.replace('sub-', '')}_desc-n4_T1w.nii.gz'}',")
        print(f"    mask_path='{brain_mask_path}',")  # Utilisation du masque recalé
        print(f"    shrink_factor=4,")
        print(f"    save_bias_field=True")
        print(f")")
        
        print(f"\nwhitestripe_path, ws_stats = whitestripe_normalize(")
        print(f"    input_path=n4_corrected_path,")
        print(f"    output_path='{ANAT_DIR / f'sub-{subject_id.replace('sub-', '')}_desc-n4_desc-ws_T1w.nii.gz'}',")
        print(f"    mask_path='{brain_mask_path}',")  # Utilisation du masque recalé
        print(f"    n_components=3")
        print(f")")
    else:
        # Version originale sans masque
        print(f"output_brain_path, output_mask_path = extract_brain(")
        print(f"    input_path='{t1_input_path}',")
        print(f"    output_path='{output_brain_path}',")
        print(f"    use_atlas_guidance=True,")
        print(f"    save_uncertainty_map=True,")
        print(f"    save_metrics=True")
        print(f")")
else:
    print("❌ La structure BIDS n'est pas valide pour SAM-NeuroAdapt v2")

### Audit Conversion BIDS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import json
import os
from pathlib import Path
from nilearn import plotting
from sklearn.cluster import KMeans

def check_bids_structure(bids_dir, subject_id):
 
    bids_dir = Path(bids_dir)
    subject_id = subject_id.replace("sub-", "")  # Assurer l'absence du préfixe
    

    required_files = [
        "dataset_description.json",
        f"sub-{subject_id}/anat/sub-{subject_id}.nii.gz",
        f"sub-{subject_id}/anat/sub-{subject_id}.json"
    ]
    

    validation_results = {
        "bids_valid": True,
        "missing_files": [],
        "warnings": [],
        "t1_properties": {},
        "json_metadata": {}
    }
    

    for file_path in required_files:
        full_path = bids_dir / file_path
        if not full_path.exists():
            validation_results["bids_valid"] = False
            validation_results["missing_files"].append(file_path)
    

    desc_path = bids_dir / "dataset_description.json"
    if desc_path.exists():
        try:
            with open(desc_path, 'r') as f:
                dataset_desc = json.load(f)
            
                required_fields = ["Name", "BIDSVersion"]
                for field in required_fields:
                    if field not in dataset_desc:
                        validation_results["warnings"].append(f"Champ requis {field} manquant dans dataset_description.json")
        except json.JSONDecodeError:
            validation_results["bids_valid"] = False
            validation_results["warnings"].append("dataset_description.json n'est pas un JSON valide")
    

    t1_path = bids_dir / f"sub-{subject_id}/anat/sub-{subject_id}.nii.gz"
    if t1_path.exists():
        try:
            t1_img = nib.load(t1_path)
            t1_data = t1_img.get_fdata()
            
        
            validation_results["t1_properties"] = {
                "shape": t1_data.shape,
                "dimensions": t1_data.ndim,
                "resolution_mm": t1_img.header.get_zooms()[:3],
                "data_type": str(t1_data.dtype),
                "min_intensity": float(np.min(t1_data)),
                "max_intensity": float(np.max(t1_data)),
                "mean_intensity": float(np.mean(t1_data)),
                "std_intensity": float(np.std(t1_data)),
                "non_zero_voxels": int(np.sum(t1_data > 0)),
                "total_voxels": int(t1_data.size),
                "non_zero_percentage": float(np.sum(t1_data > 0) / t1_data.size * 100)
            }
            
        
            if t1_data.ndim != 3:
                validation_results["bids_valid"] = False
                validation_results["warnings"].append(f"L'image T1w n'est pas 3D (dimensions: {t1_data.shape})")
            
        
            res = t1_img.header.get_zooms()[:3]
            if any(r > 2.0 for r in res):
                validation_results["warnings"].append(f"Résolution ({res} mm) supérieure à 2mm recommandée")
            
        
            if np.max(t1_data) - np.min(t1_data) < 1e-6:
                validation_results["warnings"].append("Contraste d'image très faible")
                
        
            if np.sum(t1_data > 0) / t1_data.size < 0.01:
                validation_results["warnings"].append("Moins de 1% de voxels non-nuls dans l'image")
                
        except Exception as e:
            validation_results["bids_valid"] = False
            validation_results["warnings"].append(f"Erreur lors de l'analyse de l'image T1w: {str(e)}")
    

    json_path = bids_dir / f"sub-{subject_id}/anat/sub-{subject_id}.json"
    if json_path.exists():
        try:
            with open(json_path, 'r') as f:
                metadata = json.load(f)
                validation_results["json_metadata"] = metadata             
                if "MagneticFieldStrength" not in metadata:
                    validation_results["warnings"].append("Champ MagneticFieldStrength manquant dans les métadonnées")
                
        except json.JSONDecodeError:
            validation_results["bids_valid"] = False
            validation_results["warnings"].append("Fichier de métadonnées JSON non valide")
    
    return validation_results

def analyze_t1_intensity_distribution(t1_path):
 
    try:
        t1_img = nib.load(t1_path)
        t1_data = t1_img.get_fdata()
        
    
        foreground_voxels = t1_data[t1_data > 0]
        
        if len(foreground_voxels) == 0:
            return {"error": "Pas de voxels non-nuls dans l'image"}
        
    
        p1, p25, p50, p75, p99 = np.percentile(foreground_voxels, [1, 25, 50, 75, 99])
        
    
    
        voxels_for_clustering = foreground_voxels.reshape(-1, 1)
        
    
        max_voxels = 100000  # Pour accélérer le calcul
        if len(voxels_for_clustering) > max_voxels:
            indices = np.random.choice(len(voxels_for_clustering), max_voxels, replace=False)
            voxels_for_clustering = voxels_for_clustering[indices]
        
        kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
        kmeans.fit(voxels_for_clustering)
        
    
        tissue_centers = sorted(kmeans.cluster_centers_.flatten())
        
    
        if len(tissue_centers) >= 3:
            csf_mean, gm_mean, wm_mean = tissue_centers
            gm_wm_contrast = (wm_mean - gm_mean) / (wm_mean + gm_mean) if (wm_mean + gm_mean) > 0 else 0
        else:
            gm_wm_contrast = 0
        
    
        hist, bin_edges = np.histogram(foreground_voxels, bins=100, range=(p1, p99))
        
     
        low_intensity_voxels = t1_data[(t1_data > 0) & (t1_data < p1)]
        noise_std = np.std(low_intensity_voxels) if len(low_intensity_voxels) > 0 else 0
        signal_mean = np.mean(foreground_voxels)
        snr = signal_mean / noise_std if noise_std > 0 else float('inf')
        
    
        intensity_stats = {
            "min": float(np.min(foreground_voxels)),
            "max": float(np.max(foreground_voxels)),
            "mean": float(np.mean(foreground_voxels)),
            "median": float(p50),
            "std": float(np.std(foreground_voxels)),
            "p1": float(p1),
            "p25": float(p25),
            "p75": float(p75),
            "p99": float(p99),
            "interquartile_range": float(p75 - p25),
            "histogram": {
                "counts": hist.tolist(),
                "bin_edges": bin_edges.tolist()
            },
            "tissue_centers": [float(c) for c in tissue_centers],
            "gm_wm_contrast": float(gm_wm_contrast),
            "estimated_snr": float(snr),
        
            "suitable_for_sam": gm_wm_contrast > 0.1 and snr > 5
        }
        
        return intensity_stats
    
    except Exception as e:
        return {"error": str(e)}

def visualize_t1_analysis(t1_path, intensity_stats=None):
 
    t1_img = nib.load(t1_path)
    t1_data = t1_img.get_fdata()
    

    fig = plt.figure(figsize=(18, 12))
    

    plt.subplot(2, 3, 1)
    plotting.plot_anat(
        t1_img, display_mode='ortho', cut_coords=(0, 0, 0),
        figure=fig, axes=plt.gca()
    )
    plt.title("Coupes anatomiques", fontsize=12)
    

    plt.subplot(2, 3, 2)
    foreground_voxels = t1_data[t1_data > 0]
    if len(foreground_voxels) > 0:
        plt.hist(foreground_voxels.flatten(), bins=100, color='blue', alpha=0.7)
        plt.xlabel("Intensité")
        plt.ylabel("Nombre de voxels")
        plt.title("Histogramme d'intensité (voxels non-nuls)", fontsize=12)
        
    
        if intensity_stats and "tissue_centers" in intensity_stats:
            for i, center in enumerate(intensity_stats["tissue_centers"]):
                tissue_name = ["CSF", "GM", "WM"][i] if i < 3 else f"Tissue {i+1}"
                plt.axvline(center, color=['cyan', 'green', 'red'][i % 3], 
                           linestyle='--', label=f"{tissue_name}: {center:.1f}")
            plt.legend()
    

    plt.subplot(2, 3, 3)
    slice_idx = t1_data.shape[2] // 2
    plt.imshow(t1_data[:, :, slice_idx].T, cmap='gray')
    plt.colorbar(label="Intensité")
    plt.title(f"Carte d'intensité (coupe axiale z={slice_idx})", fontsize=12)
    plt.axis('off')
    

    plt.subplot(2, 3, 4)
    mid_y = t1_data.shape[1] // 2
    mid_z = t1_data.shape[2] // 2
    intensity_profile = t1_data[:, mid_y, mid_z]
    plt.plot(intensity_profile)
    plt.xlabel("Position X")
    plt.ylabel("Intensité")
    plt.title(f"Profil d'intensité (y={mid_y}, z={mid_z})", fontsize=12)
    

    plt.subplot(2, 3, 5)
    if intensity_stats and "tissue_centers" in intensity_stats and len(intensity_stats["tissue_centers"]) >= 3:
        tissue_names = ["CSF", "GM", "WM"]
        tissue_values = intensity_stats["tissue_centers"][:3]
        
    
        plt.bar(tissue_names, tissue_values, color=['cyan', 'green', 'red'])
        plt.ylabel("Intensité moyenne")
        plt.title("Intensités des tissus estimées", fontsize=12)
        
    
        if "gm_wm_contrast" in intensity_stats:
            contrast = intensity_stats["gm_wm_contrast"]
            plt.annotate(f"Contraste GM-WM: {contrast:.3f}", 
                        xy=(0.5, 0.9), xycoords='axes fraction', 
                        ha='center', bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
    else:
        plt.text(0.5, 0.5, "Données insuffisantes pour\nestimer les tissus", 
                ha='center', va='center', transform=plt.gca().transAxes)
    

    plt.subplot(2, 3, 6)
    plt.axis('off')
    
    resolution = t1_img.header.get_zooms()[:3]
    info_text = [
        "ÉVALUATION POUR SAM-NEUROADAPT v2",
        "-----------------------------------",
        f"Dimensions: {t1_data.shape}",
        f"Résolution: {resolution[0]:.2f} × {resolution[1]:.2f} × {resolution[2]:.2f} mm³",
        f"Voxels non-nuls: {np.sum(t1_data > 0)} ({np.sum(t1_data > 0)/t1_data.size:.1%})",
    ]
    
    if intensity_stats:
        snr_text = f"SNR estimé: {intensity_stats.get('estimated_snr', 'N/A'):.1f}"
        info_text.append(snr_text)
        
        contrast_text = f"Contraste GM-WM: {intensity_stats.get('gm_wm_contrast', 'N/A'):.3f}"
        info_text.append(contrast_text)
        
        if "suitable_for_sam" in intensity_stats:
            if intensity_stats["suitable_for_sam"]:
                suitability = "✅ ADAPTÉ pour SAM-NeuroAdapt v2"
            else:
                suitability = "⚠️ POTENTIELLEMENT SOUS-OPTIMAL pour SAM-NeuroAdapt v2"
            info_text.append("")
            info_text.append(suitability)
    
    plt.text(0.1, 0.9, "\n".join(info_text), transform=plt.gca().transAxes, 
             verticalalignment='top', fontsize=11, family='monospace')
    

    plt.tight_layout()
    plt.suptitle(f"Analyse de l'image T1 pour SAM-NeuroAdapt v2: {t1_path.name}", fontsize=16)
    plt.subplots_adjust(top=0.92)
    

    plt.show()

def generate_validation_report(validation_results, intensity_stats=None):
 
    lines = [
        "=================================================================",
        "             RAPPORT DE VALIDATION SAM-NEUROADAPT v2             ",
        "================================================================="
    ]
    

    if validation_results["bids_valid"]:
        lines.append("\n✅ STRUCTURE BIDS VALIDE")
    else:
        lines.append("\n❌ STRUCTURE BIDS INVALIDE")
    

    if validation_results["missing_files"]:
        lines.append("\nFichiers manquants:")
        for file in validation_results["missing_files"]:
            lines.append(f"  - {file}")
    

    if validation_results["warnings"]:
        lines.append("\nAvertissements:")
        for warning in validation_results["warnings"]:
            lines.append(f"  - {warning}")
    

    if "t1_properties" in validation_results and validation_results["t1_properties"]:
        props = validation_results["t1_properties"]
        lines.append("\nPropriétés de l'image T1:")
        lines.append(f"  - Dimensions: {props.get('shape')}")
        lines.append(f"  - Résolution: {props.get('resolution_mm')} mm")
        lines.append(f"  - Type de données: {props.get('data_type')}")
        lines.append(f"  - Plage d'intensité: {props.get('min_intensity'):.2f} à {props.get('max_intensity'):.2f}")
        lines.append(f"  - Intensité moyenne: {props.get('mean_intensity'):.2f} ± {props.get('std_intensity'):.2f}")
        lines.append(f"  - Voxels non-nuls: {props.get('non_zero_voxels')} ({props.get('non_zero_percentage'):.1f}%)")
    

    if intensity_stats and "error" not in intensity_stats:
        lines.append("\nAnalyse d'intensité:")
        lines.append(f"  - Contraste GM-WM estimé: {intensity_stats.get('gm_wm_contrast', 'N/A'):.3f}")
        lines.append(f"  - SNR estimé: {intensity_stats.get('estimated_snr', 'N/A'):.1f}")
        
    
        lines.append("\nÉvaluation pour SAM-NeuroAdapt v2:")
        
        suitability_msgs = []
    
        if "t1_properties" in validation_results and "resolution_mm" in validation_results["t1_properties"]:
            res = validation_results["t1_properties"]["resolution_mm"]
            if any(r > 2.0 for r in res):
                suitability_msgs.append(f"⚠️ Résolution ({res} mm) supérieure à 2mm recommandée")
            else:
                suitability_msgs.append(f"✅ Résolution ({res} mm) appropriée")
        
    
        if "gm_wm_contrast" in intensity_stats:
            contrast = intensity_stats["gm_wm_contrast"]
            if contrast < 0.1:
                suitability_msgs.append(f"⚠️ Contraste GM-WM faible ({contrast:.3f} < 0.1 recommandé)")
            else:
                suitability_msgs.append(f"✅ Contraste GM-WM bon ({contrast:.3f} > 0.1)")
        
    
        if "estimated_snr" in intensity_stats:
            snr = intensity_stats["estimated_snr"]
            if snr < 5:
                suitability_msgs.append(f"⚠️ SNR estimé faible ({snr:.1f} < 5 recommandé)")
            else:
                suitability_msgs.append(f"✅ SNR estimé bon ({snr:.1f} > 5)")
        
        for msg in suitability_msgs:
            lines.append(f"  {msg}")
        
    
        if "suitable_for_sam" in intensity_stats:
            lines.append("")
            if intensity_stats["suitable_for_sam"]:
                lines.append("✅ IMAGE ADAPTÉE POUR SAM-NEUROADAPT v2")
            else:
                lines.append("⚠️ IMAGE POTENTIELLEMENT SOUS-OPTIMALE POUR SAM-NEUROADAPT v2")
                lines.append("   Des réglages manuels des seuils de confiance pourraient être nécessaires.")
    
    lines.append("\n=================================================================")
    
    return "\n".join(lines)

# Exécution de la validation approfondie
print("=== Validation Approfondie pour SAM-NeuroAdapt v2 ===")

# Vérifier que bids_dir et subject_id sont définis (depuis la cellule 5)
if 'bids_dir' not in globals() or 'subject_id' not in globals():
    bids_dir = BIDS_WORK_DIR if 'BIDS_WORK_DIR' in globals() else RESULTS_DIR / "bids_dataset"
    subject_id = SUBJECT_ID  # défini dans cell 0  # Sans le préfixe "sub-"
else:
    subject_id = subject_id.replace("sub-", "")  # Assurer l'absence du préfixe

print(f"Analyse du sujet: sub-{subject_id}")

# Chemin vers l'image T1
t1_path = bids_dir / f"sub-{subject_id}/anat/sub-{subject_id}.nii.gz"

if not t1_path.exists():
    print(f"❌ Erreur: Image T1 non trouvée à {t1_path}")
else:

    validation_results = check_bids_structure(bids_dir, subject_id)
    

    intensity_stats = analyze_t1_intensity_distribution(t1_path)
    

    report = generate_validation_report(validation_results, intensity_stats)
    print(report)
    

    visualize_t1_analysis(t1_path, intensity_stats)
    

    report_path = bids_dir / f"sub-{subject_id}/sub-{subject_id}_validation_report.txt"
    with open(report_path, 'w') as f:
        f.write(report)
    print(f"\nRapport de validation sauvegardé: {report_path}")
    

    if validation_results["bids_valid"]:
        print("\n=== Commande pour l'exécution de SAM-NeuroAdapt v2 ===")
        
    
        if 'derivatives_dir' not in globals():
            derivatives_dir = DERIVATIVES_DIR
            derivatives_dir.mkdir(parents=True, exist_ok=True)
        
    
        if 'output_brain_path' not in globals():
            output_brain_path = ANAT_DIR / f"sub-{subject_id}_desc-brain_T1w.nii.gz"
            output_brain_path.parent.mkdir(parents=True, exist_ok=True)
         
    
        

### N4 + WhiteStripe

In [ ]:
"""
Prétraitement des images T1 pour SAM-NeuroAdapt v2:
- Correction de biais N4 (SimpleITK)
- Normalisation d'intensité WhiteStripe
- Visualisation et validation des résultats
- Optimisé pour la segmentation avec SAM2
"""

import time
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.mixture import GaussianMixture
from nilearn import plotting

def n4_bias_correction(input_path, output_path, mask_path=None, shrink_factor=4, 
                       save_bias_field=True, iterations=None, convergence_threshold=1e-6):
    """
    Applique la correction de biais N4 à l'image T1, optimisée pour SAM-NeuroAdapt v2.
    
    Args:
        input_path: Chemin vers l'image d'entrée
        output_path: Chemin où enregistrer l'image corrigée
        mask_path: Chemin vers un masque optionnel
        shrink_factor: Facteur de réduction pour accélérer le traitement
        save_bias_field: Sauvegarder le champ de biais estimé
        iterations: Liste d'itérations pour chaque niveau [50,50,30,20] par défaut
        convergence_threshold: Seuil de convergence
    
    Returns:
        Path: Chemin vers l'image corrigée
    """
    print("\n--- Application de la correction de biais N4 ---")
    print(f"Image d'entrée: {input_path}")
    
    try:
        # Charger l'image
        img = sitk.ReadImage(str(input_path), sitk.sitkFloat32)
        
        # Charger le masque si fourni
        mask = None
        if mask_path and Path(mask_path).exists():
            print(f"Utilisation du masque pour N4: {mask_path}")
            mask = sitk.ReadImage(str(mask_path), sitk.sitkUInt8)
        
        # Réduction de l'image pour accélérer le traitement
        if shrink_factor > 1:
            print(f"Réduction par facteur {shrink_factor} pour accélérer le traitement")
            img_shrunk = sitk.Shrink(img, [shrink_factor] * img.GetDimension())
            if mask is not None:
                mask_shrunk = sitk.Shrink(mask, [shrink_factor] * mask.GetDimension())
            else:
                # Si pas de masque, utiliser une approche plus robuste pour générer un masque approx.
                print("Pas de masque fourni, génération d'un masque robuste...")
                # 1. Seuillage d'Otsu
                otsu_mask = sitk.OtsuThreshold(img_shrunk, 0, 1, 200)
                
                # 2. Opérations morphologiques pour améliorer le masque
                # D'abord une fermeture pour combler les trous
                close_filter = sitk.BinaryMorphologicalClosingImageFilter()
                close_filter.SetKernelRadius(3)
                closed_mask = close_filter.Execute(otsu_mask)
                
                # Puis une ouverture pour éliminer les petits artefacts
                open_filter = sitk.BinaryMorphologicalOpeningImageFilter()
                open_filter.SetKernelRadius(2)
                opened_mask = open_filter.Execute(closed_mask)
                
                # 3. Garder seulement la plus grande composante connectée
                cc_filter = sitk.ConnectedComponentImageFilter()
                labeled_mask = cc_filter.Execute(opened_mask)
                
                lss_filter = sitk.LabelShapeStatisticsImageFilter()
                lss_filter.Execute(labeled_mask)
                
                largest_label = 0
                largest_size = 0
                for label in range(1, cc_filter.GetObjectCount() + 1):
                    size = lss_filter.GetNumberOfPixels(label)
                    if size > largest_size:
                        largest_size = size
                        largest_label = label
                
                # Créer un masque contenant uniquement la plus grande composante
                mask_shrunk = sitk.BinaryThreshold(labeled_mask, largest_label, largest_label, 1, 0)
                
                # Vérifier que le masque n'est pas trop petit
                if sitk.GetArrayFromImage(mask_shrunk).sum() < img_shrunk.GetSize()[0] * img_shrunk.GetSize()[1] * img_shrunk.GetSize()[2] * 0.01:
                    print("⚠️ Masque trop petit, retour au masque Otsu original")
                    mask_shrunk = otsu_mask
        else:
            img_shrunk = img
            if mask is not None:
                mask_shrunk = mask
            else:
                mask_shrunk = sitk.OtsuThreshold(img_shrunk, 0, 1, 200)
        
        # Configurer le filtre N4
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        
        # Paramétrer les itérations
        if iterations is None:
            iterations = [50, 50, 30, 20]
        corrector.SetMaximumNumberOfIterations(iterations)
        
        # Autres paramètres
        corrector.SetConvergenceThreshold(convergence_threshold)
        corrector.SetBiasFieldFullWidthAtHalfMaximum(0.15)
        corrector.SetSplineOrder(3)
        corrector.SetNumberOfControlPoints([4, 4, 4])  # Recommandé pour les cerveaux humains
        
        # Exécuter la correction sur l'image réduite
        print(f"Exécution de N4 (peut prendre quelques minutes)...")
        print(f"Paramètres: {iterations} itérations, seuil={convergence_threshold}")
        
        start_time = time.time()
        corrected_shrunk = corrector.Execute(img_shrunk, mask_shrunk)
        
        # Obtenir le champ de biais à pleine résolution
        log_bias_field_full = corrector.GetLogBiasFieldAsImage(img)
        bias_field_full = sitk.Exp(log_bias_field_full)
        clamped_bias_field = sitk.Clamp(bias_field_full, 
                                        lowerBound=1e-5, 
                                        upperBound=1e5)
        corrected_full = img / clamped_bias_field
        end_time = time.time()
        
        print(f"✓ Correction N4 terminée en {end_time - start_time:.2f} secondes")
        
        # Sauvegarder l'image corrigée
        output_path_obj = Path(output_path) # Renommé pour éviter confusion avec l'argument
        output_path_obj.parent.mkdir(parents=True, exist_ok=True)
        sitk.WriteImage(corrected_full, str(output_path_obj))
        print(f"✓ Image corrigée sauvegardée: {output_path_obj}")
        
        # Sauvegarder le champ de biais pour visualisation
        if save_bias_field:
            out_name = output_path_obj.name
            if out_name.endswith("_T1w.nii.gz"):
                bias_name = out_name[:-len("_T1w.nii.gz")] + "_biasfield.nii.gz"
            elif out_name.endswith(".nii.gz"):
                bias_name = out_name[:-len(".nii.gz")] + "_biasfield.nii.gz"
            else:
                bias_name = output_path_obj.stem + "_biasfield.nii.gz"
            bias_path = output_path_obj.parent / bias_name
            sitk.WriteImage(bias_field_full, str(bias_path))
            print(f"✓ Champ de biais sauvegardé: {bias_path}")
        
        # Récupération de statistiques sur la correction
        stats = {
            "input_mean": sitk.GetArrayFromImage(img).mean(),
            "input_std": sitk.GetArrayFromImage(img).std(),
            "output_mean": sitk.GetArrayFromImage(corrected_full).mean(),
            "output_std": sitk.GetArrayFromImage(corrected_full).std(),
            "bias_field_mean": sitk.GetArrayFromImage(bias_field_full).mean(),
            "bias_field_std": sitk.GetArrayFromImage(bias_field_full).std(),
            "runtime_seconds": end_time - start_time
        }
        
        # Afficher quelques statistiques
        print(f"Statistiques de correction:")
        print(f"  - Moyenne avant: {stats['input_mean']:.4f}, après: {stats['output_mean']:.4f}")
        print(f"  - Écart-type avant: {stats['input_std']:.4f}, après: {stats['output_std']:.4f}")
        print(f"  - Facteur de correction moyen: {stats['bias_field_mean']:.4f} ± {stats['bias_field_std']:.4f}")
        
        return output_path_obj, stats # Retourner le Path object et les stats
    
    except Exception as e:
        print(f"❌ Erreur lors de la correction N4: {e}")
        # Imprimer la trace complète pour un meilleur débogage
        import traceback
        traceback.print_exc()
        return None, None

def whitestripe_normalize(input_path, output_path, mask_path=None, n_components=3, clip_percentiles=(1, 99), enhance_contrast=False, edge_enhancement=0):
 
    print("\n--- Application de la normalisation WhiteStripe ---")
    print(f"Image d'entrée: {input_path}")
    
    try:
        img = nib.load(str(input_path))
        data = img.get_fdata().astype(np.float32)
        
        # Charger le masque si fourni, sinon utiliser une approche plus robuste
        if mask_path and Path(mask_path).exists():
            print(f"Utilisation du masque: {mask_path}")
            brain_mask = nib.load(mask_path).get_fdata() > 0
        else:
            print("Pas de masque fourni, génération d'un masque robuste...")
            # Utiliser une approche multi-seuils au lieu d'un seuil simple
            # 1. Seuillage adaptatif
            non_zero_values = data[data > 0]
            if len(non_zero_values) > 0:
                # Calculer plusieurs seuils basés sur les percentiles
                p5 = np.percentile(non_zero_values, 5)
                p95 = np.percentile(non_zero_values, 95)
                
                # Créer un masque qui exclut les intensités très faibles et très élevées
                brain_mask = (data > p5) & (data < p95)
                
                # Opérations morphologiques pour nettoyer le masque
                from scipy.ndimage import binary_closing, binary_opening, label
                brain_mask = binary_closing(brain_mask, structure=np.ones((3,3,3)))
                brain_mask = binary_opening(brain_mask, structure=np.ones((2,2,2)))
                
                # Garder seulement la plus grande composante connectée
                labels, num = label(brain_mask)
                if num > 0:
                    from scipy.ndimage import sum as ndimage_sum
                    sizes = ndimage_sum(brain_mask, labels, range(1, num+1))
                    largest_label = np.argmax(sizes) + 1
                    brain_mask = labels == largest_label
                
                print(f"Masque robuste créé avec {np.sum(brain_mask)} voxels ({np.sum(brain_mask)/brain_mask.size:.2%} du volume)")
            else:
                # Fallback si aucun voxel non-nul
                brain_mask = data > 0
        
        # Extraction des valeurs d'intensité dans le masque
        vals = data[brain_mask].reshape(-1, 1)
        
        if len(vals) == 0:
            print("❌ Erreur: Masque vide ou ne chevauchant pas l'image")
            return None, None
        
        # 1) Clipper pour enlever les outliers (valeurs extrêmes)
        p_low, p_high = clip_percentiles
        p1, p99 = np.percentile(vals, [p_low, p_high])
        print(f"Clipping des valeurs entre {p1:.2f} et {p99:.2f} (percentiles {p_low}-{p_high})")
        vals_clipped = np.clip(vals, p1, p99)
        
        # 2) Modèle de mélange gaussien pour identifier les tissus
        start_time = time.time()
        print(f"Ajustement GMM à {n_components} composantes...")
        gmm = GaussianMixture(n_components=n_components, random_state=0, n_init=5)
        gmm.fit(vals_clipped)
        
        # Récupérer les paramètres estimés
        means = gmm.means_.flatten()
        stds = np.sqrt(gmm.covariances_.flatten())
        weights = gmm.weights_.flatten()
        
        # Identifier les composantes (par ordre croissant d'intensité)
        sorted_indices = np.argsort(means)
        if n_components >= 3:
            # Hypothèse: 0=CSF, 1=GM, 2=WM (indices après tri)
            csf_idx, gm_idx, wm_idx = sorted_indices[:3]
            tissue_types = ["CSF", "GM", "WM"]
        else:
            # Si on utilise moins de 3 composantes, identifier WM uniquement
            wm_idx = sorted_indices[-1]  # La composante la plus lumineuse
            tissue_types = ["Autre", "WM"] if n_components == 2 else ["WM"]
            
        # 3) Utiliser la matière blanche pour la normalisation
        wm_mean = means[wm_idx]
        gm_mean = means[gm_idx] if n_components >= 3 else None
        
        wm_std = stds[wm_idx]
        
        print(f"Composantes GMM identifiées:")
        for i, idx in enumerate(sorted_indices):
            tissue = tissue_types[i] if i < len(tissue_types) else f"Tissue {i}"
            print(f"  - {tissue}: μ={means[idx]:.2f}, σ={stds[idx]:.2f}, w={weights[idx]:.2f}")
        
        # 4) Affiner l'estimation en utilisant une bande étroite autour de WM
        wm_mask = (data > 0.9 * wm_mean) & (data < 1.1 * wm_mean) & brain_mask
        
        # Vérifier si le masque WM est non vide
        if np.sum(wm_mask) > 0:
            refined_wm_std = np.std(data[wm_mask])
            print(f"[WS] Statistiques WM raffinées: μ={wm_mean:.2f}, σ={refined_wm_std:.2f} (±10%)")
            # Si l'écart-type raffiné est sensiblement différent, l'utiliser
            if 0.5 * wm_std < refined_wm_std < 2.0 * wm_std:
                wm_std = refined_wm_std
        else:
            print(f"[WS] Pas assez de voxels dans la bande WM ±10%")
        
        # 5) Normalisation z-score basée sur WM
        norm_data = np.zeros_like(data)
        norm_data[brain_mask] = (data[brain_mask] - wm_mean) / wm_std
        
        # Après la normalisation z-score
        if enhance_contrast:
            # Renforcer le contraste entre GM et WM
            threshold = (wm_mean - gm_mean) / 2
            norm_data = np.tanh(norm_data * 1.5) * 1.5  # Fonction sigmoïde pour accentuer les différences
        
        # Amélioration des bords si demandée
        if edge_enhancement > 0:
            from scipy import ndimage
            edge_map = ndimage.gaussian_gradient_magnitude(norm_data, sigma=1)
            norm_data = norm_data + edge_map * edge_enhancement
    

        # Enregistrer l'image normalisée
        output_path_obj = Path(output_path)
        output_path_obj.parent.mkdir(parents=True, exist_ok=True)
        
        norm_img = nib.Nifti1Image(norm_data, img.affine, img.header)
        nib.save(norm_img, str(output_path_obj))
        print(f"✓ Image normalisée sauvegardée: {output_path_obj}")
        
        # Calcul des statistiques sur l'image normalisée
        brain_norm_data = norm_data[brain_mask]
        stats = {
            "wm_mean_raw": float(wm_mean),
            "wm_std_raw": float(wm_std),
            "mean_brain_norm": float(np.mean(brain_norm_data)),
            "std_brain_norm": float(np.std(brain_norm_data)),
            "min_norm": float(np.min(brain_norm_data)),
            "max_norm": float(np.max(brain_norm_data)),
            "runtime_seconds": time.time() - start_time
        }
        
        if n_components >= 3:
            # Inclure les statistiques des autres tissus si disponibles
            stats.update({
                "gm_mean_raw": float(means[gm_idx]),
                "gm_std_raw": float(stds[gm_idx]),
                "csf_mean_raw": float(means[csf_idx]),
                "csf_std_raw": float(stds[csf_idx]),
                "gm_wm_contrast": float((means[wm_idx] - means[gm_idx]) / (means[wm_idx] + means[gm_idx])),
            })
        
        print(f"Statistiques de normalisation:")
        print(f"  - Moyenne cerveau normalisé: {stats['mean_brain_norm']:.4f}")
        print(f"  - Écart-type cerveau normalisé: {stats['std_brain_norm']:.4f}")
        if 'gm_wm_contrast' in stats:
            print(f"  - Contraste GM-WM: {stats['gm_wm_contrast']:.4f}")
        
        return output_path_obj, stats
    
    except Exception as e:
        print(f"❌ Erreur lors de la normalisation WhiteStripe: {e}")
        import traceback
        traceback.print_exc()
        return None, None

def visualize_preprocessing(original_path, n4_path, whitestripe_path, bias_field_path=None, mask_path=None):
    """
    Visualise les résultats du prétraitement pour SAM-NeuroAdapt v2
    
    Args:
        original_path: Chemin vers l'image originale
        n4_path: Chemin vers l'image après correction N4
        whitestripe_path: Chemin vers l'image après normalisation WhiteStripe
        bias_field_path: Chemin vers le champ de biais (optionnel)
        mask_path: Chemin vers le masque cérébral (optionnel)
    """
    # Chargement des images
    orig_img = nib.load(original_path)
    n4_img = nib.load(n4_path)
    ws_img = nib.load(whitestripe_path)
    
    orig_data = orig_img.get_fdata()
    n4_data = n4_img.get_fdata()
    ws_data = ws_img.get_fdata()
    
    # Chargement du champ de biais si disponible
    if bias_field_path and Path(bias_field_path).exists():
        bias_img = nib.load(bias_field_path)
        bias_data = bias_img.get_fdata()
        has_bias_field = True
    else:
        has_bias_field = False
        
    # Chargement du masque si disponible
    if mask_path and Path(mask_path).exists():
        mask_img = nib.load(mask_path)
        mask_data = mask_img.get_fdata() > 0
        has_mask = True
    else:
        # Masque approximatif pour les analyses
        mask_data = n4_data > np.percentile(n4_data[n4_data > 0], 5)
        has_mask = False
    
    # Configuration de la figure
    plt.figure(figsize=(20, 12))
    plt.suptitle("Prétraitement pour SAM-NeuroAdapt v2", fontsize=16)
    
    # 1. Coupes anatomiques (avant/après)
    slice_z = orig_data.shape[2] // 2  # Coupe médiane axiale
    
    # Normalisation pour affichage
    def normalize_for_display(data):
        pmin, pmax = np.percentile(data[data > 0], [1, 99])
        normalized = np.clip(data, pmin, pmax)
        normalized = (normalized - pmin) / (pmax - pmin)
        return normalized
    
    # Coupes axiales
    plt.subplot(3, 4, 1)
    plt.imshow(normalize_for_display(orig_data[:, :, slice_z]).T, cmap='gray')
    plt.title("Original", fontsize=12)
    plt.axis('off')
    
    plt.subplot(3, 4, 2)
    plt.imshow(normalize_for_display(n4_data[:, :, slice_z]).T, cmap='gray')
    plt.title("Après correction N4", fontsize=12)
    plt.axis('off')
    
    plt.subplot(3, 4, 3)
    plt.imshow(ws_data[:, :, slice_z].T, cmap='gray', vmin=-3, vmax=3)
    plt.title("Après WhiteStripe", fontsize=12)
    plt.axis('off')
    
    # Visualisation du champ de biais ou masque
    plt.subplot(3, 4, 4)
    if has_bias_field:
        plt.imshow(bias_data[:, :, slice_z].T, cmap='hot')
        plt.title("Champ de biais N4", fontsize=12)
    elif has_mask:
        plt.imshow(mask_data[:, :, slice_z].T, cmap='Blues')
        plt.title("Masque cérébral", fontsize=12)
    else:
        # Différence relative
        diff = (orig_data - n4_data) / (orig_data + 1e-10)
        plt.imshow(diff[:, :, slice_z].T, cmap='coolwarm', vmin=-0.2, vmax=0.2)
        plt.title("Différence relative", fontsize=12)
    plt.axis('off')
    
    # 2. Histogrammes
    # Filtrer les données pour l'affichage des histogrammes (utiliser le masque)
    orig_values = orig_data[mask_data]
    n4_values = n4_data[mask_data]
    ws_values = ws_data[mask_data]
    
    plt.subplot(3, 4, 5)
    plt.hist(orig_values, bins=100, alpha=0.7, color='blue')
    plt.title("Histogramme Original", fontsize=12)
    plt.xlabel("Intensité")
    plt.ylabel("Fréquence")
    
    plt.subplot(3, 4, 6)
    plt.hist(n4_values, bins=100, alpha=0.7, color='green')
    plt.title("Histogramme après N4", fontsize=12)
    plt.xlabel("Intensité")
    
    plt.subplot(3, 4, 7)
    plt.hist(ws_values, bins=100, alpha=0.7, color='red')
    plt.title("Histogramme après WS", fontsize=12)
    plt.xlabel("Intensité")
    
    # Superposition des histogrammes normalisés
    plt.subplot(3, 4, 8)
    
    # Normaliser les histogrammes pour comparaison
    def norm_for_hist(vals):
        return (vals - np.mean(vals)) / np.std(vals)
    
    plt.hist(norm_for_hist(orig_values), bins=50, alpha=0.5, color='blue', label='Original')
    plt.hist(norm_for_hist(n4_values), bins=50, alpha=0.5, color='green', label='N4')
    plt.hist(ws_values, bins=50, alpha=0.5, color='red', label='WS')
    plt.title("Comparaison normalisée", fontsize=12)
    plt.xlabel("Intensité normalisée")
    plt.legend()
    
    # 3. Profils d'intensité et analyse pour SAM-NeuroAdapt v2
    # Extraire un profil à travers le cerveau
    mid_y = orig_data.shape[1] // 2
    mid_z = orig_data.shape[2] // 2
    
    plt.subplot(3, 4, 9)
    plt.plot(normalize_for_display(orig_data[:, mid_y, mid_z]), color='blue', label='Original')
    plt.plot(normalize_for_display(n4_data[:, mid_y, mid_z]), color='green', label='N4')
    plt.title("Profils d'intensité", fontsize=12)
    plt.xlabel("Position X")
    plt.ylabel("Intensité normalisée")
    plt.legend()
    
    plt.subplot(3, 4, 10)
    plt.plot(ws_data[:, mid_y, mid_z], color='red')
    plt.title("Profil WhiteStripe", fontsize=12)
    plt.xlabel("Position X")
    plt.ylabel("Z-score")
    plt.axhline(y=0, color='gray', linestyle='--')
    
    # Analyse du gradient (bords) - important pour SAM2
    plt.subplot(3, 4, 11)
    # Calculer le gradient
    from scipy import ndimage
    grad_orig = ndimage.gaussian_gradient_magnitude(normalize_for_display(orig_data[:, :, mid_z]), sigma=1)
    grad_ws = ndimage.gaussian_gradient_magnitude(normalize_for_display(ws_data[:, :, mid_z]), sigma=1)
    
    plt.imshow(grad_ws.T, cmap='viridis')
    plt.title("Gradient (bords) après WS", fontsize=12)
    plt.axis('off')
    
    # Texte d'évaluation pour SAM-NeuroAdapt v2
    plt.subplot(3, 4, 12)
    plt.axis('off')
    
    # Calcul de métriques pertinentes pour SAM2
    # 1. Contraste
    orig_contrast = np.ptp(orig_values) / (np.mean(orig_values) + 1e-10)
    n4_contrast = np.ptp(n4_values) / (np.mean(n4_values) + 1e-10)
    ws_contrast = np.ptp(ws_values) / (np.mean(ws_values) + 1e-10)
    
    # 2. Ratio signal/bruit (approximatif)
    orig_snr = np.mean(orig_values) / (np.std(orig_values) + 1e-10)
    n4_snr = np.mean(n4_values) / (np.std(n4_values) + 1e-10)
    ws_snr = abs(np.mean(ws_values)) / (np.std(ws_values) + 1e-10)
    
    # 3. Magnitude du gradient (bords)
    orig_grad_mag = np.mean(grad_orig)
    ws_grad_mag = np.mean(grad_ws)
    grad_improvement = (ws_grad_mag / (orig_grad_mag + 1e-10)) - 1.0
    
    # Texte d'évaluation
    eval_text = [
        "ÉVALUATION POUR SAM-NEUROADAPT v2",
        "------------------------------------",
        f"Contraste: Orig={orig_contrast:.2f}, N4={n4_contrast:.2f}, WS={ws_contrast:.2f}",
        f"SNR: Orig={orig_snr:.2f}, N4={n4_snr:.2f}, WS={ws_snr:.2f}",
        f"Amélioration des bords: {grad_improvement*100:.1f}%",
        "",
        "Verdict pour segmentation:",
    ]
    
    # Verdict basé sur les métriques
    if ws_contrast > 0.8 and ws_snr > 3.0 and grad_improvement > 0.1:
        eval_text.append("✅ EXCELLENT - Détection précise des bords attendue")
    elif ws_contrast > 0.5 and ws_snr > 2.0:
        eval_text.append("✓ BON - Devrait fonctionner correctement")
    else:
        eval_text.append("⚠️ MOYEN - Des ajustements de seuils pourraient être nécessaires")
    
    plt.text(0.05, 0.95, "\n".join(eval_text), transform=plt.gca().transAxes, 
             verticalalignment='top', fontsize=11, family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    # Retourner les métriques d'évaluation
    return {
        "contraste_original": orig_contrast,
        "contraste_n4": n4_contrast,
        "contraste_ws": ws_contrast,
        "snr_original": orig_snr,
        "snr_n4": n4_snr,
        "snr_ws": ws_snr,
        "amelioration_bords": grad_improvement
    }

# Fonction principale pour exécuter tout le prétraitement
def run_preprocessing_pipeline(t1_path, output_dir, subject_id=None, mask_path=None):
    """
    Exécute l'ensemble du pipeline de prétraitement pour SAM-NeuroAdapt v2
    
    Args:
        t1_path: Chemin vers l'image T1
        output_dir: Répertoire de sortie
        subject_id: Identifiant du sujet (optionnel)
        mask_path: Chemin vers un masque cérébral (optionnel)
        
    Returns:
        dict: Chemins et statistiques du prétraitement
    """
    # Assurons-nous que les chemins sont des objets Path
    t1_path = Path(t1_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Extraire l'ID du sujet depuis le chemin si non fourni
    if subject_id is None:
        # Essayer d'extraire selon convention BIDS
        if "sub-" in t1_path.name:
            parts = t1_path.name.split("_")
            for part in parts:
                if part.startswith("sub-"):
                    subject_id = part
                    break
        
        # Si toujours None, utiliser un fallback
        if subject_id is None:
            subject_id = f"sub-{t1_path.stem.split('_')[0]}"
    
    # Formater subject_id
    if not subject_id.startswith("sub-"):
        subject_id = f"sub-{subject_id}"
    
    print(f"\n=== Prétraitement pour SAM-NeuroAdapt v2: {subject_id} ===")
    
    # Définir les chemins de sortie
    n4_output_path = output_dir / f"{subject_id}_desc-n4_T1w.nii.gz"
    whitestripe_output_path = output_dir / f"{subject_id}_desc-n4_desc-ws_T1w.nii.gz"
    bias_field_path = output_dir / f"{subject_id}_desc-n4_biasfield.nii.gz"
    
    # Étape 1: Correction de biais N4
    n4_corrected_path, n4_stats = n4_bias_correction(
        input_path=t1_path,
        output_path=n4_output_path,
        mask_path=mask_path,  # Utiliser le masque s'il est fourni
        shrink_factor=1,  # Réduire pour accélérer le traitement
        save_bias_field=True, 
        iterations=[100, 80, 60, 40],  # Plus d'itérations pour une meilleure correction
        convergence_threshold=1e-5,  # Moins strict pour éviter le surajustement
        
    )
    
    # Vérification de l'exécution N4
    if n4_corrected_path is None:
        print("\n❌ La correction de biais N4 a échoué. Arrêt du pipeline.")
        return {
            "success": False,
            "stage": "n4_correction",
            "original_path": t1_path
        }
    
    # Étape 2: Normalisation WhiteStripe
    whitestripe_path, ws_stats = whitestripe_normalize(
        input_path=n4_corrected_path,
        output_path=whitestripe_output_path,
        mask_path=mask_path,  # Utiliser le masque s'il est fourni
        n_components=3,  # CSF, GM, WM 
        clip_percentiles=(2, 98),  # Élimine plus de valeurs extrêmes
        enhance_contrast=True,  # Ajouter un paramètre pour l'amélioration du contraste
        edge_enhancement=0.5  # Ajouter un paramètre pour l'amélioration des bords
    )
    
    # Vérification de l'exécution WhiteStripe
    if whitestripe_path is None:
        print("\n❌ La normalisation WhiteStripe a échoué. Utilisation de l'image N4 uniquement.")
        # Retourner quand même le résultat N4
        return {
            "success": True,
            "stage": "n4_correction",
            "original_path": t1_path,
            "n4_path": n4_corrected_path,
            "n4_stats": n4_stats
        }
    
    # Étape 3: Visualisation des résultats
    visual_metrics = visualize_preprocessing(
        original_path=t1_path,
        n4_path=n4_corrected_path,
        whitestripe_path=whitestripe_path,
        bias_field_path=bias_field_path,
        mask_path=mask_path  # Passer le masque pour visualisation
    )
    
    # Tout a réussi!
    print(f"\n✅ Prétraitement terminé avec succès")
    print(f"Image corrigée N4: {n4_corrected_path}")
    print(f"Image normalisée WhiteStripe: {whitestripe_path}")
    
    # Résumé des améliorations
    print("\nAmélioration des métriques:")
    print(f"- Contraste: {n4_stats['input_std']/n4_stats['input_mean']:.3f} → {ws_stats['std_brain_norm']:.3f}")
    print(f"- Bords: +{visual_metrics['amelioration_bords']*100:.1f}%")
    
    # Recommandation pour SAM-NeuroAdapt v2
    print("\nRecommandation pour SAM-NeuroAdapt v2:")
    if visual_metrics['contraste_ws'] > 0.8 and visual_metrics['snr_ws'] > 3.0:
        print("✓ Utilisez l'image WhiteStripe avec un seuil de confiance standard (0.80)")
    else:
        print("⚠️ Utilisez l'image WhiteStripe avec un seuil de confiance réduit (0.70)")
    
    # Retourner tous les chemins et statistiques
    return {
        "success": True,
        "stage": "complete",
        "original_path": t1_path,
        "n4_path": n4_corrected_path,
        "whitestripe_path": whitestripe_path,
        "bias_field_path": bias_field_path,
        "n4_stats": n4_stats,
        "ws_stats": ws_stats,
        "visual_metrics": visual_metrics
    }

# Exécuter le pipeline de prétraitement sur l'image T1 dans la structure BIDS
# Assurer la compatibilité avec les variables définies dans les cellules précédentes
# Chemins définis dans cell 0 (SUBJECT_RAW, T1_RAW_PATH, ANAT_DIR)
if 'T1_RAW_PATH' not in globals() or 'SUBJECT_ID' not in globals():
    raise RuntimeError("Exécuter la cellule de configuration du sujet avant le prétraitement")
subject_id = SUBJECT_ID
derivatives_dir = DERIVATIVES_DIR
anat_dir = ANAT_DIR

# Chemin de l'image T1w dans la structure BIDS
t1_path = T1_RAW_PATH  # défini dans cell 0

# Créer les dossiers de sortie pour les dérivés
# derivatives_dir et anat_dir déjà définis dans cell 0
# ANAT_DIR déjà créé dans cell 0

# Sélection robuste d'un masque optionnel pour N4/WhiteStripe.
# Les masques MNI simples trop larges peuvent inclure crâne/nuque; on les rejette
# afin que le prétraitement reste reproductible sur tous les sujets.
def _preproc_mask_quality(path, min_cm3=900.0, max_cm3=1700.0, max_components=8):
    try:
        from scipy import ndimage as _ndi
        _img = nib.load(str(path))
        _mask = _img.get_fdata() > 0.5
        _vox = float(np.prod(_img.header.get_zooms()[:3]))
        _vol_cm3 = float(np.sum(_mask) * _vox / 1000.0)
        _lbl, _ncomp = _ndi.label(_mask)
        _reasons = []
        if _vol_cm3 < min_cm3 or _vol_cm3 > max_cm3:
            _reasons.append(f"volume={_vol_cm3:.1f} cm³ hors [{min_cm3:.0f}, {max_cm3:.0f}]")
        if _ncomp > max_components:
            _reasons.append(f"composantes={_ncomp} > {max_components}")
        return len(_reasons) == 0, {"volume_cm3": _vol_cm3, "components": int(_ncomp)}, "; ".join(_reasons)
    except Exception as _e:
        return False, {}, f"erreur lecture masque: {_e}"

_preproc_mask_candidates = [
    ANAT_DIR / f"sub-{subject_id}_desc-mni_mask.nii.gz",
    ANAT_DIR / "atlases_advanced" / f"sub-{subject_id}_brain_mask_aff.nii.gz",
    ANAT_DIR / "atlases_advanced" / f"sub-{subject_id}_brain_mask_syn.nii.gz",
]
_preproc_mask_candidates += sorted((ANAT_DIR / "atlases").glob("*brain_mask*.nii.gz")) if (ANAT_DIR / "atlases").exists() else []

brain_mask_path = None
_rejected_preproc_masks = []
_seen_preproc_masks = set()
for _cand in _preproc_mask_candidates:
    if not Path(_cand).exists() or str(_cand) in _seen_preproc_masks:
        continue
    _seen_preproc_masks.add(str(_cand))
    _ok, _stats, _why = _preproc_mask_quality(_cand)
    if _ok:
        brain_mask_path = Path(_cand)
        print(f"Masque prétraitement validé: {brain_mask_path.name} "
              f"({_stats['volume_cm3']:.1f} cm³, composantes={_stats['components']})")
        break
    _rejected_preproc_masks.append((Path(_cand).name, _why))

if brain_mask_path is None:
    if _rejected_preproc_masks:
        print("⚠️ Masques de prétraitement rejetés par QC:")
        for _name, _why in _rejected_preproc_masks[:5]:
            print(f"  - {_name}: {_why}")
    print("Aucun masque cérébral valide trouvé, génération de masques approximatifs pour N4 et WhiteStripe")

# Exécuter le prétraitement
if t1_path.exists():
    # Exécution du pipeline complet
    results = run_preprocessing_pipeline(
        t1_path=t1_path,
        output_dir=anat_dir,
        subject_id=subject_id,
        mask_path=brain_mask_path if brain_mask_path is not None and brain_mask_path.exists() else None
    )
    
    # Vérification des résultats
    if results["success"]:
        print("\n=== Configuration pour l'extraction du cerveau ===")
        
        # Définir le chemin de sortie pour l'extraction du cerveau
        brain_output_path = anat_dir / f"sub-{subject_id}_desc-brain_T1w.nii.gz"
        
        # Paramétrer le chemin d'entrée en fonction du résultat du prétraitement
        if results["stage"] == "complete":
            input_path_for_extraction = results["whitestripe_path"]
            print(f"Entrée: Image WhiteStripe - {results['whitestripe_path'].name}")
        else:
            input_path_for_extraction = results["n4_path"]
            print(f"Entrée: Image N4 (WhiteStripe a échoué) - {results['n4_path'].name}")
        
        # Afficher la commande à exécuter pour l'extraction du cerveau
        print("\nCommande pour l'extraction du cerveau:")
        print(f"from preprocessing.skull_stripping import extract_brain")
        print(f"output_brain_path, output_mask_path = extract_brain(")
        print(f"    input_path='{input_path_for_extraction}',")
        print(f"    output_path='{brain_output_path}',")
        
        # Recommander des paramètres basés sur les métriques
        if "visual_metrics" in results and results["visual_metrics"]["contraste_ws"] > 0.7:
            print(f"    confidence_threshold=0.80,  # Bon contraste d'image")
        else:
            print(f"    confidence_threshold=0.70,  # Contraste d'image réduit")
            
        print(f"    use_atlas_guidance=True,")
        print(f"    save_uncertainty_map=True,")
        print(f"    save_metrics=True,")
        print(f"    debug_dir='debug_output'")
        print(f")")
    else:
        print("\n❌ Le prétraitement a échoué à l'étape:", results["stage"])
        
else:
    print(f"\n❌ Image T1 non trouvée à {t1_path}")

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import ndimage
from sklearn.mixture import GaussianMixture
import matplotlib.gridspec as gridspec
import json

def audit_sam_neuro_adapt_preproc(
    original_path, 
    n4_path, 
    whitestripe_path, 
    bias_field_path=None, 
    output_dir=None
):
  
    print("\n=== Audit du prétraitement pour SAM-NeuroAdapt v2 ===")
    
    

    paths = {
        "Original": original_path,
        "N4": n4_path,
        "WhiteStripe": whitestripe_path,
        "Bias Field": bias_field_path
    }
    
    for name, path in paths.items():
        if path and Path(path).exists():
            print(f"✓ {name} trouvé: {Path(path).name}")
        elif path:
            print(f"⚠️ {name} non trouvé: {path}")
            if name in ["Original", "N4", "WhiteStripe"]:
                raise FileNotFoundError(f"Fichier requis manquant: {name} ({path})")
    

    try:
        orig_img = nib.load(original_path)
        n4_img = nib.load(n4_path)
        ws_img = nib.load(whitestripe_path)
        
        orig_data = orig_img.get_fdata()
        n4_data = n4_img.get_fdata()
        ws_data = ws_img.get_fdata()
        
    
        bias_data = None
        if bias_field_path and Path(bias_field_path).exists():
            bias_img = nib.load(bias_field_path)
            bias_data = bias_img.get_fdata()
            print(f"✓ Champ de biais chargé")
    
    except Exception as e:
        print(f"❌ Erreur lors du chargement des images: {e}")
        raise
    


    brain_mask = n4_data > np.percentile(n4_data[n4_data > 0], 5)
    

    metrics = {}
    

    metrics["orig_mean"] = float(np.mean(orig_data[brain_mask]))
    metrics["orig_std"] = float(np.std(orig_data[brain_mask]))
    metrics["n4_mean"] = float(np.mean(n4_data[brain_mask]))
    metrics["n4_std"] = float(np.std(n4_data[brain_mask]))
    metrics["ws_mean"] = float(np.mean(ws_data[brain_mask]))
    metrics["ws_std"] = float(np.std(ws_data[brain_mask]))
    


    brain_values = n4_data[brain_mask].flatten().reshape(-1, 1)
    

    if len(brain_values) > 100000:
        indices = np.random.choice(len(brain_values), 100000, replace=False)
        brain_values = brain_values[indices]
    

    gmm = GaussianMixture(n_components=3, random_state=42, n_init=10)
    gmm.fit(brain_values)
    

    means = gmm.means_.flatten()
    stds = np.sqrt(gmm.covariances_.flatten())
    weights = gmm.weights_
    sorted_indices = np.argsort(means)
    
    if len(sorted_indices) >= 3:
    
        csf_idx, gm_idx, wm_idx = sorted_indices[:3]
        
    
        wm_mean = means[wm_idx]
        wm_mask = (n4_data > (wm_mean * 0.9)) & (n4_data < (wm_mean * 1.1)) & brain_mask
        
    
        if np.sum(wm_mask) > 0:
            metrics["wm_mean"] = float(np.mean(n4_data[wm_mask]))
            metrics["wm_std"] = float(np.std(n4_data[wm_mask]))
        else:
            metrics["wm_mean"] = float(wm_mean)
            metrics["wm_std"] = float(stds[wm_idx])
        
    
        metrics["gm_mean"] = float(means[gm_idx])
        metrics["gm_std"] = float(stds[gm_idx])
        
    
        metrics["gm_wm_contrast"] = float((means[wm_idx] - means[gm_idx]) / (means[wm_idx] + means[gm_idx]))
    else:
        metrics["wm_mean"] = float(np.nan)
        metrics["wm_std"] = float(np.nan)
        metrics["gm_mean"] = float(np.nan)
        metrics["gm_std"] = float(np.nan)
        metrics["gm_wm_contrast"] = float(np.nan)
    


    mid_z = orig_data.shape[2] // 2
    

    def calc_norm_gradient(data, slice_idx):
        slice_data = data[:, :, slice_idx]
    
        p1, p99 = np.percentile(slice_data[slice_data > 0], [1, 99])
        norm_slice = np.clip(slice_data, p1, p99)
        norm_slice = (norm_slice - p1) / (p99 - p1)
    
        return ndimage.gaussian_gradient_magnitude(norm_slice, sigma=1)
    
    grad_orig = calc_norm_gradient(orig_data, mid_z)
    grad_n4 = calc_norm_gradient(n4_data, mid_z)
    

    ws_slice = ws_data[:, :, mid_z]
    ws_min, ws_max = np.percentile(ws_slice[brain_mask[:, :, mid_z]], [1, 99])
    norm_ws_slice = np.clip(ws_slice, ws_min, ws_max)
    norm_ws_slice = (norm_ws_slice - ws_min) / (ws_max - ws_min)
    grad_ws = ndimage.gaussian_gradient_magnitude(norm_ws_slice, sigma=1)
    

    brain_slice_mask = brain_mask[:, :, mid_z]
    metrics["edge_mean_orig"] = float(np.mean(grad_orig[brain_slice_mask]))
    metrics["edge_mean_n4"] = float(np.mean(grad_n4[brain_slice_mask]))
    metrics["edge_mean_ws"] = float(np.mean(grad_ws[brain_slice_mask]))
    
    metrics["edge_max_orig"] = float(np.max(grad_orig[brain_slice_mask]))
    metrics["edge_max_n4"] = float(np.max(grad_n4[brain_slice_mask]))
    metrics["edge_max_ws"] = float(np.max(grad_ws[brain_slice_mask]))
    

    metrics["edge_improvement_n4"] = float((metrics["edge_mean_n4"] / metrics["edge_mean_orig"]) - 1.0)
    metrics["edge_improvement_ws"] = float((metrics["edge_mean_ws"] / metrics["edge_mean_orig"]) - 1.0)
    


    fig = plt.figure(figsize=(18, 14))
    gs = gridspec.GridSpec(4, 4, figure=fig)
    

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[0, 3])
    

    def normalize_for_display(data, mask=None):
        if mask is None:
            mask = data > 0
        p1, p99 = np.percentile(data[mask], [1, 99])
        normalized = np.clip(data, p1, p99)
        normalized = (normalized - p1) / (p99 - p1)
        return normalized
    

    orig_display = normalize_for_display(orig_data, brain_mask)
    n4_display = normalize_for_display(n4_data, brain_mask)
    

    ws_display = ws_data.copy()
    ws_min, ws_max = np.percentile(ws_data[brain_mask], [1, 99])
    
    ax1.imshow(orig_display[:, :, mid_z].T, cmap='gray')
    ax1.set_title("Original", fontsize=12)
    ax1.axis('off')
    
    ax2.imshow(n4_display[:, :, mid_z].T, cmap='gray')
    ax2.set_title("Après correction N4", fontsize=12)
    ax2.axis('off')
    
    ax3.imshow(ws_display[:, :, mid_z].T, cmap='gray', vmin=ws_min, vmax=ws_max)
    ax3.set_title("Après WhiteStripe", fontsize=12)
    ax3.axis('off')
    

    if bias_data is not None:
        im4 = ax4.imshow(bias_data[:, :, mid_z].T, cmap='hot')
        ax4.set_title("Champ de biais N4", fontsize=12)
        plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)
    else:
    
        diff = (orig_data - n4_data) / (np.maximum(orig_data, 1e-10))
        diff_slice = diff[:, :, mid_z]
        im4 = ax4.imshow(diff_slice.T, cmap='coolwarm', vmin=-0.2, vmax=0.2)
        ax4.set_title("Différence relative (Orig-N4)/Orig", fontsize=12)
        plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)
    ax4.axis('off')
    

    ax5 = fig.add_subplot(gs[1, 0])
    ax6 = fig.add_subplot(gs[1, 1])
    ax7 = fig.add_subplot(gs[1, 2])
    ax8 = fig.add_subplot(gs[1, 3])
    

    orig_values = orig_data[brain_mask]
    n4_values = n4_data[brain_mask]
    ws_values = ws_data[brain_mask]
    
    ax5.hist(orig_values, bins=100, color='blue', alpha=0.7)
    ax5.set_title("Histogramme Original", fontsize=12)
    ax5.set_xlabel("Intensité")
    ax5.set_ylabel("Fréquence")
    
    ax6.hist(n4_values, bins=100, color='green', alpha=0.7)
    ax6.set_title("Histogramme après N4", fontsize=12)
    ax6.set_xlabel("Intensité")
    
    ax7.hist(ws_values, bins=100, color='red', alpha=0.7)
    ax7.set_title("Histogramme après WS", fontsize=12)
    ax7.set_xlabel("Intensité")
    

    ax8.hist(orig_values, bins=50, alpha=0.5, color='blue', label='Original', 
             density=True, range=np.percentile(orig_values, [1, 99]))
    ax8.hist(n4_values, bins=50, alpha=0.5, color='green', label='N4', 
             density=True, range=np.percentile(n4_values, [1, 99]))
    ax8.hist(ws_values, bins=50, alpha=0.5, color='red', label='WS', 
             density=True, range=np.percentile(ws_values, [1, 99]))
    ax8.set_title("Histogrammes normalisés", fontsize=12)
    ax8.set_xlabel("Intensité (normalisée)")
    ax8.legend()
    

    ax9 = fig.add_subplot(gs[2, 0])
    ax10 = fig.add_subplot(gs[2, 1])
    ax11 = fig.add_subplot(gs[2, 2])
    ax12 = fig.add_subplot(gs[2, 3])
    

    im9 = ax9.imshow(grad_orig.T, cmap='viridis')
    ax9.set_title("Gradient Original", fontsize=12)
    ax9.axis('off')
    plt.colorbar(im9, ax=ax9, fraction=0.046, pad=0.04)
    
    im10 = ax10.imshow(grad_n4.T, cmap='viridis')
    ax10.set_title("Gradient après N4", fontsize=12)
    ax10.axis('off')
    plt.colorbar(im10, ax=ax10, fraction=0.046, pad=0.04)
    
    im11 = ax11.imshow(grad_ws.T, cmap='viridis')
    ax11.set_title("Gradient après WS", fontsize=12)
    ax11.axis('off')
    plt.colorbar(im11, ax=ax11, fraction=0.046, pad=0.04)
    

    mid_y = orig_data.shape[1] // 2
    ax12.plot(orig_display[:, mid_y, mid_z], 'b-', label='Original')
    ax12.plot(n4_display[:, mid_y, mid_z], 'g-', label='N4')
    

    ws_profile = ws_data[:, mid_y, mid_z]
    ws_profile_norm = (ws_profile - np.min(ws_profile)) / (np.max(ws_profile) - np.min(ws_profile))
    ax12.plot(ws_profile_norm, 'r-', label='WS')
    
    ax12.set_title("Profils d'intensité", fontsize=12)
    ax12.set_xlabel("Position X")
    ax12.set_ylabel("Intensité normalisée")
    ax12.legend()
    


    ax13 = fig.add_subplot(gs[3, :])
    ax13.axis('off')
    

    evaluation_text = []
    evaluation_text.append("ÉVALUATION POUR SAM-NEUROADAPT v2")
    evaluation_text.append("=" * 40)
    
    evaluation_text.append("\nMÉTRIQUES D'INTENSITÉ:")
    evaluation_text.append(f"• Original : moyenne = {metrics['orig_mean']:.3f}, écart-type = {metrics['orig_std']:.3f}")
    evaluation_text.append(f"• N4      : moyenne = {metrics['n4_mean']:.3f}, écart-type = {metrics['n4_std']:.3f}")
    evaluation_text.append(f"• WS      : moyenne = {metrics['ws_mean']:.3f}, écart-type = {metrics['ws_std']:.3f}")
    
    evaluation_text.append("\nTISSUS (estimés par GMM):")
    if not np.isnan(metrics.get('gm_mean', np.nan)):
        gm_wm_contrast = metrics['gm_wm_contrast']
        evaluation_text.append(f"• Matière blanche : {metrics['wm_mean']:.3f} ± {metrics['wm_std']:.3f}")
        evaluation_text.append(f"• Matière grise   : {metrics['gm_mean']:.3f} ± {metrics['gm_std']:.3f}")
        evaluation_text.append(f"• Contraste GM-WM : {gm_wm_contrast:.3f} " + 
                               ("(BON)" if gm_wm_contrast > 0.5 else "(MODÉRÉ)" if gm_wm_contrast > 0.3 else "(FAIBLE)"))
    
    evaluation_text.append("\nBORDS ET CONTOURS (crucial pour SAM2):")
    evaluation_text.append(f"• Moyenne gradient Original : {metrics['edge_mean_orig']:.4f}")
    evaluation_text.append(f"• Moyenne gradient N4      : {metrics['edge_mean_n4']:.4f} " + 
                           (f"({metrics['edge_improvement_n4']*100:+.1f}%)" if not np.isnan(metrics['edge_improvement_n4']) else ""))
    evaluation_text.append(f"• Moyenne gradient WS      : {metrics['edge_mean_ws']:.4f} " + 
                           (f"({metrics['edge_improvement_ws']*100:+.1f}%)" if not np.isnan(metrics['edge_improvement_ws']) else ""))
    
    evaluation_text.append("\nÉVALUATION POUR SAM-NEUROADAPT v2:")
    

    score = 0
    if not np.isnan(metrics.get('gm_wm_contrast', np.nan)):
        if metrics['gm_wm_contrast'] > 0.5:
            score += 2
            evaluation_text.append("✓ BON contraste tissulaire (>0.5)")
        elif metrics['gm_wm_contrast'] > 0.3:
            score += 1
            evaluation_text.append("⚠ Contraste tissulaire MOYEN (0.3-0.5)")
        else:
            evaluation_text.append("❌ Contraste tissulaire FAIBLE (<0.3)")
    
    if not np.isnan(metrics.get('edge_improvement_ws', np.nan)):
        if metrics['edge_improvement_ws'] > 0.1:
            score += 2
            evaluation_text.append("✓ Amélioration des bords significative (>10%)")
        elif metrics['edge_improvement_ws'] > -0.1:
            score += 1
            evaluation_text.append("⚠ Bords maintenus (-10% à +10%)")
        else:
            evaluation_text.append(f"❌ Dégradation des bords ({metrics['edge_improvement_ws']*100:.1f}%)")
    

    evaluation_text.append("\nRECOMMANDATION:")
    if score >= 3:
        metrics["recommended_threshold"] = 0.80
        evaluation_text.append("✓ Image prétraitée de BONNE qualité pour SAM-NeuroAdapt v2")
        evaluation_text.append("  → Utiliser un seuil de confiance standard (0.80)")
    elif score >= 1:
        metrics["recommended_threshold"] = 0.70
        evaluation_text.append("⚠ Image prétraitée de qualité MOYENNE pour SAM-NeuroAdapt v2")
        evaluation_text.append("  → Utiliser un seuil de confiance réduit (0.70)")
    else:
        metrics["recommended_threshold"] = 0.60
        evaluation_text.append("❌ Image prétraitée de qualité SOUS-OPTIMALE pour SAM-NeuroAdapt v2")
        evaluation_text.append("  → Utiliser un seuil de confiance bas (0.60) et envisager d'autres prétraitements")
    

    ax13.text(0.02, 0.98, "\n".join(evaluation_text), 
              transform=ax13.transAxes, verticalalignment='top', 
              fontsize=12, family='monospace')
    

    plt.suptitle(f"Audit du prétraitement pour SAM-NeuroAdapt v2", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    

    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        fig_path = output_dir / "audit_preprocessing.png"
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f"Figure d'audit sauvegardée: {fig_path}")
    
    plt.show()
    

    print(f"\n=== Résumé de l'audit ===")
    if not np.isnan(metrics.get('gm_wm_contrast', np.nan)):
        print(f"• Contraste GM-WM : {metrics['gm_wm_contrast']:.3f}")
    if not np.isnan(metrics.get('edge_improvement_ws', np.nan)):
        print(f"• Impact sur les bords : {metrics['edge_improvement_ws']*100:+.1f}%")
    print(f"• Seuil recommandé : {metrics['recommended_threshold']}")
    

    try:
        assert not np.isnan(metrics['ws_mean']), "La normalisation WhiteStripe a produit des résultats non valides"
        assert metrics['ws_std'] > 0, "L'écart-type après WhiteStripe est nul ou négatif"
        
        if not np.isnan(metrics.get('gm_wm_contrast', np.nan)):
            assert metrics['gm_wm_contrast'] > 0.2, "Contraste GM-WM trop faible (<0.2)"
            
        print("\n✅ Le prétraitement est valide pour la segmentation par SAM-NeuroAdapt v2")
        print(f"   (Utiliser un seuil de confiance de {metrics['recommended_threshold']})")
        
    except AssertionError as e:
        print(f"\n⚠️ Avertissement: {e}")
        print("   Le prétraitement peut ne pas être optimal pour SAM-NeuroAdapt v2")
        print(f"   (Utiliser un seuil de confiance de {metrics['recommended_threshold']} ou considérer un autre prétraitement)")
    
    return metrics

# Exécuter l'audit sur les résultats du prétraitement
try:
    
    # subject_id et derivatives_dir définis dans cell 0
    

    anat_dir = ANAT_DIR  # centralisé dans cell 0
    
    original_path = T1_RAW_PATH
    n4_path = anat_dir / f"sub-{subject_id}_desc-n4_T1w.nii.gz"
    whitestripe_path = anat_dir / f"sub-{subject_id}_desc-n4_desc-ws_T1w.nii.gz"
    bias_field_path = anat_dir / f"sub-{subject_id}_desc-n4_biasfield.nii.gz"
    if not bias_field_path.exists():
        legacy_bias_field_path = anat_dir / f"sub-{subject_id}_desc-n4_T1w.nii_biasfield.nii.gz"
        if legacy_bias_field_path.exists():
            bias_field_path = legacy_bias_field_path
   
    metrics = audit_sam_neuro_adapt_preproc(
        original_path=original_path,
        n4_path=n4_path,
        whitestripe_path=whitestripe_path,
        bias_field_path=bias_field_path,
        output_dir=anat_dir
    )

    def _audit_json_default(obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return str(obj)

    audit_json_path = anat_dir / f"sub-{subject_id}_audit_preprocessing.json"
    with open(audit_json_path, "w") as f:
        json.dump(metrics, f, indent=2, default=_audit_json_default)
    print(f"Audit JSON sauvegardé: {audit_json_path}")

except Exception as e:
    import traceback
    print(f"\n❌ Erreur lors de l'audit: {e}")
    traceback.print_exc() 

### Recalage Simple 

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.measure import label, regionprops
from scipy.ndimage import binary_erosion, binary_dilation
import ants



atlas_dir = Path("./sam_neuroadapt_v2_real_atlases/atlases")


available_atlas_paths = {
    "brain_mask": atlas_dir / "MNI152NLin2009cAsym_brainmask_2mm.nii.gz",
}


atlas_paths = {}
for name, path in available_atlas_paths.items():
    if path.exists():
        atlas_paths[name] = path
        print(f"✓ Atlas {name} trouvé: {path}")
    else:
        print(f"⚠️ Atlas {name} non trouvé: {path}")


if not atlas_paths:
    raise FileNotFoundError("Aucun atlas n'a été trouvé. Le recalage ne peut pas être effectué.")



if 'whitestripe_path' not in globals() or not Path(whitestripe_path).exists():
    # subject_id et derivatives_dir définis dans cell 0
    

    anat_dir = ANAT_DIR  # centralisé dans cell 0
    whitestripe_path = anat_dir / f"sub-{subject_id}_desc-n4_desc-ws_T1w.nii.gz"


if not Path(whitestripe_path).exists():

    n4_path = anat_dir / f"sub-{subject_id}_desc-n4_T1w.nii.gz"
    if Path(n4_path).exists():
        print(f"⚠️ Image WhiteStripe non trouvée. Utilisation de l'image N4 comme fallback: {n4_path}")
        subject_image_path = n4_path
    else:
        raise FileNotFoundError(f"Image prétraitée introuvable: ni WhiteStripe ({whitestripe_path}) ni N4 ({n4_path})")
else:
    subject_image_path = whitestripe_path


atlases_output_dir = anat_dir / "atlases"
atlases_output_dir.mkdir(parents=True, exist_ok=True)


def register_atlas_to_subject(subject_image_path, atlas_paths, output_dir, transform_type='antsRegistrationSyNQuick[a]', verbose=True):

    subject_image_path = Path(subject_image_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    

    sid = f"sub-{subject_id}" if not subject_id.startswith("sub-") else subject_id
    
    print(f"\n=== Recalage de l'atlas vers l'espace du sujet {sid} ===")
    
    results = {
        "subject_id": sid,
        "registered_atlas_paths": {},
        "metrics": {}
    }
    
    try:
    
        fixed_img = ants.image_read(str(subject_image_path))
        for atlas_name, atlas_path in atlas_paths.items():
            print(f"\nRecalage de l'atlas {atlas_name}...")
            
        
            moving_img = ants.image_read(str(atlas_path))
            
        
            output_atlas_path = output_dir / f"{sid}_space-subject_{atlas_name}.nii.gz"
            
        
            import time
            start_time = time.time()
            
            registration = ants.registration(
                fixed=fixed_img,
                moving=moving_img,
                type_of_transform=transform_type,
                verbose=verbose
            )
            
        
            warped_atlas = registration['warpedmovout']
            
        
            ants.image_write(warped_atlas, str(output_atlas_path))
            
            end_time = time.time()
            elapsed = end_time - start_time
            
        
            results["registered_atlas_paths"][atlas_name] = str(output_atlas_path)
            results["metrics"][atlas_name] = {"runtime_seconds": elapsed}
            
            print(f"✓ Atlas {atlas_name} recalé en {elapsed:.2f}s")
            print(f"  Sauvegardé dans: {output_atlas_path}")
        
        return results
    
    except Exception as e:
        print(f"❌ Erreur lors du recalage de l'atlas: {e}")
        import traceback
        traceback.print_exc()
        return results


def visualize_registered_atlas(subject_image_path, atlas_path, output_dir=None):

    print(f"\n=== Visualisation de l'atlas recalé ===")
    

    subject_img = nib.load(str(subject_image_path))
    atlas_img = nib.load(str(atlas_path))
    
    subject_data = subject_img.get_fdata()
    atlas_data = atlas_img.get_fdata() > 0.5  
    

    p1, p99 = np.percentile(subject_data[subject_data > 0], [1, 99])
    subject_display = np.clip(subject_data, p1, p99)
    subject_display = (subject_display - p1) / (p99 - p1)
    

    z_sums = np.sum(atlas_data, axis=(0, 1))
    y_sums = np.sum(atlas_data, axis=(0, 2))
    x_sums = np.sum(atlas_data, axis=(1, 2))
    
    z_slice = np.argmax(z_sums) if np.any(z_sums > 0) else subject_data.shape[2] // 2
    y_slice = np.argmax(y_sums) if np.any(y_sums > 0) else subject_data.shape[1] // 2
    x_slice = np.argmax(x_sums) if np.any(x_sums > 0) else subject_data.shape[0] // 2
    

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    

    axes[0, 0].imshow(subject_display[x_slice, :, :].T, cmap='gray')
    axes[0, 0].set_title(f"T1 - Sagittal (x={x_slice})")
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(subject_display[:, y_slice, :].T, cmap='gray')
    axes[0, 1].set_title(f"T1 - Coronal (y={y_slice})")
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(subject_display[:, :, z_slice].T, cmap='gray')
    axes[0, 2].set_title(f"T1 - Axial (z={z_slice})")
    axes[0, 2].axis('off')
    

    axes[1, 0].imshow(subject_display[x_slice, :, :].T, cmap='gray')
    axes[1, 0].imshow(atlas_data[x_slice, :, :].T, cmap='cool', alpha=0.5)
    axes[1, 0].set_title("T1 + Masque - Sagittal")
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(subject_display[:, y_slice, :].T, cmap='gray')
    axes[1, 1].imshow(atlas_data[:, y_slice, :].T, cmap='cool', alpha=0.5)
    axes[1, 1].set_title("T1 + Masque - Coronal")
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(subject_display[:, :, z_slice].T, cmap='gray')
    axes[1, 2].imshow(atlas_data[:, :, z_slice].T, cmap='cool', alpha=0.5)
    axes[1, 2].set_title("T1 + Masque - Axial")
    axes[1, 2].axis('off')
    
    plt.suptitle(f"Atlas recalé: {Path(atlas_path).name}", fontsize=16)
    plt.tight_layout()
    

    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        output_file = output_dir / f"atlas_visualization.png"
        plt.savefig(output_file, dpi=150, bbox_inches='tight')
        print(f"Visualisation sauvegardée: {output_file}")
    
    plt.show()


print(f"\nRecalage de l'atlas vers l'espace du sujet: {subject_image_path}")
registration_results = register_atlas_to_subject(
    subject_image_path=subject_image_path,
    atlas_paths=atlas_paths,
    output_dir=atlases_output_dir,
    transform_type='antsRegistrationSyNQuick[s]',  
    verbose=False  
)


if "registered_atlas_paths" in registration_results and registration_results["registered_atlas_paths"]:

    brain_mask_path = registration_results["registered_atlas_paths"].get("brain_mask")
    if brain_mask_path:
        visualize_registered_atlas(
            subject_image_path=subject_image_path,
            atlas_path=brain_mask_path,
            output_dir=atlases_output_dir
        )
    
 

### Recalage atlas Avancée


In [ ]:
import time
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ants
import logging
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("atlas_registration")


def _resample_to_1mm(img, interp_name):
    interp_codes = {
        'nearestNeighbor': 0, 'linear': 1, 'bSpline': 2,
        'cosineWindowedSinc': 3, 'lanczosWindowedSinc': 4,
        'genericLabel': 5, 'multiLabel': 6, 'gaussian': 7
    }
    code = interp_codes.get(interp_name, 0)
    return ants.resample_image(img, (1, 1, 1), use_voxels=False, interp_type=code)

def _dilate_mask(img, r=1):
    return ants.iMath(img, 'MD', r)

def _dice_jaccard(a, b):
    tp   = np.logical_and(a, b).sum()
    volA = a.sum(); volB = b.sum()
    dice = 2*tp / (volA + volB + 1e-8)
    jacc = tp / (np.logical_or(a, b).sum() + 1e-8)
    return dice, jacc

def _safe_slice(vol, z):
    return vol[:, :, z] if np.any(vol[:, :, z]) else None


def syn_atlas_registration(subject_n4_path, atlas_paths, output_dir, subject_id=None,
                                create_brain_mask=True, verbose=False):
 
    subject_n4_path, output_dir = Path(subject_n4_path), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    
    if subject_id:
        sid = subject_id
    else: 
        path_parts = str(subject_n4_path).split('/')
        for part in path_parts:
            if part.startswith("sub-"):
                sid = part.replace("sub-", "")
                break
        else:
            sid = subject_n4_path.stem.split("_")[0]
    
    print(f"\n=== Recalage cascade pour {sid} ===")
    print(f"Image source: {subject_n4_path}")
    print(f"Dossier de sortie: {output_dir}")

    
    if not subject_n4_path.exists():
        logger.error(f"L'image source n'existe pas: {subject_n4_path}")
        return {"subject_id": sid, "registered_atlas_paths": {}, "metrics": {}, "error": "Source image not found"}
    
    try:
        fixed_img = ants.image_read(str(subject_n4_path))
        print(f"Image chargée: dimensions {fixed_img.shape}, espacement {fixed_img.spacing}")
    except Exception as e:
        logger.error(f"Erreur lors du chargement de l'image source: {e}")
        return {"subject_id": sid, "registered_atlas_paths": {}, "metrics": {}, "error": str(e)}
    
    brain_mask = None
    if create_brain_mask:
        try:
            brain_mask = ants.morphology(ants.threshold_image(fixed_img, 'Otsu', 2), 'close', 5)
            #brain_mask = ants.get_mask(fixed_img, low_thresh=0.1, high_thresh=0.9)
            print(f"Masque Otsu créé: somme={np.sum(brain_mask.numpy()>0)}")
        except Exception as e:
            logger.error(f"Erreur lors de la création du masque cérébral Otsu: {e}")
            print(f"⚠️ Impossible de créer le masque Otsu: {e}")
            

    results = {"subject_id": sid, "registered_atlas_paths": {}, "metrics": {}}

    for name, path in atlas_paths.items():
        if not Path(path).exists():
            logger.error(f"Atlas {name} introuvable: {path}")
            print(f"⚠️ Atlas {name} introuvable: {path}")
            continue

        print(f"\n→ Traitement atlas **{name}**")
        
        try:
            moving_img = ants.image_read(str(path))
            print(f"Atlas {name} chargé: dimensions {moving_img.shape}")
        except Exception as e:
            logger.error(f"Erreur lors du chargement de l'atlas {name}: {e}")
            print(f"⚠️ Impossible de charger l'atlas {name}: {e}")
            continue

        
        interp = 'nearestNeighbor'

        
        # Recalage natif MNI→sujet: pas de resampling pré-optimisation.
        # Le resampling préalable sur la grille sujet peut créer un atlas décalé.
        print("Recalage natif MNI→sujet: aucun resampling pré-optimisation")

        
        local_mask = brain_mask
        try:
            if np.count_nonzero(moving_img.numpy()) < 3000:
                atlas_dil = ants.resample_image_to_target(_dilate_mask(moving_img, 1),
                                                        fixed_img, interp_type='nearestNeighbor')
                if brain_mask is not None:
                    local_mask = brain_mask * _dilate_mask(atlas_dil, 3)

            if local_mask is not None and np.count_nonzero(local_mask.numpy()) < 1000:
                print("  ▸ masque trop petit → recalage sans masque")
                local_mask = None
        except Exception as e:
            logger.error(f"Erreur lors de la création du masque local: {e}")
            print(f"⚠️ Erreur lors de la création du masque local: {e}")
            local_mask = None

        
        def _run_affine(mask, samp):
            return ants.registration(fixed=fixed_img, moving=moving_img, type_of_transform='SyN',
                                    metric='MI', radius_or_number_of_bins=32,
                                    sampling_strategy='Regular', sampling_percentage=samp,
                                    mask=mask, convergence=[1000,500,250],
                                    shrink_factors=[8,4,2], smoothing_sigmas=[4,2,1],
                                    verbose=verbose)
                                    
        
        aff = None
        t_aff = 0
        try:
            print("Exécution du recalage affine...")
            t0 = time.time()
            aff = _run_affine(local_mask, 0.10)
            t_aff = time.time() - t0
            print(f"Recalage affine terminé en {t_aff:.2f}s")
        except Exception as e:
            print(f"  ⚠️ Affine échouée: {e}")
            
            try:
                print("  ▸ Relance sans masque, samp 0.05")
                t0 = time.time()
                aff = _run_affine(None, 0.05)
                t_aff = time.time() - t0
                print(f"Recalage affine terminé en {t_aff:.2f}s")
            except Exception as e2:
                logger.error(f"Échec complet du recalage affine: {e2}")
                print(f"❌ Échec complet du recalage affine: {e2}")
                continue  

        
        if aff is None or 'warpedmovout' not in aff:
            logger.error("Recalage affine n'a pas produit de résultat valide")
            print("❌ Recalage affine n'a pas produit de résultat valide")
            continue  

        
        syn = None
        t_syn = 0
        try:
            print("Exécution du recalage non-linéaire (SyN)...")
            t1 = time.time()
            syn = ants.registration(fixed=fixed_img, moving=aff['warpedmovout'],
                                   type_of_transform='Affine', metric='CC',
                                   radius_or_number_of_bins=4, gradient_step=0.05,
                                   flow_sigma=3, total_sigma=0, mask=local_mask,
                                   convergence=[100,70,50],
                                   shrink_factors=[6,4,2], smoothing_sigmas=[2,1,0],
                                   verbose=verbose)
            t_syn = time.time() - t1
            print(f"Recalage SyN terminé en {t_syn:.2f}s")
        except Exception as e:
            logger.error(f"Erreur lors du recalage SyN: {e}")
            print(f"❌ Erreur lors du recalage SyN: {e}")
            continue  
        
        
        if syn is None or 'fwdtransforms' not in syn:
            logger.error("Recalage SyN n'a pas produit de résultat valide")
            print("❌ Recalage SyN n'a pas produit de résultat valide")
            continue  

        
        final_path = output_dir / f"sub-{sid}_{name}_syn.nii.gz"
        print(f"  ▸ Sauvegarde dans {final_path}")
        
        apply_success = False
        try:
            
            
            transformed_img = ants.apply_transforms(
                fixed=fixed_img,
                moving=moving_img,
                transformlist=syn['fwdtransforms'],
                interpolator=interp,
                verbose=verbose
            )
            
            if transformed_img is not None:
                
                ants.image_write(transformed_img, str(final_path))
                apply_success = True
                print(f"✓ Image transformée enregistrée avec succès: {final_path}")
            else:
                logger.error("La transformation a retourné None")
                print("❌ La transformation a retourné None")
        except Exception as e:
            logger.error(f"Erreur lors de l'application/sauvegarde de la transformation: {e}")
            print(f"❌ Erreur lors de l'application/sauvegarde de la transformation: {e}")
        
        
        if not final_path.exists():
            logger.error(f"Le fichier de sortie n'a pas été créé: {final_path}")
            print(f"❌ Le fichier de sortie n'a pas été créé: {final_path}")
            continue
        else:
            print(f"✓ Fichier vérifié: {final_path} existe")

        
        jac_warn = False
        try:
            warp_files = [w for w in syn['fwdtransforms'] if 'Warp' in w]
            if warp_files:
                warp = warp_files[0]  
                jac = ants.create_jacobian_determinant_image(fixed_img, warp, False)
                if (jac.numpy() < 0).any(): 
                    jac_warn = True
                    print("⚠️ Attention: Déterminant jacobien négatif détecté")
        except Exception as e:
            logger.warning(f"Impossible de vérifier le jacobien: {e}")
            print(f"⚠️ Impossible de vérifier le jacobien: {e}")

        
        if apply_success:
            results["registered_atlas_paths"][name] = str(final_path)
            results["metrics"][name] = {
                "affine_sec": round(t_aff, 2),
                "syn_sec": round(t_syn, 2),
                "jacobian_warning": jac_warn
            }
            print(f"  ✓ {name} OK – {t_aff+t_syn:.2f}s")
        else:
            print(f"❌ Échec du recalage pour {name}")

    return results


def visualize_atlas_comparison(subj_path, simple_path, adv_path, name, out_dir=None):

    subj   = nib.load(str(subj_path))           
    simple = ants.image_read(str(simple_path))  
    adv    = ants.image_read(str(adv_path))     

    
    adv_nn = ants.resample_image_to_target(adv, simple, interp_type='nearestNeighbor')

    
    s_bin = (simple.numpy() > 0).astype(np.uint8)
    a_bin = (adv_nn.numpy() > 0).astype(np.uint8)
    dice, jacc = _dice_jaccard(s_bin, a_bin)
    print(f"Dice={dice:.4f} | Jaccard={jacc:.4f}")

    
    subj_data = subj.get_fdata()
    p1, p99 = np.percentile(subj_data[subj_data > 0], [1, 99])
    subj_disp = (np.clip(subj_data, p1, p99) - p1) / (p99 - p1)

    
    z_scores = (s_bin + a_bin).reshape(-1, s_bin.shape[-1]).sum(0)
    z_ids = np.argsort(z_scores)[-3:]

    
    fig = plt.figure(figsize=(18, 12))
    gs = gridspec.GridSpec(3, len(z_ids), figure=fig)
    titles = ["Sujet", "Simple", "Avancé"]
    
    for col, z in enumerate(z_ids):
        if subj_disp[..., z].max() == 0: continue
        for row, overlay in enumerate([None, s_bin[..., z], a_bin[..., z]]):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(subj_disp[..., z].T, cmap='gray')
            if overlay is not None:
                ax.imshow(overlay.T, cmap='cool', alpha=0.5)
            if col == 0: ax.set_ylabel(titles[row], fontsize=12)
            if row == 0: ax.set_title(f'z={z}')
            ax.axis('off')
    
    plt.suptitle(f'Comparaison - {name}', fontsize=16)
    plt.tight_layout()
    
    if out_dir:
        out_file = Path(out_dir) / f'{name}_compare.png'
        plt.savefig(out_file, dpi=150)
        print(f'Figure sauvegardée → {out_file}')
    
    plt.show()


atlas_dir       = Path("./sam_neuroadapt_v2_real_atlases/atlases")
derivatives_dir = DERIVATIVES_DIR  # défini dans cell 0
subject_id      = SUBJECT_ID  # défini dans cell 0  


atlas_files = {"brain_mask": atlas_dir/"MNI152NLin2009cAsym_brainmask_2mm.nii.gz"}
atlas_paths = {k: p for k,p in atlas_files.items() if p.exists()}
print('Atlas disponibles:', atlas_paths)


n4_path = ANAT_DIR / f"sub-{subject_id}_desc-n4_T1w.nii.gz"
if not n4_path.exists():
    print(f"⚠️ Image N4 non trouvée: {n4_path}")
    
    whitestripe_path = ANAT_DIR / f"sub-{subject_id}_desc-n4_desc-ws_T1w.nii.gz"
    if whitestripe_path.exists():
        n4_path = whitestripe_path
        print(f"✓ Utilisation de l'image WhiteStripe comme alternative: {n4_path}")
    else:
        print("❌ Aucune image prétraitée trouvée!")

out_dir = ANAT_DIR / "atlases_advanced"
out_dir.mkdir(parents=True, exist_ok=True)


if n4_path.exists():
    results = syn_atlas_registration(
        subject_n4_path=n4_path, 
        atlas_paths=atlas_paths, 
        output_dir=out_dir, 
        subject_id=subject_id,  
        create_brain_mask=True
    )

    
    basic_dir = ANAT_DIR / "atlases"
    for name, adv_path in results["registered_atlas_paths"].items():
        simple_path = basic_dir / f"sub-{subject_id}_space-subject_{name}.nii.gz"
        print(f"Comparaison {name}: {simple_path} vs {adv_path}")
        
        adv_path_obj = Path(adv_path)
        if simple_path.exists() and adv_path_obj.exists():
            visualize_atlas_comparison(n4_path, simple_path, adv_path_obj, name, out_dir)
        else:
            if not simple_path.exists():
                print(f"  ⚠️ Fichier simple manquant: {simple_path}")
            if not adv_path_obj.exists():
                print(f"  ⚠️ Fichier avancé manquant: {adv_path_obj}")
else:
    print("❌ Impossible de continuer: image source manquante")

### Etat d'art approche for recalage rapide

In [ ]:
import time
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ants
import logging
import os


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("atlas_registration")


def _resample_to_1mm(img, interp_name):
    interp_codes = {
        'nearestNeighbor': 0, 'linear': 1, 'bSpline': 2,
        'cosineWindowedSinc': 3, 'lanczosWindowedSinc': 4,
        'genericLabel': 5, 'multiLabel': 6, 'gaussian': 7
    }
    code = interp_codes.get(interp_name, 0)
    return ants.resample_image(img, (1, 1, 1), use_voxels=False, interp_type=code)

def _dilate_mask(img, r=1):
    return ants.iMath(img, 'MD', r)

def _dice_jaccard(a, b):
    tp   = np.logical_and(a, b).sum()
    volA = a.sum(); volB = b.sum()
    dice = 2*tp / (volA + volB + 1e-8)
    jacc = tp / (np.logical_or(a, b).sum() + 1e-8)
    return dice, jacc

def _safe_slice(vol, z):
    return vol[:, :, z] if np.any(vol[:, :, z]) else None


def advanced_atlas_registration(subject_n4_path, atlas_paths, output_dir, subject_id=None,
                                create_brain_mask=True, verbose=False):

    subject_n4_path, output_dir = Path(subject_n4_path), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    
    if subject_id:
        sid = subject_id
    else:
        
        path_parts = str(subject_n4_path).split('/')
        for part in path_parts:
            if part.startswith("sub-"):
                sid = part.replace("sub-", "")
                break
        else:
            sid = subject_n4_path.stem.split("_")[0]
    
    print(f"\n=== Recalage cascade pour {sid} ===")
    print(f"Image source: {subject_n4_path}")
    print(f"Dossier de sortie: {output_dir}")

    
    if not subject_n4_path.exists():
        logger.error(f"L'image source n'existe pas: {subject_n4_path}")
        return {"subject_id": sid, "registered_atlas_paths": {}, "metrics": {}, "error": "Source image not found"}
    
    try:
        fixed_img = ants.image_read(str(subject_n4_path))
        print(f"Image chargée: dimensions {fixed_img.shape}, espacement {fixed_img.spacing}")
    except Exception as e:
        logger.error(f"Erreur lors du chargement de l'image source: {e}")
        return {"subject_id": sid, "registered_atlas_paths": {}, "metrics": {}, "error": str(e)}
    
    brain_mask = None
    if create_brain_mask:
        try:
            # low_thresh=0.2 échoue si Otsu < 0.2*max (ITK BinaryThreshold error)
            # On utilise un seuil absolu minimal (0.01) compatible avec toute plage
            # Otsu sur image N4 (valeurs 0-1205) — WhiteStripe a des négatifs
            # qui font monter le seuil Otsu à 0 → 76% du volume (tête entière)
            brain_mask = ants.get_mask(fixed_img)  # Otsu pur sur N4
            _otsu_n = int(np.sum(brain_mask.numpy() > 0))
            _otsu_pct = _otsu_n / brain_mask.numpy().size
            print(f"Masque Otsu N4: {_otsu_n:,} voxels ({_otsu_pct:.1%} du volume)")
            print(f"Masque Otsu créé: somme={_otsu_n}")
        except Exception as e:
            logger.error(f"Erreur lors de la création du masque cérébral Otsu: {e}")
            print(f"⚠️ Impossible de créer le masque Otsu: {e}")
            

    results = {"subject_id": sid, "registered_atlas_paths": {}, "metrics": {}}

    for name, path in atlas_paths.items():
        if not Path(path).exists():
            logger.error(f"Atlas {name} introuvable: {path}")
            print(f"⚠️ Atlas {name} introuvable: {path}")
            continue

        print(f"\n→ Traitement atlas **{name}**")
        
        try:
            moving_img = ants.image_read(str(path))
            print(f"Atlas {name} chargé: dimensions {moving_img.shape}")
        except Exception as e:
            logger.error(f"Erreur lors du chargement de l'atlas {name}: {e}")
            print(f"⚠️ Impossible de charger l'atlas {name}: {e}")
            continue

        # Charger le template T1 correspondant au masque. Point critique: on garde
        # le template dans son espace natif MNI pour l'optimisation ANTs. Le resampling
        # pré-recalage sur la grille sujet créait des atlas décalés sur plusieurs sujets.
        _mask_name = Path(path).name
        if 'UNCBCP06M' in _mask_name or 'tpl-UNCBCP06M_desc-brain_mask' in _mask_name:
            mni_t1_candidates = [Path(path).parent / 'tpl-UNCBCP06M_T1w.nii.gz']
        elif 'MNI152_T1_2mm_brain_mask' in _mask_name:
            mni_t1_candidates = [Path(path).parent / 'MNI152_T1_2mm.nii.gz']
        elif 'tpl-MNI152NLin2009cAsym' in _mask_name:
            mni_t1_candidates = [Path(path).parent / 'tpl-MNI152NLin2009cAsym_res-02_T1w.nii.gz']
        else:
            mni_t1_candidates = [
                Path(path).parent / 'MNI152NLin2009cAsym_T1w_2mm.nii.gz',
                Path(path).parent / 'tpl-MNI152NLin2009cAsym_res-02_T1w.nii.gz',
                Path(path).parent / 'MNI152_T1_2mm.nii.gz',
            ]
        moving_t1_img = None
        for _mni_t1 in mni_t1_candidates:
            if _mni_t1.exists():
                try:
                    moving_t1_img = ants.image_read(str(_mni_t1))
                    print(f'Template T1 chargé: {_mni_t1.name}')
                    break
                except Exception:
                    pass
        if moving_t1_img is None:
            print('⚠️ Template T1 non trouvé, fallback masque binaire')
            moving_t1_img = moving_img

        interp = 'nearestNeighbor'
        print("Recalage natif MNI→sujet: aucun resampling pré-optimisation")

        local_mask = brain_mask
        try:
            if np.count_nonzero(moving_img.numpy()) < 3000:
                atlas_dil = ants.resample_image_to_target(_dilate_mask(moving_img, 1),
                                                        fixed_img, interp_type='nearestNeighbor')
                if brain_mask is not None:
                    local_mask = brain_mask * _dilate_mask(atlas_dil, 3)

            if local_mask is not None and np.count_nonzero(local_mask.numpy()) < 1000:
                print("  ▸ masque trop petit → recalage sans masque")
                local_mask = None
        except Exception as e:
            logger.error(f"Erreur lors de la création du masque local: {e}")
            print(f"⚠️ Erreur lors de la création du masque local: {e}")
            local_mask = None

        
        def _run_affine(mask, samp):
            return ants.registration(
                fixed=fixed_img,
                moving=moving_t1_img,  # T1 template -> meilleurs gradients
                type_of_transform='Affine',
                aff_metric='mattes',
                aff_sampling=32,
                aff_random_sampling_rate=samp,
                mask=mask,
                aff_iterations=(1000, 500, 250, 0),
                aff_shrink_factors=(8, 4, 2, 1),
                aff_smoothing_sigmas=(4, 2, 1, 0),
                verbose=verbose,
            )

        _variants = []
        if local_mask is not None:
            _variants.append(("masked", local_mask, "aff", 0.10))
        _variants.append(("nomask", None, "aff_nomask", 0.10))

        for _variant_name, _variant_mask, _variant_suffix, _variant_samp in _variants:
            aff = None
            t_aff = 0
            try:
                print(f"Exécution du recalage affine ({_variant_name})...")
                t0 = time.time()
                aff = _run_affine(_variant_mask, _variant_samp)
                t_aff = time.time() - t0
                print(f"Recalage affine ({_variant_name}) terminé en {t_aff:.2f}s")
            except Exception as e:
                logger.error(f"Échec recalage affine ({_variant_name}) pour {name}: {e}")
                print(f"  ⚠️ Affine ({_variant_name}) échouée: {e}")
                continue

            if aff is None or 'warpedmovout' not in aff or 'fwdtransforms' not in aff:
                logger.error(f"Recalage affine ({_variant_name}) n'a pas produit de résultat valide")
                print(f"❌ Recalage affine ({_variant_name}) invalide")
                continue

            final_path = output_dir / f"sub-{sid}_{name}_{_variant_suffix}.nii.gz"
            print(f"  ▸ Sauvegarde dans {final_path}")

            apply_success = False
            try:
                transformed_img = ants.apply_transforms(
                    fixed=fixed_img,
                    moving=moving_img,
                    transformlist=aff['fwdtransforms'],
                    interpolator=interp,
                    verbose=verbose
                )
                if transformed_img is not None:
                    ants.image_write(transformed_img, str(final_path))
                    apply_success = True
                    print(f"✓ Image transformée enregistrée avec succès: {final_path}")
                else:
                    logger.error("La transformation a retourné None")
                    print("❌ La transformation a retourné None")
            except Exception as e:
                logger.error(f"Erreur lors de l'application/sauvegarde de la transformation: {e}")
                print(f"❌ Erreur lors de l'application/sauvegarde de la transformation: {e}")

            if not final_path.exists():
                logger.error(f"Le fichier de sortie n'a pas été créé: {final_path}")
                print(f"❌ Le fichier de sortie n'a pas été créé: {final_path}")
                continue
            print(f"✓ Fichier vérifié: {final_path} existe")

            jac_warn = False
            try:
                warp_files = [w for w in aff['fwdtransforms'] if 'Warp' in w]
                if warp_files:
                    warp = warp_files[0]
                    jac = ants.create_jacobian_determinant_image(fixed_img, warp, False)
                    if (jac.numpy() < 0).any():
                        jac_warn = True
                        print("⚠️ Attention: Déterminant jacobien négatif détecté")
            except Exception as e:
                logger.warning(f"Impossible de vérifier le jacobien: {e}")
                print(f"⚠️ Impossible de vérifier le jacobien: {e}")

            if apply_success:
                _result_key = name if _variant_suffix == "aff" else f"{name}_{_variant_name}"
                results["registered_atlas_paths"][_result_key] = str(final_path)
                results["metrics"][_result_key] = {
                    "affine_sec": round(t_aff, 2),
                    "variant": _variant_name,
                    "jacobian_warning": jac_warn,
                }
            else:
                print(f"❌ Échec du recalage pour {name} ({_variant_name})")

    return results


def visualize_atlas_comparison(subj_path, simple_path, adv_path, name, out_dir=None):
    """
    Affiche et compare la version « simple » vs « avancée » d'un atlas.
    """
    subj   = nib.load(str(subj_path))           
    simple = ants.image_read(str(simple_path))  
    adv    = ants.image_read(str(adv_path))     

    
    adv_nn = ants.resample_image_to_target(adv, simple, interp_type='nearestNeighbor')

    
    s_bin = (simple.numpy() > 0).astype(np.uint8)
    a_bin = (adv_nn.numpy() > 0).astype(np.uint8)
    dice, jacc = _dice_jaccard(s_bin, a_bin)
    print(f"Dice={dice:.4f} | Jaccard={jacc:.4f}")

    
    subj_data = subj.get_fdata()
    p1, p99 = np.percentile(subj_data[subj_data > 0], [1, 99])
    subj_disp = (np.clip(subj_data, p1, p99) - p1) / (p99 - p1)

    
    z_scores = (s_bin + a_bin).reshape(-1, s_bin.shape[-1]).sum(0)
    z_ids = np.argsort(z_scores)[-3:]

    
    fig = plt.figure(figsize=(18, 12))
    gs = gridspec.GridSpec(3, len(z_ids), figure=fig)
    titles = ["Sujet", "Simple", "Avancé"]
    
    for col, z in enumerate(z_ids):
        if subj_disp[..., z].max() == 0: continue
        for row, overlay in enumerate([None, s_bin[..., z], a_bin[..., z]]):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(subj_disp[..., z].T, cmap='gray')
            if overlay is not None:
                ax.imshow(overlay.T, cmap='cool', alpha=0.5)
            if col == 0: ax.set_ylabel(titles[row], fontsize=12)
            if row == 0: ax.set_title(f'z={z}')
            ax.axis('off')
    
    plt.suptitle(f'Comparaison - {name}', fontsize=16)
    plt.tight_layout()
    
    if out_dir:
        out_file = Path(out_dir) / f'{name}_compare.png'
        plt.savefig(out_file, dpi=150)
        print(f'Figure sauvegardée → {out_file}')
    
    plt.show()


atlas_dir       = Path("./sam_neuroadapt_v2_real_atlases/atlases")
derivatives_dir = DERIVATIVES_DIR  # défini dans cell 0
subject_id      = SUBJECT_ID  # défini dans cell 0  


atlas_files = {
    "brain_mask": atlas_dir/"MNI152NLin2009cAsym_brainmask_2mm.nii.gz",
    "brain_mask_mni152": atlas_dir/"MNI152_T1_2mm_brain_mask.nii.gz",
    "brain_mask_tpl": atlas_dir/"tpl-MNI152NLin2009cAsym_res-02_desc-brain_mask.nii.gz",
}
atlas_paths = {k: p for k,p in atlas_files.items() if p.exists()}
print('Atlas disponibles:', atlas_paths)


n4_path = ANAT_DIR / f"sub-{subject_id}_desc-n4_T1w.nii.gz"  # chemin centralisé
if not n4_path.exists():
    print(f"⚠️ Image N4 non trouvée: {n4_path}")
    
    whitestripe_path = ANAT_DIR / f"sub-{subject_id}_desc-n4_desc-ws_T1w.nii.gz"  # chemin centralisé
    if whitestripe_path.exists():
        n4_path = whitestripe_path
        print(f"✓ Utilisation de l'image WhiteStripe comme alternative: {n4_path}")
    else:
        print("❌ Aucune image prétraitée trouvée!")

out_dir = ANAT_DIR / "atlases_advanced"  # chemin centralisé
out_dir.mkdir(parents=True, exist_ok=True)


if n4_path.exists():
    results = advanced_atlas_registration(
        subject_n4_path=n4_path, 
        atlas_paths=atlas_paths, 
        output_dir=out_dir, 
        subject_id=subject_id,  
        create_brain_mask=True
    )

    
    basic_dir = ANAT_DIR / "atlases"  # chemin centralisé
    for name, adv_path in results["registered_atlas_paths"].items():
        simple_path = basic_dir / f"sub-{subject_id}_space-subject_{name}.nii.gz"
        print(f"Comparaison {name}: {simple_path} vs {adv_path}")
        
        adv_path_obj = Path(adv_path)
        if simple_path.exists() and adv_path_obj.exists():
            visualize_atlas_comparison(n4_path, simple_path, adv_path_obj, name, out_dir)
        else:
            if not simple_path.exists():
                print(f"  ⚠️ Fichier simple manquant: {simple_path}")
            if not adv_path_obj.exists():
                print(f"  ⚠️ Fichier avancé manquant: {adv_path_obj}")
else:
    print("❌ Impossible de continuer: image source manquante")

### Audit Recalage atlas

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from skimage.measure import label as skimage_label, regionprops
from scipy.ndimage import binary_erosion, binary_dilation, gaussian_filter, label as ndimage_label
from skimage.filters import threshold_otsu
from skimage.morphology import ball, binary_closing, binary_opening
from scipy import ndimage

def evaluate_brain_mask(subject_image_path, mask_path, output_dir=None, contours=True):
 
    
    subject_image_path = Path(subject_image_path)
    mask_path = Path(mask_path)
    
    
    if not subject_image_path.exists():
        raise FileNotFoundError(f"Image sujet introuvable: {subject_image_path}")
    if not mask_path.exists():
        raise FileNotFoundError(f"Masque introuvable: {mask_path}")
    
    
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    
    print("Chargement des images pour évaluation...")
    subject_img = nib.load(str(subject_image_path))
    subject_data = subject_img.get_fdata()
    
    mask_img = nib.load(str(mask_path))
    mask_data = mask_img.get_fdata() > 0  
    
    
    if subject_data.shape != mask_data.shape:
        raise ValueError(f"Dimensions incohérentes: sujet {subject_data.shape} vs masque {mask_data.shape}")
    
    
    t1_display = subject_data.copy()
    p1, p99 = np.percentile(t1_display[t1_display > 0], (1, 99))
    t1_display = np.clip(t1_display, p1, p99)
    t1_display = (t1_display - p1) / (p99 - p1)
    
    
    
    
    mask_ratio = np.sum(mask_data) / mask_data.size
    print(f"Ratio du volume du masque: {mask_ratio:.2%}")
    
    
    volume_score = 0
    if 0.20 <= mask_ratio <= 0.35:  
        print("✓ Volume du masque dans la plage attendue (20-35%)")
        volume_score = 2.0
    elif 0.15 <= mask_ratio <= 0.40:  
        print("⚠️ Volume du masque acceptable mais non optimal")
        volume_score = 1.0
    else:
        print("❌ Volume du masque hors de la plage attendue")
        volume_score = 0
    
    
    labeled_mask = skimage_label(mask_data)
    regions = regionprops(labeled_mask)
    roi_count = len(regions)
    
    
    connectivity_score = 2.0 if roi_count == 1 else 0
    print(f"Nombre de régions distinctes: {roi_count} {'✓' if roi_count == 1 else '❌'}")
    
    if roi_count > 1:
        
        volumes = [r.area for r in regions]
        largest_region_ratio = max(volumes) / sum(volumes)
        if largest_region_ratio > 0.95:
            print(f"⚠️ Plusieurs régions, mais la plus grande représente {largest_region_ratio:.1%} du volume")
            connectivity_score = 1.0  
        else:
            print(f"❌ Fragmentation importante: {roi_count} régions non connectées")
    
    
    try:
        mid_x = mask_data.shape[0] // 2
        left_half = mask_data[:mid_x, :, :]
        right_half_flipped = np.flip(mask_data[mid_x:, :, :], axis=0)
        
        
        min_x = min(left_half.shape[0], right_half_flipped.shape[0])
        
        
        intersection = np.sum(left_half[:min_x, :, :] & right_half_flipped[:min_x, :, :])
        union = np.sum(left_half[:min_x, :, :] | right_half_flipped[:min_x, :, :])
        symmetry_score_val = intersection / union if union > 0 else 0
        
        
        symmetry_score = 0
        if symmetry_score_val > 0.9:
            print(f"✓ Excellente symétrie gauche-droite: {symmetry_score_val:.4f}")
            symmetry_score = 2.0
        elif symmetry_score_val > 0.8:
            print(f"⚠️ Symétrie gauche-droite acceptable: {symmetry_score_val:.4f}")
            symmetry_score = 1.0
        else:
            print(f"❌ Faible symétrie gauche-droite: {symmetry_score_val:.4f}")
    except Exception as e:
        print(f"Impossible de calculer la symétrie: {e}")
        symmetry_score_val = None
        symmetry_score = 0
    
    
    try:
        
        mask_dilated = binary_dilation(mask_data, iterations=1)
        mask_eroded = binary_erosion(mask_data, iterations=1)
        mask_contour_3d = mask_dilated & ~mask_eroded
        
        
        smoothed = gaussian_filter(subject_data, sigma=0.5)
        sx = ndimage.sobel(smoothed, axis=0)
        sy = ndimage.sobel(smoothed, axis=1)
        sz = ndimage.sobel(smoothed, axis=2)
        gradient_mag = np.sqrt(sx**2 + sy**2 + sz**2)
        
        
        if np.max(gradient_mag) > 0:
            gradient_mag = gradient_mag / np.max(gradient_mag)
        
        
        
        border_gradient = gradient_mag[mask_contour_3d]
        inner_gradient = gradient_mag[mask_eroded]
        
        
        mean_border_gradient = np.mean(border_gradient) if len(border_gradient) > 0 else 0
        mean_inner_gradient = np.mean(inner_gradient) if len(inner_gradient) > 0 else 0
        
        
        if mean_inner_gradient > 0:
            edge_contrast_ratio = mean_border_gradient / mean_inner_gradient
        else:
            edge_contrast_ratio = 0
            
        
        edge_score = 0
        if edge_contrast_ratio > 1.5:
            print(f"✓ Excellente définition des bords: {edge_contrast_ratio:.2f}")
            edge_score = 3.0
        elif edge_contrast_ratio > 1.0:
            print(f"✓ Bonne définition des bords: {edge_contrast_ratio:.2f}")
            edge_score = 2.0
        elif edge_contrast_ratio > 0.8:
            print(f"⚠️ Définition acceptable des bords: {edge_contrast_ratio:.2f}")
            edge_score = 1.0
        else:
            print(f"❌ Faible définition des bords: {edge_contrast_ratio:.2f}")
    except Exception as e:
        print(f"Impossible de calculer la qualité des bords: {e}")
        edge_contrast_ratio = None
        edge_score = 0
    
    
    
    if contours:
        display_mask = mask_contour_3d if 'mask_contour_3d' in locals() else binary_dilation(mask_data, iterations=1) & ~mask_data
    else:
        display_mask = mask_data
    
    
    
    
    slice_x = np.argmax(np.sum(mask_data, axis=(1, 2)))
    slice_y = np.argmax(np.sum(mask_data, axis=(0, 2)))
    slice_z = np.argmax(np.sum(mask_data, axis=(0, 1)))
    
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    
    if contours:
        
        mask_cmap = LinearSegmentedColormap.from_list('mask_cmap', [(0, 0, 0, 0), (1, 0, 0, 1)])
    else:
        
        mask_cmap = LinearSegmentedColormap.from_list('mask_cmap', [(0, 0, 0, 0), (1, 0, 0, 0.5)])
    
    
    axes[0, 0].imshow(t1_display[slice_x, :, :].T, cmap='gray')
    axes[0, 0].set_title(f"T1 - Plan Sagittal (x={slice_x})")
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(t1_display[:, slice_y, :].T, cmap='gray')
    axes[0, 1].set_title(f"T1 - Plan Coronal (y={slice_y})")
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(t1_display[:, :, slice_z].T, cmap='gray')
    axes[0, 2].set_title(f"T1 - Plan Axial (z={slice_z})")
    axes[0, 2].axis('off')
    
    
    axes[1, 0].imshow(t1_display[slice_x, :, :].T, cmap='gray')
    axes[1, 0].imshow(display_mask[slice_x, :, :].T, cmap=mask_cmap)
    axes[1, 0].set_title("T1 + Masque - Sagittal")
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(t1_display[:, slice_y, :].T, cmap='gray')
    axes[1, 1].imshow(display_mask[:, slice_y, :].T, cmap=mask_cmap)
    axes[1, 1].set_title("T1 + Masque - Coronal")
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(t1_display[:, :, slice_z].T, cmap='gray')
    axes[1, 2].imshow(display_mask[:, :, slice_z].T, cmap=mask_cmap)
    axes[1, 2].set_title("T1 + Masque - Axial")
    axes[1, 2].axis('off')
    
    
    
    score_components = {
        "Volume": volume_score,
        "Connectivité": connectivity_score,
        "Symétrie": symmetry_score,
        "Bords": edge_score
    }
    
    
    score_weights = {
        "Volume": 2.0,
        "Connectivité": 2.0,
        "Symétrie": 2.0,
        "Bords": 3.0,
        "Évaluation visuelle": 1.0
    }
    
    
    total_auto_score = sum(score_components.values())
    max_auto_score = sum(score_weights.values()) - score_weights["Évaluation visuelle"]
    
    
    normalized_auto_score = (total_auto_score / max_auto_score) * 9
    
    
    visual_score = 1.0  
    quality_score = normalized_auto_score + visual_score
    quality_score = min(10, quality_score)  
    
    
    if quality_score >= 9:
        quality_rating = "Excellent"
    elif quality_score >= 8:
        quality_rating = "Très bon"
    elif quality_score >= 7:
        quality_rating = "Bon"
    elif quality_score >= 5:
        quality_rating = "Acceptable"
    else:
        quality_rating = "Médiocre"
    
    
    plt.suptitle(
        f"Évaluation du masque cérébral - {mask_path.name}\n"
        f"Score: {quality_score:.1f}/10 ({quality_rating}) | "
        f"Volume: {mask_ratio:.1%} | Symétrie: {symmetry_score_val:.2f} | "
        f"Définition des bords: {edge_contrast_ratio:.2f}",
        fontsize=16
    )
    
    plt.tight_layout()
    
    
    if output_dir:
        fig_path = output_dir / f"evaluation_masque_{subject_image_path.stem}_{mask_path.stem}.png"
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f"Figure sauvegardée: {fig_path}")
    
    plt.show()
    
    
    print("\n=== Détails du score ===")
    for component, weight in score_weights.items():
        if component in score_components:
            component_score = score_components[component]
            print(f"{component} ({weight} pts max): {component_score:.1f} pts")
        else:
            print(f"{component} ({weight} pts max): Évaluation manuelle requise")
    
    print(f"\nScore automatique: {normalized_auto_score:.1f}/9")
    print(f"Score total avec évaluation visuelle: {quality_score:.1f}/10 - {quality_rating}")
    
    
    metrics = {
        'mask_ratio': mask_ratio,
        'roi_count': roi_count,
        'symmetry_score': symmetry_score_val,
        'edge_contrast_ratio': edge_contrast_ratio,
        'volume_score': volume_score,
        'connectivity_score': connectivity_score,
        'symmetry_score_pts': symmetry_score,
        'edge_score': edge_score,
        'quality_score': quality_score,
        'quality_rating': quality_rating
    }
    
    return metrics



if __name__ == "__main__":
    if 'ANAT_DIR' not in globals() or 'SUBJECT_ID' not in globals():
        raise RuntimeError("Exécuter la cellule de configuration du sujet avant l'audit atlas")

    subject_id = SUBJECT_ID
    derivatives_dir = DERIVATIVES_DIR
    anat_dir = ANAT_DIR
    whitestripe_path = anat_dir / f"sub-{subject_id}_desc-n4_desc-ws_T1w.nii.gz"
    if not whitestripe_path.exists():
        whitestripe_path = anat_dir / f"sub-{subject_id}_desc-n4_T1w.nii.gz"

    atlases_dir = anat_dir / "atlases"
    advanced_dir = anat_dir / "atlases_advanced"

    _bm_aff = advanced_dir / f"sub-{subject_id}_brain_mask_aff.nii.gz"
    _bm_syn = advanced_dir / f"sub-{subject_id}_brain_mask_syn.nii.gz"
    brain_mask_advanced = _bm_aff if _bm_aff.exists() else _bm_syn
    brain_mask_simple = atlases_dir / f"sub-{subject_id}_space-subject_brain_mask.nii.gz"

    if brain_mask_advanced.exists():
        mask_path = brain_mask_advanced
        print(f"Utilisation du masque avancé: {mask_path}")
    elif brain_mask_simple.exists():
        mask_path = brain_mask_simple
        print(f"Utilisation du masque simple: {mask_path}")
    else:
        print("Aucun masque recalé trouvé!")
        mask_path = None

    if mask_path:
        metrics = evaluate_brain_mask(
            subject_image_path=whitestripe_path,
            mask_path=mask_path,
            output_dir=anat_dir,
            contours=True,
        )

        print("\nRésumé des métriques:")
        for key, value in metrics.items():
            print(f"- {key}: {value}")


### Cellule 12 : Configuration pour SAM2 et installation des dépendances

In [ ]:
# ── Fix B1 : résolution du conflit de chemin SAM2 ──────────────────────
# Le dossier sam2/ (repo) shadowe le package Python. On ajoute le repo
# à sys.path pour que 'import sam2' trouve sam2/sam2/ (le vrai package).
import sys as _sys
from pathlib import Path as _Path
_sam2_repo = (_Path('.') / 'sam2').resolve()
if _sam2_repo.exists() and str(_sam2_repo) not in _sys.path:
    _sys.path.insert(0, str(_sam2_repo))
del _sam2_repo, _Path, _sys
# ─────────────────────────────────────────────────────────────────────────

import os
import sys
import time
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
import torch
from scipy import ndimage
from skimage.morphology import ball, binary_closing, binary_opening, binary_dilation, binary_erosion


try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    print("✓ SAM2 est correctement installé")
except (ImportError, RuntimeError):
    print("⚠️ SAM2 n'est pas installé! Installation à partir de GitHub...")
    try: 
        from sam2.build_sam import build_sam2
        from sam2.sam2_image_predictor import SAM2ImagePredictor
        print("✓ SAM2 installé avec succès!")
    except Exception as e:
        print(f"❌ Erreur lors de l'installation de SAM2: {e}")
        print("⚠️ Veuillez installer SAM2 manuellement: pip install -e .")
        sys.exit(1)


try:
    from hydra import initialize_config_dir
    from hydra.core.global_hydra import GlobalHydra
    print("✓ hydra-core est correctement installé")
except (ImportError, RuntimeError):
    print("⚠️ hydra-core n'est pas installé! Installation en cours...")
    try: 
        from hydra import initialize_config_dir
        from hydra.core.global_hydra import GlobalHydra
        print("✓ hydra-core installé avec succès!")
    except Exception as e:
        print(f"❌ Erreur lors de l'installation de hydra-core: {e}")
        sys.exit(1)


if 'BASE_DIR' not in globals():
    BASE_DIR = Path('./sam_neuroadapt_v2_real_atlases')
    MODELS_DIR = BASE_DIR / 'models'
    DERIVATIVES_DIR = Path('./sam_neuroadapt_v2_real_atlases/results/bids_dataset/derivatives/sam-neuroadapt')


sam2_checkpoint_path = MODELS_DIR / "sam2.1_hiera_large.pt"
sam2_config_path = MODELS_DIR / "sam2.1_hiera_l.yaml"

if not sam2_checkpoint_path.exists():
    print("⚠️ Checkpoint SAM2 non trouvé!")
    print(f"Veuillez télécharger le modèle SAM2 depuis HuggingFace et le placer à {sam2_checkpoint_path}")
    print("URL: https://huggingface.co/facebookresearch/sam2")
else:
    print(f"✓ Checkpoint SAM2 trouvé: {sam2_checkpoint_path}")

if not sam2_config_path.exists():
    print("⚠️ Configuration SAM2 non trouvée!")
    print(f"Veuillez télécharger le fichier de configuration YAML depuis GitHub et le placer à {sam2_config_path}")
    print("URL: https://github.com/facebookresearch/sam2/blob/main/configs/sam2.1_hiera_l.yaml")
else:
    print(f"✓ Configuration SAM2 trouvée: {sam2_config_path}")


device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"✓ GPU détecté: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU non détecté, utilisation du CPU (performance réduite)")

print("\nEnvironnement prêt pour SAM-NeuroAdapt v2!")

### Cellule 13 : Implémentation de la classe SAM2BrainExtractor

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Union

@dataclass
class SegmentationParams:

    model_config_path: Optional[Path] = None
    checkpoint_path: Optional[Path] = None
    device: str = "cuda"
    
    
    initial_confidence_threshold: float = 0.85
    propagation_confidence_threshold: float = 0.70
    adapative_thresholds: bool = True
    
    
    prompt_slice_index: Optional[int] = None  
    prompt_point_coords: Optional[Tuple[int, int]] = None  
    
    
    post_process: bool = True
    closing_radius: int = 2
    largest_component_only: bool = True
    fill_holes: bool = True
    smooth_borders: bool = True
    
    
    use_atlas_guidance: bool = True
    atlas_paths: Dict[str, Path] = field(default_factory=dict)
    atlas_weight: float = 0.7
    atlas_min_coverage: float = 0.85
    atlas_min_precision: float = 0.90
    min_brain_volume_cm3: float = 950.0
    max_brain_volume_cm3: float = 1700.0
    atlas_fallback_min_cm3: float = 900.0
    atlas_fallback_max_cm3: float = 1700.0
    enable_atlas_fallback: bool = True
    
    
    max_retries: int = 3
    fallback_to_bbox: bool = True
    # Seuil adaptatif pour contrainte eap_tight (pédiatrique: 0.12, adulte: 0.40)
    # Ratio minimum atlas_contraint/sam2_mask pour activer la contrainte per-slice
    eap_tight_ratio_threshold: float = 0.40
    debug_dir: Optional[Path] = None
    collect_metrics: bool = True

    # IDÉE 1 — masquage hybride atlas (hard + transition gaussienne)
    enable_atlas_input_masking: bool = False
    atlas_mask_hard_radius_mm: float = 12.0
    atlas_mask_soft_radius_mm: float = 3.0
    atlas_mask_min_volume_cm3: float = 200.0
    atlas_mask_background_value: float = 0.0

    # IDÉE 2 — contrainte eap_tight inconditionnelle sur slice initiale
    eap_tight_always_initial: bool = False
    eap_tight_min_preservation: float = 0.20

    # IDÉE 4 — érosion atlas adaptative (atlas trop large vs âge)
    adaptive_atlas_shrink_enabled: bool = False
    adaptive_atlas_shrink_target_ratio: float = 1.15
    adaptive_atlas_shrink_trigger_ratio: float = 1.40
    adaptive_atlas_shrink_max_iters: int = 10
    adaptive_atlas_age_months: Optional[int] = None

    # IDÉE 3 — atlas comme mask_input prompt SAM2 (low-res 256×256 logits)
    enable_atlas_mask_input_prompt: bool = False
    mask_input_strength: float = 10.0
    mask_input_use_eap: bool = True

    # Vague 2 — early stopping volume-driven dans la propagation
    enable_volume_early_stopping: bool = False
    volume_early_stop_ratio: float = 1.20

    # Vague 3 — bidirectional consensus (validation par re-propagation inverse)
    enable_bidirectional_consensus: bool = False
    bidirectional_consensus_min_dice: float = 0.80  # < seuil → intersection per-slice
    bidirectional_consensus_min_range: int = 6      # propagation distance minimale pour activer

    # P1.1 — Multi-point SAM2 prompting (points pos/neg depuis l'atlas adaptatif)
    multipoint_enabled: bool = False
    multipoint_n_positive: int = 8
    multipoint_n_negative: int = 5
    multipoint_seed: int = 42

    runtime_params: Dict[str, Any] = field(default_factory=dict)


class SAM2BrainExtractor:
        
    def __init__(self, params: SegmentationParams):
        
        self.params = params
        
        
        if self.params.model_config_path is None:
            self.params.model_config_path = sam2_config_path
        if self.params.checkpoint_path is None:
            self.params.checkpoint_path = sam2_checkpoint_path
        
        
        if self.params.device == "cuda" and not torch.cuda.is_available():
            print("CUDA non disponible, utilisation du CPU")
            self.params.device = "cpu"
        
        
        if GlobalHydra.instance().is_initialized():
            GlobalHydra.instance().clear()
        
        try:
            print(f"Chargement du modèle SAM2 sur {self.params.device}...")
            start_time = time.time()
            
            config_dir_abs = self.params.model_config_path.parent.resolve()
            config_name = self.params.model_config_path.name
            
            with initialize_config_dir(version_base=None, 
                                    config_dir=str(config_dir_abs)):
                sam_model = build_sam2(config_name, 
                                    str(self.params.checkpoint_path))
            
            sam_model.to(self.params.device).eval()
            self.predictor = SAM2ImagePredictor(sam_model)
            elapsed = time.time() - start_time
            print(f"Modèle SAM2 chargé en {elapsed:.2f}s")
        except Exception as e:
            print(f"Erreur lors du chargement du modèle SAM2: {e}")
            raise
        
        
        self.metrics = {
            "model_load_time": elapsed,
            "slice_scores": [],
            "confidence_threshold_adjustments": [],
            "num_retries": 0,
            "failed_slices": [],
            "total_time": 0.0,
            "atlas_coverage": None,
            "dice_with_atlas": None,
            "atlas_precision": None,
            "atlas_volume_cm3": None,
            "pre_fallback_atlas_coverage": None,
            "pre_fallback_dice_with_atlas": None,
            "pre_fallback_mask_volume_cm3": None,
            "pre_fallback_mask_path": None,
            "fallback_used": False,
            "fallback_reason": None,
            "input_image_used": None
        }
        
        self.registered_atlases = {}
        self.anatomical_priors = {}

    def _pad_to_multiple_of_32(self, img, cval=0):

        h, w = img.shape
        pad_h = (-h) % 32
        pad_w = (-w) % 32
        
        if pad_h > 0 or pad_w > 0:
            if pad_h > 0 or pad_w > 0:
                print(f"DEBUG: Padding image from ({h}, {w}) to ({h+pad_h}, {w+pad_w}) (multiple of 32)")
        
        return np.pad(img, ((0, pad_h), (0, pad_w)), mode="constant", constant_values=cval)
    
    def pad_to_multiple(self, img, m=64, cval=0):
        """Ajoute du padding pour dimensions multiples de 'm'."""
        h, w = img.shape
        pad_h = (-h) % m
        pad_w = (-w) % m
        return np.pad(img, ((0, pad_h), (0, pad_w)), mode="constant", constant_values=cval)
    
    def _prepare_slice_for_sam(self, slice_data):
        """Prépare une coupe 2D pour SAM2."""
        
        ptp_val = np.ptp(slice_data)
        norm = (slice_data - slice_data.min()) / (ptp_val if ptp_val else 1)
        
        uint8 = (norm * 255).astype(np.uint8)
        
        padded = self._pad_to_multiple_of_32(uint8)
        
        rgb = np.stack([padded] * 3, axis=-1)
        return rgb
    
    def _build_soft_atlas_mask(self, ref_atlas_binary, voxel_zooms_mm):
        """IDÉE 1 — masque 3D float ∈ [0,1] : 1 dans atlas dilaté hard_mm,
        décroissance gaussienne sur soft_mm, 0 ailleurs.
        Utilisé pour masquer le volume d'entrée SAM2 (hard + soft fallback)."""
        z_min = float(min(voxel_zooms_mm))
        r_hard_vx = max(1, int(round(self.params.atlas_mask_hard_radius_mm / z_min)))
        hard = ndimage.binary_dilation(ref_atlas_binary, iterations=r_hard_vx)
        dist_out = ndimage.distance_transform_edt(~hard, sampling=voxel_zooms_mm)
        sigma = max(self.params.atlas_mask_soft_radius_mm / 2.0, 1e-3)
        soft = np.exp(-(dist_out ** 2) / (2.0 * sigma ** 2))
        weight = np.where(hard, 1.0, soft).astype(np.float32)
        weight[weight < 1e-3] = 0.0
        return weight

    def _atlas_slice_to_mask_input_logits(self, slice_idx, padded_shape):
        """IDÉE 3 — convertit la slice atlas pour la coupe `slice_idx` en mask_input
        tensor 1×256×256 (logits) pour SAM2 predict(mask_input=...).
        Retourne None si pas d'atlas disponible.
        padded_shape = (H, W) de l'image SAM2 (après pad multiple de 32)."""
        if not self.params.use_atlas_guidance or not self.anatomical_priors:
            return None
        key = 'eap' if self.params.mask_input_use_eap else 'brain'
        if key not in self.anatomical_priors:
            return None
        prior = self.anatomical_priors[key]
        if slice_idx < 0 or slice_idx >= prior.shape[2]:
            return None
        sl = prior[:, :, slice_idx].astype(np.float32)
        # eap est [0,1]; brain est binaire. Normaliser à [0,1].
        if sl.max() > 1.5:
            sl = sl / sl.max()
        ph, pw = padded_shape
        h, w = sl.shape
        if (h, w) != (ph, pw):
            # Pad atlas slice à la même taille que l'image SAM2
            sl_padded = np.zeros((ph, pw), dtype=np.float32)
            sl_padded[:h, :w] = sl
            sl = sl_padded
        # Downsample à 256×256 via interpolation linéaire (scipy.ndimage.zoom)
        from scipy.ndimage import zoom as _zoom
        zoom_h = 256.0 / sl.shape[0]
        zoom_w = 256.0 / sl.shape[1]
        sl_low = _zoom(sl, (zoom_h, zoom_w), order=1, mode='constant', cval=0.0)
        if sl_low.shape != (256, 256):
            # Sécurité : crop/pad si l'interpolation a produit (255 ou 257)
            tmp = np.zeros((256, 256), dtype=np.float32)
            h2, w2 = min(sl_low.shape[0], 256), min(sl_low.shape[1], 256)
            tmp[:h2, :w2] = sl_low[:h2, :w2]
            sl_low = tmp
        sl_low = np.clip(sl_low, 0.0, 1.0)
        # Convertir en logits : strength * (2*p - 1) → [-strength, +strength]
        strength = float(self.params.mask_input_strength)
        logits = (2.0 * sl_low - 1.0) * strength
        # SAM2 attend tensor 1×256×256 (np.ndarray accepté par _prep_prompts)
        return logits[None, :, :].astype(np.float32)

    @staticmethod
    def _expected_brain_volume_cm3(age_months):
        """IDÉE 4 — volume cérébral attendu (cm³) par âge en mois.
        Table interpolée linéairement (Knickmeyer 2008 / Holland 2014)."""
        if age_months is None:
            return None
        table = [(0, 400), (1, 500), (3, 650), (6, 800), (12, 950),
                 (24, 1050), (36, 1100), (60, 1200), (216, 1350)]
        if age_months <= table[0][0]:
            return float(table[0][1])
        if age_months >= table[-1][0]:
            return float(table[-1][1])
        for (a1, v1), (a2, v2) in zip(table, table[1:]):
            if a1 <= age_months <= a2:
                t = (age_months - a1) / max(a2 - a1, 1)
                return float(v1 + t * (v2 - v1))
        return float(table[-1][1])

    def _generate_anatomical_priors(self, mask_path):
        """Génère l'enveloppe anatomique probabiliste à partir du masque recalé."""
        if not self.params.use_atlas_guidance:
            return {}
        
        priors = {}
        
        try:
            
            mask_img = nib.load(str(mask_path))
            mask_data = mask_img.get_fdata() > 0.5  
            
            
            if mask_data.ndim != 3:
                print(f"⚠️ WARNING: Le masque n'est pas 3D: {mask_data.shape}")
                return {}
            
            
            # (erosion supprimee - deplacait le centroide EAP et le prompt SAM2)

            # IDÉE 4 — érosion atlas adaptative si trop large vs âge
            _age = self.params.adaptive_atlas_age_months
            _expected = self._expected_brain_volume_cm3(_age) if _age is not None else None
            if self.params.adaptive_atlas_shrink_enabled and _expected is not None:
                _vox_mm3 = float(np.prod(mask_img.header.get_zooms()[:3]))
                _atl_cm3 = float(mask_data.sum()) * _vox_mm3 / 1000.0
                _ratio = _atl_cm3 / _expected
                if _ratio > self.params.adaptive_atlas_shrink_trigger_ratio:
                    _target_cm3 = self.params.adaptive_atlas_shrink_target_ratio * _expected
                    _shrunk = mask_data.copy()
                    _iters_done = 0
                    for _i in range(self.params.adaptive_atlas_shrink_max_iters):
                        _shrunk = ndimage.binary_erosion(_shrunk, iterations=1)
                        _iters_done = _i + 1
                        if float(_shrunk.sum()) * _vox_mm3 / 1000.0 <= _target_cm3:
                            break
                    _to_cm3 = float(_shrunk.sum()) * _vox_mm3 / 1000.0
                    self.metrics["adaptive_atlas_shrink_iters"] = _iters_done
                    self.metrics["adaptive_atlas_shrink_from_cm3"] = _atl_cm3
                    self.metrics["adaptive_atlas_shrink_to_cm3"] = _to_cm3
                    self.metrics["adaptive_atlas_shrink_expected_cm3"] = _expected
                    print(f"IDÉE 4 — Atlas érodé : {_atl_cm3:.0f}→{_to_cm3:.0f}cm³ "
                          f"(expected {_expected:.0f}, ratio {_ratio:.2f}→{_to_cm3/_expected:.2f}, "
                          f"{_iters_done} itérations)")
                    mask_data = _shrunk
                else:
                    print(f"IDÉE 4 — Atlas OK (ratio {_ratio:.2f} ≤ trigger {self.params.adaptive_atlas_shrink_trigger_ratio})")

            self.registered_atlases['brain_mask'] = mask_data
            
            
            
            structure = ball(3)  
            prior = binary_dilation(mask_data, footprint=structure)
            priors['brain'] = prior
            
            
            eap = ndimage.gaussian_filter(prior.astype(float), sigma=1.5)
            
            
            if np.max(eap) > 0:
                eap = eap / np.max(eap)
            
            
            if eap.shape != mask_data.shape:
                print(f"⚠️ WARNING: Forme de l'EAP ({eap.shape}) différente du masque ({mask_data.shape})")
                return {}
                
            priors['eap'] = eap

            # EAP serrée (ball 1, sigma 0.5) pour la validation de cohérence
            # Calibrée sur les vrais contours du cerveau — évite les faux masques incohérents
            _struct_tight = ball(1)
            _prior_tight = binary_dilation(mask_data, footprint=_struct_tight)
            _eap_tight = ndimage.gaussian_filter(_prior_tight.astype(float), sigma=0.5)
            if np.max(_eap_tight) > 0:
                _eap_tight = _eap_tight / np.max(_eap_tight)
            priors['eap_tight'] = _eap_tight
            print(f"Enveloppe anatomique probabiliste générée à partir de {mask_path.name}")
            print(f"DEBUG: Forme de l'EAP: {eap.shape}")
            
            return priors
        
        except Exception as e:
            print(f"⚠️ Erreur lors de la génération des a priori anatomiques: {e}")
            import traceback
            traceback.print_exc()
            return {}
        
        except Exception as e:
            print(f"Erreur lors de la génération des a priori anatomiques: {e}")
            return {}
    
    def _sample_slice_points(
        self, slice_idx: int, n_pos: int = 8, n_neg: int = 5, rng=None,
    ):
        """P1.1 — Échantillonne points pos (intérieur érodé) + neg (coque externe).

        Retourne (point_coords, point_labels) au format SAM2 (N×2 [x,y], N int32),
        ou (None, None) si l'atlas n'est pas disponible pour cette slice.
        """
        brain_mask = self.registered_atlases.get("brain_mask")
        if brain_mask is None or slice_idx >= brain_mask.shape[2]:
            return None, None
        slice_2d = brain_mask[:, :, slice_idx].astype(bool)
        if slice_2d.sum() < 20:
            return None, None
        if rng is None:
            rng = np.random.RandomState(42)
        # Points positifs : intérieur érodé (zone de certitude anatomique)
        interior = ndimage.binary_erosion(slice_2d, iterations=3)
        pos_yx = np.array(np.where(interior)).T
        if len(pos_yx) < 5:
            pos_yx = np.array(np.where(slice_2d)).T  # fallback : mask complet
        if len(pos_yx) == 0:
            return None, None
        n_pos_actual = min(n_pos, len(pos_yx))
        pos_pts = pos_yx[rng.choice(len(pos_yx), n_pos_actual, replace=False)]
        pos_coords = pos_pts[:, ::-1]  # (y,x) → (x,y) SAM2
        # Points négatifs : coque juste au-delà du masque
        shell = ndimage.binary_dilation(slice_2d, iterations=5) & ~slice_2d
        neg_yx = np.array(np.where(shell)).T
        if len(neg_yx) >= 5:
            n_neg_actual = min(n_neg, len(neg_yx))
            neg_pts = neg_yx[rng.choice(len(neg_yx), n_neg_actual, replace=False)]
            neg_coords = neg_pts[:, ::-1]
            point_coords = np.concatenate([pos_coords, neg_coords], axis=0)
            point_labels = np.array(
                [1] * n_pos_actual + [0] * n_neg_actual, dtype=np.int32)
        else:
            point_coords = pos_coords
            point_labels = np.ones(n_pos_actual, dtype=np.int32)
        return point_coords, point_labels

    def _find_optimal_prompt_location(self, volume_data, anatomical_priors):
        """Trouve la position optimale pour le prompt initial."""
        
        if self.params.prompt_slice_index is not None and self.params.prompt_point_coords is not None:
            return self.params.prompt_slice_index, self.params.prompt_point_coords
        
        
        if 'eap' in anatomical_priors:
            eap = anatomical_priors['eap']
            
            
            slice_volumes = np.sum(eap > 0.5, axis=(0, 1))
            # Parmi les coupes avec >=90% du max EAP, choisir la plus proche du centre
            # argmax pur biaise vers les coupes périphériques (score SAM2 plus faible)
            _thr90 = np.max(slice_volumes) * 0.90
            _cands = np.where(slice_volumes >= _thr90)[0]
            _z_mid = volume_data.shape[2] // 2
            optimal_slice_idx = int(_cands[np.argmin(np.abs(_cands - _z_mid))])
            
            
            slice_eap = eap[:, :, optimal_slice_idx]
            if np.sum(slice_eap > 0.5) > 0:
                
                com = ndimage.center_of_mass(slice_eap > 0.5)
                prompt_y, prompt_x = int(round(com[0])), int(round(com[1]))
                print(f"Prompt optimal basé sur l'atlas: coupe {optimal_slice_idx}, point ({prompt_x}, {prompt_y})")
                return optimal_slice_idx, (prompt_x, prompt_y)
        
        
        slice_idx = volume_data.shape[2] // 2
        center_y, center_x = volume_data.shape[0] // 2, volume_data.shape[1] // 2
        print(f"Prompt par défaut: coupe {slice_idx}, point ({center_x}, {center_y})")
        return slice_idx, (center_x, center_y)
    
    def _validate_mask_with_eap(self, mask, slice_idx):
        """Valide un masque avec l'enveloppe anatomique probabiliste."""
        if not self.params.use_atlas_guidance or 'eap' not in self.anatomical_priors:
            return mask, 1.0  
        
        try:
            
            # Utiliser eap_tight (contours précis) pour la validation si disponible
            eap = self.anatomical_priors.get('eap_tight', self.anatomical_priors['eap'])
            
            
            if eap.ndim != 3 or slice_idx >= eap.shape[2]:
                print(f"⚠️ WARNING: EAP n'a pas les bonnes dimensions ({eap.shape}) ou slice_idx ({slice_idx}) invalide")
                return mask, 1.0  
            
            
            eap_slice = eap[:, :, slice_idx]
            
            
            if isinstance(mask, torch.Tensor):
                mask_np = mask.detach().cpu().numpy().astype(bool)
            else:
                mask_np = np.asarray(mask).astype(bool)
            
            
            eap_mask = eap_slice > 0.5
            overlap = np.sum(np.logical_and(mask_np, eap_mask))
            total_mask_area = np.sum(mask_np)
            
            
            total_eap_area = np.sum(eap_mask)
            coverage = 0.0
            precision = 0.0
            if total_mask_area > 0 and total_eap_area > 0:
                # La cohérence doit mesurer à la fois la précision et la couverture atlas.
                # Un petit masque très précis mais intra-atlas ne doit plus être validé
                # comme segmentation cérébrale complète.
                precision = overlap / total_mask_area
                coverage = overlap / total_eap_area
                consistency_score = min(precision, coverage)
                if coverage < 0.65:
                    print(f"⚠️ Sous-segmentation: masque couvre {coverage:.1%} de l'EAP "
                          f"(precision={precision:.1%})")
            else:
                consistency_score = 0.0
            
            
            # Si SAM2 couvre déjà bien l'atlas (>80%) mais s'étend au-delà
            # → masque probablement correct (atlas sous-estime le vrai cerveau)
            # Bypass uniquement si le masque n'est pas trop grand par rapport à l'EAP
            # (évite d'accepter un masque tête-entière avec couverture=1.0)
            if coverage > 0.80 and precision > 0.70 and total_mask_area <= 3.0 * total_eap_area:
                return mask_np, min(precision, coverage, 1.0)

            if consistency_score < 0.7:
                print(f"Masque incohérent (score: {consistency_score:.3f}), ajustement en cours...")
                
                
                adjusted_mask = mask_np.copy()
                
                
                low_prob_areas = np.logical_and(eap_slice < 0.3, mask_np)
                adjusted_mask[low_prob_areas] = False
                
                
                if np.sum(adjusted_mask) < 0.1 * total_mask_area:
                    print("Ajustement trop restrictif, conservation du masque original")
                    return mask_np, consistency_score
                
                return adjusted_mask, max(consistency_score, 0.7)
            
            
            return mask_np, consistency_score
        
        except Exception as e:
            
            print(f"⚠️ Erreur dans _validate_mask_with_eap: {e}")
            
            import traceback
            traceback.print_exc()
            
            return mask, 1.0

    def _adapt_confidence_threshold(self, current_threshold, score, prev_score=None):
        """Adapte dynamiquement le seuil de confiance."""
        if not self.params.adapative_thresholds:
            return current_threshold

        
        if score < current_threshold * 0.7:
            
            new_threshold = max(score * 0.9, 0.38)  
            reason = "low_score"
        elif score < current_threshold * 0.9:
            
            new_threshold = current_threshold * 0.95
            reason = "near_threshold"
        elif score > current_threshold * 1.2:
            
            new_threshold = min(current_threshold * 1.05, 0.95)  
            reason = "high_score"
        else:
            
            return current_threshold
        
        if abs(new_threshold - current_threshold) > 1e-4:
            print(f"Adaptation seuil: {current_threshold:.3f} → {new_threshold:.3f} (raison: {reason})")
            self.metrics["confidence_threshold_adjustments"].append({
                "old": current_threshold,
                "new": new_threshold,
                "reason": reason,
                "score": score
            })
            
        return new_threshold
    
    def _segment_volume(self, volume_data, prompt_slice_idx, input_point, input_label):
        """Segmente le volume complet à partir d'une coupe initiale."""
        height, width, num_slices = volume_data.shape
        
        
        volume_mask = np.zeros_like(volume_data, dtype=bool)
        uncertainty_map = np.ones_like(volume_data, dtype=float)
        
        print(f"Segmentation de la coupe initiale {prompt_slice_idx}...")
        
        
        
        initial_slice_data = volume_data[:, :, prompt_slice_idx]
        initial_slice_rgb = self._prepare_slice_for_sam(initial_slice_data)
        self.predictor.set_image(initial_slice_rgb)

        # IDÉE 3 — mask_input atlas (low-res 256×256 logits) pour cette slice initiale
        _padded_shape_init = initial_slice_rgb.shape[:2]
        _mask_input_init = None
        if self.params.enable_atlas_mask_input_prompt:
            _mask_input_init = self._atlas_slice_to_mask_input_logits(
                prompt_slice_idx, _padded_shape_init)
            if _mask_input_init is not None:
                print(f"IDÉE 3 — mask_input atlas injecté (slice {prompt_slice_idx}, "
                      f"strength={self.params.mask_input_strength})")
                self.metrics["mask_input_prompt_used"] = True

        consistency_score = 0.0  # initialisé avant la boucle
        initial_mask = None
        initial_score = 0.0
        retry_count = 0
        bbox_mode = False

        while initial_mask is None and retry_count < self.params.max_retries:
            try:
                print(f"Tentative {retry_count + 1}/{self.params.max_retries}...")


                with torch.inference_mode(), torch.autocast(self.params.device):
                    # Prompt initial : bbox dérivée de l'EAP (couvre toute la hauteur du cerveau)
                    # Évite le biais du centroïde EAP vs centroïde cerveau réel
                    _use_eap_box = (
                        not bbox_mode
                        and self.params.use_atlas_guidance
                        and 'eap' in self.anatomical_priors
                        and prompt_slice_idx < self.anatomical_priors['eap'].shape[2]
                    )
                    if _use_eap_box:
                        _eap_s0 = self.anatomical_priors['eap'][:, :, prompt_slice_idx]
                        _yi, _xi = np.where(_eap_s0 > 0.30)
                        if len(_yi) > 50:
                            _mg = 10
                            _box0 = np.array([[
                                max(0, int(_xi.min()) - _mg),
                                max(0, int(_yi.min()) - _mg),
                                min(width - 1, int(_xi.max()) + _mg),
                                min(height - 1, int(_yi.max()) + _mg),
                            ]])
                            print(f"Prompt initial (bbox EAP): {_box0}")
                            # P1.1 — Multi-point init: pos (intérieur) + neg (coque)
                            _mp0_coords, _mp0_labels = None, None
                            if getattr(self.params, "multipoint_enabled", False):
                                _mp0_rng = np.random.RandomState(
                                    getattr(self.params, "multipoint_seed", 42))
                                _mp0_coords, _mp0_labels = self._sample_slice_points(
                                    prompt_slice_idx,
                                    n_pos=getattr(self.params, "multipoint_n_positive", 8),
                                    n_neg=getattr(self.params, "multipoint_n_negative", 5),
                                    rng=_mp0_rng,
                                )
                                if _mp0_coords is not None:
                                    _n_pos0 = int(np.sum(_mp0_labels == 1))
                                    _n_neg0 = int(np.sum(_mp0_labels == 0))
                                    print(f"P1.1 multipoint init: {_n_pos0} pos + {_n_neg0} neg")
                            masks, scores, _ = self.predictor.predict(
                                point_coords=_mp0_coords,
                                point_labels=_mp0_labels,
                                box=_box0,
                                mask_input=_mask_input_init,
                                multimask_output=True,
                                return_logits=False
                            )
                        else:
                            # Fallback point si EAP trop sparse
                            masks, scores, _ = self.predictor.predict(
                                point_coords=input_point,
                                point_labels=input_label,
                                mask_input=_mask_input_init,
                                multimask_output=True,
                                return_logits=False
                            )
                    elif not bbox_mode:
                        masks, scores, _ = self.predictor.predict(
                            point_coords=input_point,
                            point_labels=input_label,
                            mask_input=_mask_input_init,
                            multimask_output=True,
                            return_logits=False
                        )
                    else:
                        center_x, center_y = input_point[0]
                        box_h, box_w = height // 2, width // 2
                        x1 = max(0, center_x - box_w // 2)
                        y1 = max(0, center_y - box_h // 2)
                        x2 = min(width, x1 + box_w)
                        y2 = min(height, y1 + box_h)
                        box = np.array([[x1, y1, x2, y2]])
                        masks, scores, _ = self.predictor.predict(
                            point_coords=None,
                            point_labels=None,
                            box=box,
                            mask_input=_mask_input_init,
                            multimask_output=True,
                            return_logits=False
                        )
                
                
                if masks is not None and len(masks) > 0:
                    
                    
                    best_mask_idx = scores.argmax()
 
                    
                    if isinstance(masks[best_mask_idx], torch.Tensor):
                        candidate_mask = masks[best_mask_idx].squeeze().cpu().numpy()[:height, :width]
                    else:
                        candidate_mask = masks[best_mask_idx].squeeze()[:height, :width]
                    
                    candidate_score = scores[best_mask_idx]
                    # ── Si le masque best-score est trop petit (< 5%),
                    # ── chercher parmi les 3 masques SAM2 un qui couvre mieux le cerveau
                    _ratio_check = np.sum(candidate_mask) / candidate_mask.size
                    if (_ratio_check < 0.05 or _ratio_check > 0.25) and len(masks) > 1:
                        for _alt in range(len(masks)):
                            if _alt == int(best_mask_idx): continue
                            _m = masks[_alt]
                            if isinstance(_m, torch.Tensor):
                                _m = _m.squeeze().cpu().numpy()[:height, :width]
                            else:
                                _m = _m.squeeze()[:height, :width]
                            _r = np.sum(_m) / _m.size
                            _s = float(scores[_alt])
                            if 0.05 <= _r <= 0.25 and _s >= 0.50:
                                print(f"Masque alt idx={_alt}: ratio={_r:.2%}, "
                                      f"score={_s:.3f} → plus grande couverture sélectionnée")
                                candidate_mask = _m
                                candidate_score = _s
                                break

                    print(f"Score initial: {candidate_score:.3f}")
                    print(f"DEBUG: Type de masque: {type(candidate_mask)}, forme: {candidate_mask.shape}")
                    
                    
                    if self.params.use_atlas_guidance and 'eap' in self.anatomical_priors:
                        try:
                            candidate_mask, consistency_score = self._validate_mask_with_eap(
                                candidate_mask, prompt_slice_idx)
                            print(f"Cohérence avec atlas: {consistency_score:.3f}")
                            
                            
                            candidate_score = candidate_score * 0.85 + consistency_score * 0.15
                        except Exception as e:
                            print(f"⚠️ Erreur lors de la validation avec l'EAP: {e}")
                            
                            
                    
                    mask_ratio = np.sum(candidate_mask) / candidate_mask.size

                    # IDÉE 2 — appliquer eap_tight INCONDITIONNELLEMENT sur la slice initiale
                    # (pas seulement si mask_ratio > 0.50). Garde-fou de préservation pour éviter
                    # l'effondrement du masque si l'atlas est trop strict.
                    if (self.params.eap_tight_always_initial
                            and self.params.use_atlas_guidance
                            and 'eap_tight' in self.anatomical_priors
                            and 0 <= prompt_slice_idx < self.anatomical_priors['eap_tight'].shape[2]):
                        _et_always = self.anatomical_priors['eap_tight'][:, :, prompt_slice_idx]
                        _cm_always = candidate_mask & (_et_always > 0.25)
                        _before_always = int(candidate_mask.sum())
                        _after_always = int(_cm_always.sum())
                        if _before_always > 0:
                            _preserved = _after_always / _before_always
                            if _preserved >= self.params.eap_tight_min_preservation:
                                if _after_always != _before_always:
                                    print(f"IDÉE 2 — eap_tight init: {_before_always}→{_after_always} "
                                          f"({_preserved:.1%} préservé, seuil={self.params.eap_tight_min_preservation:.0%})")
                                candidate_mask = _cm_always
                                mask_ratio = np.sum(candidate_mask) / candidate_mask.size
                            else:
                                print(f"IDÉE 2 — eap_tight init ignorée "
                                      f"(préservation {_preserved:.1%} < {self.params.eap_tight_min_preservation:.0%}) "
                                      f"— masque original conservé")

                    # ── Bypass haute-confiance : accepter un petit masque si score
                    # ── ET cohérence atlas sont élevés (ex: seed cérébral précis)
                    if mask_ratio < 0.01 and candidate_score >= 0.80 and consistency_score >= 0.85:
                        print(f"Masque petit ({mask_ratio:.2%}) mais confiant "
                              f"(score={candidate_score:.3f}, cohérence={consistency_score:.3f}) "
                              f"→ accepté comme seed précis")
                        initial_mask = candidate_mask
                        initial_score = candidate_score
                        uncertainty_map[:, :, prompt_slice_idx] = 1.0 - initial_score
                        break
                    if mask_ratio < 0.01:
                        print(f"Masque trop petit ({mask_ratio:.2%}), nouvelle tentative...")
                        if self.params.fallback_to_bbox and not bbox_mode:
                            print("Passage au mode boîte englobante")
                            bbox_mode = True
                            continue
                    elif mask_ratio > 0.50:
                        # Tenter la contrainte atlas avant de rejeter (pédiatrique: réduit l'oversegmentation)
                        if self.params.use_atlas_guidance and 'eap_tight' in self.anatomical_priors:
                            _et_init = self.anatomical_priors['eap_tight'][:, :, prompt_slice_idx]
                            _cm_init = candidate_mask & (_et_init > 0.25)
                            if _cm_init.sum() > self.params.eap_tight_ratio_threshold * candidate_mask.sum():
                                candidate_mask = _cm_init
                                mask_ratio = np.sum(candidate_mask) / candidate_mask.size
                                if mask_ratio <= 0.50:
                                    print(f"Contrainte atlas initiale: masque ramené à {mask_ratio:.1%}")
                                    initial_mask = candidate_mask
                                    initial_score = candidate_score
                                    uncertainty_map[:, :, prompt_slice_idx] = 1.0 - initial_score
                                    break
                        print(f"Masque trop grand ({mask_ratio:.2%}), nouvelle tentative...")
                    else:
                        
                        initial_mask = candidate_mask
                        initial_score = candidate_score
                        uncertainty_map[:, :, prompt_slice_idx] = 1.0 - initial_score
                        break
            
            except Exception as e:
                print(f"Erreur durant la segmentation: {e}")
                if self.params.fallback_to_bbox and not bbox_mode:
                    print("Passage au mode boîte englobante après erreur")
                    bbox_mode = True
                    continue
            
            retry_count += 1
        
        
        if initial_mask is None:
            print(f"Échec de segmentation après {retry_count} tentatives.")
            self.metrics["num_retries"] = retry_count
            return None, None
        
        
        volume_mask[:, :, prompt_slice_idx] = initial_mask
        self.metrics["num_retries"] = retry_count
        self.metrics["slice_scores"].append({
            "slice": prompt_slice_idx, 
            "score": float(initial_score),
            "stage": "initial"
        })
        
        
        print(f"Propagation forward (coupes {prompt_slice_idx+1} → {num_slices-1})...")
        volume_mask, uncertainty_forward = self._propagate_mask(
            volume_data=volume_data,
            volume_mask=volume_mask,
            uncertainty_map=uncertainty_map,
            start_slice=prompt_slice_idx,
            direction='forward',
            initial_score=initial_score
        )
        
        
        if uncertainty_forward is not None:
            uncertainty_map[:, :, prompt_slice_idx+1:] = uncertainty_forward[:, :, prompt_slice_idx+1:]
        
        
        print(f"Propagation backward (coupes {prompt_slice_idx-1} → 0)...")
        volume_mask, uncertainty_backward = self._propagate_mask(
            volume_data=volume_data,
            volume_mask=volume_mask,
            uncertainty_map=uncertainty_map,
            start_slice=prompt_slice_idx,
            direction='backward',
            initial_score=initial_score
        )
        
        
        if uncertainty_backward is not None:
            uncertainty_map[:, :, :prompt_slice_idx] = uncertainty_backward[:, :, :prompt_slice_idx]


        if np.sum(volume_mask) < volume_data.size * 0.001:
            print("Masque final presque vide. Segmentation échouée.")
            return None, None

        # Vague 3 — bidirectional consensus : valider par re-propagation depuis l'extrémité haute
        if self.params.enable_bidirectional_consensus:
            volume_mask = self._apply_bidirectional_consensus(
                volume_data, volume_mask, uncertainty_map, prompt_slice_idx, initial_score)

        return volume_mask, uncertainty_map

    def _apply_bidirectional_consensus(self, volume_data, volume_mask_v1, uncertainty_map,
                                        prompt_slice_idx, initial_score):
        """Vague 3 — pass de validation inverse + consensus per-slice.

        1. Trouve top_slice et bottom_slice non-vides dans v1
        2. Re-propage forward depuis bottom et backward depuis top (cross-direction)
        3. Pour chaque slice intermédiaire, compare v1 vs v2 :
           - Si Dice(v1, v2) >= min → garde v1
           - Sinon → intersection v1 ∩ v2 (consensus conservateur)
        """
        slice_vols = np.sum(volume_mask_v1, axis=(0, 1))
        non_empty = np.where(slice_vols > 100)[0]
        if len(non_empty) < self.params.bidirectional_consensus_min_range:
            self.metrics["bidirectional_consensus_skipped"] = "too_few_slices"
            return volume_mask_v1

        top_slice = int(non_empty.max())
        bottom_slice = int(non_empty.min())
        # Exiger un range de propagation minimal de chaque côté
        if (top_slice - prompt_slice_idx < self.params.bidirectional_consensus_min_range
                or prompt_slice_idx - bottom_slice < self.params.bidirectional_consensus_min_range):
            self.metrics["bidirectional_consensus_skipped"] = "insufficient_range"
            return volume_mask_v1

        print(f"Vague 3 — bidirectional consensus : re-propagation top={top_slice} → 0, "
              f"bottom={bottom_slice} → fin (range {bottom_slice}..{top_slice})")

        # Re-propagation backward depuis top (couvre top → 0)
        volume_mask_v2_bwd = np.zeros_like(volume_mask_v1)
        volume_mask_v2_bwd[:, :, top_slice] = volume_mask_v1[:, :, top_slice]
        try:
            volume_mask_v2_bwd, _ = self._propagate_mask(
                volume_data=volume_data,
                volume_mask=volume_mask_v2_bwd,
                uncertainty_map=np.ones_like(volume_data, dtype=float),
                start_slice=top_slice,
                direction='backward',
                initial_score=initial_score
            )
        except Exception as _exc:
            print(f"⚠️ Vague 3 — re-propagation backward échouée ({_exc}), consensus skip")
            self.metrics["bidirectional_consensus_skipped"] = f"backward_exception:{_exc}"
            return volume_mask_v1

        # Re-propagation forward depuis bottom (couvre bottom → fin)
        volume_mask_v2_fwd = np.zeros_like(volume_mask_v1)
        volume_mask_v2_fwd[:, :, bottom_slice] = volume_mask_v1[:, :, bottom_slice]
        try:
            volume_mask_v2_fwd, _ = self._propagate_mask(
                volume_data=volume_data,
                volume_mask=volume_mask_v2_fwd,
                uncertainty_map=np.ones_like(volume_data, dtype=float),
                start_slice=bottom_slice,
                direction='forward',
                initial_score=initial_score
            )
        except Exception as _exc:
            print(f"⚠️ Vague 3 — re-propagation forward échouée ({_exc}), consensus skip")
            self.metrics["bidirectional_consensus_skipped"] = f"forward_exception:{_exc}"
            return volume_mask_v1

        # Pour chaque slice, consensus
        intersected_count = 0
        agreed_count = 0
        consensus_dice_total = 0.0
        consensus_dice_n = 0
        volume_mask_out = volume_mask_v1.copy()
        for slice_idx in range(volume_mask_v1.shape[2]):
            if slice_idx == prompt_slice_idx:
                continue  # garder le seed initial intact
            m1 = volume_mask_v1[:, :, slice_idx]
            # v2 dépend du côté du prompt
            if slice_idx > prompt_slice_idx:
                m2 = volume_mask_v2_bwd[:, :, slice_idx]
            else:
                m2 = volume_mask_v2_fwd[:, :, slice_idx]
            m1_sum, m2_sum = int(m1.sum()), int(m2.sum())
            if m1_sum == 0 and m2_sum == 0:
                continue
            if m1_sum == 0 or m2_sum == 0:
                # Une seule passe a un masque → garder v1 (existant)
                continue
            inter = int(np.logical_and(m1, m2).sum())
            dice_slice = 2.0 * inter / (m1_sum + m2_sum)
            consensus_dice_total += dice_slice
            consensus_dice_n += 1
            if dice_slice < self.params.bidirectional_consensus_min_dice:
                # Désaccord → intersection conservatrice
                volume_mask_out[:, :, slice_idx] = np.logical_and(m1, m2)
                intersected_count += 1
            else:
                agreed_count += 1

        mean_dice = consensus_dice_total / max(consensus_dice_n, 1)
        print(f"Vague 3 — consensus appliqué : {agreed_count} slices accord, "
              f"{intersected_count} intersections (Dice moyen v1/v2 = {mean_dice:.3f})")
        self.metrics["bidirectional_consensus_applied"] = True
        self.metrics["bidirectional_consensus_slices_intersected"] = int(intersected_count)
        self.metrics["bidirectional_consensus_slices_agreed"] = int(agreed_count)
        self.metrics["bidirectional_consensus_mean_dice_v1_v2"] = float(mean_dice)
        return volume_mask_out
    
    def _propagate_mask(self, volume_data, volume_mask, uncertainty_map, start_slice, direction, initial_score):
       
        height, width, num_slices = volume_data.shape
        
        
        if direction == 'forward':
            slice_indices = range(start_slice + 1, num_slices)
        else:  
            slice_indices = range(start_slice - 1, -1, -1)
        
        
        if not slice_indices:
            return volume_mask, uncertainty_map
        
        
        prev_mask = volume_mask[:, :, start_slice].copy()
        prev_score = initial_score


        current_threshold = self.params.propagation_confidence_threshold

        # Vague 2 — volume-driven early stopping : pré-calcul du seuil cumulé
        _vol_stop_voxels = None
        if (self.params.enable_volume_early_stopping
                and getattr(self, '_voxel_volume_mm3', None)
                and self.params.max_brain_volume_cm3 > 0):
            _vol_stop_cm3 = self.params.max_brain_volume_cm3 * self.params.volume_early_stop_ratio
            _vol_stop_voxels = int(_vol_stop_cm3 * 1000.0 / self._voxel_volume_mm3)
            print(f"Vague 2 — early stopping armé : stop si volume > {_vol_stop_cm3:.0f} cm³ "
                  f"({_vol_stop_voxels} voxels)")

        for slice_idx in slice_indices:
            # Vague 2 — early stopping volume-driven
            if _vol_stop_voxels is not None and int(volume_mask.sum()) > _vol_stop_voxels:
                _curr_cm3 = int(volume_mask.sum()) * self._voxel_volume_mm3 / 1000.0
                print(f"Vague 2 — early stop déclenché (slice {slice_idx}, vol={_curr_cm3:.0f} cm³ "
                      f"> {self.params.max_brain_volume_cm3 * self.params.volume_early_stop_ratio:.0f} cm³)")
                self.metrics["volume_early_stop_triggered"] = True
                self.metrics["volume_early_stop_slice"] = int(slice_idx)
                self.metrics["volume_early_stop_volume_cm3"] = float(_curr_cm3)
                # Remplir le reste avec des masques vides
                for _rest in slice_indices[slice_indices.index(slice_idx):]:
                    volume_mask[:, :, _rest] = np.zeros_like(prev_mask, dtype=bool)
                    uncertainty_map[:, :, _rest] = 1.0
                break

            consistency_score = 0.0
            # ── Vérification atlas EN TÊTE DE BOUCLE ──────────────────
            # Si la coupe est hors atlas, masque vide et on passe à la suivante
            # (s'applique MÊME si SAM2 aurait un bon score)
            _in_atlas = True  # par défaut : pas de restriction
            if self.params.use_atlas_guidance:
                if 'brain_mask' in self.registered_atlases:
                    _atl_bin = self.registered_atlases['brain_mask']
                    if slice_idx < _atl_bin.shape[2]:
                        _in_atlas = bool(_atl_bin[:, :, slice_idx].any())
                    else:
                        _in_atlas = False
                elif 'eap' in self.anatomical_priors:
                    _eap_chk = self.anatomical_priors['eap']
                    if slice_idx < _eap_chk.shape[2]:
                        _in_atlas = bool((_eap_chk[:, :, slice_idx] > 0.10).any())
                    else:
                        _in_atlas = False
            if not _in_atlas:
                volume_mask[:, :, slice_idx] = np.zeros_like(prev_mask, dtype=bool)
                uncertainty_map[:, :, slice_idx] = 1.0
                continue  # skip SAM2 complètement pour les coupes hors cerveau
            # ─────────────────────────────────────────────────────────
            current_slice_data = volume_data[:, :, slice_idx]
            current_slice_rgb = self._prepare_slice_for_sam(current_slice_data)

            try:

                self.predictor.set_image(current_slice_rgb)

                # IDÉE 3 — mask_input atlas pour cette slice (propagation)
                _mask_input_prop = None
                if self.params.enable_atlas_mask_input_prompt:
                    _mask_input_prop = self._atlas_slice_to_mask_input_logits(
                        slice_idx, current_slice_rgb.shape[:2])

                if np.sum(prev_mask) > 0:

                    y_indices, x_indices = np.where(prev_mask)


                    min_y, max_y = np.min(y_indices), np.max(y_indices)
                    min_x, max_x = np.min(x_indices), np.max(x_indices)


                    margin_y = int((max_y - min_y) * 0.05)
                    margin_x = int((max_x - min_x) * 0.05)

                    min_y = max(0, min_y - margin_y)
                    max_y = min(height - 1, max_y + margin_y)
                    min_x = max(0, min_x - margin_x)
                    max_x = min(width - 1, max_x + margin_x)


                    box = np.array([[min_x, min_y, max_x, max_y]])

                    print(f"Propagation avec boîte: {box}")
                    # P1.1 — Multi-point propagation (4 pos + 3 neg depuis atlas)
                    _mp_coords_p, _mp_labels_p = None, None
                    if getattr(self.params, "multipoint_enabled", False):
                        _mp_rng_p = np.random.RandomState(
                            getattr(self.params, "multipoint_seed", 42) + slice_idx)
                        _mp_coords_p, _mp_labels_p = self._sample_slice_points(
                            slice_idx, n_pos=4, n_neg=3, rng=_mp_rng_p)

                    with torch.inference_mode(), torch.autocast(self.params.device):
                        masks, scores, _ = self.predictor.predict(
                            point_coords=_mp_coords_p,
                            point_labels=_mp_labels_p,
                            box=box,
                            mask_input=_mask_input_prop,
                            multimask_output=True,
                            return_logits=False
                        )
                else:

                    center_y, center_x = height // 2, width // 2
                    point_coords = np.array([[center_x, center_y]])
                    point_labels = np.array([1])

                    with torch.inference_mode(), torch.autocast(self.params.device):
                        masks, scores, _ = self.predictor.predict(
                            point_coords=point_coords,
                            point_labels=point_labels,
                            mask_input=_mask_input_prop,
                            multimask_output=True,
                            return_logits=False
                        )
                
                
                if masks is not None and len(masks) > 0:
                    best_mask_idx = np.argmax(scores)
                    
                    
                    if isinstance(masks[best_mask_idx], torch.Tensor):
                        current_mask = masks[best_mask_idx].squeeze().cpu().numpy()
                    else:
                        current_mask = masks[best_mask_idx].squeeze()
                    
                    
                    if current_mask.shape[0] > height or current_mask.shape[1] > width:
                        current_mask = current_mask[:height, :width]
                    
                    current_score = scores[best_mask_idx]
                    
                    print(f"Coupe {slice_idx} ({direction}): score={current_score:.3f}")
                    
                    
                    if self.params.use_atlas_guidance and 'eap' in self.anatomical_priors:
                        try:
                            current_mask, consistency_score = self._validate_mask_with_eap(
                                current_mask, slice_idx)
                            print(f"Cohérence avec atlas: {consistency_score:.3f}")
                            
                            
                            current_score = current_score * 0.85 + consistency_score * 0.15
                        except Exception as e:
                            print(f"⚠️ Erreur lors de la validation avec l'EAP: {e}")
                    
                    
                    current_threshold = self._adapt_confidence_threshold(
                        current_threshold, current_score, prev_score)
                    
                    
                    if current_score >= current_threshold:
                        # Contraindre à eap_tight pour limiter la sur-segmentation
                        if self.params.use_atlas_guidance and 'eap_tight' in self.anatomical_priors:
                            _et_s = self.anatomical_priors['eap_tight'][:, :, slice_idx]
                            _cm_c = current_mask & (_et_s > 0.25)
                            if _cm_c.sum() > self.params.eap_tight_ratio_threshold * current_mask.sum():
                                current_mask = _cm_c
                        volume_mask[:, :, slice_idx] = current_mask
                        uncertainty_map[:, :, slice_idx] = 1.0 - current_score
                        
                        prev_mask = current_mask.copy()
                        prev_score = current_score
                    elif consistency_score >= 0.85 and current_score >= 0.50:
                        # Rescue atlas : atlas très confiant, SAM2 a trouvé quelque chose
                        print(f"  ↑ Rescue atlas (cohérence={consistency_score:.3f}, score={current_score:.3f})")
                        volume_mask[:, :, slice_idx] = current_mask
                        uncertainty_map[:, :, slice_idx] = 1.0 - current_score * 0.7
                        prev_mask = current_mask.copy()
                        prev_score = current_score * 0.85
                    else:
                        # Vérifier si on est dans la région cérébrale (atlas binaire exact)
                        _atlas_here = 0
                        if self.params.use_atlas_guidance:
                            # Priorité : masque binaire de l'atlas (True/False exact)
                            if 'brain_mask' in self.registered_atlases:
                                _bin = self.registered_atlases['brain_mask']
                                if slice_idx < _bin.shape[2]:
                                    _atlas_here = int(_bin[:, :, slice_idx].sum())
                            elif 'eap' in self.anatomical_priors:
                                _eap_s = self.anatomical_priors['eap']
                                if slice_idx < _eap_s.shape[2]:
                                    _atlas_here = int((_eap_s[:, :, slice_idx] > 0.10).sum())
                        if _atlas_here == 0 and self.params.use_atlas_guidance:
                            # Hors atlas : masque vide (pas de propagation hors cerveau)
                            volume_mask[:, :, slice_idx] = np.zeros_like(prev_mask, dtype=bool)
                        else:
                            print(f"Score insuffisant, propagation du masque précédent")
                            volume_mask[:, :, slice_idx] = prev_mask
                        
                        
                        prev_slice = slice_idx-1 if direction == 'forward' else slice_idx+1
                        uncertainty_map[:, :, slice_idx] = np.clip(
                            np.maximum(1.0 - current_score,
                                       uncertainty_map[:, :, prev_slice] + 0.1),
                            0.0, 1.0
                        )
                        
                        self.metrics["failed_slices"].append({
                            "slice": slice_idx, 
                            "stage": direction,
                            "score": float(current_score)
                        })
                else:
                    
                    print(f"Aucun masque généré pour coupe {slice_idx}, propagation du masque précédent")
                    volume_mask[:, :, slice_idx] = prev_mask
                    
                    
                    prev_slice = slice_idx-1 if direction == 'forward' else slice_idx+1
                    uncertainty_map[:, :, slice_idx] = np.minimum(
                        uncertainty_map[:, :, prev_slice] + 0.2, 1.0)
                    
                    self.metrics["failed_slices"].append({
                        "slice": slice_idx, 
                        "stage": direction,
                        "score": 0.0
                    })
            
            except Exception as e:
                
                print(f"Erreur lors du traitement de la coupe {slice_idx}: {e}")
                volume_mask[:, :, slice_idx] = prev_mask
                uncertainty_map[:, :, slice_idx] = 1.0  
                
                self.metrics["failed_slices"].append({
                    "slice": slice_idx, 
                    "stage": direction,
                    "score": -1.0
                })
        
        return volume_mask, uncertainty_map

    def _atlas_quality_metrics(self, mask, voxel_volume_mm3=None):
        """Calcule les métriques de couverture par rapport à l'atlas recalé."""
        metrics = {
            "atlas_coverage": None,
            "dice_with_atlas": None,
            "atlas_precision": None,
            "atlas_volume_cm3": None,
            "mask_volume_cm3": None
        }
        atlas = self.registered_atlases.get('brain_mask')
        if atlas is None:
            return metrics
        atlas_mask = np.asarray(atlas).astype(bool)
        mask_bool = np.asarray(mask).astype(bool)
        if atlas_mask.shape != mask_bool.shape:
            print(f"⚠️ Atlas et masque de formes différentes: {atlas_mask.shape} vs {mask_bool.shape}")
            return metrics
        atlas_sum = int(np.sum(atlas_mask))
        mask_sum = int(np.sum(mask_bool))
        if atlas_sum == 0 or mask_sum == 0:
            return metrics
        intersection = int(np.sum(mask_bool & atlas_mask))
        metrics["atlas_coverage"] = float(intersection / atlas_sum)
        metrics["atlas_precision"] = float(intersection / mask_sum)
        metrics["dice_with_atlas"] = float(2.0 * intersection / (mask_sum + atlas_sum))
        if voxel_volume_mm3 is not None:
            metrics["atlas_volume_cm3"] = float(atlas_sum * voxel_volume_mm3 / 1000.0)
            metrics["mask_volume_cm3"] = float(mask_sum * voxel_volume_mm3 / 1000.0)
        return metrics

    def _apply_atlas_fallback_if_needed(self, mask, voxel_volume_mm3=None):
        """Remplace un masque SAM2 sous-segmenté par l'atlas avancé recalé si celui-ci est plausible."""
        if not self.params.enable_atlas_fallback or not self.params.use_atlas_guidance:
            return mask
        atlas = self.registered_atlases.get('brain_mask')
        if atlas is None:
            return mask
        metrics_before = self._atlas_quality_metrics(mask, voxel_volume_mm3)
        self.metrics["pre_fallback_atlas_coverage"] = metrics_before.get("atlas_coverage")
        self.metrics["pre_fallback_dice_with_atlas"] = metrics_before.get("dice_with_atlas")
        atlas_volume_cm3 = metrics_before.get("atlas_volume_cm3")
        mask_volume_cm3 = metrics_before.get("mask_volume_cm3")
        atlas_valid = True
        if atlas_volume_cm3 is not None:
            atlas_valid = (
                self.params.atlas_fallback_min_cm3
                <= atlas_volume_cm3
                <= self.params.atlas_fallback_max_cm3
            )
        if not atlas_valid:
            print(f"⚠️ Fallback atlas désactivé: volume atlas non plausible ({atlas_volume_cm3:.1f} cm³)")
            return mask
        low_volume = (
            mask_volume_cm3 is not None
            and mask_volume_cm3 < self.params.min_brain_volume_cm3
        )
        high_volume = (
            mask_volume_cm3 is not None
            and mask_volume_cm3 > self.params.max_brain_volume_cm3
        )
        low_coverage = (
            metrics_before.get("atlas_coverage") is not None
            and metrics_before["atlas_coverage"] < self.params.atlas_min_coverage
        )
        low_precision = (
            metrics_before.get("atlas_precision") is not None
            and metrics_before["atlas_precision"] < self.params.atlas_min_precision
        )
        if low_volume or high_volume or low_coverage or low_precision:
            reasons = []
            if low_volume:
                reasons.append(f"volume={mask_volume_cm3:.1f} cm³ < {self.params.min_brain_volume_cm3:.0f}")
            if high_volume:
                reasons.append(f"volume={mask_volume_cm3:.1f} cm³ > {self.params.max_brain_volume_cm3:.0f}")
            if low_coverage:
                reasons.append(f"couverture_atlas={metrics_before['atlas_coverage']:.1%} < {self.params.atlas_min_coverage:.0%}")
            if low_precision:
                reasons.append(f"precision_atlas={metrics_before['atlas_precision']:.1%} < {self.params.atlas_min_precision:.0%}")
            print("⚠️ Masque SAM2 anatomiquement non plausible; fallback vers l'atlas avancé recalé ("
                  + "; ".join(reasons) + ")")
            self.metrics["fallback_used"] = True
            self.metrics["fallback_reason"] = "; ".join(reasons)
            return np.asarray(atlas).astype(bool)
        return mask

    def _post_process_mask(self, mask):
      
        print("Application du post-traitement morphologique...")
        
        
        binary_mask = mask.astype(bool)
        processed_mask = binary_mask.copy()
        
        
        if self.params.closing_radius > 0:
            print(f"Fermeture binaire (rayon={self.params.closing_radius})...")
            struct_elem = ball(self.params.closing_radius)
            processed_mask = binary_closing(processed_mask, footprint=struct_elem)
        
        
        if self.params.largest_component_only:
            print("Extraction de la plus grande composante connectée...")
            labeled_mask, num_labels = ndimage.label(processed_mask)
            
            if num_labels > 1:
                print(f"Nombre de composantes détectées: {num_labels}")
                
                
                component_sizes = ndimage.sum(processed_mask, labeled_mask, range(1, num_labels + 1))
                
                # Sélection par meilleur overlap avec l'atlas (pas par taille brute)
                # Évite de sélectionner un grand composant dans le crâne/scalp
                if self.params.use_atlas_guidance and 'eap' in self.anatomical_priors:
                    eap_binary = self.anatomical_priors['eap'] > 0.40
                    atlas_overlaps = [
                        int(np.sum(np.logical_and(labeled_mask == lbl, eap_binary)))
                        for lbl in range(1, num_labels + 1)
                    ]
                    # Combiner overlap atlas (priorité) + taille (critère secondaire)
                    scores = [
                        atlas_overlaps[i] * 1000 + component_sizes[i]
                        for i in range(len(component_sizes))
                    ]
                    best_label = int(np.argmax(scores)) + 1
                    print(f"Composante conservée (atlas-guided): {best_label} "
                          f"(taille: {component_sizes[best_label-1]:.0f}, "
                          f"overlap_atlas: {atlas_overlaps[best_label-1]})")
                else:
                    best_label = int(np.argmax(component_sizes)) + 1
                    print(f"Composante conservée (taille): {best_label} "
                          f"(taille: {component_sizes[best_label-1]:.0f})")
                processed_mask = (labeled_mask == best_label)
        
        
        if self.params.use_atlas_guidance and 'eap' in self.anatomical_priors:
            print("Application de la contrainte anatomique...")
            # Utiliser eap_tight (contours serrés) pour une contrainte précise
            eap = self.anatomical_priors.get('eap_tight', self.anatomical_priors['eap'])
            
            exclude_mask = (eap < 0.50) & processed_mask
            exclude_count = np.sum(exclude_mask)
            
            if exclude_count > 0:
                print(f"Exclusion de {exclude_count} voxels hors contrainte anatomique (eap_tight>0.50)")
                processed_mask[exclude_mask] = False
                
                if np.sum(processed_mask) < 0.30 * np.sum(mask):
                    print("Contrainte trop restrictive (seuil 0.30), retour au masque initial")
                    processed_mask = mask.copy()
        
        
        if self.params.fill_holes:
            print("Remplissage des trous...")
            processed_mask = ndimage.binary_fill_holes(processed_mask)
        
        
        if self.params.smooth_borders:
            print("Lissage des bords...")
            struct_elem_smooth = ball(1)
            _before_smooth = processed_mask.copy()
            opened_mask = binary_opening(processed_mask, footprint=struct_elem_smooth)
            processed_mask = binary_closing(opened_mask, footprint=struct_elem_smooth)
            atlas_mask = self.registered_atlases.get('brain_mask')
            if atlas_mask is not None:
                atlas_vox = np.sum(atlas_mask)
                if atlas_vox > 0 and np.sum(processed_mask) < 0.85 * atlas_vox and np.sum(processed_mask) < np.sum(_before_smooth):
                    print("Lissage annulé: il accentue une sous-segmentation par rapport à l'atlas")
                    processed_mask = _before_smooth
        
        # ── Garde-fou volume global (B2) ─────────────────────────────────────
        # Un cerveau adulte normal = 1200-1600 cm³. Si le masque dépasse 1800 cm³
        # (ratio > ~18%), on applique une contrainte EAP plus stricte (> 0.40).
        _mask_ratio_check = np.sum(processed_mask) / processed_mask.size
        if _mask_ratio_check > 0.18 and 'eap' in self.anatomical_priors:
            _eap_g = self.anatomical_priors['eap']
            _strict = processed_mask & (_eap_g > 0.35)
            if np.sum(_strict) > self.params.eap_tight_ratio_threshold * np.sum(processed_mask):
                print(f"Garde-fou volume: {_mask_ratio_check:.2%} > 18%, "
                      f"contrainte EAP > 0.40 appliquée "
                      f"({np.sum(processed_mask):,} → {np.sum(_strict):,} voxels)")
                processed_mask = _strict
        # ─────────────────────────────────────────────────────────────────────
        # Sélection finale (smooth_borders peut re-fragmenter)
        labeled_final, n_final = ndimage.label(processed_mask)
        print(f"Composantes connexes finales: {n_final}")
        if n_final > 1:
            sizes_final = ndimage.sum(processed_mask, labeled_final, range(1, n_final + 1))
            largest_final = int(np.argmax(sizes_final)) + 1
            processed_mask = (labeled_final == largest_final)
            print(f"Composante finale retenue: {np.sum(processed_mask)} voxels")

        # ── Ouverture finale ball(1) ─────────────────────────────────────────
        # Retire les protrusions de surface < 2mm sans toucher au corps du masque
        # → réduit la sur-segmentation de surface (~200 cm³) et fait baisser HD95
        # (différent du smooth_borders : pas de closing après → réduction permanente)
        atlas_mask = self.registered_atlases.get('brain_mask')
        _skip_final_opening = False
        if atlas_mask is not None:
            atlas_vox = np.sum(atlas_mask)
            if atlas_vox > 0 and np.sum(processed_mask) < 0.85 * atlas_vox:
                _skip_final_opening = True
                print("Ouverture finale ignorée: masque déjà sous-couvrant vs atlas")

        if not _skip_final_opening:
            print("Ouverture finale ball(1) — retrait protrusions de surface...")
            _before_open = int(np.sum(processed_mask))
            processed_mask = binary_opening(processed_mask, footprint=ball(1))
            # Réextraire la plus grande composante (ouverture peut fragmenter)
            _lbl_o, _n_o = ndimage.label(processed_mask)
            if _n_o > 1:
                _s_o = ndimage.sum(processed_mask, _lbl_o, range(1, _n_o + 1))
                processed_mask = (_lbl_o == int(np.argmax(_s_o)) + 1)
            _after_open = int(np.sum(processed_mask))
            print(f"Ouverture: {_before_open:,} → {_after_open:,} voxels "
                  f"(retrait: {_before_open - _after_open:,} voxels = "
                  f"{(_before_open - _after_open) / 1000:.1f} cm³)")
        # ─────────────────────────────────────────────────────────────────────
        print("Post-traitement terminé.")
        return processed_mask
    
    def extract_brain(self, input_path, output_path, save_metrics=True, save_uncertainty_map=True):
        """Extraction du cerveau à partir d'une image T1."""
        total_start_time = time.time()
        
        
        input_path = Path(input_path)
        output_path = Path(output_path)
        self.metrics["input_image_used"] = str(input_path)
        
        
        base_name = output_path.name.split('_desc-brain')[0]
        mask_path = output_path.parent / f"{base_name}_desc-brain_mask.nii.gz"
        uncertainty_path = output_path.parent / f"{base_name}_desc-brain_uncertainty.nii.gz"
        
        
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        print(f"Extraction du cerveau pour {input_path.name}...")
        
        
        try:
            img = nib.load(str(input_path))
            data = np.ascontiguousarray(img.get_fdata(dtype=np.float32))
            voxel_volume_mm3 = float(np.prod(img.header.get_zooms()[:3]))
            # Vague 2 — exposer voxel_volume_mm3 à _propagate_mask pour early stopping
            self._voxel_volume_mm3 = voxel_volume_mm3
            print(f"Image chargée: dimensions {data.shape}")
        except Exception as e:
            print(f"Erreur lors du chargement de l'image: {e}")
            return None, None
        
        
        if self.params.use_atlas_guidance:
             
            atlas_advanced_dir = output_path.parent / "atlases_advanced"
            atlas_simple_dir = output_path.parent / "atlases"
            
            
            # Priorite : atlas passé par self.params, puis atlas exact du sujet courant.
            # Pas de fallback nom court/glob ici: ces fichiers peuvent être stale.
            _base_subject_id = base_name[4:] if base_name.startswith("sub-") else base_name
            _prefix_full = base_name if base_name.startswith("sub-") else f"sub-{_base_subject_id}"
            if self.params.atlas_paths and 'brain_mask' in self.params.atlas_paths:
                atlas_path_advanced = Path(str(self.params.atlas_paths['brain_mask']))
            else:
                _atlas_candidates = [
                    atlas_advanced_dir / f'{_prefix_full}_brain_mask_aff_nomask.nii.gz',
                    atlas_advanced_dir / f'{_prefix_full}_brain_mask_mni152_aff_nomask.nii.gz',
                    atlas_advanced_dir / f'{_prefix_full}_brain_mask_tpl_aff_nomask.nii.gz',
                    atlas_advanced_dir / f'{_prefix_full}_brain_mask_aff.nii.gz',
                    atlas_advanced_dir / f'{_prefix_full}_brain_mask_mni152_aff.nii.gz',
                    atlas_advanced_dir / f'{_prefix_full}_brain_mask_tpl_aff.nii.gz',
                    atlas_advanced_dir / f'{_prefix_full}_brain_mask_syn.nii.gz',
                    atlas_advanced_dir / f'{_prefix_full}_brain_mask_mni152_syn.nii.gz',
                    atlas_advanced_dir / f'{_prefix_full}_brain_mask_tpl_syn.nii.gz',
                ]
                atlas_path_advanced = None
                for _c in _atlas_candidates:
                    if Path(_c).exists():
                        atlas_path_advanced = Path(_c)
                        break
            atlas_path_simple = atlas_simple_dir / f"{_prefix_full}_space-subject_brain_mask.nii.gz"
            
            
            if atlas_path_advanced is not None and atlas_path_advanced.exists():
                atlas_path = atlas_path_advanced
                print(f"Utilisation de l'atlas avancé: {atlas_path}")
            elif atlas_path_simple.exists():
                atlas_path = atlas_path_simple
                print(f"Utilisation de l'atlas simple: {atlas_path}")
            else:
                print(f"⚠️ Aucun atlas recalé exact trouvé pour {_prefix_full} dans {atlas_advanced_dir} ou {atlas_simple_dir}")
                print("Vérifiez que l'atlas est correctement recalé et placé dans le bon répertoire.")
                print(' atlas_path_simple', atlas_path_simple)
                print(' atlas_path_advanced', atlas_path_advanced)
                print("Aucun atlas recalé trouvé! Le guidage par atlas sera désactivé.")
                self.params.use_atlas_guidance = False
                atlas_path = None
        
        
        if self.params.use_atlas_guidance and atlas_path is not None:
            self.anatomical_priors = self._generate_anatomical_priors(atlas_path)

            if not self.anatomical_priors:
                print("Aucun a priori anatomique généré. Guidage atlas limité.")

        # IDÉE 1 — masquage hybride du volume d'entrée par atlas dilaté + transition gaussienne
        # Doit être appliqué AVANT _find_optimal_prompt_location et AVANT la normalisation per-slice
        # (la min-max recalibrera ensuite la dynamique sur le cerveau — bonus contrast stretching).
        if self.params.enable_atlas_input_masking and self.anatomical_priors and 'brain' in self.anatomical_priors:
            _atl_bin = self.registered_atlases.get('brain_mask')
            _atl_vol_cm3 = (float(_atl_bin.sum()) * voxel_volume_mm3 / 1000.0) if _atl_bin is not None else 0.0
            if _atl_bin is None or _atl_vol_cm3 < self.params.atlas_mask_min_volume_cm3:
                print(f"IDÉE 1 — atlas trop petit ou absent ({_atl_vol_cm3:.0f}cm³ < {self.params.atlas_mask_min_volume_cm3}cm³) → masquage désactivé (garde-fou)")
                self.metrics["atlas_input_masking_applied"] = False
                self.metrics["atlas_input_masking_skipped_reason"] = "atlas_too_small_or_missing"
            else:
                try:
                    _zooms = img.header.get_zooms()[:3]
                    _w = self._build_soft_atlas_mask(_atl_bin, _zooms)
                    _bg = float(self.params.atlas_mask_background_value)
                    data = (data * _w + _bg * (1.0 - _w)).astype(np.float32)
                    self.metrics["atlas_input_masking_applied"] = True
                    self.metrics["atlas_input_masking_atlas_volume_cm3"] = _atl_vol_cm3
                    self.metrics["atlas_input_masking_hard_mm"] = float(self.params.atlas_mask_hard_radius_mm)
                    self.metrics["atlas_input_masking_soft_mm"] = float(self.params.atlas_mask_soft_radius_mm)
                    print(f"IDÉE 1 — masquage hybride appliqué (atlas={_atl_vol_cm3:.0f}cm³, "
                          f"hard={self.params.atlas_mask_hard_radius_mm}mm, soft={self.params.atlas_mask_soft_radius_mm}mm)")
                except Exception as _exc:
                    print(f"⚠️ IDÉE 1 — masquage échoué ({_exc}), volume original conservé")
                    self.metrics["atlas_input_masking_applied"] = False
                    self.metrics["atlas_input_masking_skipped_reason"] = f"exception:{_exc}"

        prompt_slice_idx, (prompt_x, prompt_y) = self._find_optimal_prompt_location(
            data, self.anatomical_priors)
        
        
        input_point = np.array([[prompt_x, prompt_y]])
        input_label = np.array([1])  
        
        
        segmentation_start = time.time()
        volume_mask, uncertainty_map = self._segment_volume(
            volume_data=data,
            prompt_slice_idx=prompt_slice_idx,
            input_point=input_point,
            input_label=input_label
        )
        
        if volume_mask is None:
            print("Échec de la segmentation du volume.")
            return None, None
        
        self.metrics["segmentation_time"] = time.time() - segmentation_start
        print(f"Segmentation terminée en {self.metrics['segmentation_time']:.2f}s")
        
        
        if self.params.post_process:
            post_start = time.time()
            try:
                volume_mask = self._post_process_mask(volume_mask)
                self.metrics["post_processing_time"] = time.time() - post_start
            except Exception as e:
                print(f"Erreur lors du post-traitement: {e}")
                self.metrics["post_processing_error"] = str(e)

        _pre_fallback_mask = np.asarray(volume_mask).astype(bool).copy()
        _pre_fallback_metrics = self._atlas_quality_metrics(_pre_fallback_mask, voxel_volume_mm3)
        self.metrics["pre_fallback_mask_volume_cm3"] = _pre_fallback_metrics.get("mask_volume_cm3")
        volume_mask = self._apply_atlas_fallback_if_needed(volume_mask, voxel_volume_mm3)
        if self.metrics.get("fallback_used"):
            try:
                _pre_fb_path = output_path.parent / f"{base_name}_desc-brain_mask_pre-fallback.nii.gz"
                _pre_fb_img = nib.Nifti1Image(_pre_fallback_mask.astype(np.uint8), img.affine, img.header)
                _pre_fb_img.set_data_dtype(np.uint8)
                nib.save(_pre_fb_img, _pre_fb_path)
                self.metrics["pre_fallback_mask_path"] = str(_pre_fb_path)
                print(f"Masque pré-fallback sauvegardé: {_pre_fb_path}")
            except Exception as _e_pre_fb:
                self.metrics["pre_fallback_save_error"] = str(_e_pre_fb)
        _atlas_metrics = self._atlas_quality_metrics(volume_mask, voxel_volume_mm3)
        for _k, _v in _atlas_metrics.items():
            if _k != "mask_volume_cm3":
                self.metrics[_k] = _v
        
        
        try:
            
            brain_data = data.copy()
            brain_data[~volume_mask] = 0
            
            
            brain_img = nib.Nifti1Image(brain_data, img.affine, img.header)
            brain_img.header['descrip'] = 'SAM2-NeuroAdapt Brain Extracted'
            nib.save(brain_img, output_path)
            print(f"Cerveau extrait sauvegardé: {output_path}")
            
            
            mask_img = nib.Nifti1Image(volume_mask.astype(np.uint8), img.affine, img.header)
            mask_img.header['descrip'] = 'SAM2-NeuroAdapt Brain Mask'
            nib.save(mask_img, mask_path)
            print(f"Masque cérébral sauvegardé: {mask_path}")
            
            
            if save_uncertainty_map and uncertainty_map is not None:
                uncertainty_img = nib.Nifti1Image(uncertainty_map.astype(np.float32), 
                                                 img.affine, img.header)
                uncertainty_img.header['descrip'] = 'SAM2-NeuroAdapt Uncertainty Map'
                nib.save(uncertainty_img, uncertainty_path)
                print(f"Carte d'incertitude sauvegardée: {uncertainty_path}")
        
        except Exception as e:
            print(f"Erreur lors de la sauvegarde des résultats: {e}")
            return None, None
        
        
        self.metrics["total_time"] = time.time() - total_start_time
        mask_ratio = np.sum(volume_mask) / volume_mask.size
        try:
            mask_volume_mm3 = np.sum(volume_mask) * voxel_volume_mm3
        except:
            mask_volume_mm3 = "N/A"
        
        self.metrics["mask_ratio"] = float(mask_ratio)
        self.metrics["mask_volume_mm3"] = float(mask_volume_mm3) if isinstance(mask_volume_mm3, (int, float, np.floating)) else mask_volume_mm3
        self.metrics["mask_volume_cm3"] = float(mask_volume_mm3 / 1000.0) if isinstance(mask_volume_mm3, (int, float, np.floating)) else None
        
        
        _mask_volume_cm3 = self.metrics["mask_volume_cm3"]
        _atlas_cov = self.metrics.get("atlas_coverage")
        if _mask_volume_cm3 is not None and _mask_volume_cm3 < self.params.min_brain_volume_cm3:
            quality = "Poor - undersegmentation"
        elif _mask_volume_cm3 is not None and _mask_volume_cm3 > self.params.max_brain_volume_cm3:
            quality = "Poor - oversegmentation"
        elif _atlas_cov is not None and _atlas_cov < self.params.atlas_min_coverage:
            quality = "Poor - atlas undercoverage"
        elif mask_ratio > 0.6:
            quality = "Suspicious - mask potentially too large"
        elif self.metrics.get("fallback_used"):
            quality = "Good - atlas fallback applied"
        elif self.metrics["num_retries"] >= self.params.max_retries:
            quality = "Fair - Required multiple retries"
        elif len(self.metrics["failed_slices"]) > data.shape[2] * 0.15:
            quality = "Fair - Propagation failed for several slices"
        else:
            quality = "Good"
        
        self.metrics["quality_rating"] = quality
        
        print(f"Extraction complète en {self.metrics['total_time']:.2f}s")
        print(f"Ratio du masque: {mask_ratio:.2%}")
        print(f"Évaluation de la qualité: {quality}")
         
        
        if save_metrics:
            try:
                import json
                metrics_path = output_path.parent / f"{base_name}_desc-brain_metrics.json"
                
                
                def json_serializable(obj):
                    if isinstance(obj, (np.integer, np.int64, np.int32)):
                        return int(obj)
                    elif isinstance(obj, (np.floating, np.float64, np.float32)):
                        if np.isnan(obj) or np.isinf(obj):
                            return None
                        return float(obj)
                    elif isinstance(obj, np.ndarray):
                        return obj.tolist()
                    elif isinstance(obj, np.bool_):
                        return bool(obj)
                    elif obj is None or isinstance(obj, (str, int, float, bool, list, dict)):
                        return obj
                    else:
                        return str(obj)
                
                
                clean_metrics = {}
                for k, v in self.metrics.items():
                    if isinstance(v, dict):
                        clean_metrics[k] = {sk: json_serializable(sv) for sk, sv in v.items()}
                    elif isinstance(v, list):
                        clean_metrics[k] = [json_serializable(item) if not isinstance(item, dict) 
                                        else {sk: json_serializable(sv) for sk, sv in item.items()} 
                                        for item in v]
                    else:
                        clean_metrics[k] = json_serializable(v)
                
                
                with open(metrics_path, 'w') as f:
                    json.dump(clean_metrics, f, indent=2)
                print(f"Métriques sauvegardées: {metrics_path}")
            except Exception as e:
                print(f"⚠️ Erreur lors de la sauvegarde des métriques: {e}")
            
        
        return output_path, mask_path



def extract_brain(
    input_path,
    output_path,
    confidence_threshold=None,
    use_atlas_guidance=True,
    atlas_weight=0.7,
    atlas_paths=None,
    post_process=True,
    save_metrics=True,
    save_uncertainty_map=True,
    debug_dir=None
):
    
    params = SegmentationParams(
        debug_dir=debug_dir,
        use_atlas_guidance=use_atlas_guidance,
        atlas_weight=atlas_weight,
        post_process=post_process
    )
    
    
    if atlas_paths:
        params.atlas_paths = atlas_paths
    
    
    if confidence_threshold is not None:
        params.initial_confidence_threshold = confidence_threshold
        params.propagation_confidence_threshold = confidence_threshold * 0.9
    
    
    try:
        extractor = SAM2BrainExtractor(params=params)
        brain_path, mask_path = extractor.extract_brain(
            input_path=input_path,
            output_path=output_path,
            save_metrics=save_metrics,
            save_uncertainty_map=save_uncertainty_map
        )
        return brain_path, mask_path
    except Exception as e:
        print(f"Erreur lors de l'extraction du cerveau: {e}")
        import traceback
        traceback.print_exc()
        return None, None
# P1.1-MULTIPOINT-INSTALLED


### Cellule 14 : Exécution de l'extraction du cerveau

In [ ]:
# Vérification que les variables pointent bien vers le sujet actif
subject_id = SUBJECT_ID
derivatives_dir = DERIVATIVES_DIR
anat_dir = ANAT_DIR


# subject_id, derivatives_dir, anat_dir → définis dans cell 0
# anat_dir pointe vers ANAT_DIR = outputs/multisubject_eval/{SUBJECT_RAW}/anat


whitestripe_path = anat_dir / f"sub-{subject_id}_desc-n4_desc-ws_T1w.nii.gz"
n4_path = anat_dir / f"sub-{subject_id}_desc-n4_T1w.nii.gz"


if n4_path.exists():
    input_image_path = n4_path
    print(f"Utilisation de l'image N4 pour SAM2 (préserve les bords anatomiques): {n4_path.name}")
    if whitestripe_path.exists():
        print(f"WhiteStripe disponible pour QC uniquement: {whitestripe_path.name}")
elif whitestripe_path.exists():
    input_image_path = whitestripe_path
    print(f"Fallback WhiteStripe uniquement car N4 est absente: {whitestripe_path.name}")
else:
    raise FileNotFoundError(f"Aucune image prétraitée trouvée pour sub-{subject_id}")


atlas_advanced_dir = anat_dir / "atlases_advanced"
atlas_simple_dir = anat_dir / "atlases"


atlas_paths = {}


# Priorité au résultat de la cellule SOTA (_aff), puis SyN si disponible.
# Le masque atlas doit porter le nom complet du sujet courant; un nom court
# ou un glob peut être stale et ne doit pas guider SAM2.
def _atlas_quality(path, min_cm3=900.0, max_cm3=1700.0, max_components=8):
    try:
        import nibabel as _nib
        from scipy import ndimage as _ndi
        _img = _nib.load(str(path))
        _mask = _img.get_fdata() > 0.5
        _vox = float(np.prod(_img.header.get_zooms()[:3]))
        _vol_cm3 = float(np.sum(_mask) * _vox / 1000.0)
        _lbl, _ncomp = _ndi.label(_mask)
        _reason = []
        if _vol_cm3 < min_cm3 or _vol_cm3 > max_cm3:
            _reason.append(f"volume={_vol_cm3:.1f} cm³ hors [{min_cm3:.0f}, {max_cm3:.0f}]")
        if _ncomp > max_components:
            _reason.append(f"composantes={_ncomp} > {max_components}")
        if _mask.any():
            _pts = np.where(_mask)
            _bbox = [(int(p.min()), int(p.max())) for p in _pts]
            _extent_ratio = max((b - a + 1) / s for (a, b), s in zip(_bbox, _mask.shape))
            if _extent_ratio > 0.85:
                _reason.append(f"bbox trop étendue={_extent_ratio:.1%}")
        else:
            _reason.append("masque vide")
        return len(_reason) == 0, {"volume_cm3": _vol_cm3, "components": int(_ncomp)}, "; ".join(_reason)
    except Exception as _e:
        return False, {}, f"erreur lecture atlas: {_e}"

# Recherche atlas robuste : uniquement le nom complet du sujet courant.
# On évite volontairement les noms courts/globaux pour ne pas reprendre un fichier stale.
_selected = None
_atlas_candidates = [
    atlas_advanced_dir / f"sub-{subject_id}_brain_mask_aff_nomask.nii.gz",
    atlas_advanced_dir / f"sub-{subject_id}_brain_mask_mni152_aff_nomask.nii.gz",
    atlas_advanced_dir / f"sub-{subject_id}_brain_mask_tpl_aff_nomask.nii.gz",
    atlas_advanced_dir / f"sub-{subject_id}_brain_mask_aff.nii.gz",
    atlas_advanced_dir / f"sub-{subject_id}_brain_mask_mni152_aff.nii.gz",
    atlas_advanced_dir / f"sub-{subject_id}_brain_mask_tpl_aff.nii.gz",
    atlas_advanced_dir / f"sub-{subject_id}_brain_mask_syn.nii.gz",
    atlas_advanced_dir / f"sub-{subject_id}_brain_mask_mni152_syn.nii.gz",
    atlas_advanced_dir / f"sub-{subject_id}_brain_mask_tpl_syn.nii.gz",
]

_rejected_atlases = []
for _c in _atlas_candidates:
    if not Path(_c).exists():
        continue
    _ok, _stats, _why = _atlas_quality(Path(_c))
    if _ok:
        _selected = Path(_c)
        print(f"Atlas candidat validé: {_selected.name} "
              f"(volume={_stats['volume_cm3']:.1f} cm³, composantes={_stats['components']})")
        break
    _rejected_atlases.append((Path(_c).name, _why))

if _selected is None and _rejected_atlases:
    print("⚠️ Atlas recalés rejetés par QC anatomique:")
    for _name, _why in _rejected_atlases[:5]:
        print(f"  - {_name}: {_why}")

if _selected:
    atlas_paths["brain_mask"] = _selected
    _atlas_data = nib.load(str(_selected)).get_fdata()
    _ratio = np.sum(_atlas_data > 0.5) / _atlas_data.size
    _ok, _stats, _why = _atlas_quality(_selected)
    print(f"Atlas sélectionné : {_selected.name}  ({_ratio:.1%} du volume, "
          f"{_stats.get('volume_cm3', float('nan')):.1f} cm³)")
else:
    print("⚠️ Aucun atlas recalé valide trouvé! Le guidage par atlas sera limité.")


brain_output_path = anat_dir / f"sub-{subject_id}_desc-brain_T1w.nii.gz"


debug_dir = anat_dir / "debug_sam"
debug_dir.mkdir(parents=True, exist_ok=True)


try:
    import json
    audit_file = list(anat_dir.glob(f"sub-{subject_id}*audit_preprocessing.json"))
    if audit_file:
        with open(audit_file[0], 'r') as f:
            audit_metrics = json.load(f)
        contraste = audit_metrics.get("contraste_ws", 0.5)
        snr = audit_metrics.get("snr_ws", 2.0)
        edge_ws = audit_metrics.get("edge_improvement_ws")
        print(f"Métriques d'audit trouvées: contraste={contraste:.2f}, SNR={snr:.2f}")
        if edge_ws is not None and edge_ws < 0:
            print(f"QC WhiteStripe: bords dégradés ({edge_ws*100:.1f}%) -> N4 conservée pour SAM2")
        
        
        if contraste > 0.8 and snr > 3.0:
            confidence_threshold = 0.80
            print("Image de bonne qualité → seuil de confiance élevé (0.80)")
        else:
            confidence_threshold = 0.70
            print("Image de qualité moyenne → seuil de confiance réduit (0.70)")
    else:
        confidence_threshold = 0.75
        print("Pas de métrique d'audit → seuil de confiance par défaut (0.75)")
except Exception as e:
    confidence_threshold = 0.75
    print(f"Erreur lors de la lecture des métriques d'audit: {e}")
    print("Utilisation du seuil de confiance par défaut (0.75)")

print("\n=== Extraction du cerveau avec SAM-NeuroAdapt v2 ===")
print(f"Image source: {input_image_path}")
print(f"Image de sortie: {brain_output_path}")
print(f"Guidage par atlas: {'Activé' if atlas_paths else 'Limité'}")
print(f"Seuil de confiance: {confidence_threshold}")
print("Note méthodologique: SAM2-NeuroAdapt utilise SAM2 avec contrôle atlas; l'atlas recalé sert de garde-fou en cas de sous-segmentation.")


start_time = time.time()

brain_path, mask_path = extract_brain(
    input_path=input_image_path,
    output_path=brain_output_path,
    confidence_threshold=confidence_threshold,
    use_atlas_guidance=bool(atlas_paths),
    atlas_paths=atlas_paths,
    atlas_weight=0.7,
    post_process=True,
    save_metrics=True,
    save_uncertainty_map=True,
    debug_dir=debug_dir
)

elapsed_time = time.time() - start_time

if brain_path and mask_path:
    print(f"\n✅ Extraction réussie en {elapsed_time:.2f}s!")
    print(f"Cerveau extrait: {brain_path}")
    print(f"Masque cérébral: {mask_path}")
    
    
    uncertainty_path = anat_dir / f"sub-{subject_id}_desc-brain_uncertainty.nii.gz"
    if uncertainty_path.exists():
        print(f"Carte d'incertitude: {uncertainty_path}")
else:
    print(f"\n❌ Échec de l'extraction du cerveau ({elapsed_time:.2f}s)")

### Cellule 15 : Visualisation et évaluation des résultats

In [ ]:
subject_id = SUBJECT_ID
anat_dir = ANAT_DIR

if 'brain_path' not in globals() or not brain_path:
    
    brain_path = anat_dir / f"sub-{subject_id}_desc-brain_T1w.nii.gz"
    mask_path = anat_dir / f"sub-{subject_id}_desc-brain_mask.nii.gz"
    uncertainty_path = anat_dir / f"sub-{subject_id}_desc-brain_uncertainty.nii.gz"


all_files_exist = brain_path.exists() and mask_path.exists()
has_uncertainty = uncertainty_path.exists() if 'uncertainty_path' in globals() else False

if not all_files_exist:
    raise FileNotFoundError(f"Fichiers d'extraction du cerveau non trouvés: {brain_path} / {mask_path}")


print("Chargement des images pour visualisation...")
brain_img = nib.load(str(brain_path))
brain_data = brain_img.get_fdata()

mask_img = nib.load(str(mask_path))
mask_data = mask_img.get_fdata() > 0  


display_image_label = "Image SAM2 input: N4"
if n4_path.exists():
    display_image_path = n4_path
    orig_img = nib.load(str(n4_path))
else:
    raw_path = Path(T1_RAW_PATH) if 'T1_RAW_PATH' in globals() else None
    if raw_path is not None and raw_path.exists():
        display_image_path = raw_path
        display_image_label = "Image anatomique brute"
        orig_img = nib.load(str(raw_path))
    else:
        display_image_path = brain_path
        display_image_label = "Image cerveau extrait"
        print("⚠️ Image N4/brute non trouvée, utilisation de l'image du cerveau")
        orig_img = brain_img

print(f"Image affichée pour l'anatomie: {display_image_label} ({display_image_path.name})")
orig_data = orig_img.get_fdata()


atlas_img = None
if atlas_paths and "brain_mask" in atlas_paths:
    atlas_img = nib.load(str(atlas_paths["brain_mask"]))
    atlas_data = atlas_img.get_fdata() > 0.5  


uncertainty_data = None
if has_uncertainty:
    uncertainty_img = nib.load(str(uncertainty_path))
    uncertainty_data = uncertainty_img.get_fdata()


def normalize_for_display(image_data):
    finite_positive = image_data[np.isfinite(image_data) & (image_data > 0)]
    if finite_positive.size == 0:
        return np.zeros_like(image_data, dtype=float)
    p1, p99 = np.percentile(finite_positive, [1, 99])
    if p99 <= p1:
        p99 = p1 + 1e-6
    norm_data = np.clip(image_data, p1, p99)
    norm_data = (norm_data - p1) / (p99 - p1)
    return norm_data

orig_display = normalize_for_display(orig_data)
brain_display = normalize_for_display(brain_data)


slice_x = np.argmax(np.sum(mask_data, axis=(1, 2)))
slice_y = np.argmax(np.sum(mask_data, axis=(0, 2)))
slice_z = mask_data.shape[2] // 2

print(f"Coupes optimales: Sagittal x={slice_x}, Coronal y={slice_y}, Axial z={slice_z}")


plt.figure(figsize=(18, 12))


plt.subplot(3, 3, 1)
plt.imshow(orig_display[slice_x, :, :].T, cmap='gray')
plt.title(f"{display_image_label} - Sagittal (x={slice_x})")
plt.axis('off')

plt.subplot(3, 3, 2)
plt.imshow(orig_display[:, slice_y, :].T, cmap='gray')
plt.title(f"{display_image_label} - Coronal (y={slice_y})")
plt.axis('off')

plt.subplot(3, 3, 3)
plt.imshow(orig_display[:, :, slice_z].T, cmap='gray')
plt.title(f"{display_image_label} - Axial (z={slice_z})")
plt.axis('off')


plt.subplot(3, 3, 4)
plt.imshow(brain_display[slice_x, :, :].T, cmap='gray')
plt.title("Cerveau extrait - Sagittal")
plt.axis('off')

plt.subplot(3, 3, 5)
plt.imshow(brain_display[:, slice_y, :].T, cmap='gray')
plt.title("Cerveau extrait - Coronal")
plt.axis('off')

plt.subplot(3, 3, 6)
plt.imshow(brain_display[:, :, slice_z].T, cmap='gray')
plt.title("Cerveau extrait - Axial")
plt.axis('off')


plt.subplot(3, 3, 7)
plt.imshow(orig_display[slice_x, :, :].T, cmap='gray')
plt.imshow(mask_data[slice_x, :, :].T, cmap='hot', alpha=0.5)
if atlas_img is not None:
    
    plt.imshow(atlas_data[slice_x, :, :].T, cmap='winter', alpha=0.3)
plt.title("Masque superposé - Sagittal")
plt.axis('off')

plt.subplot(3, 3, 8)
plt.imshow(orig_display[:, slice_y, :].T, cmap='gray')
plt.imshow(mask_data[:, slice_y, :].T, cmap='hot', alpha=0.5)
if atlas_img is not None:
    plt.imshow(atlas_data[:, slice_y, :].T, cmap='winter', alpha=0.3)
plt.title("Masque superposé - Coronal")
plt.axis('off')

plt.subplot(3, 3, 9)
plt.imshow(orig_display[:, :, slice_z].T, cmap='gray')
plt.imshow(mask_data[:, :, slice_z].T, cmap='hot', alpha=0.5)
if atlas_img is not None:
    plt.imshow(atlas_data[:, :, slice_z].T, cmap='winter', alpha=0.3)
plt.title("Masque superposé - Axial")
plt.axis('off')

plt.suptitle("Résultats de l'extraction du cerveau avec SAM-NeuroAdapt v2", fontsize=16)
plt.tight_layout()


import matplotlib.patches as mpatches
if atlas_img is not None:
    red_patch = mpatches.Patch(color='red', alpha=0.5, label='Masque final SAM2/atlas')
    green_patch = mpatches.Patch(color='green', alpha=0.3, label='Atlas recalé')
    plt.legend(handles=[red_patch, green_patch], loc='lower center', ncol=2)

plt.show()


if uncertainty_data is not None:
    plt.figure(figsize=(18, 6))
    
    plt.subplot(1, 3, 1)
    plt.imshow(uncertainty_data[slice_x, :, :].T, cmap='viridis')
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(f"Incertitude - Sagittal (x={slice_x})")
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(uncertainty_data[:, slice_y, :].T, cmap='viridis')
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(f"Incertitude - Coronal (y={slice_y})")
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(uncertainty_data[:, :, slice_z].T, cmap='viridis')
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(f"Incertitude - Axial (z={slice_z})")
    plt.axis('off')
    
    plt.suptitle("Carte d'incertitude par coupe de la segmentation", fontsize=16)
    plt.tight_layout()
    plt.show()


try:
    
    metrics_path = anat_dir / f"sub-{subject_id}_desc-brain_metrics.json"
    if metrics_path.exists():
        with open(metrics_path, 'r') as f:
            metrics = json.load(f)
        
        
        print("\n=== Métriques d'extraction du cerveau ===")
        print(f"Temps total: {metrics.get('total_time', 'N/A'):.2f}s")
        print(f"Ratio du masque: {metrics.get('mask_ratio', 'N/A'):.2%}")
        
        if 'mask_volume_mm3' in metrics:
            vol_mm3 = metrics['mask_volume_mm3']
            print(f"Volume cérébral: {vol_mm3:.2f} mm³ ({vol_mm3/1000:.2f} cm³)")
        
        print(f"Qualité estimée: {metrics.get('quality_rating', 'N/A')}")
        print(f"Image SAM2 utilisée: {metrics.get('input_image_used', 'N/A')}")
        print(f"Fallback atlas utilisé: {metrics.get('fallback_used', False)}")
        if metrics.get('atlas_coverage') is not None:
            print(f"Couverture atlas finale: {metrics['atlas_coverage']:.2%}")
        if metrics.get('dice_with_atlas') is not None:
            print(f"Dice final avec atlas: {metrics['dice_with_atlas']:.4f}")
        if metrics.get('fallback_reason'):
            print(f"Raison fallback: {metrics['fallback_reason']}")
          
        if 'slice_scores' in metrics:
            avg_score = np.mean([s['score'] for s in metrics['slice_scores']])
            print(f"Score moyen: {avg_score:.4f}")
        
        if atlas_img is not None:
            
            intersection = np.sum(mask_data & atlas_data)
            dice = 2 * intersection / (np.sum(mask_data) + np.sum(atlas_data))
            print(f"Dice avec atlas: {dice:.4f}")
    else:
        print(f"\n⚠️ Fichier de métriques non trouvé: {metrics_path}")
except Exception as e:
    print(f"\n⚠️ Erreur lors du chargement des métriques: {e}")

### B3 — Évaluation vs Gold-Standard (sujet courant)
Calcul de Dice, HD95, ASSD par rapport au masque expert NFBS.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# B3 — Évaluation vs Gold-Standard (sujet courant)
# Compute Dice, HD95, ASSD against the manually validated NFBS brain mask.
# ═══════════════════════════════════════════════════════════════════════════

from scipy.ndimage import distance_transform_edt
from nibabel.processing import resample_from_to

def compute_metrics(pred_mask_path, gs_mask_path):
    # Calcule Dice, HD95, ASSD entre masque prédit et gold-standard.
    pred_img = nib.load(str(pred_mask_path))
    gs_img   = nib.load(str(gs_mask_path))

    # ── Alignement world-space : resample prédit dans l'espace GS ────────
    _affines_match = np.allclose(pred_img.affine, gs_img.affine, atol=1e-3)
    _shapes_match  = (pred_img.shape == gs_img.shape)
    if not _affines_match or not _shapes_match:
        pred_img = resample_from_to(pred_img, gs_img, order=0, cval=0)

    voxel_spacing = tuple(float(v) for v in gs_img.header.get_zooms()[:3])
    pred = (pred_img.get_fdata() > 0.5).astype(bool)
    gs   = (gs_img.get_fdata()  > 0.5).astype(bool)


    # Dice
    intersection = np.sum(pred & gs)
    union_sum    = np.sum(pred) + np.sum(gs)
    dice = float(2.0 * intersection / union_sum) if union_sum > 0 else 0.0

    # Volumes
    voxel_vol    = float(np.prod(voxel_spacing))
    vol_pred_cm3 = float(np.sum(pred) * voxel_vol / 1000)
    vol_gs_cm3   = float(np.sum(gs)   * voxel_vol / 1000)

    # Bords pour HD95 / ASSD
    pred_border = pred ^ ndimage.binary_erosion(pred)
    gs_border   = gs   ^ ndimage.binary_erosion(gs)
    dist_gs_map   = distance_transform_edt(~gs,   sampling=voxel_spacing)
    dist_pred_map = distance_transform_edt(~pred, sampling=voxel_spacing)
    d_p2g = dist_gs_map[pred_border]
    d_g2p = dist_pred_map[gs_border]

    if len(d_p2g) == 0 or len(d_g2p) == 0:
        hd95 = float('inf')
        assd = float('inf')
    else:
        hd95 = float(max(np.percentile(d_p2g, 95), np.percentile(d_g2p, 95)))
        assd = float((np.mean(d_p2g) + np.mean(d_g2p)) / 2)

    return {"dice": dice, "hd95": hd95, "assd": assd,
            "vol_pred_cm3": vol_pred_cm3, "vol_gs_cm3": vol_gs_cm3}

# ── Évaluation B3 — sujet défini dans cell 0 (SUBJECT_RAW) ─────────
mask_path = ANAT_DIR / f"sub-{SUBJECT_ID}_desc-brain_mask.nii.gz"
_subj_raw = SUBJECT_RAW
_gs_mask  = GS_MASK_PATH

print(f"\n═══ B3 — Évaluation vs Gold-Standard (sub-{_subj_raw}) ═══")
print(f"Masque prédit  : {mask_path}")
print(f"Gold-standard  : {_gs_mask}  (existe: {_gs_mask.exists()})")

if mask_path and Path(str(mask_path)).exists() and _gs_mask.exists():
    m = compute_metrics(mask_path, _gs_mask)
    print(f"\n  Dice          : {m['dice']:.4f}  (cible ≥ 0.92)")
    print(f"  HD95          : {m['hd95']:.2f} mm")
    print(f"  ASSD          : {m['assd']:.2f} mm")
    print(f"  Volume prédit : {m['vol_pred_cm3']:.1f} cm³  (plage QC: 950–1 700 cm³)")
    print(f"  Volume expert : {m['vol_gs_cm3']:.1f} cm³")
    print()
    if m['dice'] >= 0.92:
        print("  ✅ Dice OK — qualité suffisante pour publication")
    else:
        print("  ⚠️  Dice < 0.92 — corrections supplémentaires des paramètres requises")
    if 950 <= m['vol_pred_cm3'] <= 1700:
        print("  ✅ Volume dans la plage QC anatomique")
    elif m['vol_pred_cm3'] < 950:
        print(f"  ⚠️  Volume {m['vol_pred_cm3']:.0f} cm³ < 950 cm³ — sous-segmentation possible")
    else:
        print(f"  ⚠️  Volume {m['vol_pred_cm3']:.0f} cm³ > 1 700 cm³ — sur-segmentation persistante")
else:
    print("⚠️  Fichiers introuvables — vérifier que l'extraction a bien été exécutée.")

### M1-NFBS-v4 — Évaluation externe adulte gold-standard
Même moteur adulte externe adaptatif figé v4 que IBSR18/LPBA40: atlas externes uniquement, no-LOO, réparation affine/orientation, raffinements et adjudication no-gold.


In [ ]:
# ============================================================================
# M1-NFBS-v4 - Evaluation externe adulte SAM-NeuroAdapt sans LOO
# ============================================================================
import importlib
import os
from pathlib import Path

import scripts.adult_candidate_adjudicator as _m1_adult_candidate_adjudicator
import scripts.adult_external_refinement as _m1_adult_external_refinement
import scripts.adult_t1_adaptive_atlas as _m1_adult_t1_adaptive_atlas
import scripts.adult_t1_boundary_refinement as _m1_adult_t1_boundary_refinement
import scripts.adult_t1_strong_refinement as _m1_adult_t1_strong_refinement
import scripts.adaptive_mask_fusion as _m1_adaptive_mask_fusion
import scripts.external_atlas_config as _m1_external_atlas_config
import scripts.m1_external_adaptive_runner as _m1_external_adaptive_runner

importlib.reload(_m1_adult_candidate_adjudicator)
importlib.reload(_m1_adult_external_refinement)
importlib.reload(_m1_adult_t1_adaptive_atlas)
importlib.reload(_m1_adult_t1_boundary_refinement)
importlib.reload(_m1_adult_t1_strong_refinement)
importlib.reload(_m1_adaptive_mask_fusion)
importlib.reload(_m1_external_atlas_config)
_m1_external_adaptive_runner = importlib.reload(_m1_external_adaptive_runner)
run_m1_external_adaptive_gold = _m1_external_adaptive_runner.run_m1_external_adaptive_gold

RUN_M1_NFBS_V4 = os.environ.get("RUN_M1_NFBS_V4", "0").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}
M1_EXT_DATASET = "NFBS"
M1_EXT_PROFILE = "adult_t1"
M1_EXT_MANIFEST = Path("datasets/manifests/NFBS.csv")
M1_EXT_OUT_BASE = Path("outputs/nfbs_m1_eval_v4_external_templates_frozen")

_nfbs_n_subjects = os.environ.get("NFBS_N_SUBJECTS", "5").strip()
NFBS_N_SUBJECTS = None if _nfbs_n_subjects.lower() in {"", "none", "all", "full"} else int(_nfbs_n_subjects)
_nfbs_subjects = os.environ.get("NFBS_SUBJECTS", "").strip()
NFBS_SUBJECTS = [s.strip() for s in _nfbs_subjects.split(",") if s.strip()] or None
_nfbs_min_smoke = os.environ.get("NFBS_MIN_SMOKE_MEAN_DICE", "").strip()
NFBS_MIN_SMOKE_MEAN_DICE = None if _nfbs_min_smoke.lower() in {"", "none", "off"} else float(_nfbs_min_smoke)
NFBS_REPRO_SEED = int(os.environ.get("SAM_NEUROADAPT_SEED", "20260517"))
NFBS_ARCHIVE_PREVIOUS = os.environ.get("M1_EXT_ARCHIVE_PREVIOUS", "1").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}

# Garde-fous explicites: atlas externes uniquement; aucun LOO, aucun SOTA-prior.
M1_EXT_ATLAS_MODE = "external_adaptive_no_loo"
M1_EXT_USE_LOO_ATLAS_LIBRARY = False
M1_EXT_ENABLE_ATLAS_FALLBACK = False

print(f"NFBS manifest: {M1_EXT_MANIFEST} | exists={M1_EXT_MANIFEST.exists()}")
print(f"NFBS run: n_subjects={NFBS_N_SUBJECTS} | subjects={NFBS_SUBJECTS} | min_smoke_mean_dice={NFBS_MIN_SMOKE_MEAN_DICE} | seed={NFBS_REPRO_SEED}")

if RUN_M1_NFBS_V4:
    NFBS_M1_RESULTS = run_m1_external_adaptive_gold(
        globals(),
        dataset=M1_EXT_DATASET,
        profile_filter=M1_EXT_PROFILE,
        manifest=M1_EXT_MANIFEST,
        out_base=M1_EXT_OUT_BASE,
        n_subjects=NFBS_N_SUBJECTS,
        subjects=NFBS_SUBJECTS,
        force_clean_outputs=True,
        force_atlas_registration=True,
        force_extraction=True,
        reproducibility_seed=NFBS_REPRO_SEED,
        archive_previous_outputs=NFBS_ARCHIVE_PREVIOUS,
        deterministic_threads=True,
        min_smoke_mean_dice=NFBS_MIN_SMOKE_MEAN_DICE,
        target_mean_dice=0.98,
    )
else:
    print("M1-NFBS-v4 pret. Mettre RUN_M1_NFBS_V4=1 puis reexecuter cette cellule.")
    print("Watcher: python3 scripts/watch_m1_progress.py --dataset NFBS --version v4")


### M1-ADNI — Validation externe pilote (20 sujets ADNI)
Copie indépendante de M1 pour tester SAM-NeuroAdapt sur ADNI BIDS sans gold-standard manuel. Les sorties sont séparées de M1 NFBS (`outputs/adni_m1_eval`) et les métriques sont des QC anatomiques/atlas, pas un Dice expert.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# M1-ADNI — Copie indépendante de M1 pour 20 sujets ADNI BIDS
# Objectif : tester la robustesse externe sans toucher au M1 NFBS.
# Important : ADNI n'a pas ici de masque expert comparable NFBS; cette cellule
# produit donc des métriques QC anatomiques/atlas, pas un Dice gold-standard.
# Sorties : outputs/adni_m1_eval/*
# ═══════════════════════════════════════════════════════════════════════════

import time, json, traceback, shutil, math
from pathlib import Path
import nibabel as _nib_adni
import numpy as np
import pandas as pd
from scipy import ndimage as _ndi_adni

_REQUIRED_M1_ADNI_SYMBOLS = [
    "run_preprocessing_pipeline",
    "advanced_atlas_registration",
    "extract_brain",
]
_missing_m1_adni = [name for name in _REQUIRED_M1_ADNI_SYMBOLS if name not in globals()]
if _missing_m1_adni:
    raise RuntimeError(
        "Exécuter d'abord les cellules 1-30 avant M1-ADNI. Symboles manquants: "
        + ", ".join(_missing_m1_adni)
    )

# ── Configuration ADNI ─────────────────────────────────────────────────────
ADNI_BIDS_DIR = Path("data/bids")
ADNI_METADATA_CSV = Path("datasets/Adni/T1_3D_SAGITAL_MRI/New_T1_3D_SAGI_5_17_2025.csv")
ADNI_OUT_BASE = Path("outputs/adni_m1_eval")
ADNI_OUT_BASE.mkdir(parents=True, exist_ok=True)

ADNI_N_SUBJECTS = 20
ADNI_SELECTION_MODE = "stratified_metadata"  # stratified_metadata | sorted | random
ADNI_RANDOM_SEED = 42
ADNI_SUBJECTS = None  # liste optionnelle: ["002_S_6007", "sub-003_S_6259", ...]

FORCE_ADNI_PREPROCESSING = False
FORCE_ADNI_ATLAS_REGISTRATION = True
FORCE_ADNI_EXTRACTION = True
CLEAN_ADNI_STALE_OUTPUTS = True
DELETE_ADNI_SHORT_STALE_ATLASES = True
STOP_ADNI_IF_NO_VALID_ATLAS = True

_ADNI_ATLAS_BASE = Path("./sam_neuroadapt_v2_real_atlases/atlases")
_ADNI_ATLAS_FILES = {
    "brain_mask": _ADNI_ATLAS_BASE / "MNI152NLin2009cAsym_brainmask_2mm.nii.gz",
    "brain_mask_mni152": _ADNI_ATLAS_BASE / "MNI152_T1_2mm_brain_mask.nii.gz",
    "brain_mask_tpl": _ADNI_ATLAS_BASE / "tpl-MNI152NLin2009cAsym_res-02_desc-brain_mask.nii.gz",
}
_ADNI_ATLAS_FILES = {k: p for k, p in _ADNI_ATLAS_FILES.items() if Path(p).exists()}

_ADNI_FINAL_CSV = ADNI_OUT_BASE / "sam_neuroadapt_adni20_results.csv"
_ADNI_LIVE_CSV = ADNI_OUT_BASE / "sam_neuroadapt_adni20_results_live.csv"
_ADNI_PROGRESS_JSON = ADNI_OUT_BASE / "m1_adni_progress.json"
_ADNI_SELECTION_JSON = ADNI_OUT_BASE / "adni20_selection.json"

if CLEAN_ADNI_STALE_OUTPUTS:
    for _p in [_ADNI_FINAL_CSV, _ADNI_LIVE_CSV, _ADNI_PROGRESS_JSON]:
        if _p.exists():
            _p.unlink()


def _m1adni_atomic_write_text(path, text):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text)
    tmp.replace(path)


def _m1adni_write_progress(stage, subject=None, idx=None, total=None, rows=None):
    payload = {
        "stage": stage,
        "subject": subject,
        "index": idx,
        "total": total,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "results_csv": str(_ADNI_LIVE_CSV),
    }
    _m1adni_atomic_write_text(_ADNI_PROGRESS_JSON, json.dumps(payload, indent=2, ensure_ascii=False))
    if rows is not None:
        pd.DataFrame(rows).to_csv(_ADNI_LIVE_CSV, index=False)


def _m1adni_load_metadata():
    if not ADNI_METADATA_CSV.exists():
        return {}
    try:
        _df_meta = pd.read_csv(ADNI_METADATA_CSV)
        _meta = {}
        for _, _row in _df_meta.iterrows():
            _sid = str(_row.get("Subject", "")).strip()
            if not _sid or _sid in _meta:
                continue
            _meta[_sid] = {
                "group": str(_row.get("Group", "")).strip(),
                "sex": str(_row.get("Sex", "")).strip(),
                "age": _row.get("Age", ""),
                "visit": str(_row.get("Visit", "")).strip(),
                "image_data_id": str(_row.get("Image Data ID", "")).strip(),
                "description": str(_row.get("Description", "")).strip(),
            }
        return _meta
    except Exception as _e:
        print(f"⚠️ Métadonnées ADNI non lues: {_e}")
        return {}


def _m1adni_discover_t1s():
    if not ADNI_BIDS_DIR.exists():
        raise FileNotFoundError(f"ADNI_BIDS_DIR introuvable: {ADNI_BIDS_DIR}")
    _meta = _m1adni_load_metadata()
    _paths = sorted(ADNI_BIDS_DIR.glob("sub-*/anat/*_T1w.nii.gz"))
    _paths += sorted(ADNI_BIDS_DIR.glob("sub-*/ses-*/anat/*_T1w.nii.gz"))
    _records = []
    _seen = set()
    for _path in _paths:
        _parts = _path.parts
        _sub_label = next((p for p in _parts if p.startswith("sub-")), None)
        _ses_label = next((p for p in _parts if p.startswith("ses-")), None)
        if _sub_label is None:
            continue
        _subject_raw = _sub_label.replace("sub-", "", 1)
        _subject_key = _subject_raw if _ses_label is None else f"{_subject_raw}_{_ses_label.replace('ses-', '', 1)}"
        if _subject_key in _seen:
            continue
        _seen.add(_subject_key)
        _m = _meta.get(_subject_raw, {})
        _records.append({
            "subject": _subject_key,
            "subject_raw": _subject_raw,
            "session": _ses_label.replace("ses-", "", 1) if _ses_label else "",
            "subject_id": f"{_subject_key}_T1w",
            "t1_path": _path,
            "group": _m.get("group", ""),
            "sex": _m.get("sex", ""),
            "age": _m.get("age", ""),
            "visit": _m.get("visit", ""),
            "image_data_id": _m.get("image_data_id", ""),
            "description": _m.get("description", ""),
        })
    if not _records:
        raise RuntimeError(f"Aucun T1w ADNI trouvé dans {ADNI_BIDS_DIR}")
    return _records


def _m1adni_select_subjects(records):
    _records = list(records)
    if ADNI_SUBJECTS:
        _wanted = [str(s).replace("sub-", "", 1) for s in ADNI_SUBJECTS]
        _by_subject = {r["subject"]: r for r in _records}
        _missing = [s for s in _wanted if s not in _by_subject]
        if _missing:
            raise RuntimeError(f"Sujets ADNI demandés introuvables: {_missing}")
        return [_by_subject[s] for s in _wanted]

    if ADNI_SELECTION_MODE == "random":
        _rng = np.random.default_rng(ADNI_RANDOM_SEED)
        _idx = _rng.choice(len(_records), size=min(ADNI_N_SUBJECTS, len(_records)), replace=False)
        return [_records[int(i)] for i in _idx]

    if ADNI_SELECTION_MODE == "stratified_metadata" and any(r.get("group") for r in _records):
        _rng = np.random.default_rng(ADNI_RANDOM_SEED)
        _groups = [g for g in ["CN", "MCI", "AD"] if any(r.get("group") == g for r in _records)]
        _groups += sorted({r.get("group") for r in _records if r.get("group") and r.get("group") not in _groups})
        if _groups:
            _selected, _selected_ids = [], set()
            _base = ADNI_N_SUBJECTS // len(_groups)
            _remainder = ADNI_N_SUBJECTS % len(_groups)
            for _gi, _group in enumerate(_groups):
                _pool = [r for r in _records if r.get("group") == _group]
                _take = min(len(_pool), _base + (1 if _gi < _remainder else 0))
                if _take <= 0:
                    continue
                _choice = _rng.choice(len(_pool), size=_take, replace=False)
                for _ci in _choice:
                    _rec = _pool[int(_ci)]
                    _selected.append(_rec)
                    _selected_ids.add(_rec["subject"])
            if len(_selected) < min(ADNI_N_SUBJECTS, len(_records)):
                _remaining = [r for r in _records if r["subject"] not in _selected_ids]
                _need = min(ADNI_N_SUBJECTS, len(_records)) - len(_selected)
                _choice = _rng.choice(len(_remaining), size=_need, replace=False)
                _selected.extend(_remaining[int(i)] for i in _choice)
            return sorted(_selected, key=lambda r: (r.get("group", ""), r["subject"]))

    return _records[:min(ADNI_N_SUBJECTS, len(_records))]


def _m1adni_image_info(path):
    _img = _nib_adni.load(str(path))
    return {
        "shape": "x".join(str(int(v)) for v in _img.shape[:3]),
        "zooms": "x".join(f"{float(v):.3g}" for v in _img.header.get_zooms()[:3]),
        "orientation": "".join(_nib_adni.aff2axcodes(_img.affine)),
    }


def _m1adni_mask_stats(path):
    _img = _nib_adni.load(str(path))
    _mask = _img.get_fdata() > 0.5
    _vox = float(np.prod(_img.header.get_zooms()[:3]))
    _vol_cm3 = float(np.sum(_mask) * _vox / 1000.0)
    _lbl, _ncomp = _ndi_adni.label(_mask)
    if _mask.any():
        _pts = np.where(_mask)
        _bbox = [(int(p.min()), int(p.max())) for p in _pts]
        _extent_ratio = max((b - a + 1) / s for (a, b), s in zip(_bbox, _mask.shape))
        _centroid = tuple(float(p.mean() / s) for p, s in zip(_pts, _mask.shape))
    else:
        _bbox = None
        _extent_ratio = 0.0
        _centroid = None
    return {
        "volume_cm3": _vol_cm3,
        "components": int(_ncomp),
        "bbox": _bbox,
        "max_extent_ratio": float(_extent_ratio),
        "centroid_rel": _centroid,
    }


def _m1adni_mask_quality(path, min_cm3=900.0, max_cm3=1700.0, max_components=8):
    try:
        _stats = _m1adni_mask_stats(path)
        _vol_cm3 = _stats["volume_cm3"]
        _ncomp = _stats["components"]
        _extent_ratio = _stats["max_extent_ratio"]
        _reason = []
        if _vol_cm3 < min_cm3 or _vol_cm3 > max_cm3:
            _reason.append(f"volume={_vol_cm3:.1f} cm³ hors [{min_cm3:.0f}, {max_cm3:.0f}]")
        if _ncomp > max_components:
            _reason.append(f"composantes={_ncomp} > {max_components}")
        if _vol_cm3 <= 0:
            _reason.append("masque vide")
        if _extent_ratio > 0.85:
            _reason.append(f"bbox trop étendue={_extent_ratio:.1%}")
        _volume_score = max(0.0, 1.0 - abs(_vol_cm3 - 1300.0) / 500.0)
        _extent_score = max(0.0, 1.0 - max(0.0, _extent_ratio - 0.72) / 0.13)
        _component_score = 1.0 if _ncomp == 1 else max(0.0, 1.0 - (_ncomp - 1) / max_components)
        _stats["qc_score"] = float(0.55 * _volume_score + 0.25 * _extent_score + 0.20 * _component_score)
        return len(_reason) == 0, _stats, "; ".join(_reason)
    except Exception as _e:
        return False, {}, f"erreur lecture masque: {_e}"


def _m1adni_clean_subject_artifacts(_anat_dir, _subject_raw, _subject_id):
    if not CLEAN_ADNI_STALE_OUTPUTS:
        return
    _deleted = []
    _subject_bids = f"sub-{_subject_id}"
    for _name in [
        f"{_subject_bids}_desc-brain_T1w.nii.gz",
        f"{_subject_bids}_desc-brain_mask.nii.gz",
        f"{_subject_bids}_desc-brain_uncertainty.nii.gz",
        f"{_subject_bids}_desc-brain_metrics.json",
    ]:
        _p = _anat_dir / _name
        if _p.exists():
            _p.unlink()
            _deleted.append(_p.name)
    _dbg = _anat_dir / "debug_sam"
    if FORCE_ADNI_EXTRACTION and _dbg.exists():
        shutil.rmtree(_dbg)
        _deleted.append("debug_sam/")
    _adv = _anat_dir / "atlases_advanced"
    if DELETE_ADNI_SHORT_STALE_ATLASES and _adv.exists():
        for _p in list(_adv.glob(f"sub-{_subject_raw}_brain_mask*.nii.gz")):
            _p.unlink()
            _deleted.append(_p.name)
    if FORCE_ADNI_ATLAS_REGISTRATION and _adv.exists():
        for _p in list(_adv.glob(f"sub-{_subject_id}_brain_mask*_aff*.nii.gz")) + list(_adv.glob(f"sub-{_subject_id}_brain_mask*_syn.nii.gz")):
            if _p.exists():
                _p.unlink()
                _deleted.append(_p.name)
    if _deleted:
        print(f"    Nettoyage anti-stale ADNI: {len(_deleted)} fichier(s) supprimé(s)")


def _m1adni_select_preproc_mask(_anat_dir, _subject_id):
    _candidates = [
        _anat_dir / f"sub-{_subject_id}_desc-mni_mask.nii.gz",
        _anat_dir / "atlases_advanced" / f"sub-{_subject_id}_brain_mask_aff_nomask.nii.gz",
        _anat_dir / "atlases_advanced" / f"sub-{_subject_id}_brain_mask_mni152_aff_nomask.nii.gz",
        _anat_dir / "atlases_advanced" / f"sub-{_subject_id}_brain_mask_tpl_aff_nomask.nii.gz",
        _anat_dir / "atlases_advanced" / f"sub-{_subject_id}_brain_mask_aff.nii.gz",
        _anat_dir / "atlases_advanced" / f"sub-{_subject_id}_brain_mask_mni152_aff.nii.gz",
        _anat_dir / "atlases_advanced" / f"sub-{_subject_id}_brain_mask_tpl_aff.nii.gz",
    ]
    _atlas_dir = _anat_dir / "atlases"
    if _atlas_dir.exists():
        _candidates += sorted(_atlas_dir.glob("*brain_mask*.nii.gz"))
    _seen, _rejected = set(), []
    for _cand in _candidates:
        if not _cand.exists() or str(_cand) in _seen:
            continue
        _seen.add(str(_cand))
        _ok, _stats, _why = _m1adni_mask_quality(_cand)
        if _ok:
            print(f"    Masque prétraitement validé: {_cand.name} ({_stats['volume_cm3']:.1f} cm³)")
            return _cand
        _rejected.append((_cand.name, _why))
    if _rejected:
        print("    ⚠️ Masques prétraitement ADNI rejetés:")
        for _name, _why in _rejected[:4]:
            print(f"      - {_name}: {_why}")
    return None


def _m1adni_registered_atlas_candidates(_anat_dir, _subject_id):
    _adv_dir = _anat_dir / "atlases_advanced"
    _names = ["brain_mask", "brain_mask_mni152", "brain_mask_tpl"]
    _paths = []
    for _suffix in ["aff_nomask", "aff", "syn"]:
        for _name in _names:
            _paths.append(_adv_dir / f"sub-{_subject_id}_{_name}_{_suffix}.nii.gz")
    return _paths


def _m1adni_atlas_image_alignment_score(_atlas_path, _image_path):
    try:
        _atlas_img = _nib_adni.load(str(_atlas_path))
        _image_img = _nib_adni.load(str(_image_path))
        _mask = _atlas_img.get_fdata() > 0.5
        _data = _image_img.get_fdata(dtype=np.float32)
        if _mask.shape != _data.shape:
            return 0.0, {"intensity_ratio": None}
        _shell = _ndi_adni.binary_dilation(_mask, iterations=3) & (~_mask)
        if np.sum(_mask) < 1000 or np.sum(_shell) < 1000:
            return 0.0, {"intensity_ratio": None}
        _inside_med = float(np.nanmedian(_data[_mask]))
        _outside_med = float(np.nanmedian(_data[_shell]))
        _ratio = _inside_med / (_outside_med + 1e-6)
        _score = float(np.clip((_ratio - 1.10) / 0.90, 0.0, 1.0))
        return _score, {"intensity_ratio": float(_ratio), "inside_median": _inside_med, "shell_median": _outside_med}
    except Exception as _e:
        return 0.0, {"intensity_ratio": None, "image_score_error": str(_e)}


def _m1adni_select_registered_atlas(_anat_dir, _subject_id, _image_path=None):
    _valid, _rejected = [], []
    for _cand in _m1adni_registered_atlas_candidates(_anat_dir, _subject_id):
        if not _cand.exists():
            continue
        _ok, _stats, _why = _m1adni_mask_quality(_cand)
        if _ok:
            if _image_path is not None:
                _img_score, _img_stats = _m1adni_atlas_image_alignment_score(_cand, _image_path)
            else:
                _img_score, _img_stats = 0.0, {}
            _stats.update(_img_stats)
            _stats["image_alignment_score"] = float(_img_score)
            _stats["selection_score"] = float(0.45 * _stats.get("qc_score", 0.0) + 0.55 * _img_score)
            _valid.append((_cand, _stats))
        else:
            _rejected.append((_cand.name, _why))
    if _valid:
        _valid.sort(key=lambda item: item[1].get("selection_score", item[1].get("qc_score", 0.0)), reverse=True)
        _cand, _stats = _valid[0]
        print(f"    Atlas validé: {_cand.name} ({_stats['volume_cm3']:.1f} cm³, score={_stats.get('selection_score', 0.0):.3f})")
        return _cand, _stats, _rejected
    if _rejected:
        print("    ⚠️ Atlas ADNI exacts rejetés par QC:")
        for _name, _why in _rejected[:8]:
            print(f"      - {_name}: {_why}")
    else:
        print(f"    ⚠️ Aucun atlas exact trouvé pour sub-{_subject_id}")
    return None, {}, _rejected


def _m1adni_confidence_from_audit(_anat_dir, _subject_id, _pp=None):
    _audit = None
    _audit_files = list(_anat_dir.glob(f"sub-{_subject_id}*audit_preprocessing.json"))
    if _audit_files:
        with open(_audit_files[0]) as _f_a:
            _audit = json.load(_f_a)
    elif _pp and _pp.get("visual_metrics"):
        _audit = _pp["visual_metrics"]
    if _audit:
        _recommended = _audit.get("recommended_threshold")
        if _recommended is not None:
            _confidence = float(_recommended)
        else:
            _contraste = _audit.get("contraste_ws", 0.5)
            _snr = _audit.get("snr_ws", 2.0)
            _confidence = 0.80 if (_contraste > 0.8 and _snr > 3.0) else 0.70
        _edge_ws = _audit.get("edge_improvement_ws", _audit.get("amelioration_bords"))
        if _edge_ws is not None and _edge_ws < 0:
            print(f"    QC WhiteStripe: bords dégradés ({_edge_ws*100:.1f}%) -> N4 conservée pour SAM2")
        print(f"    Seuil confiance={_confidence:.2f}")
        return _confidence
    print("    Pas d'audit exploitable -> seuil confiance=0.75")
    return 0.75


def _m1adni_qc_status(row):
    if row.get("error"):
        return "Error"
    _vol = row.get("vol_pred_cm3")
    _atlas_vol = row.get("atlas_volume_cm3")
    _components = row.get("components")
    _coverage = row.get("atlas_coverage")
    _precision = row.get("atlas_precision")
    _dice_atlas = row.get("dice_with_atlas")
    _extent = row.get("max_extent_ratio")
    _reasons = []
    try:
        if _vol is None or math.isnan(float(_vol)):
            _reasons.append("volume missing")
        elif float(_vol) < 950:
            _reasons.append("low volume")
        elif float(_vol) > 1700:
            _reasons.append("high volume")
    except Exception:
        _reasons.append("volume invalid")
    try:
        if _components is not None and not math.isnan(float(_components)) and int(_components) > 8:
            _reasons.append("fragmented")
    except Exception:
        pass
    try:
        if _coverage is not None and not math.isnan(float(_coverage)) and float(_coverage) < 0.85:
            _reasons.append("low atlas coverage")
    except Exception:
        pass
    try:
        if _precision is not None and not math.isnan(float(_precision)) and float(_precision) < 0.85:
            _reasons.append("low atlas precision")
    except Exception:
        pass
    try:
        if _dice_atlas is not None and not math.isnan(float(_dice_atlas)) and float(_dice_atlas) < 0.90:
            _reasons.append("low atlas Dice")
    except Exception:
        pass
    try:
        if _extent is not None and not math.isnan(float(_extent)) and float(_extent) > 0.85:
            _reasons.append("wide bbox")
    except Exception:
        pass
    try:
        if _vol and _atlas_vol and not math.isnan(float(_vol)) and not math.isnan(float(_atlas_vol)) and float(_atlas_vol) > 0:
            _ratio = float(_vol) / float(_atlas_vol)
            if _ratio < 0.85:
                _reasons.append("mask << atlas")
            elif _ratio > 1.15:
                _reasons.append("mask >> atlas")
    except Exception:
        pass
    if _reasons:
        return "Review - " + "; ".join(_reasons[:3])
    if str(row.get("fallback_used")).lower() in {"true", "1"}:
        return "Good - atlas fallback applied"
    return "Good - QC only"


# ── Sélection des sujets ────────────────────────────────────────────────────
_all_adni_records = _m1adni_discover_t1s()
ADNI_SELECTED_SUBJECTS = _m1adni_select_subjects(_all_adni_records)
_selection_payload = {
    "bids_dir": str(ADNI_BIDS_DIR),
    "metadata_csv": str(ADNI_METADATA_CSV) if ADNI_METADATA_CSV.exists() else "",
    "selection_mode": ADNI_SELECTION_MODE,
    "random_seed": ADNI_RANDOM_SEED,
    "n_requested": ADNI_N_SUBJECTS,
    "n_available": len(_all_adni_records),
    "n_selected": len(ADNI_SELECTED_SUBJECTS),
    "subjects": [{k: (str(v) if isinstance(v, Path) else v) for k, v in r.items()} for r in ADNI_SELECTED_SUBJECTS],
}
_ADNI_SELECTION_JSON.write_text(json.dumps(_selection_payload, indent=2, ensure_ascii=False))

print(f"ADNI disponibles: {len(_all_adni_records)} | sélectionnés: {len(ADNI_SELECTED_SUBJECTS)}")
print(f"Mode sélection: {ADNI_SELECTION_MODE} | seed={ADNI_RANDOM_SEED}")
print(f"Atlas candidats: {list(_ADNI_ATLAS_FILES.keys())}")
print("Sujets ADNI:", [r["subject"] for r in ADNI_SELECTED_SUBJECTS])
print(f"Sélection sauvegardée: {_ADNI_SELECTION_JSON}")

# ── Boucle ADNI ─────────────────────────────────────────────────────────────
print("\n═══ M1-ADNI — Pipeline SAM-NeuroAdapt robuste sur 20 sujets ADNI ═══\n")
_m1adni_write_progress("started", total=len(ADNI_SELECTED_SUBJECTS), rows=[])
_results_adni = []

for _idx, _rec in enumerate(ADNI_SELECTED_SUBJECTS, 1):
    _sid = _rec["subject"]
    _subject_raw = _rec["subject_raw"]
    _subject_id = _rec["subject_id"]
    _subject_bids = f"sub-{_subject_id}"
    _t1_path = Path(_rec["t1_path"])
    _out_dir = ADNI_OUT_BASE / _sid
    _anat_dir = _out_dir / "anat"
    _out_dir.mkdir(parents=True, exist_ok=True)
    _anat_dir.mkdir(parents=True, exist_ok=True)
    _t0 = time.time()

    print(f"\n[{_idx:02d}/{len(ADNI_SELECTED_SUBJECTS)}] {_sid}  group={_rec.get('group','')} age={_rec.get('age','')}")
    _m1adni_write_progress("subject_started", _sid, _idx, len(ADNI_SELECTED_SUBJECTS), _results_adni)

    try:
        if not _t1_path.exists():
            raise FileNotFoundError(f"T1 ADNI introuvable: {_t1_path}")
        _img_info = _m1adni_image_info(_t1_path)
        _m1adni_clean_subject_artifacts(_anat_dir, _subject_raw, _subject_id)

        print(f"  [{_sid}] Étape 1/3 : N4 + WhiteStripe...")
        _m1adni_write_progress("preprocessing", _sid, _idx, len(ADNI_SELECTED_SUBJECTS), _results_adni)
        _n4_path = _anat_dir / f"{_subject_bids}_desc-n4_T1w.nii.gz"
        _ws_path = _anat_dir / f"{_subject_bids}_desc-n4_desc-ws_T1w.nii.gz"
        _pp = None
        if FORCE_ADNI_PREPROCESSING or not _n4_path.exists() or not _ws_path.exists():
            _preproc_mask = _m1adni_select_preproc_mask(_anat_dir, _subject_id)
            _pp = run_preprocessing_pipeline(
                t1_path=_t1_path,
                output_dir=_anat_dir,
                subject_id=_subject_id,
                mask_path=_preproc_mask if _preproc_mask is not None and _preproc_mask.exists() else None,
            )
            if not _pp.get("success"):
                raise RuntimeError(f"Preprocessing failed: {_pp}")
            _n4_path = Path(_pp.get("n4_path")) if _pp.get("n4_path") else _n4_path
            _ws_path = Path(_pp.get("whitestripe_path")) if _pp.get("whitestripe_path") else _ws_path
        else:
            print(f"    Cache prétraitement réutilisé: {_n4_path.name}, {_ws_path.name}")

        if _n4_path.exists():
            _input_image_path = _n4_path
            print(f"    Image SAM2: N4 ({_n4_path.name})")
        elif _ws_path.exists():
            _input_image_path = _ws_path
            print(f"    Image SAM2 fallback: WhiteStripe ({_ws_path.name})")
        else:
            raise RuntimeError("Preprocessing failed — N4/WS introuvable")

        print(f"  [{_sid}] Étape 2/3 : Recalage atlas multi-template...")
        _m1adni_write_progress("atlas_registration", _sid, _idx, len(ADNI_SELECTED_SUBJECTS), _results_adni)
        _atlas_adv_dir = _anat_dir / "atlases_advanced"
        _atlas_adv_dir.mkdir(parents=True, exist_ok=True)

        _selected_atlas, _atlas_stats, _rejected_atlases = _m1adni_select_registered_atlas(
            _anat_dir, _subject_id, _n4_path
        )
        if FORCE_ADNI_ATLAS_REGISTRATION or _selected_atlas is None:
            try:
                advanced_atlas_registration(
                    subject_n4_path=_n4_path,
                    atlas_paths=_ADNI_ATLAS_FILES,
                    output_dir=_atlas_adv_dir,
                    subject_id=_subject_id,
                    create_brain_mask=True,
                    verbose=False,
                )
            except Exception as _e_reg:
                print(f"    ⚠️ Recalage atlas échoué: {_e_reg}")
            _selected_atlas, _atlas_stats, _rejected_atlases = _m1adni_select_registered_atlas(
                _anat_dir, _subject_id, _n4_path
            )

        if _selected_atlas is None:
            _msg = "Aucun atlas ADNI recalé exact valide après recalage multi-template"
            if STOP_ADNI_IF_NO_VALID_ATLAS:
                raise RuntimeError(_msg + " — extraction SAM2 non lancée pour éviter une sur-segmentation")
            print(f"    ⚠️ {_msg} -> guidage atlas désactivé")
            _atlas_paths_subj = {}
        else:
            _atlas_paths_subj = {"brain_mask": _selected_atlas}

        _confidence = _m1adni_confidence_from_audit(_anat_dir, _subject_id, _pp)

        print(f"  [{_sid}] Étape 3/3 : Extraction SAM-NeuroAdapt...")
        _m1adni_write_progress("sam2_extraction", _sid, _idx, len(ADNI_SELECTED_SUBJECTS), _results_adni)
        _brain_out = _anat_dir / f"{_subject_bids}_desc-brain_T1w.nii.gz"
        _mask_out = _anat_dir / f"{_subject_bids}_desc-brain_mask.nii.gz"
        _debug_dir = _anat_dir / "debug_sam"
        _debug_dir.mkdir(parents=True, exist_ok=True)

        if FORCE_ADNI_EXTRACTION or not _mask_out.exists():
            _b_path, _m_path = extract_brain(
                input_path=_input_image_path,
                output_path=_brain_out,
                confidence_threshold=_confidence,
                use_atlas_guidance=bool(_atlas_paths_subj),
                atlas_paths=_atlas_paths_subj,
                atlas_weight=0.7,
                post_process=True,
                save_metrics=True,
                save_uncertainty_map=True,
                debug_dir=_debug_dir,
            )
        else:
            _b_path, _m_path = _brain_out, _mask_out
            print(f"    Cache extraction réutilisé: {_mask_out.name}")

        if _m_path is None or not Path(str(_m_path)).exists():
            raise RuntimeError("Extraction ADNI échouée — masque introuvable")

        _mask_stats = _m1adni_mask_stats(_m_path)
        _metrics_path = _anat_dir / f"{_subject_bids}_desc-brain_metrics.json"
        _extract_metrics = {}
        if _metrics_path.exists():
            with open(_metrics_path) as _fm:
                _extract_metrics = json.load(_fm)

        _m = {
            "subject": _sid,
            "dataset": "ADNI",
            "gold_standard_available": False,
            "group": _rec.get("group", ""),
            "sex": _rec.get("sex", ""),
            "age": _rec.get("age", ""),
            "visit": _rec.get("visit", ""),
            "image_data_id": _rec.get("image_data_id", ""),
            "t1_path": str(_t1_path),
            "input_image_used": str(_input_image_path),
            "mask_path": str(_m_path),
            "brain_path": str(_b_path),
            "input_shape": _img_info.get("shape"),
            "input_zooms": _img_info.get("zooms"),
            "input_orientation": _img_info.get("orientation"),
            "vol_pred_cm3": _mask_stats.get("volume_cm3"),
            "components": _mask_stats.get("components"),
            "max_extent_ratio": _mask_stats.get("max_extent_ratio"),
            "centroid_rel": json.dumps(_mask_stats.get("centroid_rel")),
            "atlas_path": str(_selected_atlas) if _selected_atlas is not None else "",
            "atlas_volume_cm3": _atlas_stats.get("volume_cm3"),
            "atlas_components": _atlas_stats.get("components"),
            "atlas_qc_score": _atlas_stats.get("qc_score"),
            "atlas_selection_score": _atlas_stats.get("selection_score"),
            "atlas_image_alignment_score": _atlas_stats.get("image_alignment_score"),
            "atlas_intensity_ratio": _atlas_stats.get("intensity_ratio"),
            "dice_with_atlas": _extract_metrics.get("dice_with_atlas"),
            "atlas_coverage": _extract_metrics.get("atlas_coverage"),
            "atlas_precision": _extract_metrics.get("atlas_precision"),
            "pre_fallback_dice_with_atlas": _extract_metrics.get("pre_fallback_dice_with_atlas"),
            "pre_fallback_atlas_coverage": _extract_metrics.get("pre_fallback_atlas_coverage"),
            "fallback_used": _extract_metrics.get("fallback_used"),
            "fallback_reason": _extract_metrics.get("fallback_reason"),
            "extractor_quality_rating": _extract_metrics.get("quality_rating"),
            "mask_volume_cm3_from_extractor": _extract_metrics.get("mask_volume_cm3"),
            "total_time_s": round(time.time() - _t0, 1),
            "error": "",
        }
        _m["mask_to_atlas_volume_ratio"] = (
            float(_m["vol_pred_cm3"]) / float(_m["atlas_volume_cm3"])
            if _m.get("vol_pred_cm3") is not None and _m.get("atlas_volume_cm3") not in (None, 0) else np.nan
        )
        _m["qc_status"] = _m1adni_qc_status(_m)

    except Exception as _e:
        traceback.print_exc()
        _m = {
            "subject": _sid,
            "dataset": "ADNI",
            "gold_standard_available": False,
            "group": _rec.get("group", ""),
            "sex": _rec.get("sex", ""),
            "age": _rec.get("age", ""),
            "visit": _rec.get("visit", ""),
            "t1_path": str(_t1_path),
            "error": str(_e),
            "qc_status": "Error",
            "total_time_s": round(time.time() - _t0, 1),
        }

    _results_adni.append(_m)
    _m1adni_write_progress("subject_finished", _sid, _idx, len(ADNI_SELECTED_SUBJECTS), _results_adni)
    _err = _m.get("error", "")
    if _err:
        print(f"  → ERREUR: {_err}")
    else:
        print(
            f"  → QC={_m.get('qc_status')}  "
            f"Vol={_m.get('vol_pred_cm3', float('nan')):.1f} cm³  "
            f"AtlasVol={_m.get('atlas_volume_cm3', float('nan')):.1f} cm³  "
            f"DiceAtlas={_m.get('dice_with_atlas', float('nan')):.4f}  "
            f"CovAtlas={_m.get('atlas_coverage', float('nan')):.1%}  "
            f"t={_m.get('total_time_s', 0):.0f}s"
        )

_df_adni = pd.DataFrame(_results_adni)
_df_adni.to_csv(_ADNI_FINAL_CSV, index=False)
_m1adni_write_progress("finished", total=len(ADNI_SELECTED_SUBJECTS), rows=_results_adni)

_df_adni_ok = _df_adni[_df_adni["error"].fillna("").astype(str).str.len() == 0]
print(f"\n═══ Résultats M1-ADNI QC (N={len(_df_adni_ok)}/{len(ADNI_SELECTED_SUBJECTS)} sujets sans erreur) ═══")
if len(_df_adni_ok):
    _cols = [
        "subject", "group", "age", "qc_status", "vol_pred_cm3", "atlas_volume_cm3",
        "mask_to_atlas_volume_ratio", "dice_with_atlas", "atlas_coverage", "atlas_precision",
        "fallback_used", "input_orientation", "total_time_s",
    ]
    _cols = [c for c in _cols if c in _df_adni_ok.columns]
    print(_df_adni_ok[_cols].to_string(index=False))
    print(f"\nVolume masque : {_df_adni_ok.vol_pred_cm3.mean():.1f} ± {_df_adni_ok.vol_pred_cm3.std():.1f} cm³")
    if "dice_with_atlas" in _df_adni_ok:
        print(f"Dice atlas    : {_df_adni_ok.dice_with_atlas.mean():.4f} ± {_df_adni_ok.dice_with_atlas.std():.4f}")
    if "atlas_coverage" in _df_adni_ok:
        print(f"Couverture atlas : {_df_adni_ok.atlas_coverage.mean():.1%} ± {_df_adni_ok.atlas_coverage.std():.1%}")
    print("\nStatuts QC:")
    print(_df_adni_ok["qc_status"].value_counts(dropna=False).to_string())

print(f"\nRésultats ADNI sauvegardés: {_ADNI_FINAL_CSV}")
print(f"Suivi live ADNI: {_ADNI_LIVE_CSV} | {_ADNI_PROGRESS_JSON}")
print("Note: pas de Dice gold-standard sur ADNI dans cette cellule; utiliser une revue visuelle ou un masque expert si disponible.")


### M1-IBSR18 — Évaluation externe gold-standard
Même pipeline que M1 NFBS, avec `IBSR18_N_SUBJECTS = None` par défaut pour le run publication et sorties séparées dans `outputs/ibsr18_m1_eval_v4_external_templates_frozen`.


In [ ]:
# ============================================================================
# M1-IBSR18 - Evaluation externe SAM-NeuroAdapt v4 templates externes sans LOO
# ============================================================================
import importlib
import os
from pathlib import Path

import scripts.adult_candidate_adjudicator as _m1_adult_candidate_adjudicator
import scripts.adaptive_mask_fusion as _m1_adaptive_mask_fusion
import scripts.external_atlas_config as _m1_external_atlas_config
import scripts.m1_external_adaptive_runner as _m1_external_adaptive_runner

importlib.reload(_m1_adult_candidate_adjudicator)
importlib.reload(_m1_adaptive_mask_fusion)
importlib.reload(_m1_external_atlas_config)
_m1_external_adaptive_runner = importlib.reload(_m1_external_adaptive_runner)
run_m1_external_adaptive_gold = _m1_external_adaptive_runner.run_m1_external_adaptive_gold

M1_EXT_DATASET = "IBSR18"
M1_EXT_PROFILE = "adult_t1"
M1_EXT_MANIFEST = Path("datasets/manifests/IBSR18.csv")
M1_EXT_OUT_BASE = Path("outputs/ibsr18_m1_eval_v4_external_templates_frozen")

# Run publication: pas de seuil bloquant; utiliser un run diagnostic separe pour le smoke.
IBSR18_N_SUBJECTS = None
IBSR18_SUBJECTS = None
IBSR18_REPRO_SEED = int(os.environ.get("SAM_NEUROADAPT_SEED", "20260517"))
IBSR18_ARCHIVE_PREVIOUS = os.environ.get("M1_EXT_ARCHIVE_PREVIOUS", "1").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}

# Garde-fous explicites: aucun atlas IBSR/LOO ne doit guider l'inference.
M1_EXT_ATLAS_MODE = "external_adaptive_no_loo"
M1_EXT_USE_LOO_ATLAS_LIBRARY = False
M1_EXT_ENABLE_ATLAS_FALLBACK = False

IBSR18_M1_RESULTS = run_m1_external_adaptive_gold(
    globals(),
    dataset=M1_EXT_DATASET,
    profile_filter=M1_EXT_PROFILE,
    manifest=M1_EXT_MANIFEST,
    out_base=M1_EXT_OUT_BASE,
    n_subjects=IBSR18_N_SUBJECTS,
    subjects=IBSR18_SUBJECTS,
    force_clean_outputs=True,
    force_atlas_registration=True,
    force_extraction=True,
    reproducibility_seed=IBSR18_REPRO_SEED,
    archive_previous_outputs=IBSR18_ARCHIVE_PREVIOUS,
    deterministic_threads=True,
    min_smoke_mean_dice=None,
    target_mean_dice=0.98,
)


### M1-LPBA40 — Évaluation externe gold-standard
Pipeline adulte externe stabilisé no-LOO. Déposez les fichiers bruts sous `data/LPBA40/**/Sxx.native.mri.hdr` + `.img/.img.gz` et `Sxx.native.brain.mask.hdr` + `.img/.img.gz`; la cellule standardise automatiquement vers des NIfTI runtime-ready dans `datasets/standardized/LPBA40` si le manifest manque ou pointe encore vers Analyze `.hdr`. Sorties: `outputs/lpba40_m1_eval_v4_external_templates_frozen`.


In [ ]:
# ============================================================================
# M1-LPBA40 - Evaluation externe SAM-NeuroAdapt v4 templates externes sans LOO
# ============================================================================
import importlib
import os
from pathlib import Path

import scripts.adult_candidate_adjudicator as _m1_adult_candidate_adjudicator
import scripts.adaptive_mask_fusion as _m1_adaptive_mask_fusion
import scripts.external_atlas_config as _m1_external_atlas_config
import scripts.m1_external_adaptive_runner as _m1_external_adaptive_runner

importlib.reload(_m1_adult_candidate_adjudicator)
importlib.reload(_m1_adaptive_mask_fusion)
importlib.reload(_m1_external_atlas_config)
_m1_external_adaptive_runner = importlib.reload(_m1_external_adaptive_runner)
run_m1_external_adaptive_gold = _m1_external_adaptive_runner.run_m1_external_adaptive_gold

M1_EXT_DATASET = "LPBA40"
M1_EXT_PROFILE = "adult_t1"
M1_EXT_MANIFEST = Path("datasets/manifests/LPBA40.csv")
M1_EXT_OUT_BASE = Path("outputs/lpba40_m1_eval_v4_external_templates_frozen")

# Smoke test par defaut. Utiliser LPBA40_N_SUBJECTS=all pour full run.
_lpba_n_subjects = os.environ.get("LPBA40_N_SUBJECTS", "5").strip()
LPBA40_N_SUBJECTS = None if _lpba_n_subjects.lower() in {"", "none", "all", "full"} else int(_lpba_n_subjects)
_lpba_subjects = os.environ.get("LPBA40_SUBJECTS", "").strip()
LPBA40_SUBJECTS = [s.strip() for s in _lpba_subjects.split(",") if s.strip()] or None
LPBA40_AUTO_PREPARE = os.environ.get("LPBA40_AUTO_PREPARE", "1").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}
_lpba_min_smoke = os.environ.get("LPBA40_MIN_SMOKE_MEAN_DICE", "0.97").strip()
LPBA40_MIN_SMOKE_MEAN_DICE = None if _lpba_min_smoke.lower() in {"", "none", "off"} else float(_lpba_min_smoke)
LPBA40_REPRO_SEED = int(os.environ.get("SAM_NEUROADAPT_SEED", "20260517"))
LPBA40_ARCHIVE_PREVIOUS = os.environ.get("M1_EXT_ARCHIVE_PREVIOUS", "1").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}
LPBA40_RAW_ROOT = Path("data/LPBA40")
LPBA40_RAW_T1_COUNT = len(list(LPBA40_RAW_ROOT.rglob("*native.mri.hdr"))) if LPBA40_RAW_ROOT.exists() else 0
LPBA40_RAW_MASK_COUNT = len(list(LPBA40_RAW_ROOT.rglob("*native.brain.mask.hdr"))) if LPBA40_RAW_ROOT.exists() else 0
LPBA40_RAW_T1_IMG_COUNT = len(list(LPBA40_RAW_ROOT.rglob("*native.mri.img*"))) if LPBA40_RAW_ROOT.exists() else 0
LPBA40_RAW_MASK_IMG_COUNT = len(list(LPBA40_RAW_ROOT.rglob("*native.brain.mask.img*"))) if LPBA40_RAW_ROOT.exists() else 0

# Garde-fous explicites: aucun atlas LPBA/LOO ne doit guider l'inference.
M1_EXT_ATLAS_MODE = "external_adaptive_no_loo"
M1_EXT_USE_LOO_ATLAS_LIBRARY = False
M1_EXT_ENABLE_ATLAS_FALLBACK = False

print(f"LPBA40 manifest: {M1_EXT_MANIFEST} | exists={M1_EXT_MANIFEST.exists()}")
print(f"LPBA40 raw: {LPBA40_RAW_ROOT} | T1 hdr={LPBA40_RAW_T1_COUNT}, img={LPBA40_RAW_T1_IMG_COUNT} | mask hdr={LPBA40_RAW_MASK_COUNT}, img={LPBA40_RAW_MASK_IMG_COUNT}")
print(f"LPBA40 run: n_subjects={LPBA40_N_SUBJECTS} | subjects={LPBA40_SUBJECTS} | min_smoke_mean_dice={LPBA40_MIN_SMOKE_MEAN_DICE} | seed={LPBA40_REPRO_SEED}")
if not M1_EXT_MANIFEST.exists() and (not LPBA40_RAW_ROOT.exists() or LPBA40_RAW_T1_COUNT == 0 or LPBA40_RAW_MASK_COUNT == 0 or LPBA40_RAW_T1_IMG_COUNT == 0 or LPBA40_RAW_MASK_IMG_COUNT == 0):
    raise FileNotFoundError(
        "LPBA40 brut introuvable. Placer les donnees sous data/LPBA40 avec "
        "**/Sxx.native.mri.hdr, **/Sxx.native.mri.img(.gz), "
        "**/Sxx.native.brain.mask.hdr et **/Sxx.native.brain.mask.img(.gz), puis relancer la cellule."
    )

LPBA40_M1_RESULTS = run_m1_external_adaptive_gold(
    globals(),
    dataset=M1_EXT_DATASET,
    profile_filter=M1_EXT_PROFILE,
    manifest=M1_EXT_MANIFEST,
    out_base=M1_EXT_OUT_BASE,
    n_subjects=LPBA40_N_SUBJECTS,
    subjects=LPBA40_SUBJECTS,
    force_clean_outputs=True,
    force_atlas_registration=True,
    force_extraction=True,
    auto_prepare_lpba40=LPBA40_AUTO_PREPARE,
    reproducibility_seed=LPBA40_REPRO_SEED,
    archive_previous_outputs=LPBA40_ARCHIVE_PREVIOUS,
    deterministic_threads=True,
    min_smoke_mean_dice=LPBA40_MIN_SMOKE_MEAN_DICE,
    target_mean_dice=0.98,
)


### M1-iSeg2017 — Évaluation externe pédiatrique gold-standard
Même pipeline externe que M1-IBSR18/LPBA40, adapté au stress test iSeg-2017 6 mois avec l'atlas UNC-BCP 06M et les 10 sujets localement labellisés.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# M1-iSeg2017 — Évaluation externe pédiatrique SAM-NeuroAdapt
# Stress test 6 mois: T1-only, atlas UNC-BCP 06M, gold = iSeg label > 0.
# Les sujets officiels 11-23 n'ont pas de labels locaux: métriques seulement
# sur datasets/iSeg2017/training_manifest.csv.
# ═══════════════════════════════════════════════════════════════════════════
import os, json, time, traceback, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as _nib_iseg
from scipy import ndimage as _ndi_iseg
from scripts.iseg2017_robust_refinement import (
    apply_pediatric_t2_refinement as _iseg_apply_pediatric_t2_refinement,
    robust_atlas_image_alignment_score as _iseg_robust_atlas_image_alignment_score,
)
from scripts.adult_external_refinement import apply_adult_external_refinement as _iseg_apply_wide_adult_style_refinement
from scripts.iseg2017_adaptive_atlas import build_iseg_adaptive_guidance_atlas as _iseg_build_adaptive_guidance_atlas
from scripts.adaptive_mask_fusion import apply_adaptive_mask_fusion as _iseg_apply_adaptive_mask_fusion
warnings.filterwarnings("ignore")

_REQUIRED_ISEG_SYMBOLS = [
    "run_preprocessing_pipeline",
    "advanced_atlas_registration",
    "SegmentationParams",
    "SAM2BrainExtractor",
    "compute_metrics",
]
_missing = [name for name in _REQUIRED_ISEG_SYMBOLS if name not in globals()]
if _missing:
    raise RuntimeError(
        "Exécuter d'abord les cellules 1-30 avant M1-iSeg2017. Symboles manquants: "
        + ", ".join(_missing)
    )

ISEG_DATASET = "iSeg2017"
ISEG_ROOT = Path("datasets/iSeg2017")
ISEG_MANIFEST = ISEG_ROOT / "training_manifest.csv"
ISEG_OUT_BASE = Path("outputs/iseg2017_m1_eval")
ISEG_OUT_BASE.mkdir(parents=True, exist_ok=True)

ISEG_N_SUBJECTS = None  # None = 10 sujets labellisés; 1 = smoke-test rapide
ISEG_SUBJECTS = None  # ex: ["sub-ISEG001", "ISEG002"]

FORCE_ISEG_SOURCE_CONVERSION = True
FORCE_ISEG_PREPROCESSING = True
FORCE_ISEG_ATLAS_REGISTRATION = True
FORCE_ISEG_EXTRACTION = True
CLEAN_ISEG_STALE_OUTPUTS = True
DELETE_ISEG_SHORT_STALE_ATLASES = True
STOP_ISEG_IF_NO_VALID_ATLAS = True

# Volumes nourrisson 6 mois observés localement: 703-921 cm3, atlas 06M: ~786 cm3.
ISEG_SPACING_OVERRIDE = None
ISEG_MASK_MIN_CM3 = 550.0
ISEG_MASK_MAX_CM3 = 1100.0
ISEG_MASK_MAX_COMPONENTS = 4
ISEG_MASK_MAX_EXTENT = 0.96
ISEG_ATLAS_QC_MIN_CM3 = 550.0
ISEG_ATLAS_QC_MAX_CM3 = 1100.0
ISEG_ATLAS_MAX_COMPONENTS = 1
ISEG_ATLAS_NOMASK_BONUS = 0.03
ISEG_MIN_ATLAS_IMAGE_SCORE = 0.05
ISEG_MIN_ATLAS_SELECTION_SCORE = 0.25
ISEG_MIN_BRAIN_VOLUME_CM3 = 550.0
ISEG_MAX_BRAIN_VOLUME_CM3 = 1100.0
ISEG_ATLAS_FALLBACK_MIN_CM3 = 550.0
ISEG_ATLAS_FALLBACK_MAX_CM3 = 1100.0
ISEG_ATLAS_MIN_COVERAGE = 0.85
ISEG_ATLAS_MIN_PRECISION = 0.90
# Fallback atlas désactivé par défaut: sur iSeg il remplaçait 3 masques SAM par
# un atlas 06M moins précis. Il ne doit rester qu'un recours d'urgence explicite.
ISEG_ENABLE_ATLAS_FALLBACK = False
ISEG_ENABLE_ATLAS_REFINEMENT = True
ISEG_REFINEMENT_ATLAS_DILATION = 1
ISEG_REFINEMENT_MIN_KEEP_RATIO = 0.85
ISEG_REFINEMENT_MIN_ATLAS_COVERAGE = 0.85
ISEG_REFINEMENT_MIN_PRECISION_GAIN = 0.01
ISEG_ENABLE_BOUNDARY_REFINEMENT = True
ISEG_BOUNDARY_REFINEMENT_WIDTH = 1
ISEG_BOUNDARY_ATLAS_DILATION = 1
ISEG_BOUNDARY_INTENSITY_PERCENTILE = 8.0
ISEG_BOUNDARY_MAX_ADDED_CM3 = 35.0
ISEG_BOUNDARY_MAX_VOLUME_RATIO = 1.06
ISEG_BOUNDARY_MIN_ATLAS_PRECISION = 0.92

# Raffinement pédiatrique T1/T2: récupération large de la périphérie CSF/GM
# dans une enveloppe atlas dilatée, sans jamais lire le gold mask.
ISEG_ENABLE_PEDIATRIC_T2_REFINEMENT = True
ISEG_PEDIATRIC_T2_SHELL_WIDTH = 20
ISEG_PEDIATRIC_T2_ATLAS_DILATION = 15
ISEG_PEDIATRIC_T2_T1_PERCENTILE = 3.0
ISEG_PEDIATRIC_T2_T2_PERCENTILE = 3.0
ISEG_PEDIATRIC_T2_MAX_ADDED_CM3 = 500.0
ISEG_PEDIATRIC_T2_MIN_ATLAS_PRECISION = 0.75

# Passe supplémentaire "adult-style élargie" adaptée à iSeg: elle garde les
# mêmes garde-fous atlas/T1 que les cellules adultes, mais avec volumes 6 mois.
# Elle reste post-hoc et ne lit jamais le gold mask.
ISEG_ENABLE_WIDE_ADULT_STYLE_REFINEMENT = True
ISEG_WIDE_ADULT_STYLE_REFINEMENT_CONFIG = {
    "enabled": True,
    "min_volume_cm3": ISEG_MIN_BRAIN_VOLUME_CM3,
    "max_volume_cm3": ISEG_MAX_BRAIN_VOLUME_CM3,
    "prune_enabled": True,
    "prune_atlas_dilation": 2,
    "prune_if_atlas_precision_below": 0.78,
    "prune_if_volume_ratio_above": 1.22,
    "prune_min_keep_ratio": 0.82,
    "recover_enabled": True,
    "recover_shell_width": 4,
    "recover_atlas_dilation": 4,
    "recover_intensity_percentile": 3.0,
    "recover_max_added_cm3": 160.0,
    "recover_max_volume_ratio": 1.16,
    "recover_min_atlas_precision": 0.74,
    "min_atlas_dice_gain": 0.001,
    "backup_suffix": "_pre-iseg-wide-refine",
}

# Atlas adaptatif sans LOO: construit a partir des variantes UNC-BCP 06M
# recalees et du signal T1/T2 du sujet. Il ne lit jamais le gold mask.
ISEG_ENABLE_ADAPTIVE_GUIDANCE_ATLAS = True
ISEG_ADAPTIVE_GUIDANCE_CONFIG = {
    "min_volume_cm3": ISEG_ATLAS_QC_MIN_CM3,
    "max_volume_cm3": ISEG_ATLAS_QC_MAX_CM3,
    "seed_prefer": "aff",
    "shell_width": 24,
    "envelope_dilation": 3,
    "t1_percentile": 3.0,
    "t2_percentile": 3.0,
    "target_from_envelope_ratio": 0.90,
    "target_seed_gain_ratio": 1.12,
    "target_max_cm3": 980.0,
    "max_added_cm3": 260.0,
    "min_added_cm3": 5.0,
}

# Fusion adaptive finale sans gold mask: sélectionne entre SAM courant, atlas
# adaptatif tissue-pruned, vote atlas et fallback anatomique via QC.
ISEG_ENABLE_ADAPTIVE_MASK_FUSION = True
ISEG_ADAPTIVE_MASK_FUSION_CONFIG = {
    "enabled": True,
    "min_volume_cm3": ISEG_MIN_BRAIN_VOLUME_CM3,
    "soft_min_volume_cm3": 720.0,
    "soft_max_volume_cm3": 900.0,
    "max_volume_cm3": ISEG_MAX_BRAIN_VOLUME_CM3,
    "tissue_percentiles": [8.0, 10.0, 12.0, 15.0, 18.0],
    "use_t2_gate": False,
    "coverage_target": 0.84,
    "coverage_sigma": 0.13,
    "catastrophic_min_precision": 0.50,
    "catastrophic_min_coverage": 0.45,
    "catastrophic_min_atlas_dice": 0.55,
    "catastrophic_max_uncertainty_mean": 0.25,
    "undercoverage_min_precision": 0.96,
    "undercoverage_max_coverage": 0.82,
    "undercoverage_max_volume_ratio": 0.86,
    "current_mask_bonus": 0.08,
    "tissue_undercoverage_bonus": 0.07,
    "selection_margin": 0.015,
    "backup_suffix": "_pre-adaptive-fusion",
}

_ISEG_ATLAS_BASE = ISEG_ROOT / "atlas" / "UNC_BCP_06M"
_ISEG_ATLAS_FILES = {
    "brain_mask": _ISEG_ATLAS_BASE / "tpl-UNCBCP06M_desc-brain_mask.nii.gz",
}
for _name, _path in _ISEG_ATLAS_FILES.items():
    if not Path(_path).exists():
        raise FileNotFoundError(f"Atlas iSeg2017 06M introuvable ({_name}): {_path}")
if not (_ISEG_ATLAS_BASE / "tpl-UNCBCP06M_T1w.nii.gz").exists():
    raise FileNotFoundError("Template T1 UNC-BCP 06M introuvable; requis pour le recalage atlas iSeg2017")

_ISEG_FINAL_CSV = ISEG_OUT_BASE / "sam_neuroadapt_iseg2017_results.csv"
_ISEG_LIVE_CSV = ISEG_OUT_BASE / "sam_neuroadapt_iseg2017_results_live.csv"
_ISEG_PROGRESS_JSON = ISEG_OUT_BASE / "m1_iseg2017_progress.json"
_ISEG_SELECTION_JSON = ISEG_OUT_BASE / "iseg2017_selection.json"


def _iseg_atomic_write_text(path, text):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text)
    tmp.replace(path)


def _iseg_write_progress(stage, subject=None, idx=None, total=None, rows=None):
    payload = {
        "dataset": ISEG_DATASET,
        "stage": stage,
        "subject": subject,
        "idx": idx,
        "total": total,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    if rows is not None:
        payload["n_rows"] = len(rows)
        payload["n_errors"] = int(sum(1 for r in rows if str(r.get("error", ""))))
        payload["n_dice"] = int(sum(1 for r in rows if pd.notna(r.get("dice", np.nan))))
        try:
            pd.DataFrame(rows).to_csv(_ISEG_LIVE_CSV, index=False)
        except Exception as exc:
            payload["live_csv_error"] = str(exc)
    _iseg_atomic_write_text(_ISEG_PROGRESS_JSON, json.dumps(payload, indent=2, ensure_ascii=False))


def _iseg_natural_key(value):
    import re
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", str(value))]


def _iseg_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"1", "true", "yes", "y"}


def _iseg_discover_records():
    if not ISEG_MANIFEST.exists():
        raise FileNotFoundError(f"Manifest iSeg2017 introuvable: {ISEG_MANIFEST}")
    df = pd.read_csv(ISEG_MANIFEST)
    required = {"subject", "split", "has_gold", "t1_path", "t2_path", "brain_mask_path", "label_path"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise RuntimeError(f"Manifest iSeg2017 invalide, colonnes manquantes: {missing}")
    df = df[(df["split"].astype(str) == "train_labeled") & df["has_gold"].map(_iseg_bool)].copy()
    if len(df) == 0:
        raise RuntimeError("Aucun sujet iSeg2017 labellisé disponible pour les métriques")
    records = []
    for _, row in df.sort_values("subject", key=lambda s: s.map(_iseg_natural_key)).iterrows():
        subject = str(row["subject"])
        subject_raw = subject.replace("sub-", "", 1)
        rec = {
            "subject": subject,
            "subject_raw": subject_raw,
            "subject_id": f"{subject_raw}_T1w",
            "t1_path": Path(str(row["t1_path"])),
            "t2_path": Path(str(row["t2_path"])),
            "gs_mask_path": Path(str(row["brain_mask_path"])),
            "label_path": Path(str(row["label_path"])),
            "split": str(row["split"]),
        }
        for key in ["t1_path", "t2_path", "gs_mask_path", "label_path"]:
            if not rec[key].exists():
                raise FileNotFoundError(f"{subject}: fichier {key} introuvable: {rec[key]}")
        records.append(rec)
    return records


def _iseg_select_records(records):
    records = list(records)
    if ISEG_SUBJECTS:
        wanted = [str(s).replace("sub-", "", 1) for s in ISEG_SUBJECTS]
        by_key = {}
        for rec in records:
            keys = {rec["subject"], rec["subject_raw"], rec["subject_id"]}
            keys |= {k.replace("sub-", "", 1) for k in list(keys)}
            for key in keys:
                by_key[key] = rec
        missing = [s for s in wanted if s not in by_key]
        if missing:
            raise RuntimeError(f"Sujets demandés introuvables dans iSeg2017: {missing}")
        return [by_key[s] for s in wanted]
    if ISEG_N_SUBJECTS is None:
        return records
    return records[:min(int(ISEG_N_SUBJECTS), len(records))]


def _iseg_save_nifti_3d(src_path, dst_path, is_mask=False, spacing_override=None):
    src_path = Path(src_path)
    dst_path = Path(dst_path)
    if dst_path.exists() and not FORCE_ISEG_SOURCE_CONVERSION:
        return dst_path
    img = _nib_iseg.load(str(src_path))
    data = np.asanyarray(img.dataobj)
    while data.ndim > 3 and data.shape[-1] == 1:
        data = data[..., 0]
    if data.ndim != 3:
        raise RuntimeError(f"Image iSeg non 3D après squeeze: {src_path} shape={data.shape}")
    affine = np.array(img.affine, dtype=float)
    zooms = tuple(float(z) for z in img.header.get_zooms()[:3])
    if spacing_override is not None:
        sx, sy, sz = [float(v) for v in spacing_override]
        affine = np.diag([sx, sy, sz, 1.0])
        zooms = (sx, sy, sz)
    if is_mask:
        out_data = (data > 0).astype(np.uint8)
        out_dtype = np.uint8
    elif np.issubdtype(data.dtype, np.integer):
        out_data = np.rint(data).astype(np.int16)
        out_dtype = np.int16
    else:
        out_data = np.nan_to_num(data.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        out_dtype = np.float32
    out_img = _nib_iseg.Nifti1Image(out_data, affine)
    out_img.header.set_zooms(zooms)
    out_img.set_data_dtype(out_dtype)
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    _nib_iseg.save(out_img, str(dst_path))
    return dst_path


def _iseg_prepare_source_images(rec, anat_dir):
    source_dir = Path(anat_dir) / "source"
    source_dir.mkdir(parents=True, exist_ok=True)
    subject_id = rec["subject_id"]
    t1_out = source_dir / f"sub-{subject_id}.nii.gz"
    t2_out = source_dir / f"sub-{subject_id}_T2w.nii.gz"
    mask_out = source_dir / f"sub-{subject_id}_gold-brain_mask.nii.gz"
    label_out = source_dir / f"sub-{subject_id}_desc-label_seg.nii.gz"
    t1_out = _iseg_save_nifti_3d(rec["t1_path"], t1_out, is_mask=False, spacing_override=ISEG_SPACING_OVERRIDE)
    t2_out = _iseg_save_nifti_3d(rec["t2_path"], t2_out, is_mask=False, spacing_override=ISEG_SPACING_OVERRIDE)
    mask_out = _iseg_save_nifti_3d(rec["gs_mask_path"], mask_out, is_mask=True, spacing_override=ISEG_SPACING_OVERRIDE)
    label_out = _iseg_save_nifti_3d(rec["label_path"], label_out, is_mask=False, spacing_override=ISEG_SPACING_OVERRIDE)
    return t1_out, mask_out, t2_out, label_out


def _iseg_image_info(path):
    img = _nib_iseg.load(str(path))
    return {
        "shape": "x".join(str(int(v)) for v in img.shape[:3]),
        "zooms": "x".join(f"{float(v):g}" for v in img.header.get_zooms()[:3]),
        "orientation": "".join(_nib_iseg.aff2axcodes(img.affine)),
    }


def _iseg_mask_quality(path, min_cm3=None, max_cm3=None, max_components=None, max_extent_ratio=None):
    min_cm3 = ISEG_MASK_MIN_CM3 if min_cm3 is None else float(min_cm3)
    max_cm3 = ISEG_MASK_MAX_CM3 if max_cm3 is None else float(max_cm3)
    max_components = ISEG_MASK_MAX_COMPONENTS if max_components is None else int(max_components)
    max_extent_ratio = ISEG_MASK_MAX_EXTENT if max_extent_ratio is None else float(max_extent_ratio)
    try:
        img = _nib_iseg.load(str(path))
        mask = img.get_fdata() > 0.5
        vox = float(np.prod(img.header.get_zooms()[:3]))
        vol_cm3 = float(np.sum(mask) * vox / 1000.0)
        lbl, ncomp = _ndi_iseg.label(mask)
        reason = []
        if vol_cm3 < min_cm3 or vol_cm3 > max_cm3:
            reason.append(f"volume={vol_cm3:.1f} cm3 hors [{min_cm3:.0f}, {max_cm3:.0f}]")
        if ncomp > max_components:
            reason.append(f"composantes={ncomp} > {max_components}")
        if mask.any():
            pts = np.where(mask)
            bbox = [(int(p.min()), int(p.max())) for p in pts]
            extent_ratios = [(b - a + 1) / s for (a, b), s in zip(bbox, mask.shape)]
            extent_ratio = max(extent_ratios)
            centroid = tuple(float(p.mean() / s) for p, s in zip(pts, mask.shape))
            if extent_ratio > max_extent_ratio:
                reason.append(f"bbox trop etendue={extent_ratio:.1%} > {max_extent_ratio:.0%}")
        else:
            bbox = None
            extent_ratio = 0.0
            centroid = None
            reason.append("masque vide")
        volume_center = (min_cm3 + max_cm3) / 2.0
        volume_half_range = max(1.0, (max_cm3 - min_cm3) / 2.0)
        volume_score = max(0.0, 1.0 - abs(vol_cm3 - volume_center) / volume_half_range)
        extent_score = max(0.0, 1.0 - max(0.0, extent_ratio - 0.72) / max(0.01, max_extent_ratio - 0.72))
        component_score = 1.0 if ncomp == 1 else max(0.0, 1.0 - (ncomp - 1) / max_components)
        score = 0.60 * volume_score + 0.20 * extent_score + 0.20 * component_score
        return len(reason) == 0, {
            "volume_cm3": vol_cm3,
            "components": int(ncomp),
            "bbox": bbox,
            "max_extent_ratio": float(extent_ratio),
            "centroid_rel": centroid,
            "qc_score": float(score),
        }, "; ".join(reason)
    except Exception as e:
        return False, {}, f"erreur lecture masque: {e}"


def _iseg_clean_subject_artifacts(anat_dir, subject_id):
    if not CLEAN_ISEG_STALE_OUTPUTS:
        return
    anat_dir = Path(anat_dir)
    deleted = []
    patterns = [
        f"sub-{subject_id}_desc-brain*.nii.gz",
        f"sub-{subject_id}_desc-uncertainty*.nii.gz",
        f"sub-{subject_id}_desc-brain_metrics.json",
    ]
    atlas_dir = anat_dir / "atlases_advanced"
    if FORCE_ISEG_ATLAS_REGISTRATION and atlas_dir.exists():
        patterns += [f"atlases_advanced/sub-{subject_id}_brain_mask_*.nii.gz"]
    if FORCE_ISEG_EXTRACTION:
        for pattern in patterns:
            for p in anat_dir.glob(pattern):
                try:
                    p.unlink()
                    deleted.append(p.name)
                except FileNotFoundError:
                    pass
    if DELETE_ISEG_SHORT_STALE_ATLASES and atlas_dir.exists():
        for p in atlas_dir.glob(f"sub-{subject_id}_brain_mask_*.nii.gz"):
            try:
                if p.stat().st_size < 4096:
                    p.unlink()
                    deleted.append(p.name)
            except FileNotFoundError:
                pass
    if deleted:
        print(f"    Nettoyage anti-stale: {len(deleted)} fichier(s) supprimé(s)")


def _iseg_select_preproc_mask(anat_dir, subject_id):
    candidates = []
    adv_dir = Path(anat_dir) / "atlases_advanced"
    if adv_dir.exists():
        for suffix in ["aff_nomask", "aff", "syn"]:
            candidates.append(adv_dir / f"sub-{subject_id}_brain_mask_{suffix}.nii.gz")
    rejected, seen = [], set()
    for cand in candidates:
        if not cand.exists() or str(cand) in seen:
            continue
        seen.add(str(cand))
        ok, stats, why = _iseg_mask_quality(cand)
        if ok:
            print(f"    Masque preprocessing validé: {cand.name} ({stats['volume_cm3']:.1f} cm3, composantes={stats['components']})")
            return cand
        rejected.append((cand.name, why))
    if rejected:
        print("    Masques preprocessing rejetés:")
        for name, why in rejected[:5]:
            print(f"      - {name}: {why}")
    return None


def _iseg_registered_atlas_candidates(anat_dir, subject_id):
    adv_dir = Path(anat_dir) / "atlases_advanced"
    paths = [
        adv_dir / f"sub-{subject_id}_brain_mask_adaptive.nii.gz",
    ]
    paths.extend(sorted(adv_dir.glob(f"sub-{subject_id}_brain_mask_*_adaptive.nii.gz")))
    for suffix in ["aff_nomask", "aff", "syn"]:
        paths.append(adv_dir / f"sub-{subject_id}_brain_mask_{suffix}.nii.gz")
    # Deduplication stable.
    out, seen = [], set()
    for p in paths:
        key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out




def _iseg_adaptive_fusion_atlas_paths(anat_dir, subject_id, selected_atlas=None, adaptive_path=None):
    paths = [selected_atlas, adaptive_path]
    adv_dir = Path(anat_dir) / "atlases_advanced"
    for suffix in ["adaptive", "aff", "aff_nomask", "syn"]:
        paths.append(adv_dir / f"sub-{subject_id}_brain_mask_{suffix}.nii.gz")
    out, seen = [], set()
    for path in paths:
        if not path:
            continue
        path = Path(path)
        if not path.exists():
            continue
        key = str(path.resolve())
        if key in seen:
            continue
        seen.add(key)
        out.append(path)
    return out

def _iseg_atlas_variant_bonus(atlas_path):
    name = Path(atlas_path).name
    if "_adaptive" in name:
        return 0.12
    if "_aff_nomask" in name:
        return float(ISEG_ATLAS_NOMASK_BONUS)
    return 0.0


def _iseg_atlas_image_alignment_score(atlas_path, image_path):
    return _iseg_robust_atlas_image_alignment_score(atlas_path, image_path)


def _iseg_select_registered_atlas(anat_dir, subject_id, image_path=None):
    valid, rejected = [], []
    for cand in _iseg_registered_atlas_candidates(anat_dir, subject_id):
        if not cand.exists():
            continue
        ok, stats, why = _iseg_mask_quality(
            cand,
            min_cm3=ISEG_ATLAS_QC_MIN_CM3,
            max_cm3=ISEG_ATLAS_QC_MAX_CM3,
            max_components=ISEG_ATLAS_MAX_COMPONENTS,
        )
        if ok:
            if image_path is not None:
                img_score, img_stats = _iseg_atlas_image_alignment_score(cand, image_path)
            else:
                img_score, img_stats = 0.0, {}
            stats.update(img_stats)
            stats["image_alignment_score"] = float(img_score)
            variant_bonus = _iseg_atlas_variant_bonus(cand)
            stats["variant_bonus"] = float(variant_bonus)
            stats["selection_score_base"] = float(0.55 * stats.get("qc_score", 0.0) + 0.45 * img_score)
            stats["selection_score"] = float(np.clip(stats["selection_score_base"] + variant_bonus, 0.0, 1.0))
            weak = []
            if image_path is not None and img_score < ISEG_MIN_ATLAS_IMAGE_SCORE:
                weak.append(f"score_image={img_score:.3f} < {ISEG_MIN_ATLAS_IMAGE_SCORE:.3f}")
            if stats["selection_score"] < ISEG_MIN_ATLAS_SELECTION_SCORE:
                weak.append(f"score_global={stats['selection_score']:.3f} < {ISEG_MIN_ATLAS_SELECTION_SCORE:.3f}")
            if weak:
                rejected.append((cand.name, "atlas recalé trop faible: " + "; ".join(weak)))
                continue
            valid.append((cand, stats))
        else:
            rejected.append((cand.name, why))
    if valid:
        valid.sort(key=lambda item: item[1].get("selection_score", item[1].get("qc_score", 0.0)), reverse=True)
        cand, stats = valid[0]
        print(f"    Atlas 06M validé: {cand.name} ({stats['volume_cm3']:.1f} cm3, composantes={stats['components']}, score={stats.get('selection_score', stats.get('qc_score', 0.0)):.3f}, ratio_int={stats.get('intensity_ratio')})")
        if len(valid) > 1:
            print("    Autres atlas 06M plausibles:")
            for alt, alt_stats in valid[1:3]:
                print(f"      - {alt.name}: {alt_stats['volume_cm3']:.1f} cm3, score={alt_stats.get('selection_score', alt_stats.get('qc_score', 0.0)):.3f}")
        return cand, stats, rejected
    if rejected:
        print("    Atlas 06M rejetés par QC anatomique/alignement:")
        for name, why in rejected[:8]:
            print(f"      - {name}: {why}")
    else:
        print(f"    Aucun atlas 06M exact trouvé pour sub-{subject_id}")
    return None, {}, rejected


def _iseg_confidence_from_audit(anat_dir, subject_id, pp=None):
    try:
        audit = None
        if isinstance(pp, dict) and pp.get("visual_metrics"):
            audit = pp["visual_metrics"]
        recommended = audit.get("recommended_threshold") if isinstance(audit, dict) else None
        if recommended is not None:
            confidence = float(np.clip(recommended, 0.65, 0.85))
        else:
            confidence = 0.72
        if isinstance(audit, dict):
            edge_ws = float(audit.get("edge_degradation_ws", 0.0) or 0.0)
            if edge_ws > 0.15:
                print(f"    QC WhiteStripe: bords dégradés ({edge_ws*100:.1f}%) -> N4 conservée pour SAM2")
        print(f"    Seuil confiance={confidence:.2f}")
        return confidence
    except Exception:
        print("    Pas d'audit exploitable -> seuil confiance=0.72")
        return 0.72


def _iseg_extract_brain(
    input_path,
    output_path,
    confidence_threshold=None,
    use_atlas_guidance=True,
    atlas_weight=0.75,
    atlas_paths=None,
    post_process=True,
    save_metrics=True,
    save_uncertainty_map=True,
    debug_dir=None,
):
    params = SegmentationParams(
        debug_dir=debug_dir,
        use_atlas_guidance=use_atlas_guidance,
        atlas_weight=atlas_weight,
        post_process=post_process,
    )
    params.min_brain_volume_cm3 = ISEG_MIN_BRAIN_VOLUME_CM3
    params.max_brain_volume_cm3 = ISEG_MAX_BRAIN_VOLUME_CM3
    params.atlas_fallback_min_cm3 = ISEG_ATLAS_FALLBACK_MIN_CM3
    params.atlas_fallback_max_cm3 = ISEG_ATLAS_FALLBACK_MAX_CM3
    params.atlas_min_coverage = ISEG_ATLAS_MIN_COVERAGE
    params.atlas_min_precision = ISEG_ATLAS_MIN_PRECISION
    params.enable_atlas_fallback = bool(atlas_paths) and ISEG_ENABLE_ATLAS_FALLBACK
    if atlas_paths:
        params.atlas_paths = atlas_paths
    if confidence_threshold is not None:
        params.initial_confidence_threshold = float(confidence_threshold)
        params.propagation_confidence_threshold = float(confidence_threshold) * 0.9
    try:
        extractor = SAM2BrainExtractor(params=params)
        return extractor.extract_brain(
            input_path=input_path,
            output_path=output_path,
            save_metrics=save_metrics,
            save_uncertainty_map=save_uncertainty_map,
        )
    except Exception as e:
        print(f"Erreur lors de l'extraction du cerveau iSeg2017: {e}")
        traceback.print_exc()
        return None, None


def _iseg_nii_with_suffix(path, suffix):
    path = Path(path)
    name = path.name
    if name.endswith(".nii.gz"):
        return path.with_name(name[:-7] + suffix + ".nii.gz")
    return path.with_name(path.stem + suffix + path.suffix)


def _iseg_largest_component(mask):
    mask = np.asarray(mask).astype(bool)
    lbl, ncomp = _ndi_iseg.label(mask)
    if ncomp <= 1:
        return mask
    counts = np.bincount(lbl.ravel())
    counts[0] = 0
    return lbl == int(np.argmax(counts))


def _iseg_mask_overlap_stats(mask, atlas, zooms):
    mask = np.asarray(mask).astype(bool)
    atlas = np.asarray(atlas).astype(bool)
    inter = int(np.logical_and(mask, atlas).sum())
    mask_n = int(mask.sum())
    atlas_n = int(atlas.sum())
    voxel_vol = float(np.prod(zooms))
    return {
        "volume_cm3": float(mask_n * voxel_vol / 1000.0),
        "atlas_coverage": float(inter / (atlas_n + 1e-8)),
        "atlas_precision": float(inter / (mask_n + 1e-8)),
        "atlas_dice": float((2.0 * inter) / (mask_n + atlas_n + 1e-8)),
        "voxels": mask_n,
    }


def _iseg_apply_local_boundary_refinement(mask, atlas, mask_img, input_image_path, resample_from_to):
    stats = {
        "boundary_refinement_enabled": bool(ISEG_ENABLE_BOUNDARY_REFINEMENT),
        "boundary_refinement_used": False,
        "boundary_refinement_reason": None,
    }
    if not ISEG_ENABLE_BOUNDARY_REFINEMENT:
        stats["boundary_refinement_reason"] = "disabled"
        return mask, stats
    try:
        input_image_path = Path(input_image_path)
        if not input_image_path.exists():
            stats["boundary_refinement_reason"] = "input_missing"
            return mask, stats
        mask = np.asarray(mask).astype(bool)
        atlas = np.asarray(atlas).astype(bool)
        if mask.sum() == 0:
            stats["boundary_refinement_reason"] = "empty_mask"
            return mask, stats
        input_img = _nib_iseg.load(str(input_image_path))
        input_for_mask = input_img
        if input_img.shape[:3] != mask_img.shape[:3] or not np.allclose(input_img.affine, mask_img.affine, atol=1e-3):
            input_for_mask = resample_from_to(input_img, mask_img, order=1, cval=0)
        data = np.nan_to_num(input_for_mask.get_fdata(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        structure = _ndi_iseg.generate_binary_structure(3, 1)
        atlas_allowed = atlas
        if int(ISEG_BOUNDARY_ATLAS_DILATION) > 0:
            atlas_allowed = _ndi_iseg.binary_dilation(atlas, structure=structure, iterations=int(ISEG_BOUNDARY_ATLAS_DILATION))
        shell = _ndi_iseg.binary_dilation(mask, structure=structure, iterations=int(ISEG_BOUNDARY_REFINEMENT_WIDTH)) & (~mask) & atlas_allowed
        in_values = data[mask]
        in_values = in_values[np.isfinite(in_values)]
        if in_values.size == 0:
            stats["boundary_refinement_reason"] = "no_finite_intensity"
            return mask, stats
        pct = float(ISEG_BOUNDARY_INTENSITY_PERCENTILE)
        threshold = float(np.percentile(in_values, pct))
        additions = shell & (data >= threshold)
        if int(additions.sum()) == 0:
            stats.update({
                "boundary_refinement_reason": "no_candidate_voxels",
                "boundary_refinement_intensity_threshold": threshold,
                "boundary_refinement_candidate_voxels": int(shell.sum()),
            })
            return mask, stats
        proposal = _iseg_largest_component(mask | additions)
        proposal = _ndi_iseg.binary_fill_holes(proposal).astype(bool)
        zooms = tuple(float(v) for v in mask_img.header.get_zooms()[:3])
        voxel_vol_cm3 = float(np.prod(zooms) / 1000.0)
        before = _iseg_mask_overlap_stats(mask, atlas, zooms)
        after = _iseg_mask_overlap_stats(proposal, atlas, zooms)
        added_voxels = int(np.logical_and(proposal, ~mask).sum())
        added_cm3 = float(added_voxels * voxel_vol_cm3)
        volume_ratio = float(after["volume_cm3"] / max(before["volume_cm3"], 1e-8))
        stats.update({
            "boundary_refinement_intensity_percentile": pct,
            "boundary_refinement_intensity_threshold": threshold,
            "boundary_refinement_candidate_voxels": int(shell.sum()),
            "boundary_refinement_added_voxels": added_voxels,
            "boundary_refinement_added_cm3": added_cm3,
            "boundary_refinement_volume_before_cm3": before["volume_cm3"],
            "boundary_refinement_volume_after_cm3": after["volume_cm3"],
            "boundary_refinement_volume_ratio": volume_ratio,
            "boundary_refinement_atlas_precision_before": before["atlas_precision"],
            "boundary_refinement_atlas_precision_after": after["atlas_precision"],
            "boundary_refinement_atlas_coverage_before": before["atlas_coverage"],
            "boundary_refinement_atlas_coverage_after": after["atlas_coverage"],
        })
        reject = []
        if after["volume_cm3"] < ISEG_MIN_BRAIN_VOLUME_CM3:
            reject.append(f"volume_after={after['volume_cm3']:.1f} cm3 < {ISEG_MIN_BRAIN_VOLUME_CM3:.0f}")
        if after["volume_cm3"] > ISEG_MAX_BRAIN_VOLUME_CM3:
            reject.append(f"volume_after={after['volume_cm3']:.1f} cm3 > {ISEG_MAX_BRAIN_VOLUME_CM3:.0f}")
        if added_cm3 > float(ISEG_BOUNDARY_MAX_ADDED_CM3):
            reject.append(f"added_cm3={added_cm3:.1f} > {ISEG_BOUNDARY_MAX_ADDED_CM3:.1f}")
        if volume_ratio > float(ISEG_BOUNDARY_MAX_VOLUME_RATIO):
            reject.append(f"volume_ratio={volume_ratio:.3f} > {ISEG_BOUNDARY_MAX_VOLUME_RATIO:.3f}")
        if after["atlas_precision"] < float(ISEG_BOUNDARY_MIN_ATLAS_PRECISION):
            reject.append(f"atlas_precision_after={after['atlas_precision']:.3f} < {ISEG_BOUNDARY_MIN_ATLAS_PRECISION:.3f}")
        if reject:
            stats["boundary_refinement_reason"] = "; ".join(reject)
            return mask, stats
        stats["boundary_refinement_used"] = True
        stats["boundary_refinement_reason"] = "applied"
        return proposal, stats
    except Exception as e:
        stats["boundary_refinement_reason"] = f"erreur boundary refinement: {e}"
        return mask, stats


def _iseg_refine_mask_with_atlas(mask_path, brain_path, input_image_path, atlas_path, metrics_path=None):
    stats = {
        "refinement_enabled": bool(ISEG_ENABLE_ATLAS_REFINEMENT),
        "refinement_used": False,
        "refinement_reason": None,
    }
    if not ISEG_ENABLE_ATLAS_REFINEMENT:
        stats["refinement_reason"] = "disabled"
        return stats
    if atlas_path is None or not Path(atlas_path).exists():
        stats["refinement_reason"] = "atlas_missing"
        return stats
    try:
        from nibabel.processing import resample_from_to as _iseg_resample_from_to
        mask_path = Path(mask_path)
        brain_path = Path(brain_path) if brain_path is not None else None
        input_image_path = Path(input_image_path)
        atlas_path = Path(atlas_path)
        mask_img = _nib_iseg.load(str(mask_path))
        mask = mask_img.get_fdata() > 0.5
        atlas_img = _nib_iseg.load(str(atlas_path))
        if atlas_img.shape[:3] != mask_img.shape[:3] or not np.allclose(atlas_img.affine, mask_img.affine, atol=1e-3):
            atlas_img = _iseg_resample_from_to(atlas_img, mask_img, order=0, cval=0)
        atlas = atlas_img.get_fdata() > 0.5
        zooms = tuple(float(v) for v in mask_img.header.get_zooms()[:3])
        before = _iseg_mask_overlap_stats(mask, atlas, zooms)
        structure = _ndi_iseg.generate_binary_structure(3, 1)
        allowed = atlas
        if int(ISEG_REFINEMENT_ATLAS_DILATION) > 0:
            allowed = _ndi_iseg.binary_dilation(atlas, structure=structure, iterations=int(ISEG_REFINEMENT_ATLAS_DILATION))
        refined = _iseg_largest_component(mask & allowed)
        refined = _ndi_iseg.binary_fill_holes(refined).astype(bool)
        strict_after = _iseg_mask_overlap_stats(refined, atlas, zooms)
        after = strict_after
        reject = []
        if after["voxels"] == 0:
            reject.append("masque raffiné vide")
        if after["volume_cm3"] < ISEG_MIN_BRAIN_VOLUME_CM3:
            reject.append(f"volume_after={after['volume_cm3']:.1f} cm3 < {ISEG_MIN_BRAIN_VOLUME_CM3:.0f}")
        if after["volume_cm3"] > ISEG_MAX_BRAIN_VOLUME_CM3:
            reject.append(f"volume_after={after['volume_cm3']:.1f} cm3 > {ISEG_MAX_BRAIN_VOLUME_CM3:.0f}")
        keep_ratio = after["voxels"] / max(1, before["voxels"])
        if keep_ratio < float(ISEG_REFINEMENT_MIN_KEEP_RATIO):
            reject.append(f"keep_ratio={keep_ratio:.3f} < {ISEG_REFINEMENT_MIN_KEEP_RATIO:.3f}")
        if after["atlas_coverage"] < float(ISEG_REFINEMENT_MIN_ATLAS_COVERAGE):
            reject.append(f"atlas_coverage_after={after['atlas_coverage']:.3f} < {ISEG_REFINEMENT_MIN_ATLAS_COVERAGE:.3f}")
        precision_gain = after["atlas_precision"] - before["atlas_precision"]
        already_atlas_constrained = before["atlas_precision"] >= float(ISEG_BOUNDARY_MIN_ATLAS_PRECISION)
        if precision_gain < float(ISEG_REFINEMENT_MIN_PRECISION_GAIN) and not already_atlas_constrained:
            reject.append(f"precision_gain={precision_gain:.3f} < {ISEG_REFINEMENT_MIN_PRECISION_GAIN:.3f}")
        stats.update({
            "refinement_dilation": int(ISEG_REFINEMENT_ATLAS_DILATION),
            "refinement_volume_before_cm3": before["volume_cm3"],
            "atlas_refinement_volume_after_cm3": strict_after["volume_cm3"],
            "atlas_refinement_keep_ratio": float(keep_ratio),
            "refinement_atlas_coverage_before": before["atlas_coverage"],
            "atlas_refinement_atlas_coverage_after": strict_after["atlas_coverage"],
            "refinement_atlas_precision_before": before["atlas_precision"],
            "atlas_refinement_atlas_precision_after": strict_after["atlas_precision"],
            "refinement_atlas_dice_before": before["atlas_dice"],
            "atlas_refinement_atlas_dice_after": strict_after["atlas_dice"],
            "refinement_precision_gain": float(precision_gain),
        })
        if reject:
            stats["refinement_reason"] = "; ".join(reject)
            print(f"    Raffinement atlas iSeg non appliqué: {stats['refinement_reason']}")
            return stats
        refined, boundary_stats = _iseg_apply_local_boundary_refinement(refined, atlas, mask_img, input_image_path, _iseg_resample_from_to)
        stats.update(boundary_stats)
        after = _iseg_mask_overlap_stats(refined, atlas, zooms)
        stats.update({
            "refinement_volume_after_cm3": after["volume_cm3"],
            "refinement_keep_ratio": float(after["voxels"] / max(1, before["voxels"])),
            "refinement_atlas_coverage_after": after["atlas_coverage"],
            "refinement_atlas_precision_after": after["atlas_precision"],
            "refinement_atlas_dice_after": after["atlas_dice"],
        })
        backup_mask = _iseg_nii_with_suffix(mask_path, "_pre-refine")
        if not backup_mask.exists() or FORCE_ISEG_EXTRACTION:
            _nib_iseg.save(mask_img, str(backup_mask))
        out_hdr = mask_img.header.copy()
        out_img = _nib_iseg.Nifti1Image(refined.astype(np.uint8), mask_img.affine, out_hdr)
        out_img.header.set_zooms(zooms)
        out_img.set_data_dtype(np.uint8)
        _nib_iseg.save(out_img, str(mask_path))
        if brain_path is not None and input_image_path.exists():
            input_img = _nib_iseg.load(str(input_image_path))
            refine_img_for_input = out_img
            if input_img.shape[:3] != out_img.shape[:3] or not np.allclose(input_img.affine, out_img.affine, atol=1e-3):
                refine_img_for_input = _iseg_resample_from_to(out_img, input_img, order=0, cval=0)
            refined_input = refine_img_for_input.get_fdata() > 0.5
            brain_data = np.nan_to_num(input_img.get_fdata(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0) * refined_input.astype(np.float32)
            brain_hdr = input_img.header.copy()
            brain_img = _nib_iseg.Nifti1Image(brain_data.astype(np.float32), input_img.affine, brain_hdr)
            brain_img.set_data_dtype(np.float32)
            _nib_iseg.save(brain_img, str(brain_path))
        stats["refinement_used"] = True
        stats["refinement_reason"] = "applied"
        stats["refinement_backup_mask"] = str(backup_mask)
        label = "Raffinement atlas+frontière iSeg appliqué" if stats.get("boundary_refinement_used") else "Raffinement atlas iSeg appliqué"
        print(
            f"    {label}: volume {before['volume_cm3']:.1f}->{after['volume_cm3']:.1f} cm3, "
            f"atlas_precision {before['atlas_precision']:.3f}->{after['atlas_precision']:.3f}, coverage={after['atlas_coverage']:.3f}"
        )
        if metrics_path is not None and Path(metrics_path).exists():
            try:
                payload = json.loads(Path(metrics_path).read_text())
                payload.update(stats)
                Path(metrics_path).write_text(json.dumps(payload, indent=2, ensure_ascii=False))
            except Exception as exc:
                print(f"    Impossible de mettre à jour les métriques de raffinement iSeg: {exc}")
        return stats
    except Exception as e:
        stats["refinement_reason"] = f"erreur raffinement atlas iSeg: {e}"
        print(f"    {stats['refinement_reason']}")
        traceback.print_exc()
        return stats


_ALL_ISEG_RECORDS = _iseg_discover_records()
ISEG_SELECTED_RECORDS = _iseg_select_records(_ALL_ISEG_RECORDS)
_selection_payload = {
    "dataset": ISEG_DATASET,
    "root": str(ISEG_ROOT),
    "manifest": str(ISEG_MANIFEST),
    "n_requested": ISEG_N_SUBJECTS,
    "n_available": len(_ALL_ISEG_RECORDS),
    "n_selected": len(ISEG_SELECTED_RECORDS),
    "subjects": [
        {k: (str(v) if isinstance(v, Path) else v) for k, v in rec.items()}
        for rec in ISEG_SELECTED_RECORDS
    ],
    "atlas_files": {k: str(v) for k, v in _ISEG_ATLAS_FILES.items()},
}
_ISEG_SELECTION_JSON.write_text(json.dumps(_selection_payload, indent=2, ensure_ascii=False))

print(f"iSeg2017 labellisés disponibles: {len(_ALL_ISEG_RECORDS)} | sélectionnés: {len(ISEG_SELECTED_RECORDS)}")
print(f"Nombre configurable: {ISEG_N_SUBJECTS} (None = tous les sujets labellisés)")
print(f"Atlas pédiatrique 06M: {_ISEG_ATLAS_FILES['brain_mask']}")
print("Sujets:", [r["subject"] for r in ISEG_SELECTED_RECORDS])
print(f"Sélection sauvegardée: {_ISEG_SELECTION_JSON}")

print("\n═══ M1-iSeg2017 — Pipeline SAM-NeuroAdapt pédiatrique robuste ═══\n")
_iseg_write_progress("started", total=len(ISEG_SELECTED_RECORDS), rows=[])
_results_iseg = []

for _idx, _rec in enumerate(ISEG_SELECTED_RECORDS, 1):
    _sid = _rec["subject"]
    _subject_id = _rec["subject_id"]
    _subject_bids = f"sub-{_subject_id}"
    _out_dir = ISEG_OUT_BASE / _sid
    _anat_dir = _out_dir / "anat"
    _out_dir.mkdir(parents=True, exist_ok=True)
    _anat_dir.mkdir(parents=True, exist_ok=True)
    _t0 = time.time()

    print(f"\n[{_idx:02d}/{len(ISEG_SELECTED_RECORDS)}] {_sid}")
    _iseg_write_progress("subject_started", _sid, _idx, len(ISEG_SELECTED_RECORDS), _results_iseg)

    try:
        _source_t1, _gs_mask, _source_t2, _source_label = _iseg_prepare_source_images(_rec, _anat_dir)
        if not _source_t1.exists() or not _gs_mask.exists():
            raise FileNotFoundError(f"T1 ou gold-standard manquant pour {_sid}")
        _img_info = _iseg_image_info(_source_t1)
        _gs_info = _iseg_image_info(_gs_mask)

        _iseg_clean_subject_artifacts(_anat_dir, _subject_id)

        print(f"  [{_sid}] Étape 1/3 : N4 + WhiteStripe...")
        _iseg_write_progress("preprocessing", _sid, _idx, len(ISEG_SELECTED_RECORDS), _results_iseg)
        _n4_path = _anat_dir / f"{_subject_bids}_desc-n4_T1w.nii.gz"
        _ws_path = _anat_dir / f"{_subject_bids}_desc-n4_desc-ws_T1w.nii.gz"
        _pp = None
        if FORCE_ISEG_PREPROCESSING or not _n4_path.exists() or not _ws_path.exists():
            _preproc_mask = _iseg_select_preproc_mask(_anat_dir, _subject_id)
            _pp = run_preprocessing_pipeline(
                t1_path=_source_t1,
                output_dir=_anat_dir,
                subject_id=_subject_id,
                mask_path=_preproc_mask if _preproc_mask is not None and _preproc_mask.exists() else None,
            )
            if not _pp.get("success"):
                raise RuntimeError(f"Preprocessing iSeg failed: {_pp}")
            _n4_path = Path(_pp.get("n4_path")) if _pp.get("n4_path") else _n4_path
            _ws_path = Path(_pp.get("whitestripe_path")) if _pp.get("whitestripe_path") else _ws_path
        else:
            print(f"    Cache preprocessing réutilisé: {_n4_path.name}, {_ws_path.name}")

        if _n4_path.exists():
            _input_image_path = _n4_path
            print(f"    Image SAM2: N4 ({_n4_path.name})")
        elif _ws_path.exists():
            _input_image_path = _ws_path
            print(f"    Image SAM2 fallback: WhiteStripe ({_ws_path.name})")
        else:
            raise RuntimeError("Preprocessing iSeg failed — N4/WS introuvable")

        print(f"  [{_sid}] Étape 2/3 : Recalage atlas UNC-BCP 06M...")
        _iseg_write_progress("atlas_registration", _sid, _idx, len(ISEG_SELECTED_RECORDS), _results_iseg)
        _atlas_adv_dir = _anat_dir / "atlases_advanced"
        _atlas_adv_dir.mkdir(parents=True, exist_ok=True)
        _adaptive_atlas_stats = {}
        if ISEG_ENABLE_ADAPTIVE_GUIDANCE_ATLAS:
            try:
                _adaptive_atlas_stats = _iseg_build_adaptive_guidance_atlas(
                    anat_dir=_anat_dir,
                    subject_id=_subject_id,
                    input_image_path=_n4_path,
                    t2_path=_source_t2,
                    force=False,
                    config=ISEG_ADAPTIVE_GUIDANCE_CONFIG,
                )
            except Exception as _e_adapt:
                print(f"    Atlas adaptatif iSeg non disponible avant recalage: {_e_adapt}")
                _adaptive_atlas_stats = {"adaptive_atlas_reason": str(_e_adapt)}
        _selected_atlas, _atlas_stats, _rejected_atlases = _iseg_select_registered_atlas(
            _anat_dir, _subject_id, _n4_path
        )
        if FORCE_ISEG_ATLAS_REGISTRATION or _selected_atlas is None:
            try:
                advanced_atlas_registration(
                    subject_n4_path=_n4_path,
                    atlas_paths=_ISEG_ATLAS_FILES,
                    output_dir=_atlas_adv_dir,
                    subject_id=_subject_id,
                    create_brain_mask=True,
                    verbose=False,
                )
            except Exception as _e_reg:
                print(f"    Recalage atlas iSeg échoué: {_e_reg}")
            if ISEG_ENABLE_ADAPTIVE_GUIDANCE_ATLAS:
                try:
                    _adaptive_atlas_stats = _iseg_build_adaptive_guidance_atlas(
                        anat_dir=_anat_dir,
                        subject_id=_subject_id,
                        input_image_path=_n4_path,
                        t2_path=_source_t2,
                        force=True,
                        config=ISEG_ADAPTIVE_GUIDANCE_CONFIG,
                    )
                    if _adaptive_atlas_stats.get("adaptive_atlas_used"):
                        print(
                            "    Atlas adaptatif iSeg prêt: "
                            f"{Path(_adaptive_atlas_stats['output_path']).name} "
                            f"vol={_adaptive_atlas_stats.get('adaptive_volume_cm3', float('nan')):.1f} cm3 "
                            f"(+{_adaptive_atlas_stats.get('adaptive_added_cm3', 0.0):.1f})"
                        )
                except Exception as _e_adapt:
                    print(f"    Construction atlas adaptatif iSeg échouée: {_e_adapt}")
                    _adaptive_atlas_stats = {"adaptive_atlas_reason": str(_e_adapt)}
            _selected_atlas, _atlas_stats, _rejected_atlases = _iseg_select_registered_atlas(
                _anat_dir, _subject_id, _n4_path
            )

        if _selected_atlas is None:
            _msg = "Aucun atlas UNC-BCP 06M recalé valide après recalage"
            if STOP_ISEG_IF_NO_VALID_ATLAS:
                raise RuntimeError(_msg + " — extraction SAM2 non lancée pour éviter une sur-segmentation")
            print(f"    {_msg} -> extraction SAM2 sans fallback atlas")
            _atlas_paths_subj = {}
        else:
            _atlas_paths_subj = {"brain_mask": _selected_atlas}

        _confidence = _iseg_confidence_from_audit(_anat_dir, _subject_id, _pp)

        print(f"  [{_sid}] Étape 3/3 : Extraction SAM-NeuroAdapt...")
        _iseg_write_progress("sam2_extraction", _sid, _idx, len(ISEG_SELECTED_RECORDS), _results_iseg)
        _brain_out = _anat_dir / f"{_subject_bids}_desc-brain_T1w.nii.gz"
        _mask_out = _anat_dir / f"{_subject_bids}_desc-brain_mask.nii.gz"
        _debug_dir = _anat_dir / "debug_sam"
        _debug_dir.mkdir(parents=True, exist_ok=True)

        if FORCE_ISEG_EXTRACTION or not _mask_out.exists():
            _b_path, _m_path = _iseg_extract_brain(
                input_path=_input_image_path,
                output_path=_brain_out,
                confidence_threshold=_confidence,
                use_atlas_guidance=bool(_atlas_paths_subj),
                atlas_paths=_atlas_paths_subj,
                atlas_weight=0.75,
                post_process=True,
                save_metrics=True,
                save_uncertainty_map=True,
                debug_dir=_debug_dir,
            )
        else:
            _b_path, _m_path = _brain_out, _mask_out
            print(f"    Cache extraction réutilisé: {_mask_out.name}")

        if _m_path is None or not Path(str(_m_path)).exists():
            raise RuntimeError("Extraction iSeg échouée — masque introuvable")

        _metrics_path = _anat_dir / f"{_subject_bids}_desc-brain_metrics.json"
        _refinement_stats = {}
        if _selected_atlas is not None:
            _refinement_stats = _iseg_refine_mask_with_atlas(
                mask_path=_m_path,
                brain_path=_b_path,
                input_image_path=_input_image_path,
                atlas_path=_selected_atlas,
                metrics_path=_metrics_path,
            )
        _pediatric_refinement_stats = {}
        if _selected_atlas is not None:
            _pediatric_refinement_stats = _iseg_apply_pediatric_t2_refinement(
                mask_path=_m_path,
                brain_path=_b_path,
                input_image_path=_input_image_path,
                t2_path=_source_t2,
                atlas_path=_selected_atlas,
                metrics_path=_metrics_path,
                config={
                    "enabled": ISEG_ENABLE_PEDIATRIC_T2_REFINEMENT,
                    "shell_width": ISEG_PEDIATRIC_T2_SHELL_WIDTH,
                    "atlas_dilation": ISEG_PEDIATRIC_T2_ATLAS_DILATION,
                    "t1_percentile": ISEG_PEDIATRIC_T2_T1_PERCENTILE,
                    "t2_percentile": ISEG_PEDIATRIC_T2_T2_PERCENTILE,
                    "max_added_cm3": ISEG_PEDIATRIC_T2_MAX_ADDED_CM3,
                    "max_volume_cm3": ISEG_MAX_BRAIN_VOLUME_CM3,
                    "min_atlas_precision": ISEG_PEDIATRIC_T2_MIN_ATLAS_PRECISION,
                    "allow_atlas_dice_decrease_on_undercoverage": True,
                    "target_min_atlas_volume_ratio": 0.93,
                    "min_recovery_added_cm3": 25.0,
                    "max_atlas_dice_drop": 0.14,
                },
            )
            if _pediatric_refinement_stats.get("pediatric_refinement_used"):
                print(
                    f"    Raffinement pédiatrique T1/T2 appliqué: +"
                    f"{_pediatric_refinement_stats.get('pediatric_refinement_added_cm3', 0.0):.1f} cm3"
                )
            elif _pediatric_refinement_stats:
                print(f"    Raffinement pédiatrique T1/T2 non appliqué: {_pediatric_refinement_stats.get('pediatric_refinement_reason')}")

        _iseg_wide_refinement_stats = {}
        if ISEG_ENABLE_WIDE_ADULT_STYLE_REFINEMENT and _selected_atlas is not None:
            _iseg_wide_refinement_stats = _iseg_apply_wide_adult_style_refinement(
                mask_path=_m_path,
                brain_path=_b_path,
                input_image_path=_input_image_path,
                atlas_path=_selected_atlas,
                metrics_path=_metrics_path,
                config=ISEG_WIDE_ADULT_STYLE_REFINEMENT_CONFIG,
            )
            if _iseg_wide_refinement_stats.get("adult_refinement_used"):
                print(
                    "    Raffinement iSeg élargi atlas/T1 appliqué: "
                    f"{_iseg_wide_refinement_stats.get('adult_refinement_reason')}"
                )
            elif _iseg_wide_refinement_stats:
                print(
                    "    Raffinement iSeg élargi atlas/T1 non appliqué: "
                    f"{_iseg_wide_refinement_stats.get('adult_refinement_reason')}"
                )


        _adaptive_fusion_stats = {}
        if ISEG_ENABLE_ADAPTIVE_MASK_FUSION and _selected_atlas is not None:
            try:
                _fusion_atlas_paths = _iseg_adaptive_fusion_atlas_paths(
                    _anat_dir,
                    _subject_id,
                    selected_atlas=_selected_atlas,
                    adaptive_path=_adaptive_atlas_stats.get("output_path"),
                )
                _adaptive_ref = _adaptive_atlas_stats.get("output_path")
                if not _adaptive_ref or not Path(str(_adaptive_ref)).exists():
                    _adaptive_ref = _selected_atlas
                _adaptive_fusion_stats = _iseg_apply_adaptive_mask_fusion(
                    mask_path=_m_path,
                    brain_path=_b_path,
                    input_image_path=_input_image_path,
                    atlas_paths=_fusion_atlas_paths,
                    reference_atlas_path=_adaptive_ref,
                    seed_atlas_path=_anat_dir / "atlases_advanced" / f"sub-{_subject_id}_brain_mask_aff.nii.gz",
                    uncertainty_path=_anat_dir / f"{_subject_bids}_desc-brain_uncertainty.nii.gz",
                    t2_path=_source_t2,
                    metrics_path=_metrics_path,
                    config=ISEG_ADAPTIVE_MASK_FUSION_CONFIG,
                )
                if _adaptive_fusion_stats.get("adaptive_fusion_used"):
                    print(
                        "    Fusion adaptive finale appliquée: "
                        f"{_adaptive_fusion_stats.get('adaptive_fusion_selected_candidate')} "
                        f"({ _adaptive_fusion_stats.get('adaptive_fusion_reason') })"
                    )
                elif _adaptive_fusion_stats:
                    print(
                        "    Fusion adaptive finale non appliquée: "
                        f"{_adaptive_fusion_stats.get('adaptive_fusion_reason')}"
                    )
            except Exception as _e_fusion:
                print(f"    Fusion adaptive finale échouée: {_e_fusion}")
                _adaptive_fusion_stats = {"adaptive_fusion_reason": str(_e_fusion)}

        _m = compute_metrics(_m_path, _gs_mask)
        _extract_metrics = {}
        if _metrics_path.exists():
            with open(_metrics_path) as _fm:
                _extract_metrics = json.load(_fm)
        _quality_rating = _extract_metrics.get("quality_rating")
        if _pediatric_refinement_stats.get("pediatric_refinement_used"):
            _quality_rating = "Good - pediatric T1/T2 refinement applied"
        if _iseg_wide_refinement_stats.get("adult_refinement_used"):
            _quality_rating = "Good - pediatric T1/T2 + wide atlas/T1 refinement applied"
        if _adaptive_fusion_stats.get("adaptive_fusion_used"):
            _quality_rating = "Good - adaptive mask fusion applied"
        _m.update({
            "subject": _sid,
            "dataset": ISEG_DATASET,
            "split": _rec.get("split"),
            "source_t1_path": str(_source_t1),
            "source_t2_path": str(_source_t2),
            "gold_mask_path": str(_gs_mask),
            "label_path": str(_source_label),
            "t2_path": str(_source_t2),
            "original_t1_path": str(_rec["t1_path"]),
            "original_t2_path": str(_rec["t2_path"]),
            "original_gold_mask_path": str(_rec["gs_mask_path"]),
            "original_label_path": str(_rec["label_path"]),
            "input_shape": _img_info.get("shape"),
            "input_zooms": _img_info.get("zooms"),
            "input_orientation": _img_info.get("orientation"),
            "gold_shape": _gs_info.get("shape"),
            "gold_zooms": _gs_info.get("zooms"),
            "gold_orientation": _gs_info.get("orientation"),
            "input_image_used": str(_input_image_path),
            "mask_path": str(_m_path),
            "brain_path": str(_b_path),
            "atlas_path": str(_selected_atlas) if _selected_atlas is not None else "",
            "atlas_guidance_used": bool(_atlas_paths_subj),
            "atlas_rejected_count": len(_rejected_atlases),
            "atlas_volume_cm3": _atlas_stats.get("volume_cm3"),
            "atlas_components": _atlas_stats.get("components"),
            "atlas_qc_score": _atlas_stats.get("qc_score"),
            "atlas_selection_score": _atlas_stats.get("selection_score"),
            "atlas_image_alignment_score": _atlas_stats.get("image_alignment_score"),
            "atlas_intensity_ratio": _atlas_stats.get("intensity_ratio"),
            "adaptive_atlas_path": _adaptive_atlas_stats.get("output_path"),
            "adaptive_atlas_used": _adaptive_atlas_stats.get("adaptive_atlas_used"),
            "adaptive_atlas_reason": _adaptive_atlas_stats.get("adaptive_atlas_reason"),
            "adaptive_atlas_seed_variant": _adaptive_atlas_stats.get("seed_variant"),
            "adaptive_atlas_seed_volume_cm3": _adaptive_atlas_stats.get("seed_volume_cm3"),
            "adaptive_atlas_envelope_volume_cm3": _adaptive_atlas_stats.get("envelope_volume_cm3"),
            "adaptive_atlas_volume_cm3": _adaptive_atlas_stats.get("adaptive_volume_cm3"),
            "adaptive_atlas_added_cm3": _adaptive_atlas_stats.get("adaptive_added_cm3"),
            "dice_with_atlas": _extract_metrics.get("dice_with_atlas"),
            "atlas_coverage": _extract_metrics.get("atlas_coverage"),
            "atlas_precision": _extract_metrics.get("atlas_precision"),
            "pre_fallback_dice_with_atlas": _extract_metrics.get("pre_fallback_dice_with_atlas"),
            "pre_fallback_atlas_coverage": _extract_metrics.get("pre_fallback_atlas_coverage"),
            "fallback_used": _extract_metrics.get("fallback_used"),
            "fallback_reason": _extract_metrics.get("fallback_reason"),
            "refinement_used": _extract_metrics.get("refinement_used", _refinement_stats.get("refinement_used")),
            "refinement_reason": _extract_metrics.get("refinement_reason", _refinement_stats.get("refinement_reason")),
            "refinement_volume_before_cm3": _extract_metrics.get("refinement_volume_before_cm3", _refinement_stats.get("refinement_volume_before_cm3")),
            "refinement_volume_after_cm3": _extract_metrics.get("refinement_volume_after_cm3", _refinement_stats.get("refinement_volume_after_cm3")),
            "refinement_keep_ratio": _extract_metrics.get("refinement_keep_ratio", _refinement_stats.get("refinement_keep_ratio")),
            "refinement_atlas_coverage_before": _extract_metrics.get("refinement_atlas_coverage_before", _refinement_stats.get("refinement_atlas_coverage_before")),
            "refinement_atlas_coverage_after": _extract_metrics.get("refinement_atlas_coverage_after", _refinement_stats.get("refinement_atlas_coverage_after")),
            "refinement_atlas_precision_before": _extract_metrics.get("refinement_atlas_precision_before", _refinement_stats.get("refinement_atlas_precision_before")),
            "refinement_atlas_precision_after": _extract_metrics.get("refinement_atlas_precision_after", _refinement_stats.get("refinement_atlas_precision_after")),
            "refinement_precision_gain": _extract_metrics.get("refinement_precision_gain", _refinement_stats.get("refinement_precision_gain")),
            "boundary_refinement_used": _extract_metrics.get("boundary_refinement_used", _refinement_stats.get("boundary_refinement_used")),
            "boundary_refinement_reason": _extract_metrics.get("boundary_refinement_reason", _refinement_stats.get("boundary_refinement_reason")),
            "boundary_refinement_added_cm3": _extract_metrics.get("boundary_refinement_added_cm3", _refinement_stats.get("boundary_refinement_added_cm3")),
            "boundary_refinement_volume_before_cm3": _extract_metrics.get("boundary_refinement_volume_before_cm3", _refinement_stats.get("boundary_refinement_volume_before_cm3")),
            "boundary_refinement_volume_after_cm3": _extract_metrics.get("boundary_refinement_volume_after_cm3", _refinement_stats.get("boundary_refinement_volume_after_cm3")),
            "boundary_refinement_volume_ratio": _extract_metrics.get("boundary_refinement_volume_ratio", _refinement_stats.get("boundary_refinement_volume_ratio")),
            "boundary_refinement_intensity_threshold": _extract_metrics.get("boundary_refinement_intensity_threshold", _refinement_stats.get("boundary_refinement_intensity_threshold")),
            "boundary_refinement_atlas_precision_after": _extract_metrics.get("boundary_refinement_atlas_precision_after", _refinement_stats.get("boundary_refinement_atlas_precision_after")),
            "pediatric_refinement_used": _extract_metrics.get("pediatric_refinement_used", _pediatric_refinement_stats.get("pediatric_refinement_used")),
            "pediatric_refinement_reason": _extract_metrics.get("pediatric_refinement_reason", _pediatric_refinement_stats.get("pediatric_refinement_reason")),
            "pediatric_refinement_added_cm3": _extract_metrics.get("pediatric_refinement_added_cm3", _pediatric_refinement_stats.get("pediatric_refinement_added_cm3")),
            "pediatric_refinement_proposed_added_cm3": _extract_metrics.get("pediatric_refinement_proposed_added_cm3", _pediatric_refinement_stats.get("pediatric_refinement_proposed_added_cm3")),
            "pediatric_refinement_proposed_volume_after_cm3": _extract_metrics.get("pediatric_refinement_proposed_volume_after_cm3", _pediatric_refinement_stats.get("pediatric_refinement_proposed_volume_after_cm3")),
            "pediatric_refinement_proposed_atlas_precision_after": _extract_metrics.get("pediatric_refinement_proposed_atlas_precision_after", _pediatric_refinement_stats.get("pediatric_refinement_proposed_atlas_precision_after")),
            "pediatric_refinement_proposed_atlas_coverage_after": _extract_metrics.get("pediatric_refinement_proposed_atlas_coverage_after", _pediatric_refinement_stats.get("pediatric_refinement_proposed_atlas_coverage_after")),
            "pediatric_refinement_volume_before_cm3": _extract_metrics.get("pediatric_refinement_volume_before_cm3", _pediatric_refinement_stats.get("pediatric_refinement_volume_before_cm3")),
            "pediatric_refinement_volume_after_cm3": _extract_metrics.get("pediatric_refinement_volume_after_cm3", _pediatric_refinement_stats.get("pediatric_refinement_volume_after_cm3")),
            "pediatric_refinement_volume_ratio": _extract_metrics.get("pediatric_refinement_volume_ratio", _pediatric_refinement_stats.get("pediatric_refinement_volume_ratio")),
            "pediatric_refinement_atlas_precision_after": _extract_metrics.get("pediatric_refinement_atlas_precision_after", _pediatric_refinement_stats.get("pediatric_refinement_atlas_precision_after")),
            "pediatric_refinement_atlas_coverage_after": _extract_metrics.get("pediatric_refinement_atlas_coverage_after", _pediatric_refinement_stats.get("pediatric_refinement_atlas_coverage_after")),
            "pediatric_refinement_t1_threshold": _extract_metrics.get("pediatric_refinement_t1_threshold", _pediatric_refinement_stats.get("pediatric_refinement_t1_threshold")),
            "pediatric_refinement_t2_threshold": _extract_metrics.get("pediatric_refinement_t2_threshold", _pediatric_refinement_stats.get("pediatric_refinement_t2_threshold")),
            "iseg_wide_refinement_used": _extract_metrics.get("adult_refinement_used", _iseg_wide_refinement_stats.get("adult_refinement_used")),
            "iseg_wide_refinement_reason": _extract_metrics.get("adult_refinement_reason", _iseg_wide_refinement_stats.get("adult_refinement_reason")),
            "iseg_wide_refinement_volume_before_cm3": _extract_metrics.get("adult_refinement_volume_before_cm3", _iseg_wide_refinement_stats.get("adult_refinement_volume_before_cm3")),
            "iseg_wide_refinement_volume_after_cm3": _extract_metrics.get("adult_refinement_volume_after_cm3", _iseg_wide_refinement_stats.get("adult_refinement_volume_after_cm3")),
            "iseg_wide_refinement_atlas_precision_after": _extract_metrics.get("adult_refinement_atlas_precision_after", _iseg_wide_refinement_stats.get("adult_refinement_atlas_precision_after")),
            "iseg_wide_refinement_atlas_coverage_after": _extract_metrics.get("adult_refinement_atlas_coverage_after", _iseg_wide_refinement_stats.get("adult_refinement_atlas_coverage_after")),
            "iseg_wide_refinement_atlas_dice_after": _extract_metrics.get("adult_refinement_atlas_dice_after", _iseg_wide_refinement_stats.get("adult_refinement_atlas_dice_after")),
            "adaptive_fusion_used": _extract_metrics.get("adaptive_fusion_used", _adaptive_fusion_stats.get("adaptive_fusion_used")),
            "adaptive_fusion_reason": _extract_metrics.get("adaptive_fusion_reason", _adaptive_fusion_stats.get("adaptive_fusion_reason")),
            "adaptive_fusion_selected_candidate": _extract_metrics.get("adaptive_fusion_selected_candidate", _adaptive_fusion_stats.get("adaptive_fusion_selected_candidate")),
            "adaptive_fusion_raw_bad": _extract_metrics.get("adaptive_fusion_raw_bad", _adaptive_fusion_stats.get("adaptive_fusion_raw_bad")),
            "adaptive_fusion_raw_undercoverage": _extract_metrics.get("adaptive_fusion_raw_undercoverage", _adaptive_fusion_stats.get("adaptive_fusion_raw_undercoverage")),
            "adaptive_fusion_current_atlas_dice": _extract_metrics.get("adaptive_fusion_current_atlas_dice", _adaptive_fusion_stats.get("adaptive_fusion_current_atlas_dice")),
            "adaptive_fusion_current_atlas_coverage": _extract_metrics.get("adaptive_fusion_current_atlas_coverage", _adaptive_fusion_stats.get("adaptive_fusion_current_atlas_coverage")),
            "adaptive_fusion_current_atlas_precision": _extract_metrics.get("adaptive_fusion_current_atlas_precision", _adaptive_fusion_stats.get("adaptive_fusion_current_atlas_precision")),
            "adaptive_fusion_current_uncertainty_mean": _extract_metrics.get("adaptive_fusion_current_uncertainty_mean", _adaptive_fusion_stats.get("adaptive_fusion_current_uncertainty_mean")),
            "adaptive_fusion_selected_score": _extract_metrics.get("adaptive_fusion_selected_score", _adaptive_fusion_stats.get("adaptive_fusion_selected_score")),
            "adaptive_fusion_current_score": _extract_metrics.get("adaptive_fusion_current_score", _adaptive_fusion_stats.get("adaptive_fusion_current_score")),
            "adaptive_fusion_volume_after_cm3": _extract_metrics.get("adaptive_fusion_volume_after_cm3", _adaptive_fusion_stats.get("adaptive_fusion_volume_after_cm3")),
            "adaptive_fusion_qc_path": _extract_metrics.get("adaptive_fusion_qc_path", _adaptive_fusion_stats.get("adaptive_fusion_qc_path")),
            "adaptive_fusion_probability_path": _extract_metrics.get("adaptive_fusion_probability_path", _adaptive_fusion_stats.get("adaptive_fusion_probability_path")),
            "adaptive_fusion_uncertainty_path": _extract_metrics.get("adaptive_fusion_uncertainty_path", _adaptive_fusion_stats.get("adaptive_fusion_uncertainty_path")),
            "quality_rating": _quality_rating,
            "mask_volume_cm3_from_extractor": _extract_metrics.get("mask_volume_cm3"),
            "total_time_s": round(time.time() - _t0, 1),
            "error": "",
        })

    except Exception as _e:
        traceback.print_exc()
        _m = {
            "subject": _sid,
            "dataset": ISEG_DATASET,
            "split": _rec.get("split"),
            "source_t1_path": str(_rec.get("t1_path", "")),
            "gold_mask_path": str(_rec.get("gs_mask_path", "")),
            "t2_path": str(_rec.get("t2_path", "")),
            "label_path": str(_rec.get("label_path", "")),
            "error": str(_e),
            "dice": float("nan"),
            "hd95": float("nan"),
            "assd": float("nan"),
            "total_time_s": round(time.time() - _t0, 1),
        }

    _results_iseg.append(_m)
    _iseg_write_progress("subject_finished", _sid, _idx, len(ISEG_SELECTED_RECORDS), _results_iseg)
    _d = _m.get("dice", float("nan"))
    _h = _m.get("hd95", float("nan"))
    _err = _m.get("error", "")
    if _err:
        print(f"  → ERREUR: {_err}")
    else:
        print(f"  → Dice={_d:.4f}  HD95={_h:.2f} mm  Vol={_m.get('vol_pred_cm3', float('nan')):.1f} cm3  GS={_m.get('vol_gs_cm3', float('nan')):.1f} cm3  AtlasVol={_m.get('atlas_volume_cm3', float('nan')):.1f} cm3  PedAdd={_m.get('pediatric_refinement_added_cm3', 0) or 0:.1f} cm3  Wide={_m.get('iseg_wide_refinement_used', False)}  t={_m.get('total_time_s', 0):.0f}s")

_df_iseg = pd.DataFrame(_results_iseg)
_df_iseg.to_csv(_ISEG_FINAL_CSV, index=False)
_iseg_write_progress("finished", total=len(ISEG_SELECTED_RECORDS), rows=_results_iseg)

_df_ok = _df_iseg[_df_iseg.get("error", "").fillna("").astype(str).str.len() == 0].copy() if "error" in _df_iseg.columns else _df_iseg.copy()
_df_ok["dice"] = pd.to_numeric(_df_ok.get("dice"), errors="coerce")
_df_ok = _df_ok.dropna(subset=["dice"])
print(f"\n═══ Résultats M1-iSeg2017 (N={len(_df_ok)}/{len(ISEG_SELECTED_RECORDS)} sujets) ═══")
if len(_df_ok):
    _cols = [c for c in ["subject", "dice", "hd95", "assd", "vol_pred_cm3", "vol_gs_cm3", "atlas_volume_cm3", "atlas_selection_score", "refinement_used", "pediatric_refinement_used", "pediatric_refinement_added_cm3", "pediatric_refinement_proposed_added_cm3", "iseg_wide_refinement_used", "iseg_wide_refinement_reason", "fallback_used", "adaptive_fusion_used", "adaptive_fusion_selected_candidate", "quality_rating"] if c in _df_ok.columns]
    print(_df_ok[_cols].to_string(index=False))
    print(f"\nDice  : {_df_ok.dice.mean():.4f} ± {_df_ok.dice.std():.4f}")
    print(f"HD95  : {_df_ok.hd95.mean():.2f} ± {_df_ok.hd95.std():.2f} mm")
    print(f"ASSD  : {_df_ok.assd.mean():.2f} ± {_df_ok.assd.std():.2f} mm")
    print(f"Volume prédit : {_df_ok.vol_pred_cm3.mean():.1f} ± {_df_ok.vol_pred_cm3.std():.1f} cm3")
    print(f"Volume gold   : {_df_ok.vol_gs_cm3.mean():.1f} ± {_df_ok.vol_gs_cm3.std():.1f} cm3")
print(f"\nRésultats sauvegardés: {_ISEG_FINAL_CSV}")
print(f"Suivi live: {_ISEG_LIVE_CSV} | {_ISEG_PROGRESS_JSON}")


### M1-ADULT-DATASETS-v4 — Autres datasets adultes gold-standard
Lance exactement le moteur adulte externe adaptatif figé v4 sur les datasets adultes disponibles hors iSeg-2017. Par défaut: smoke test sur `NFBS`, `MRBrainS13` et `SynthStripT1` adulte, sans LOO, sans SOTA-prior, avec réparation affine/orientation comme IBSR18/LPBA40.


In [ ]:
import os
os.environ["RUN_M1_ADULT_AVAILABLE_V4"] = "1"
os.environ["M1_ADULT_N_SUBJECTS"] = "5"
os.environ["M1_ADULT_RUN_DATASETS"] = "MRBrainS13,SynthStripT1,NFBS"
os.environ["RUN_M1_SYNTHSTRIPT1_V4"]="1"
os.environ["SYNTHSTRIPT1_N_SUBJECTS"]="5"

In [ ]:
# ============================================================================
# M1-ADULT-DATASETS-v4 - Autres datasets adultes sans LOO
# ============================================================================
import importlib
import os
from pathlib import Path

import nibabel as _m1ad_nib
import numpy as _m1ad_np
import pandas as pd

import scripts.adult_candidate_adjudicator as _m1ad_adjudicator
import scripts.adult_external_refinement as _m1ad_external_refinement
import scripts.adult_t1_adaptive_atlas as _m1ad_adaptive_atlas
import scripts.adult_t1_boundary_refinement as _m1ad_boundary_refinement
import scripts.adult_t1_strong_refinement as _m1ad_strong_refinement
import scripts.adaptive_mask_fusion as _m1ad_adaptive_fusion
import scripts.external_atlas_config as _m1ad_external_atlas_config
import scripts.m1_external_adaptive_runner as _m1ad_runner

importlib.reload(_m1ad_adjudicator)
importlib.reload(_m1ad_external_refinement)
importlib.reload(_m1ad_adaptive_atlas)
importlib.reload(_m1ad_boundary_refinement)
importlib.reload(_m1ad_strong_refinement)
importlib.reload(_m1ad_adaptive_fusion)
importlib.reload(_m1ad_external_atlas_config)
_m1ad_runner = importlib.reload(_m1ad_runner)
run_m1_external_adaptive_gold = _m1ad_runner.run_m1_external_adaptive_gold

# Datasets adultes exploitables dans datasets/manifests avec T1 + masque gold.
# Exclus volontairement:
# - IBSR18/LPBA40: cellules v4 dediees deja figées.
# - iSeg2017/BOBs: profils infantiles.
# - BrainMetShare/SynthStrip non-T1: pas de gold adulte T1 exploitable ici.
M1_ADULT_AVAILABLE_DATASETS = ("NFBS", "MRBrainS13", "SynthStripT1")
M1_ADULT_OUT_BASES = {
    "NFBS": Path("outputs/nfbs_m1_eval_v4_external_templates_frozen"),
    "MRBrainS13": Path("outputs/mrbrains13_m1_eval_v4_external_templates_frozen"),
    "SynthStripT1": Path("outputs/synthstript1_m1_eval_v4_external_templates_frozen"),
}

# Lancement:
#   RUN_M1_ADULT_AVAILABLE_V4=1 pour executer.
#   M1_ADULT_RUN_DATASETS=NFBS,MRBrainS13 pour limiter.
#   M1_ADULT_N_SUBJECTS=5 par defaut; mettre all/full/none pour full run.
RUN_M1_ADULT_AVAILABLE_V4 = os.environ.get("RUN_M1_ADULT_AVAILABLE_V4", "0").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}
_m1ad_requested = os.environ.get("M1_ADULT_RUN_DATASETS", "").strip()
M1_ADULT_RUN_DATASETS = tuple(
    ds.strip() for ds in _m1ad_requested.split(",") if ds.strip()
) or M1_ADULT_AVAILABLE_DATASETS

_m1ad_n_subjects = os.environ.get("M1_ADULT_N_SUBJECTS", "5").strip()
M1_ADULT_N_SUBJECTS = None if _m1ad_n_subjects.lower() in {"", "none", "all", "full"} else int(_m1ad_n_subjects)
_m1ad_min_smoke = os.environ.get("M1_ADULT_MIN_SMOKE_MEAN_DICE", "").strip()
M1_ADULT_MIN_SMOKE_MEAN_DICE = None if _m1ad_min_smoke.lower() in {"", "none", "off"} else float(_m1ad_min_smoke)
M1_ADULT_REPRO_SEED = int(os.environ.get("SAM_NEUROADAPT_SEED", "20260517"))
M1_ADULT_ARCHIVE_PREVIOUS = os.environ.get("M1_EXT_ARCHIVE_PREVIOUS", "1").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}

# Garde-fous explicites: aucun atlas interne, LOO, HD-BET, SynthStrip ou FSL ne guide M1.
M1_EXT_ATLAS_MODE = "external_adaptive_no_loo"
M1_EXT_USE_LOO_ATLAS_LIBRARY = False
M1_EXT_ENABLE_ATLAS_FALLBACK = False


def _m1ad_parse_subjects(dataset):
    raw = os.environ.get(f"{dataset.upper()}_SUBJECTS", "").strip()
    return [item.strip() for item in raw.split(",") if item.strip()] or None


def _m1ad_n_subjects_for(dataset):
    raw = os.environ.get(f"{dataset.upper()}_N_SUBJECTS", "").strip()
    if not raw:
        return M1_ADULT_N_SUBJECTS
    return None if raw.lower() in {"", "none", "all", "full"} else int(raw)


def _m1ad_bool(value, default=False):
    if value is None:
        return default
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on", "oui"}


def _m1ad_preflight_dataset(dataset):
    manifest = Path("datasets/manifests") / f"{dataset}.csv"
    if not manifest.exists():
        raise FileNotFoundError(f"Manifest introuvable pour {dataset}: {manifest}")
    df = pd.read_csv(manifest)
    if "profile" in df.columns:
        df = df[df["profile"].astype(str).eq("adult_t1")]
    if "has_gold" in df.columns:
        df = df[df["has_gold"].map(lambda value: _m1ad_bool(value, True))]
    if "qc_only" in df.columns:
        df = df[~df["qc_only"].map(lambda value: _m1ad_bool(value, False))]
    if df.empty:
        raise RuntimeError(f"{dataset}: aucun sujet adulte gold-standard exploitable dans {manifest}")

    path_errors = []
    for row in df.itertuples(index=False):
        t1 = Path(str(getattr(row, "input_path", "") or getattr(row, "t1_path", "")))
        gold = Path(str(getattr(row, "brain_mask_path", "") or getattr(row, "gold_mask_path", "")))
        if not t1.exists() or not gold.exists():
            path_errors.append((getattr(row, "subject", "?"), str(t1), str(gold)))
    if path_errors:
        sample = "; ".join(f"{sid}: t1={t1} gold={gold}" for sid, t1, gold in path_errors[:3])
        raise FileNotFoundError(f"{dataset}: chemins T1/gold manquants dans le manifest. Exemples: {sample}")

    first = df.iloc[0]
    t1 = Path(str(first.get("input_path") or first.get("t1_path")))
    gold = Path(str(first.get("brain_mask_path") or first.get("gold_mask_path")))
    t1_img = _m1ad_nib.load(str(t1))
    gold_img = _m1ad_nib.load(str(gold))
    same_shape = tuple(t1_img.shape[:3]) == tuple(gold_img.shape[:3])
    same_affine = _m1ad_np.allclose(t1_img.affine, gold_img.affine, atol=1e-3)
    volumes = pd.to_numeric(df.get("mask_volume_cm3"), errors="coerce").dropna()
    vol_msg = "n/a"
    if len(volumes):
        vol_msg = f"min={volumes.min():.1f}, median={volumes.median():.1f}, max={volumes.max():.1f} cm3"
    print(
        f"{dataset}: {len(df)} sujets adult_t1 gold | volumes {vol_msg} | "
        f"source/gold shape_ok={same_shape}, affine_ok={same_affine}, "
        f"axcodes={''.join(_m1ad_nib.aff2axcodes(t1_img.affine))}->{''.join(_m1ad_nib.aff2axcodes(gold_img.affine))}"
    )
    return manifest, len(df)


_m1ad_invalid = sorted(set(M1_ADULT_RUN_DATASETS) - set(M1_ADULT_AVAILABLE_DATASETS))
if _m1ad_invalid:
    raise ValueError(f"Datasets adultes v4 non supportes dans cette cellule: {_m1ad_invalid}. Supportes: {M1_ADULT_AVAILABLE_DATASETS}")

print("M1-ADULT-DATASETS-v4 preflight")
print(f"Datasets demandes: {M1_ADULT_RUN_DATASETS} | n_subjects={M1_ADULT_N_SUBJECTS} | seed={M1_ADULT_REPRO_SEED}")
_m1ad_manifests = {}
for _dataset in M1_ADULT_RUN_DATASETS:
    _manifest, _count = _m1ad_preflight_dataset(_dataset)
    _m1ad_manifests[_dataset] = _manifest

if RUN_M1_ADULT_AVAILABLE_V4:
    M1_ADULT_AVAILABLE_RESULTS = {}
    for _dataset in M1_ADULT_RUN_DATASETS:
        print(f"\n===== M1 adulte v4: {_dataset} =====")
        M1_ADULT_AVAILABLE_RESULTS[_dataset] = run_m1_external_adaptive_gold(
            globals(),
            dataset=_dataset,
            profile_filter="adult_t1",
            manifest=_m1ad_manifests[_dataset],
            out_base=M1_ADULT_OUT_BASES[_dataset],
            n_subjects=_m1ad_n_subjects_for(_dataset),
            subjects=_m1ad_parse_subjects(_dataset),
            force_clean_outputs=True,
            force_atlas_registration=True,
            force_extraction=True,
            reproducibility_seed=M1_ADULT_REPRO_SEED,
            archive_previous_outputs=M1_ADULT_ARCHIVE_PREVIOUS,
            deterministic_threads=True,
            min_smoke_mean_dice=M1_ADULT_MIN_SMOKE_MEAN_DICE,
            target_mean_dice=0.98,
        )
else:
    print("Cellule prete. Mettre RUN_M1_ADULT_AVAILABLE_V4=1 puis reexecuter pour lancer.")
    print("Watchers:")
    print("  python3 scripts/watch_m1_progress.py --dataset NFBS --version v4")
    print("  python3 scripts/watch_m1_progress.py --dataset MRBrainS13 --version v4")
    print("  python3 scripts/watch_m1_progress.py --dataset SynthStripT1 --version v4")


### M2-IBSR18-SOTA — Comparaison externe SOTA
Baselines FSL BET, SynthStrip et HD-BET sur exactement les sujets validés par M1-IBSR18.


In [ ]:
# ============================================================================
# M2-IBSR18-SOTA - comparaison externe robuste sur les memes sujets que M1
# Methodes: FSL BET | SynthStrip | HD-BET, selon les outils disponibles.
# Le gold sert uniquement au calcul final des metriques, jamais au guidage.
# ============================================================================
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

if "compute_metrics" not in globals():
    try:
        from scripts.m1_external_adaptive_runner import compute_metrics
    except Exception as exc:
        raise RuntimeError("compute_metrics introuvable. Executer les cellules de base ou importer scripts.m1_external_adaptive_runner.") from exc

try:
    from scipy import stats as _m2ibsr_stats
except Exception:
    _m2ibsr_stats = None
from scripts.sota_comparison_stats import add_sam_paired_stats

M2_IBSR18_DATASET = "IBSR18"
M2_IBSR18_M1_FINAL_CSV = Path("outputs/ibsr18_m1_eval_v4_external_templates_frozen/sam_neuroadapt_ibsr18_adultt1_results.csv")
M2_IBSR18_M1_LIVE_CSV = Path("outputs/ibsr18_m1_eval_v4_external_templates_frozen/sam_neuroadapt_ibsr18_adultt1_results_live.csv")
M2_IBSR18_OUT_BASE = Path("outputs/ibsr18_sota_eval_v4_external_templates_frozen")
M2_IBSR18_LOG_DIR = M2_IBSR18_OUT_BASE / "logs"
M2_IBSR18_PROGRESS_JSON = M2_IBSR18_OUT_BASE / "m2_ibsr18_sota_progress.json"
M2_IBSR18_LONG_CSV = M2_IBSR18_OUT_BASE / "all_methods_long.csv"
M2_IBSR18_SUMMARY_CSV = M2_IBSR18_OUT_BASE / "method_summary.csv"
M2_IBSR18_COMPARISON_CSV = M2_IBSR18_OUT_BASE / "comparison_table.csv"
M2_IBSR18_OUT_BASE.mkdir(parents=True, exist_ok=True)
M2_IBSR18_LOG_DIR.mkdir(parents=True, exist_ok=True)

# Execution control. Leave False for preflight only; set True to run the SOTA comparison.
RUN_M2_IBSR18_SOTA = True
M2_IBSR18_N_SUBJECTS = None          # None = all valid M1 rows; int = smoke subset
M2_IBSR18_SUBJECTS = None            # Example: ["IBSR_1_24", "IBSR_2_4"]
M2_IBSR18_REQUIRE_FINAL_M1 = False   # True = refuse live CSV / partial run
M2_IBSR18_REQUIRE_CLEAN_M1 = True    # keep rows with no error and valid Dice
M2_IBSR18_FORCE_RERUN = os.environ.get("M2_IBSR18_FORCE_RERUN", "0").strip().lower() in {"1", "true", "yes"}
M2_IBSR18_REQUIRE_ALL_TOOLS = True
M2_IBSR18_REQUIRE_ALL_COMPLETE = True

M2_IBSR18_SYNTHSTRIP_NO_CSF = os.environ.get("SYNTHSTRIP_NO_CSF", "0").strip().lower() in {"1", "true", "yes", "on"}
M2_IBSR18_HDBET_DEVICE = os.environ.get("HDBET_DEVICE")
M2_IBSR18_HDBET_DISABLE_TTA = os.environ.get("HDBET_DISABLE_TTA", "1").strip().lower() not in {"0", "false", "no"}
M2_IBSR18_HDBET_TIMEOUT_S = int(os.environ.get("HDBET_TIMEOUT_S", "900"))


def _m2ibsr_resolve_executable(env_name, candidates):
    env_value = os.environ.get(env_name)
    ordered = ([env_value] if env_value else []) + list(candidates)
    for cand in ordered:
        if not cand:
            continue
        text = str(cand)
        found = shutil.which(text) if os.path.sep not in text else text
        if found and Path(found).exists():
            return str(Path(found))
    return None


def _m2ibsr_command_prefix(executable):
    path = Path(str(executable))
    if path.exists():
        try:
            first = path.read_bytes()[:128].lower()
            if first.startswith(b"#!") and b"python" in first:
                return [sys.executable, str(path)]
        except Exception:
            pass
    return [str(executable)]


def _m2ibsr_resolve_m1_csv():
    final_exists = M2_IBSR18_M1_FINAL_CSV.exists()
    live_exists = M2_IBSR18_M1_LIVE_CSV.exists()
    if final_exists:
        return M2_IBSR18_M1_FINAL_CSV, "final"
    if live_exists and not M2_IBSR18_REQUIRE_FINAL_M1:
        return M2_IBSR18_M1_LIVE_CSV, "live"
    if live_exists and M2_IBSR18_REQUIRE_FINAL_M1:
        raise FileNotFoundError(f"CSV final M1 absent: {M2_IBSR18_M1_FINAL_CSV}. Live disponible: {M2_IBSR18_M1_LIVE_CSV}")
    raise FileNotFoundError(f"Aucun CSV M1 IBSR18 trouve: {M2_IBSR18_M1_FINAL_CSV} ou {M2_IBSR18_M1_LIVE_CSV}")


def _m2ibsr_progress_stage():
    progress = M2_IBSR18_M1_LIVE_CSV.parent / "m1_ibsr18_adultt1_progress.json"
    if not progress.exists():
        return None
    try:
        return json.loads(progress.read_text())
    except Exception:
        return None


def _m2ibsr_prepare_m1_dataframe(m1_csv, subjects=None, n_subjects=None, require_clean_m1=True):
    df = pd.read_csv(m1_csv)
    aliases = {
        "source_t1_path": ["source_t1_path", "input_path", "t1_path"],
        "gold_mask_path": ["gold_mask_path", "brain_mask_path", "mask_path"],
    }
    for target, candidates in aliases.items():
        if target not in df.columns:
            for cand in candidates:
                if cand in df.columns:
                    df[target] = df[cand]
                    break
    required = ["subject", "source_t1_path", "gold_mask_path"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"CSV M1 invalide, colonnes manquantes: {missing}")
    if require_clean_m1:
        err = df["error"].fillna("").astype(str).str.strip() if "error" in df.columns else pd.Series([""] * len(df), index=df.index)
        df = df[err == ""].copy()
        df = df[pd.to_numeric(df.get("dice"), errors="coerce").notna()].copy()
    if subjects:
        wanted = [str(s) for s in subjects]
        df = df[df["subject"].astype(str).isin(wanted)].copy()
        missing_subjects = sorted(set(wanted) - set(df["subject"].astype(str)))
        if missing_subjects:
            raise RuntimeError(f"Sujets demandes absents du CSV M1 propre: {missing_subjects}")
    if n_subjects is not None:
        df = df.head(int(n_subjects)).copy()
    if df.empty:
        raise RuntimeError(f"Aucun sujet M1 exploitable dans {m1_csv}")
    for col in ["source_t1_path", "gold_mask_path"]:
        absent = [str(v) for v in df[col].astype(str) if not Path(str(v)).exists()]
        if absent:
            raise FileNotFoundError(f"Chemins absents dans {col}: {absent[:5]}")
    return df.reset_index(drop=True)


def _m2ibsr_atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    tmp.replace(path)


def _m2ibsr_write_progress(stage, method=None, subject=None, index=None, total=None):
    _m2ibsr_atomic_json(M2_IBSR18_PROGRESS_JSON, {
        "updated": time.strftime("%Y-%m-%d %H:%M:%S"),
        "stage": stage,
        "dataset": M2_IBSR18_DATASET,
        "method": method,
        "subject": subject,
        "index": index,
        "total": total,
        "long_csv": str(M2_IBSR18_LONG_CSV),
    })


def _m2ibsr_run_command(cmd, timeout_s, log_prefix):
    log_prefix = Path(log_prefix)
    log_prefix.parent.mkdir(parents=True, exist_ok=True)
    meta = {"cmd": shlex.join([str(c) for c in cmd]), "returncode": None, "runtime_s": None, "stderr_tail": ""}
    log_prefix.with_suffix(".cmd.txt").write_text(meta["cmd"] + "\n")
    t0 = time.time()
    try:
        proc = subprocess.run([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=int(timeout_s))
        meta["returncode"] = int(proc.returncode)
        meta["runtime_s"] = round(time.time() - t0, 2)
        stdout, stderr = proc.stdout or "", proc.stderr or ""
    except subprocess.TimeoutExpired as exc:
        meta["returncode"] = -999
        meta["runtime_s"] = round(time.time() - t0, 2)
        stdout = exc.stdout.decode() if isinstance(exc.stdout, bytes) else (exc.stdout or "")
        stderr = exc.stderr.decode() if isinstance(exc.stderr, bytes) else (exc.stderr or "")
        stderr += f"\nTIMEOUT after {timeout_s}s"
    log_prefix.with_suffix(".stdout.txt").write_text(stdout)
    log_prefix.with_suffix(".stderr.txt").write_text(stderr)
    meta["stderr_tail"] = "\n".join(str(stderr).splitlines()[-10:])
    return meta


def _m2ibsr_canonicalize_mask(candidate_paths, canonical_mask):
    canonical_mask = Path(canonical_mask)
    for cand in candidate_paths:
        cand = Path(cand)
        if cand.exists():
            if cand.resolve() != canonical_mask.resolve():
                if canonical_mask.exists():
                    canonical_mask.unlink()
                shutil.move(str(cand), str(canonical_mask))
            return canonical_mask
    return None


def _m2ibsr_clean_subject(out_dir, subject, tag):
    if not M2_IBSR18_FORCE_RERUN:
        return
    for path in Path(out_dir).glob(f"{subject}_{tag}*"):
        if path.is_file():
            path.unlink()


_BET_BIN = _m2ibsr_resolve_executable("FSL_BET_BIN", [shutil.which("bet"), "/home/nabolitains/fsl/share/fsl/bin/bet"])
_SYNTHSTRIP_BIN = _m2ibsr_resolve_executable("SYNTHSTRIP_BIN", ["resources/tools/mri_synthstrip", "mri_synthstrip"])
_SYNTHSTRIP_MODEL = Path(os.environ.get("SYNTHSTRIP_WT", "resources/models/synthstrip.1.pt"))
_HDBET_BIN = _m2ibsr_resolve_executable("HDBET_BIN", [Path(".venv/bin/hd-bet"), Path(".venv/bin/hd_bet"), Path("venv_wsl/bin/hd-bet"), shutil.which("hd-bet"), shutil.which("hd_bet")])
if M2_IBSR18_HDBET_DEVICE is None:
    try:
        import torch as _m2ibsr_torch
        M2_IBSR18_HDBET_DEVICE = "cuda" if _m2ibsr_torch.cuda.is_available() else "cpu"
    except Exception:
        M2_IBSR18_HDBET_DEVICE = "cpu"


def _m2ibsr_run_bet(t1_path, out_dir, subject):
    _m2ibsr_clean_subject(out_dir, subject, "bet")
    out_brain = out_dir / f"{subject}_bet_brain.nii.gz"
    out_mask = out_dir / f"{subject}_bet_mask.nii.gz"
    if out_mask.exists() and not M2_IBSR18_FORCE_RERUN:
        return out_mask, {"runtime_s": 0.0, "cmd": "cache", "returncode": 0}
    cmd = [str(_BET_BIN), str(t1_path), str(out_brain), "-m", "-R"]
    meta = _m2ibsr_run_command(cmd, 180, M2_IBSR18_LOG_DIR / f"{subject}_fsl_bet")
    mask = _m2ibsr_canonicalize_mask([out_mask, out_dir / f"{subject}_bet_brain_mask.nii.gz"], out_mask)
    if meta["returncode"] != 0 and mask is None:
        meta["error"] = f"FSL BET failed rc={meta['returncode']}: {meta['stderr_tail']}"
    return mask, meta


def _m2ibsr_run_synthstrip(t1_path, out_dir, subject):
    _m2ibsr_clean_subject(out_dir, subject, "synthstrip")
    out_brain = out_dir / f"{subject}_synthstrip_brain.nii.gz"
    out_mask = out_dir / f"{subject}_synthstrip_mask.nii.gz"
    if out_mask.exists() and not M2_IBSR18_FORCE_RERUN:
        return out_mask, {"runtime_s": 0.0, "cmd": "cache", "returncode": 0}
    cmd = _m2ibsr_command_prefix(_SYNTHSTRIP_BIN) + ["-i", str(t1_path), "-o", str(out_brain), "-m", str(out_mask), "--model", str(_SYNTHSTRIP_MODEL)]
    if M2_IBSR18_SYNTHSTRIP_NO_CSF:
        cmd.append("--no-csf")
    meta = _m2ibsr_run_command(cmd, 600, M2_IBSR18_LOG_DIR / f"{subject}_synthstrip")
    mask = out_mask if out_mask.exists() else None
    if meta["returncode"] != 0 and mask is None:
        meta["error"] = f"SynthStrip failed rc={meta['returncode']}: {meta['stderr_tail']}"
    return mask, meta


def _m2ibsr_run_hdbet(t1_path, out_dir, subject):
    _m2ibsr_clean_subject(out_dir, subject, "hdbet")
    out_stub = out_dir / f"{subject}_hdbet.nii.gz"
    out_mask = out_dir / f"{subject}_hdbet_mask.nii.gz"
    if out_mask.exists() and not M2_IBSR18_FORCE_RERUN:
        return out_mask, {"runtime_s": 0.0, "cmd": "cache", "returncode": 0}
    cmd = [str(_HDBET_BIN), "-i", str(t1_path), "-o", str(out_stub), "-device", str(M2_IBSR18_HDBET_DEVICE), "--save_bet_mask", "--no_bet_image"]
    if M2_IBSR18_HDBET_DISABLE_TTA:
        cmd.append("--disable_tta")
    meta = _m2ibsr_run_command(cmd, M2_IBSR18_HDBET_TIMEOUT_S, M2_IBSR18_LOG_DIR / f"{subject}_hdbet")
    mask = _m2ibsr_canonicalize_mask([
        out_mask,
        out_dir / f"{subject}_hdbet_bet.nii.gz",
        out_dir / f"{subject}_hdbet_brain_bet.nii.gz",
        *sorted(out_dir.glob(f"{subject}_hdbet*_bet.nii.gz")),
    ], out_mask)
    if meta["returncode"] != 0 and mask is None:
        meta["error"] = f"HD-BET failed rc={meta['returncode']}: {meta['stderr_tail']}"
    elif mask is None:
        meta["error"] = "HD-BET finished but mask not found"
    return mask, meta


def _m2ibsr_available_baselines():
    baselines = {}
    missing = []
    if _BET_BIN:
        baselines["FSL BET"] = _m2ibsr_run_bet
    else:
        missing.append("FSL BET")
    if _SYNTHSTRIP_BIN and _SYNTHSTRIP_MODEL.exists() and _SYNTHSTRIP_MODEL.stat().st_size > 10_000_000:
        baselines["SynthStrip"] = _m2ibsr_run_synthstrip
    else:
        missing.append("SynthStrip")
    if _HDBET_BIN:
        baselines["HD-BET"] = _m2ibsr_run_hdbet
    else:
        missing.append("HD-BET")
    if missing and M2_IBSR18_REQUIRE_ALL_TOOLS:
        raise FileNotFoundError("Outils SOTA manquants: " + ", ".join(missing))
    if not baselines:
        raise FileNotFoundError("Aucun baseline SOTA disponible. Installer FSL BET, SynthStrip ou HD-BET.")
    return baselines, missing


def _m2ibsr_mean_std(values, digits=4):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().values
    if len(vals) == 0:
        return "N/D"
    if len(vals) == 1:
        return f"{vals[0]:.{digits}f} +/- 0.{'0' * digits}"
    return f"{np.mean(vals):.{digits}f} +/- {np.std(vals, ddof=1):.{digits}f}"


def run_m2_ibsr18_sota():
    m1_csv, csv_kind = _m2ibsr_resolve_m1_csv()
    df_m1 = _m2ibsr_prepare_m1_dataframe(
        m1_csv,
        subjects=M2_IBSR18_SUBJECTS,
        n_subjects=M2_IBSR18_N_SUBJECTS,
        require_clean_m1=M2_IBSR18_REQUIRE_CLEAN_M1,
    )
    baselines, missing_tools = _m2ibsr_available_baselines()
    progress = _m2ibsr_progress_stage()
    print("M2-IBSR18-SOTA preflight")
    print(f"CSV M1      : {m1_csv} ({csv_kind}) | sujets={len(df_m1)}")
    if progress:
        print(f"M1 stage    : {progress.get('stage')} | current={progress.get('current_subject')} | {progress.get('current_index')}/{progress.get('total')}")
    print(f"FSL BET     : {_BET_BIN}")
    print(f"SynthStrip  : bin={_SYNTHSTRIP_BIN} | model={_SYNTHSTRIP_MODEL} | no_csf={M2_IBSR18_SYNTHSTRIP_NO_CSF}")
    print(f"HD-BET      : {_HDBET_BIN} | device={M2_IBSR18_HDBET_DEVICE} | TTA={'off' if M2_IBSR18_HDBET_DISABLE_TTA else 'on'}")
    if missing_tools:
        print("Outils ignores: " + ", ".join(missing_tools))

    if M2_IBSR18_FORCE_RERUN:
        for path in [M2_IBSR18_LONG_CSV, M2_IBSR18_SUMMARY_CSV, M2_IBSR18_COMPARISON_CSV, M2_IBSR18_PROGRESS_JSON]:
            if path.exists():
                path.unlink()
        for method in baselines:
            for path in [M2_IBSR18_OUT_BASE / f"{method.replace(' ', '_')}_results.csv", M2_IBSR18_OUT_BASE / f"{method.replace(' ', '_')}_results_live.csv"]:
                if path.exists():
                    path.unlink()

    previous_lookup = {}
    if M2_IBSR18_LONG_CSV.exists() and not M2_IBSR18_FORCE_RERUN:
        try:
            prev = pd.read_csv(M2_IBSR18_LONG_CSV)
            prev["error"] = prev.get("error", "").fillna("").astype(str)
            for cached_row in prev[prev.get("method", "").isin(baselines.keys())].to_dict("records"):
                key = (str(cached_row.get("method", "")), str(cached_row.get("subject", "")))
                mask = Path(str(cached_row.get("mask_path", "")))
                if key[0] and key[1] and mask.exists() and cached_row.get("error", "").strip() in {"", "nan", "None"}:
                    previous_lookup[key] = cached_row
        except Exception as exc:
            print(f"Cache SOTA ignore pour relance propre: {exc}")

    long_rows = []
    for row in df_m1.itertuples(index=False):
        rd = row._asdict()
        long_rows.append({
            "dataset": M2_IBSR18_DATASET,
            "subject": str(rd["subject"]),
            "method": "SAM-NeuroAdapt",
            "source_t1_path": str(rd["source_t1_path"]),
            "gold_mask_path": str(rd["gold_mask_path"]),
            "mask_path": str(rd.get("output_mask_path", "")),
            "runtime_s": rd.get("total_time_s", np.nan),
            "dice": rd.get("dice", np.nan),
            "hd95": rd.get("hd95", np.nan),
            "assd": rd.get("assd", np.nan),
            "vol_pred_cm3": rd.get("vol_pred_cm3", np.nan),
            "vol_gs_cm3": rd.get("vol_gs_cm3", np.nan),
            "qc_status": rd.get("qc_status", ""),
            "diagnostic_flags": rd.get("diagnostic_flags", ""),
            "error": "",
        })

    baseline_rows = {method: [] for method in baselines}
    total = len(df_m1)
    _m2ibsr_write_progress("started", total=total)
    for index, row in enumerate(df_m1.itertuples(index=False), 1):
        rd = row._asdict()
        subject = str(rd["subject"])
        safe_subject = re.sub(r"[^A-Za-z0-9_.-]+", "_", subject).strip("_")
        t1_path = Path(str(rd["source_t1_path"]))
        gold_path = Path(str(rd["gold_mask_path"]))
        subject_dir = M2_IBSR18_OUT_BASE / safe_subject
        subject_dir.mkdir(parents=True, exist_ok=True)
        for method, runner in baselines.items():
            _m2ibsr_write_progress("running", method, subject, index, total)
            cached_result = previous_lookup.get((method, subject))
            if cached_result is not None:
                result = dict(cached_result)
                baseline_rows[method].append(result)
                long_rows.append(result)
                pd.DataFrame(baseline_rows[method]).to_csv(M2_IBSR18_OUT_BASE / f"{method.replace(' ', '_')}_results_live.csv", index=False)
                pd.DataFrame(long_rows).to_csv(M2_IBSR18_LONG_CSV, index=False)
                dice_value = pd.to_numeric(pd.Series([result.get("dice")]), errors="coerce").iloc[0]
                dice_text = f"{dice_value:.4f}" if pd.notna(dice_value) else "ERROR"
                print(f"{method} | {subject}: Dice={dice_text} HD95={result.get('hd95', np.nan)} cache")
                continue
            result = {
                "dataset": M2_IBSR18_DATASET,
                "subject": subject,
                "method": method,
                "source_t1_path": str(t1_path),
                "gold_mask_path": str(gold_path),
                "mask_path": "",
                "runtime_s": np.nan,
                "returncode": np.nan,
                "error": "",
            }
            try:
                mask_path, meta = runner(t1_path, subject_dir, safe_subject)
                result["runtime_s"] = meta.get("runtime_s", np.nan)
                result["returncode"] = meta.get("returncode", np.nan)
                result["cmd"] = meta.get("cmd", "")
                if meta.get("error"):
                    result["error"] = meta["error"]
                if mask_path and Path(mask_path).exists():
                    result["mask_path"] = str(mask_path)
                    result.update(compute_metrics(mask_path, gold_path))
                elif not result["error"]:
                    result["error"] = "mask_not_found"
            except Exception as exc:
                result["error"] = str(exc)
            baseline_rows[method].append(result)
            long_rows.append(result)
            pd.DataFrame(baseline_rows[method]).to_csv(M2_IBSR18_OUT_BASE / f"{method.replace(' ', '_')}_results_live.csv", index=False)
            pd.DataFrame(long_rows).to_csv(M2_IBSR18_LONG_CSV, index=False)
            dice_value = pd.to_numeric(pd.Series([result.get("dice")]), errors="coerce").iloc[0]
            dice_text = f"{dice_value:.4f}" if pd.notna(dice_value) else "ERROR"
            print(f"{method} | {subject}: Dice={dice_text} HD95={result.get('hd95', np.nan)} err={result.get('error', '')}")

    for method, rows in baseline_rows.items():
        pd.DataFrame(rows).to_csv(M2_IBSR18_OUT_BASE / f"{method.replace(' ', '_')}_results.csv", index=False)
    long_df = pd.DataFrame(long_rows)
    for col in ["dice", "hd95", "assd", "vol_pred_cm3", "vol_gs_cm3", "runtime_s"]:
        if col in long_df.columns:
            long_df[col] = pd.to_numeric(long_df[col], errors="coerce")

    ok = long_df[(long_df["error"].fillna("").astype(str).str.strip() == "") & long_df["dice"].notna()].copy()
    summary = (
        ok.groupby("method")
        .agg(
            n=("dice", "count"),
            dice_mean=("dice", "mean"),
            dice_std=("dice", "std"),
            dice_median=("dice", "median"),
            dice_min=("dice", "min"),
            hd95_mean=("hd95", "mean"),
            assd_mean=("assd", "mean"),
            runtime_mean_s=("runtime_s", "mean"),
        )
        .sort_values("dice_mean", ascending=False)
        .reset_index()
    )
    long_df, summary = add_sam_paired_stats(long_df, summary, baseline_method="SAM-NeuroAdapt")
    long_df.to_csv(M2_IBSR18_LONG_CSV, index=False)
    summary.to_csv(M2_IBSR18_SUMMARY_CSV, index=False)

    sam = long_df[long_df["method"].eq("SAM-NeuroAdapt")][["subject", "dice", "hd95", "assd"]].rename(columns={"dice": "sam_dice", "hd95": "sam_hd95", "assd": "sam_assd"})
    comparison_rows = []
    for method in summary["method"].astype(str):
        dfm = long_df[long_df["method"].eq(method)].copy()
        paired = sam.merge(dfm[["subject", "dice", "hd95"]], on="subject", how="inner").dropna(subset=["sam_dice", "dice"])
        delta = float((paired["dice"] - paired["sam_dice"]).mean()) if len(paired) else np.nan
        p_text = "N/D"
        if method != "SAM-NeuroAdapt" and len(paired) >= 2 and _m2ibsr_stats is not None:
            try:
                p_text = f"{float(_m2ibsr_stats.wilcoxon(paired['sam_dice'], paired['dice'], zero_method='wilcox').pvalue):.3g}"
            except Exception:
                p_text = "N/D"
        comparison_rows.append({
            "method": method,
            "dice": _m2ibsr_mean_std(dfm["dice"], 4),
            "hd95_mm": _m2ibsr_mean_std(dfm["hd95"], 2),
            "assd_mm": _m2ibsr_mean_std(dfm["assd"], 2),
            "n": int(pd.to_numeric(dfm["dice"], errors="coerce").notna().sum()),
            "paired_with_sam": int(len(paired)),
            "delta_dice_vs_sam": "--" if method == "SAM-NeuroAdapt" else (f"{delta:+.4f}" if np.isfinite(delta) else "N/D"),
            "p_wilcoxon_dice": "--" if method == "SAM-NeuroAdapt" else p_text,
        })
    comparison = pd.DataFrame(comparison_rows)
    comparison.to_csv(M2_IBSR18_COMPARISON_CSV, index=False)
    _m2ibsr_write_progress("finished", total=total)

    if M2_IBSR18_REQUIRE_ALL_COMPLETE:
        failures = long_df[(long_df["method"] != "SAM-NeuroAdapt") & (long_df["error"].fillna("").astype(str).str.strip() != "")]
        if len(failures):
            raise RuntimeError("M2-IBSR18-SOTA incomplet: " + "; ".join(f"{r.method}/{r.subject}: {r.error}" for r in failures.itertuples(index=False)))

    print(f"\nM2-IBSR18-SOTA termine: {M2_IBSR18_LONG_CSV}")
    display(comparison)
    return {
        "long": long_df,
        "summary": summary,
        "comparison": comparison,
        "long_csv": M2_IBSR18_LONG_CSV,
        "summary_csv": M2_IBSR18_SUMMARY_CSV,
        "comparison_csv": M2_IBSR18_COMPARISON_CSV,
        "out_base": M2_IBSR18_OUT_BASE,
    }


_m2ibsr_m1_csv, _m2ibsr_csv_kind = _m2ibsr_resolve_m1_csv()
_m2ibsr_preview = _m2ibsr_prepare_m1_dataframe(
    _m2ibsr_m1_csv,
    subjects=M2_IBSR18_SUBJECTS,
    n_subjects=M2_IBSR18_N_SUBJECTS,
    require_clean_m1=M2_IBSR18_REQUIRE_CLEAN_M1,
)
_m2ibsr_baselines, _m2ibsr_missing = _m2ibsr_available_baselines()
print("M2-IBSR18-SOTA pret")
print(f"CSV M1: {_m2ibsr_m1_csv} ({_m2ibsr_csv_kind}) | sujets exploitables={len(_m2ibsr_preview)}")
print("Baselines disponibles: " + ", ".join(_m2ibsr_baselines.keys()))
if _m2ibsr_missing:
    print("Baselines ignorees: " + ", ".join(_m2ibsr_missing))
print("Mettre RUN_M2_IBSR18_SOTA=True puis reexecuter cette cellule pour lancer la comparaison.")

if RUN_M2_IBSR18_SOTA:
    M2_IBSR18_SOTA_RESULTS = run_m2_ibsr18_sota()


### M2-LPBA40-SOTA — Comparaison externe SOTA
Baselines FSL BET, SynthStrip et HD-BET sur exactement les sujets validés par M1-LPBA40.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# M2-LPBA40-SOTA — Baselines externes sur les mêmes sujets que M1-LPBA40
# Méthodes: FSL BET | SynthStrip | HD-BET
# Entrées/gold repris depuis le CSV M1 externe pour garantir un appariement strict.
# ═══════════════════════════════════════════════════════════════════════════
import sys, os, json, time, shlex, shutil, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
from scipy import stats
warnings.filterwarnings("ignore")

if "compute_metrics" not in globals():
    raise RuntimeError("Exécuter d'abord la cellule B3 / compute_metrics.")

M2_EXT_DATASET = "LPBA40"
M2_EXT_M1_CSV = Path("outputs/lpba40_m1_eval_v4_external_templates_frozen/sam_neuroadapt_lpba40_adultt1_results.csv")
M2_EXT_M1_LIVE_CSV = Path("outputs/lpba40_m1_eval_v4_external_templates_frozen/sam_neuroadapt_lpba40_adultt1_results_live.csv")
M2_EXT_OUT_BASE = Path("outputs/lpba40_sota_eval_v4_external_templates_frozen")
M2_EXT_LOG_DIR = M2_EXT_OUT_BASE / "logs"
M2_EXT_PROGRESS_JSON = M2_EXT_OUT_BASE / "m2_lpba40_sota_progress.json"
M2_EXT_LONG_CSV = M2_EXT_OUT_BASE / "all_methods_long.csv"
M2_EXT_COMPARISON_CSV = M2_EXT_OUT_BASE / "comparison_table.csv"
M2_EXT_OUT_BASE.mkdir(parents=True, exist_ok=True)
M2_EXT_LOG_DIR.mkdir(parents=True, exist_ok=True)

# None = exactement les sujets validés dans M1 externe; int = sous-échantillon; liste = sujets exacts.
M2_EXT_N_SUBJECTS = None
M2_EXT_SUBJECTS = None
REQUIRE_COMPLETE_EXTERNAL_M1 = True
FORCE_EXTERNAL_BASELINE_RERUN = True
REQUIRE_ALL_EXTERNAL_BASELINES_COMPLETE = True

HDBET_DEVICE = os.environ.get("HDBET_DEVICE")
HDBET_DISABLE_TTA = os.environ.get("HDBET_DISABLE_TTA", "0") == "1"
HDBET_TIMEOUT_S = int(os.environ.get("HDBET_TIMEOUT_S", "900"))


def _m2ext_resolve_executable(env_name, candidates):
    env_value = os.environ.get(env_name)
    ordered = ([env_value] if env_value else []) + list(candidates)
    for cand in ordered:
        if not cand:
            continue
        cand = str(cand)
        found = shutil.which(cand) if os.path.sep not in cand else cand
        if found and Path(found).exists():
            return str(Path(found))
    return None


_BET_BIN = _m2ext_resolve_executable("FSL_BET_BIN", [
    shutil.which("bet"),
    "/home/nabolitains/fsl/share/fsl/bin/bet",
])
_SYNTHSTRIP_BIN = _m2ext_resolve_executable("SYNTHSTRIP_BIN", [
    Path("resources/tools/mri_synthstrip"),
    shutil.which("mri_synthstrip"),
])
_SYNTHSTRIP_WT = Path(os.environ.get("SYNTHSTRIP_WT", "resources/models/synthstrip.1.pt"))
_HDBET_BIN = _m2ext_resolve_executable("HDBET_BIN", [
    Path(".venv/bin/hd-bet"),
    Path(".venv/bin/hd_bet"),
    Path("venv_wsl/bin/hd-bet"),
    shutil.which("hd-bet"),
    shutil.which("hd_bet"),
    "/mnt/d/CHARKO-RESEARCH/FEDERATED-LEARNING/V2/venv_wsl/bin/hd-bet",
])
if HDBET_DEVICE is None:
    try:
        import torch
        HDBET_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        HDBET_DEVICE = "cuda"

print(f"═══ M2-LPBA40-SOTA — Préflight ═══")
print(f"FSL BET    : {_BET_BIN}")
print(f"SynthStrip : bin={_SYNTHSTRIP_BIN} | weights={_SYNTHSTRIP_WT}")
print(f"HD-BET     : {_HDBET_BIN} | device={HDBET_DEVICE} | TTA={'off' if HDBET_DISABLE_TTA else 'on'}")
if _BET_BIN is None:
    raise FileNotFoundError("FSL BET introuvable. Définir FSL_BET_BIN ou corriger l'installation FSL.")
if _SYNTHSTRIP_BIN is None or not _SYNTHSTRIP_WT.exists() or _SYNTHSTRIP_WT.stat().st_size < 10_000_000:
    raise FileNotFoundError("SynthStrip introuvable. Attendu: resources/tools/mri_synthstrip et resources/models/synthstrip.1.pt")
if _HDBET_BIN is None:
    raise FileNotFoundError("HD-BET introuvable. Définir HDBET_BIN ou installer hd-bet dans venv_wsl.")

if not M2_EXT_M1_CSV.exists():
    raise FileNotFoundError(f"Résultats M1-LPBA40 finaux introuvables: {M2_EXT_M1_CSV}. Relancer la cellule M1-LPBA40 jusqu'à la fin.")
_df_m1_ext = pd.read_csv(M2_EXT_M1_CSV)
if "subject" not in _df_m1_ext.columns:
    raise RuntimeError(f"CSV M1-LPBA40 invalide: colonne subject absente")
_df_m1_ext["dice"] = pd.to_numeric(_df_m1_ext.get("dice"), errors="coerce")
_errors = _df_m1_ext[_df_m1_ext.get("error", "").fillna("").astype(str).str.len() > 0] if "error" in _df_m1_ext.columns else pd.DataFrame()
_bad = _df_m1_ext[_df_m1_ext["dice"].isna()]
if REQUIRE_COMPLETE_EXTERNAL_M1 and (len(_errors) or len(_bad)):
    raise RuntimeError(f"M2-LPBA40-SOTA arrêté: M1 externe non propre. errors={len(_errors)}, dice_nan={len(_bad)}")

_df_m1_ext = _df_m1_ext.sort_values("subject").reset_index(drop=True)
if M2_EXT_SUBJECTS:
    wanted = [str(s) for s in M2_EXT_SUBJECTS]
    _df_m1_ext = _df_m1_ext[_df_m1_ext["subject"].astype(str).isin(wanted)].copy()
    missing = sorted(set(wanted) - set(_df_m1_ext["subject"].astype(str)))
    if missing:
        raise RuntimeError(f"Sujets demandés absents du M1-LPBA40: {missing}")
elif M2_EXT_N_SUBJECTS is not None:
    _df_m1_ext = _df_m1_ext.head(int(M2_EXT_N_SUBJECTS)).copy()
if len(_df_m1_ext) == 0:
    raise RuntimeError(f"Aucun sujet M1-LPBA40 disponible pour SOTA")

for col in ["source_t1_path", "gold_mask_path"]:
    if col not in _df_m1_ext.columns:
        raise RuntimeError(f"M1-LPBA40 doit contenir la colonne {col}")
for _, row in _df_m1_ext.iterrows():
    if not Path(str(row["source_t1_path"])).exists() or not Path(str(row["gold_mask_path"])).exists():
        raise FileNotFoundError(f"Source/gold absent pour {row['subject']}")


def _m2ext_atomic_json(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    tmp.replace(path)


def _m2ext_write_progress(stage, method=None, subject=None, idx=None, total=None):
    _m2ext_atomic_json(M2_EXT_PROGRESS_JSON, {
        "stage": stage,
        "dataset": M2_EXT_DATASET,
        "method": method,
        "subject": subject,
        "index": idx,
        "total": total,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "results_csv": str(M2_EXT_LONG_CSV),
    })


def _m2ext_clean_subject(out_dir, subj_id, label):
    if not FORCE_EXTERNAL_BASELINE_RERUN:
        return
    for pattern in [f"{subj_id}_{label}*.nii.gz", f"{subj_id}_{label}*.json", "dataset.json", "plans.json", "predict_from_raw_data_args.json"]:
        for p in Path(out_dir).glob(pattern):
            try:
                p.unlink()
            except FileNotFoundError:
                pass


def _m2ext_run_command(cmd, timeout_s, log_prefix):
    log_prefix = Path(log_prefix)
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)
        runtime = time.time() - t0
        stdout = proc.stdout or ""
        stderr = proc.stderr or ""
        returncode = int(proc.returncode)
    except subprocess.TimeoutExpired as e:
        runtime = time.time() - t0
        stdout = e.stdout.decode() if isinstance(e.stdout, bytes) else (e.stdout or "")
        stderr = e.stderr.decode() if isinstance(e.stderr, bytes) else (e.stderr or "")
        stderr += f"\nTIMEOUT after {timeout_s}s"
        returncode = 124
    log_prefix.with_suffix(".stdout.log").write_text(stdout)
    log_prefix.with_suffix(".stderr.log").write_text(stderr)
    return {
        "cmd": shlex.join([str(c) for c in cmd]),
        "returncode": returncode,
        "runtime_s": round(runtime, 2),
        "stdout_log": str(log_prefix.with_suffix(".stdout.log")),
        "stderr_log": str(log_prefix.with_suffix(".stderr.log")),
        "stderr_tail": "\n".join(stderr.splitlines()[-8:]),
    }


def _m2ext_canonicalize_mask(candidate_paths, canonical_mask):
    for cand in candidate_paths:
        cand = Path(cand)
        if cand.exists():
            canonical_mask = Path(canonical_mask)
            if cand.resolve() != canonical_mask.resolve():
                if canonical_mask.exists():
                    canonical_mask.unlink()
                shutil.move(str(cand), str(canonical_mask))
            return canonical_mask
    return None


def _m2ext_repair_mask_spatial_reference(mask_path, reference_path):
    mask_path = Path(mask_path)
    reference_path = Path(reference_path)
    info = {"spatial_repair_checked": False, "spatial_repair_used": False, "spatial_repair_reason": "not_checked"}
    if not mask_path.exists() or not reference_path.exists():
        info["spatial_repair_reason"] = "missing_mask_or_reference"
        return info
    try:
        mask_img = nib.load(str(mask_path))
        ref_img = nib.load(str(reference_path))
    except Exception as exc:
        info["spatial_repair_reason"] = f"load_failed:{exc}"
        return info
    info["spatial_repair_checked"] = True
    info["spatial_repair_old_axcodes"] = "".join(nib.aff2axcodes(mask_img.affine))
    info["spatial_repair_ref_axcodes"] = "".join(nib.aff2axcodes(ref_img.affine))
    if mask_img.shape[:3] != ref_img.shape[:3]:
        info["spatial_repair_reason"] = "shape_mismatch"
        return info
    mask_zooms = tuple(float(v) for v in mask_img.header.get_zooms()[:3])
    ref_zooms = tuple(float(v) for v in ref_img.header.get_zooms()[:3])
    if not np.allclose(mask_zooms, ref_zooms, atol=1e-3):
        info["spatial_repair_reason"] = "spacing_mismatch"
        return info
    if np.allclose(mask_img.affine, ref_img.affine, atol=1e-4):
        info["spatial_repair_reason"] = "already_aligned"
        return info
    data = np.asanyarray(mask_img.dataobj).copy()
    header = mask_img.header.copy()
    header.set_data_dtype(mask_img.get_data_dtype())
    fixed = nib.Nifti1Image(data, ref_img.affine, header)
    qform_code = int(ref_img.header["qform_code"]) if int(ref_img.header["qform_code"]) > 0 else 1
    sform_code = int(ref_img.header["sform_code"]) if int(ref_img.header["sform_code"]) > 0 else 1
    fixed.set_qform(ref_img.affine, code=qform_code)
    fixed.set_sform(ref_img.affine, code=sform_code)
    fixed.set_data_dtype(mask_img.get_data_dtype())
    nib.save(fixed, str(mask_path))
    info["spatial_repair_used"] = True
    info["spatial_repair_reason"] = "affine_repaired"
    info["spatial_repair_new_axcodes"] = "".join(nib.aff2axcodes(ref_img.affine))
    return info


def _m2ext_run_bet(t1_path, out_dir, subj_id):
    _m2ext_clean_subject(out_dir, subj_id, "bet")
    out_brain = out_dir / f"{subj_id}_bet_brain.nii.gz"
    out_mask = out_dir / f"{subj_id}_bet_mask.nii.gz"
    if out_mask.exists() and not FORCE_EXTERNAL_BASELINE_RERUN:
        return out_mask, {"runtime_s": 0.0, "cmd": "cache"}
    cmd = [_BET_BIN, str(t1_path), str(out_brain), "-m", "-R"]
    meta = _m2ext_run_command(cmd, 180, M2_EXT_LOG_DIR / f"{subj_id}_fsl_bet")
    mask = _m2ext_canonicalize_mask([out_mask, out_dir / f"{subj_id}_bet_brain_mask.nii.gz"], out_mask)
    if meta["returncode"] != 0 and mask is None:
        meta["error"] = f"FSL BET failed rc={meta['returncode']}: {meta['stderr_tail']}"
    return mask, meta


def _m2ext_run_synthstrip(t1_path, out_dir, subj_id):
    _m2ext_clean_subject(out_dir, subj_id, "synthstrip")
    out_brain = out_dir / f"{subj_id}_synthstrip_brain.nii.gz"
    out_mask = out_dir / f"{subj_id}_synthstrip_mask.nii.gz"
    if out_mask.exists() and not FORCE_EXTERNAL_BASELINE_RERUN:
        return out_mask, {"runtime_s": 0.0, "cmd": "cache"}
    cmd = [str(_SYNTHSTRIP_BIN), "-i", str(t1_path), "-o", str(out_brain), "-m", str(out_mask), "--model", str(_SYNTHSTRIP_WT)]
    meta = _m2ext_run_command(cmd, 600, M2_EXT_LOG_DIR / f"{subj_id}_synthstrip")
    mask = out_mask if out_mask.exists() else None
    if meta["returncode"] != 0 and mask is None:
        meta["error"] = f"SynthStrip failed rc={meta['returncode']}: {meta['stderr_tail']}"
    return mask, meta


def _m2ext_run_hdbet(t1_path, out_dir, subj_id):
    _m2ext_clean_subject(out_dir, subj_id, "hdbet")
    out_stub = out_dir / f"{subj_id}_hdbet.nii.gz"
    out_mask = out_dir / f"{subj_id}_hdbet_mask.nii.gz"
    if out_mask.exists() and not FORCE_EXTERNAL_BASELINE_RERUN:
        return out_mask, {"runtime_s": 0.0, "cmd": "cache"}
    cmd = [_HDBET_BIN, "-i", str(t1_path), "-o", str(out_stub), "-device", HDBET_DEVICE, "--save_bet_mask", "--no_bet_image"]
    if HDBET_DISABLE_TTA:
        cmd.append("--disable_tta")
    meta = _m2ext_run_command(cmd, HDBET_TIMEOUT_S, M2_EXT_LOG_DIR / f"{subj_id}_hdbet")
    mask = _m2ext_canonicalize_mask([
        out_mask,
        out_dir / f"{subj_id}_hdbet_bet.nii.gz",
        out_dir / f"{subj_id}_hdbet_brain_bet.nii.gz",
        *sorted(out_dir.glob(f"{subj_id}_hdbet*_bet.nii.gz")),
    ], out_mask)
    if meta["returncode"] != 0 and mask is None:
        meta["error"] = f"HD-BET failed rc={meta['returncode']}: {meta['stderr_tail']}"
    elif mask is None:
        meta["error"] = "HD-BET finished but mask not found; expected suffix *_bet.nii.gz"
    return mask, meta


BASELINES_EXT = {
    "FSL BET": _m2ext_run_bet,
    "SynthStrip": _m2ext_run_synthstrip,
    "HD-BET": _m2ext_run_hdbet,
}

if FORCE_EXTERNAL_BASELINE_RERUN:
    for p in [M2_EXT_LONG_CSV, M2_EXT_COMPARISON_CSV, M2_EXT_PROGRESS_JSON]:
        if p.exists():
            p.unlink()
    for name in BASELINES_EXT:
        for p in [M2_EXT_OUT_BASE / f"{name.replace(' ', '_')}_results.csv", M2_EXT_OUT_BASE / f"{name.replace(' ', '_')}_results_live.csv"]:
            if p.exists():
                p.unlink()

print(f"\n═══ M2-LPBA40-SOTA — Baselines externes sur {len(_df_m1_ext)} sujets ═══\n")
_m2ext_write_progress("started", total=len(_df_m1_ext))
_bl_results = {name: [] for name in BASELINES_EXT}
_long_rows = []

for idx, row in enumerate(_df_m1_ext.itertuples(index=False), 1):
    sid = str(row.subject)
    safe_sid = sid.replace("/", "_")
    t1_path = Path(str(row.source_t1_path))
    gold_path = Path(str(row.gold_mask_path))
    bl_dir = M2_EXT_OUT_BASE / safe_sid
    bl_dir.mkdir(parents=True, exist_ok=True)
    for bl_name, bl_fn in BASELINES_EXT.items():
        _m2ext_write_progress("running", bl_name, sid, idx, len(_df_m1_ext))
        try:
            mask, meta = bl_fn(t1_path, bl_dir, safe_sid)
            if mask and Path(str(mask)).exists():
                repair = _m2ext_repair_mask_spatial_reference(mask, gold_path)
                metrics = compute_metrics(mask, gold_path)
                metrics.update({"subject": sid, "dataset": M2_EXT_DATASET, "method": bl_name, "mask_path": str(mask), "error": ""})
                metrics.update(repair)
            else:
                metrics = {"subject": sid, "dataset": M2_EXT_DATASET, "method": bl_name, "dice": float("nan"), "hd95": float("nan"), "assd": float("nan"), "vol_pred_cm3": float("nan"), "vol_gs_cm3": float("nan"), "mask_path": "", "error": meta.get("error", "mask missing")}
            metrics.update({k: meta.get(k) for k in ["cmd", "returncode", "runtime_s", "stdout_log", "stderr_log"]})
        except Exception as exc:
            metrics = {"subject": sid, "dataset": M2_EXT_DATASET, "method": bl_name, "dice": float("nan"), "hd95": float("nan"), "assd": float("nan"), "vol_pred_cm3": float("nan"), "vol_gs_cm3": float("nan"), "mask_path": "", "error": str(exc), "cmd": "", "returncode": None, "runtime_s": None, "stdout_log": "", "stderr_log": ""}
        _bl_results[bl_name].append(metrics)
        _long_rows.append(metrics)
        pd.DataFrame(_bl_results[bl_name]).to_csv(M2_EXT_OUT_BASE / f"{bl_name.replace(' ', '_')}_results_live.csv", index=False)
        pd.DataFrame(_long_rows).to_csv(M2_EXT_LONG_CSV, index=False)
        d = pd.to_numeric(pd.Series([metrics.get("dice")]), errors="coerce").iloc[0]
        d_txt = f"{d:.4f}" if pd.notna(d) else "ERREUR"
        print(f"  [{bl_name}] {sid} → Dice={d_txt}  t={metrics.get('runtime_s')}s  err={metrics.get('error','')}")

for bl_name, rows in _bl_results.items():
    pd.DataFrame(rows).to_csv(M2_EXT_OUT_BASE / f"{bl_name.replace(' ', '_')}_results.csv", index=False)
pd.DataFrame(_long_rows).to_csv(M2_EXT_LONG_CSV, index=False)


def _m2ext_mean_std(values, digits=4):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().values
    if len(vals) == 0:
        return "N/D"
    if len(vals) == 1:
        return f"{vals[0]:.{digits}f} ± 0.{'0'*digits}"
    return f"{np.mean(vals):.{digits}f} ± {np.std(vals, ddof=1):.{digits}f}"

_summary = []
_sam = _df_m1_ext.copy()
_summary.append({"Méthode": "SAM-NeuroAdapt", "Dice": _m2ext_mean_std(_sam["dice"], 4), "HD95 (mm)": _m2ext_mean_std(_sam["hd95"], 2), "ASSD (mm)": _m2ext_mean_std(_sam["assd"], 2), "N": int(_sam["dice"].notna().sum()), "N paired": int(_sam["dice"].notna().sum()), "delta_dice_vs_sam": "--", "p_wilcoxon_dice": "--"})

for bl_name, rows in _bl_results.items():
    dfb = pd.DataFrame(rows)
    dfb["dice"] = pd.to_numeric(dfb["dice"], errors="coerce")
    paired = _sam[["subject", "dice"]].rename(columns={"dice": "sam_dice"}).merge(dfb[["subject", "dice"]], on="subject", how="inner").dropna()
    delta = float((paired["dice"] - paired["sam_dice"]).mean()) if len(paired) else float("nan")
    if len(paired) >= 2:
        try:
            pval = float(stats.wilcoxon(paired["sam_dice"], paired["dice"], zero_method="wilcox").pvalue)
            ptxt = f"{pval:.3g}"
        except Exception:
            ptxt = "N/D"
    else:
        ptxt = "N/D"
    _summary.append({"Méthode": bl_name, "Dice": _m2ext_mean_std(dfb["dice"], 4), "HD95 (mm)": _m2ext_mean_std(dfb["hd95"], 2), "ASSD (mm)": _m2ext_mean_std(dfb["assd"], 2), "N": int(dfb["dice"].notna().sum()), "N paired": int(len(paired)), "delta_dice_vs_sam": f"{delta:+.4f}" if np.isfinite(delta) else "N/D", "p_wilcoxon_dice": ptxt})

_summary_df = pd.DataFrame(_summary)
_summary_df.to_csv(M2_EXT_COMPARISON_CSV, index=False)
print("\n═══ Résumé comparatif externe ═══")
print(_summary_df.to_string(index=False))
print(f"\nRésultats longs: {M2_EXT_LONG_CSV}")
print(f"Résumé: {M2_EXT_COMPARISON_CSV}")
_m2ext_write_progress("finished", total=len(_df_m1_ext))

_failures = []
if REQUIRE_ALL_EXTERNAL_BASELINES_COMPLETE:
    for bl_name, rows in _bl_results.items():
        dfb = pd.DataFrame(rows)
        n_bad = int(dfb["dice"].isna().sum()) if "dice" in dfb.columns else len(dfb)
        if n_bad:
            _failures.append(f"{bl_name}: {n_bad} masque(s)/métrique(s) manquant(s)")
if _failures:
    raise RuntimeError(f"M2-LPBA40-SOTA incomplet: " + "; ".join(_failures))


### M2-iSeg2017-SOTA — Comparaison externe SOTA pédiatrique
Baselines FSL BET, SynthStrip et HD-BET sur exactement les sujets validés par M1-iSeg2017.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# M2-iSeg2017-SOTA — Baselines externes sur les mêmes sujets que M1-iSeg2017
# Méthodes: FSL BET | SynthStrip | HD-BET
# Entrées/gold repris depuis le CSV M1-iSeg2017 pour garantir un appariement strict.
# ═══════════════════════════════════════════════════════════════════════════
import sys, os, json, time, shlex, shutil, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
warnings.filterwarnings("ignore")

if "compute_metrics" not in globals():
    raise RuntimeError("Exécuter d'abord la cellule B3 / compute_metrics.")

M2_ISEG_DATASET = "iSeg2017"
M2_ISEG_M1_CSV = Path("outputs/iseg2017_m1_eval/sam_neuroadapt_iseg2017_results.csv")
M2_ISEG_M1_LIVE_CSV = Path("outputs/iseg2017_m1_eval/sam_neuroadapt_iseg2017_results_live.csv")
M2_ISEG_OUT_BASE = Path("outputs/iseg2017_sota_eval")
M2_ISEG_LOG_DIR = M2_ISEG_OUT_BASE / "logs"
M2_ISEG_PROGRESS_JSON = M2_ISEG_OUT_BASE / "m2_iseg2017_sota_progress.json"
M2_ISEG_LONG_CSV = M2_ISEG_OUT_BASE / "all_methods_long.csv"
M2_ISEG_COMPARISON_CSV = M2_ISEG_OUT_BASE / "comparison_table.csv"
M2_ISEG_OUT_BASE.mkdir(parents=True, exist_ok=True)
M2_ISEG_LOG_DIR.mkdir(parents=True, exist_ok=True)

M2_ISEG_N_SUBJECTS = None  # None = exactement les sujets validés dans M1-iSeg2017
M2_ISEG_SUBJECTS = None
REQUIRE_COMPLETE_ISEG_M1 = True
FORCE_ISEG_BASELINE_RERUN = True
REQUIRE_ALL_ISEG_BASELINES_COMPLETE = True

HDBET_DEVICE = os.environ.get("HDBET_DEVICE")
HDBET_DISABLE_TTA = os.environ.get("HDBET_DISABLE_TTA", "0") == "1"
HDBET_TIMEOUT_S = int(os.environ.get("HDBET_TIMEOUT_S", "900"))


def _m2iseg_resolve_executable(env_name, candidates):
    env_value = os.environ.get(env_name)
    ordered = ([env_value] if env_value else []) + list(candidates)
    for cand in ordered:
        if not cand:
            continue
        cand = str(cand)
        found = shutil.which(cand) if os.path.sep not in cand else cand
        if found and Path(found).exists():
            return str(Path(found))
    return None


_BET_BIN = _m2iseg_resolve_executable("FSL_BET_BIN", [
    shutil.which("bet"),
    "/home/nabolitains/fsl/share/fsl/bin/bet",
])
_SYNTHSTRIP_PY = Path(os.environ.get("SYNTHSTRIP_PY", "resources/synthstrip_standalone.py"))
_SYNTHSTRIP_WT = Path(os.environ.get("SYNTHSTRIP_WT", "resources/models/synthstrip.1.pt"))
_HDBET_BIN = _m2iseg_resolve_executable("HDBET_BIN", [
    Path(".venv/bin/hd-bet"),
    Path(".venv/bin/hd_bet"),
    Path("venv_wsl/bin/hd-bet"),
    shutil.which("hd-bet"),
    shutil.which("hd_bet"),
    "/mnt/d/CHARKO-RESEARCH/FEDERATED-LEARNING/V2/venv_wsl/bin/hd-bet",
])
if HDBET_DEVICE is None:
    try:
        import torch
        HDBET_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        HDBET_DEVICE = "cuda"

print(f"═══ M2-iSeg2017-SOTA — Préflight ═══")
print(f"FSL BET    : {_BET_BIN}")
print(f"SynthStrip : script={_SYNTHSTRIP_PY} | weights={_SYNTHSTRIP_WT}")
print(f"HD-BET     : {_HDBET_BIN} | device={HDBET_DEVICE} | TTA={'off' if HDBET_DISABLE_TTA else 'on'}")
if _BET_BIN is None:
    raise FileNotFoundError("FSL BET introuvable. Définir FSL_BET_BIN ou corriger l'installation FSL.")
if not _SYNTHSTRIP_PY.exists() or not _SYNTHSTRIP_WT.exists():
    raise FileNotFoundError("SynthStrip standalone ou poids introuvables.")
if _HDBET_BIN is None:
    raise FileNotFoundError("HD-BET introuvable. Définir HDBET_BIN ou installer hd-bet dans venv_wsl.")

if not M2_ISEG_M1_CSV.exists():
    raise FileNotFoundError(f"Résultats M1-iSeg2017 finaux introuvables: {M2_ISEG_M1_CSV}. Relancer M1-iSeg2017 jusqu'à la fin.")
_df_m1_iseg = pd.read_csv(M2_ISEG_M1_CSV)
if "subject" not in _df_m1_iseg.columns:
    raise RuntimeError("CSV M1-iSeg2017 invalide: colonne subject absente")
_df_m1_iseg["dice"] = pd.to_numeric(_df_m1_iseg.get("dice"), errors="coerce")
_errors = _df_m1_iseg[_df_m1_iseg.get("error", "").fillna("").astype(str).str.len() > 0] if "error" in _df_m1_iseg.columns else pd.DataFrame()
_bad = _df_m1_iseg[_df_m1_iseg["dice"].isna()]
if REQUIRE_COMPLETE_ISEG_M1 and (len(_errors) or len(_bad)):
    raise RuntimeError(f"M2-iSeg2017-SOTA arrêté: M1-iSeg2017 non propre. errors={len(_errors)}, dice_nan={len(_bad)}")

_df_m1_iseg = _df_m1_iseg.sort_values("subject").reset_index(drop=True)
if M2_ISEG_SUBJECTS:
    wanted = [str(s) for s in M2_ISEG_SUBJECTS]
    _df_m1_iseg = _df_m1_iseg[_df_m1_iseg["subject"].astype(str).isin(wanted)].copy()
    missing = sorted(set(wanted) - set(_df_m1_iseg["subject"].astype(str)))
    if missing:
        raise RuntimeError(f"Sujets demandés absents du M1-iSeg2017: {missing}")
elif M2_ISEG_N_SUBJECTS is not None:
    _df_m1_iseg = _df_m1_iseg.head(int(M2_ISEG_N_SUBJECTS)).copy()
if len(_df_m1_iseg) == 0:
    raise RuntimeError("Aucun sujet M1-iSeg2017 disponible pour SOTA")

for col in ["source_t1_path", "gold_mask_path"]:
    if col not in _df_m1_iseg.columns:
        raise RuntimeError(f"M1-iSeg2017 doit contenir la colonne {col}")
for _, row in _df_m1_iseg.iterrows():
    if not Path(str(row["source_t1_path"])).exists() or not Path(str(row["gold_mask_path"])).exists():
        raise FileNotFoundError(f"Source/gold absent pour {row['subject']}")


def _m2iseg_atomic_json(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    tmp.replace(path)


def _m2iseg_write_progress(stage, method=None, subject=None, idx=None, total=None):
    _m2iseg_atomic_json(M2_ISEG_PROGRESS_JSON, {
        "dataset": M2_ISEG_DATASET,
        "stage": stage,
        "method": method,
        "subject": subject,
        "idx": idx,
        "total": total,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    })


def _m2iseg_clean_subject(out_dir, subj_id, label):
    if not FORCE_ISEG_BASELINE_RERUN:
        return
    for pattern in [f"{subj_id}_{label}*.nii.gz", f"{subj_id}_{label}*.json", "dataset.json", "plans.json", "predict_from_raw_data_args.json"]:
        for p in Path(out_dir).glob(pattern):
            try:
                p.unlink()
            except FileNotFoundError:
                pass


def _m2iseg_run_command(cmd, timeout_s, log_prefix):
    log_prefix = Path(log_prefix)
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)
        runtime = time.time() - t0
        stdout = proc.stdout or ""
        stderr = proc.stderr or ""
        returncode = int(proc.returncode)
    except subprocess.TimeoutExpired as e:
        runtime = time.time() - t0
        stdout = e.stdout.decode() if isinstance(e.stdout, bytes) else (e.stdout or "")
        stderr = e.stderr.decode() if isinstance(e.stderr, bytes) else (e.stderr or "")
        stderr += f"\nTIMEOUT after {timeout_s}s"
        returncode = 124
    log_prefix.with_suffix(".stdout.log").write_text(stdout)
    log_prefix.with_suffix(".stderr.log").write_text(stderr)
    return {
        "cmd": shlex.join([str(c) for c in cmd]),
        "returncode": returncode,
        "runtime_s": round(runtime, 2),
        "stdout_log": str(log_prefix.with_suffix(".stdout.log")),
        "stderr_log": str(log_prefix.with_suffix(".stderr.log")),
        "stderr_tail": "\n".join(stderr.splitlines()[-8:]),
    }


def _m2iseg_canonicalize_mask(candidate_paths, canonical_mask):
    for cand in candidate_paths:
        cand = Path(cand)
        if cand.exists():
            canonical_mask = Path(canonical_mask)
            if cand.resolve() != canonical_mask.resolve():
                if canonical_mask.exists():
                    canonical_mask.unlink()
                shutil.move(str(cand), str(canonical_mask))
            return canonical_mask
    return None


def _m2iseg_run_bet(t1_path, out_dir, subj_id):
    _m2iseg_clean_subject(out_dir, subj_id, "bet")
    out_brain = out_dir / f"{subj_id}_bet_brain.nii.gz"
    out_mask = out_dir / f"{subj_id}_bet_mask.nii.gz"
    if out_mask.exists() and not FORCE_ISEG_BASELINE_RERUN:
        return out_mask, {"runtime_s": 0.0, "cmd": "cache"}
    cmd = [_BET_BIN, str(t1_path), str(out_brain), "-m", "-R"]
    meta = _m2iseg_run_command(cmd, 180, M2_ISEG_LOG_DIR / f"{subj_id}_fsl_bet")
    mask = _m2iseg_canonicalize_mask([out_mask, out_dir / f"{subj_id}_bet_brain_mask.nii.gz"], out_mask)
    if meta["returncode"] != 0 and mask is None:
        meta["error"] = f"FSL BET failed rc={meta['returncode']}: {meta['stderr_tail']}"
    return mask, meta


def _m2iseg_run_synthstrip(t1_path, out_dir, subj_id):
    _m2iseg_clean_subject(out_dir, subj_id, "synthstrip")
    out_brain = out_dir / f"{subj_id}_synthstrip_brain.nii.gz"
    out_mask = out_dir / f"{subj_id}_synthstrip_mask.nii.gz"
    if out_mask.exists() and not FORCE_ISEG_BASELINE_RERUN:
        return out_mask, {"runtime_s": 0.0, "cmd": "cache"}
    cmd = [sys.executable, str(_SYNTHSTRIP_PY), "-i", str(t1_path), "-o", str(out_brain), "-m", str(out_mask), "--model", str(_SYNTHSTRIP_WT)]
    meta = _m2iseg_run_command(cmd, 300, M2_ISEG_LOG_DIR / f"{subj_id}_synthstrip")
    mask = out_mask if out_mask.exists() else None
    if meta["returncode"] != 0 and mask is None:
        meta["error"] = f"SynthStrip failed rc={meta['returncode']}: {meta['stderr_tail']}"
    return mask, meta


def _m2iseg_run_hdbet(t1_path, out_dir, subj_id):
    _m2iseg_clean_subject(out_dir, subj_id, "hdbet")
    out_stub = out_dir / f"{subj_id}_hdbet.nii.gz"
    out_mask = out_dir / f"{subj_id}_hdbet_mask.nii.gz"
    if out_mask.exists() and not FORCE_ISEG_BASELINE_RERUN:
        return out_mask, {"runtime_s": 0.0, "cmd": "cache"}
    cmd = [_HDBET_BIN, "-i", str(t1_path), "-o", str(out_stub), "-device", HDBET_DEVICE, "--save_bet_mask", "--no_bet_image"]
    if HDBET_DISABLE_TTA:
        cmd.append("--disable_tta")
    meta = _m2iseg_run_command(cmd, HDBET_TIMEOUT_S, M2_ISEG_LOG_DIR / f"{subj_id}_hdbet")
    mask = _m2iseg_canonicalize_mask([
        out_mask,
        out_dir / f"{subj_id}_hdbet_bet.nii.gz",
        out_dir / f"{subj_id}_hdbet_brain_bet.nii.gz",
        *sorted(out_dir.glob(f"{subj_id}_hdbet*_bet.nii.gz")),
    ], out_mask)
    if meta["returncode"] != 0 and mask is None:
        meta["error"] = f"HD-BET failed rc={meta['returncode']}: {meta['stderr_tail']}"
    elif mask is None:
        meta["error"] = "HD-BET finished but mask not found; expected suffix *_bet.nii.gz"
    return mask, meta


BASELINES_ISEG = {
    "FSL BET": _m2iseg_run_bet,
    "SynthStrip": _m2iseg_run_synthstrip,
    "HD-BET": _m2iseg_run_hdbet,
}

if FORCE_ISEG_BASELINE_RERUN:
    for p in [M2_ISEG_LONG_CSV, M2_ISEG_COMPARISON_CSV, M2_ISEG_PROGRESS_JSON]:
        if p.exists():
            p.unlink()
    for name in BASELINES_ISEG:
        for p in [M2_ISEG_OUT_BASE / f"{name.replace(' ', '_')}_results.csv", M2_ISEG_OUT_BASE / f"{name.replace(' ', '_')}_results_live.csv"]:
            if p.exists():
                p.unlink()

print(f"\n═══ M2-iSeg2017-SOTA — Baselines externes sur {len(_df_m1_iseg)} sujets ═══\n")
_m2iseg_write_progress("started", total=len(_df_m1_iseg))
_bl_results = {name: [] for name in BASELINES_ISEG}
_long_rows = []

for idx, row in enumerate(_df_m1_iseg.itertuples(index=False), 1):
    sid = str(row.subject)
    safe_sid = sid.replace("/", "_")
    t1_path = Path(str(row.source_t1_path))
    gold_path = Path(str(row.gold_mask_path))
    bl_dir = M2_ISEG_OUT_BASE / safe_sid
    bl_dir.mkdir(parents=True, exist_ok=True)
    for bl_name, bl_fn in BASELINES_ISEG.items():
        _m2iseg_write_progress("running", bl_name, sid, idx, len(_df_m1_iseg))
        try:
            mask, meta = bl_fn(t1_path, bl_dir, safe_sid)
            if mask and Path(str(mask)).exists():
                metrics = compute_metrics(mask, gold_path)
                metrics.update({"subject": sid, "dataset": M2_ISEG_DATASET, "method": bl_name, "mask_path": str(mask), "error": ""})
            else:
                metrics = {"subject": sid, "dataset": M2_ISEG_DATASET, "method": bl_name, "dice": float("nan"), "hd95": float("nan"), "assd": float("nan"), "vol_pred_cm3": float("nan"), "vol_gs_cm3": float("nan"), "mask_path": "", "error": meta.get("error", "mask missing")}
            metrics.update({k: meta.get(k) for k in ["cmd", "returncode", "runtime_s", "stdout_log", "stderr_log"]})
        except Exception as exc:
            metrics = {"subject": sid, "dataset": M2_ISEG_DATASET, "method": bl_name, "dice": float("nan"), "hd95": float("nan"), "assd": float("nan"), "vol_pred_cm3": float("nan"), "vol_gs_cm3": float("nan"), "mask_path": "", "error": str(exc), "cmd": "", "returncode": None, "runtime_s": None, "stdout_log": "", "stderr_log": ""}
        _bl_results[bl_name].append(metrics)
        _long_rows.append(metrics)
        pd.DataFrame(_bl_results[bl_name]).to_csv(M2_ISEG_OUT_BASE / f"{bl_name.replace(' ', '_')}_results_live.csv", index=False)
        pd.DataFrame(_long_rows).to_csv(M2_ISEG_LONG_CSV, index=False)
        d = pd.to_numeric(pd.Series([metrics.get("dice")]), errors="coerce").iloc[0]
        d_txt = f"{d:.4f}" if pd.notna(d) else "ERREUR"
        print(f"  [{bl_name}] {sid} → Dice={d_txt}  t={metrics.get('runtime_s')}s  err={metrics.get('error','')}")

for bl_name, rows in _bl_results.items():
    pd.DataFrame(rows).to_csv(M2_ISEG_OUT_BASE / f"{bl_name.replace(' ', '_')}_results.csv", index=False)
pd.DataFrame(_long_rows).to_csv(M2_ISEG_LONG_CSV, index=False)


def _m2iseg_mean_std(values, digits=4):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().values
    if len(vals) == 0:
        return "N/D"
    if len(vals) == 1:
        return f"{vals[0]:.{digits}f} ± 0.{'0'*digits}"
    return f"{np.mean(vals):.{digits}f} ± {np.std(vals, ddof=1):.{digits}f}"

_summary = []
_sam = _df_m1_iseg.copy()
_summary.append({"Méthode": "SAM-NeuroAdapt", "Dice": _m2iseg_mean_std(_sam["dice"], 4), "HD95 (mm)": _m2iseg_mean_std(_sam["hd95"], 2), "ASSD (mm)": _m2iseg_mean_std(_sam["assd"], 2), "N": int(_sam["dice"].notna().sum()), "N paired": int(_sam["dice"].notna().sum()), "ΔDice vs SAM": "—", "p (Dice)": "—"})

for bl_name, rows in _bl_results.items():
    dfb = pd.DataFrame(rows)
    dfb["dice"] = pd.to_numeric(dfb["dice"], errors="coerce")
    paired = _sam[["subject", "dice"]].rename(columns={"dice": "sam_dice"}).merge(dfb[["subject", "dice"]], on="subject", how="inner").dropna()
    delta = float((paired["dice"] - paired["sam_dice"]).mean()) if len(paired) else float("nan")
    if len(paired) >= 2:
        try:
            pval = float(stats.wilcoxon(paired["sam_dice"], paired["dice"], zero_method="wilcox").pvalue)
            ptxt = f"{pval:.3g}"
        except Exception:
            ptxt = "N/D"
    else:
        ptxt = "N/D"
    _summary.append({"Méthode": bl_name, "Dice": _m2iseg_mean_std(dfb["dice"], 4), "HD95 (mm)": _m2iseg_mean_std(dfb["hd95"], 2), "ASSD (mm)": _m2iseg_mean_std(dfb["assd"], 2), "N": int(dfb["dice"].notna().sum()), "N paired": int(len(paired)), "ΔDice vs SAM": f"{delta:+.4f}" if np.isfinite(delta) else "N/D", "p (Dice)": ptxt})

_summary_df = pd.DataFrame(_summary)
_summary_df.to_csv(M2_ISEG_COMPARISON_CSV, index=False)
print("\n═══ Résumé comparatif externe iSeg2017 ═══")
print(_summary_df.to_string(index=False))
print(f"\nRésultats longs: {M2_ISEG_LONG_CSV}")
print(f"Résumé: {M2_ISEG_COMPARISON_CSV}")
_m2iseg_write_progress("finished", total=len(_df_m1_iseg))

_failures = []
if REQUIRE_ALL_ISEG_BASELINES_COMPLETE:
    for bl_name, rows in _bl_results.items():
        dfb = pd.DataFrame(rows)
        n_bad = int(dfb["dice"].isna().sum()) if "dice" in dfb.columns else len(dfb)
        if n_bad:
            _failures.append(f"{bl_name}: {n_bad} masque(s)/métrique(s) manquant(s)")
if _failures:
    raise RuntimeError("M2-iSeg2017-SOTA incomplet: " + "; ".join(_failures))


### M2 — Comparaison avec baselines (FSL BET, SynthStrip, HD-BET)
Tableau comparatif Dice / HD95 / ASSD + test statistique Wilcoxon sur 20 sujets test.

In [ ]:
import sys
import os
import json
from pathlib import Path
import numpy as np
# ═══════════════════════════════════════════════════════════════════════════
# M2 — Comparaison formelle avec baselines sur les 20 sujets test NFBS
# Baselines : FSL BET | SynthStrip | HD-BET
# Test statistique : Wilcoxon signed-rank (paired, α=0.05)
# Reproductibilité : nettoyage anti-stale, logs de commandes, appariement sujet.
# ═══════════════════════════════════════════════════════════════════════════
import subprocess, shutil, time, shlex

if '_GS_BASE' not in dir():
    _GS_BASE  = Path("datasets/NFBS/NFBS_Dataset/NFBS_Dataset")
    if not _GS_BASE.exists():
        _GS_BASE = Path("gold_standart/NFBS_Dataset/NFBS_Dataset")
with open("splits.json") as _f_m2:
    TEST_SUBJECTS = json.load(_f_m2)["test"]
if '_OUT_BASE' not in dir():
    _OUT_BASE = Path("outputs/multisubject_eval")
if 'compute_metrics' not in dir():
    raise RuntimeError("Exécuter d'abord la cellule B3 (compute_metrics) puis M1.")
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings("ignore")

# ── Garde-fous reproductibilité ──────────────────────────────────────────
REQUIRE_COMPLETE_M1 = True
FORCE_BASELINE_RERUN = True
REQUIRE_ALL_BASELINES_COMPLETE = True

# HD-BET : par défaut, comparaison sincère = TTA activée; mettre HDBET_DISABLE_TTA=1
# dans l'environnement uniquement pour un smoke-test rapide.
HDBET_DEVICE = os.environ.get("HDBET_DEVICE")
HDBET_DISABLE_TTA = os.environ.get("HDBET_DISABLE_TTA", "0") == "1"
HDBET_TIMEOUT_S = int(os.environ.get("HDBET_TIMEOUT_S", "900"))

_M1_CSV = _OUT_BASE / "sam_neuroadapt_results.csv"
if not _M1_CSV.exists():
    raise FileNotFoundError(f"Résultats M1 introuvables: {_M1_CSV}")
_df_m1 = pd.read_csv(_M1_CSV)
_expected = set(TEST_SUBJECTS)
_observed = set(_df_m1.get("subject", pd.Series(dtype=str)).astype(str))
_missing = sorted(_expected - _observed)
_extra = sorted(_observed - _expected)
_errors = _df_m1[_df_m1.get("error", "").fillna("").astype(str).str.len() > 0] if "error" in _df_m1.columns else pd.DataFrame()
_bad_dice = _df_m1[_df_m1["dice"].isna()] if "dice" in _df_m1.columns else _df_m1
if REQUIRE_COMPLETE_M1 and (_missing or _extra or len(_errors) or len(_bad_dice)):
    raise RuntimeError(
        "M2 arrêté: M1 n'est pas complet/propre. "
        f"missing={_missing}, extra={_extra}, errors={len(_errors)}, dice_nan={len(_bad_dice)}. "
        "Relancer M1 avant les baselines."
    )

# ── Configuration des baselines ────────────────────────────────────────────
_BL_OUT_BASE = Path("outputs/baselines_eval")
_BL_LOG_DIR = _BL_OUT_BASE / "logs"
_BL_OUT_BASE.mkdir(parents=True, exist_ok=True)
_BL_LOG_DIR.mkdir(parents=True, exist_ok=True)
_M2_PROGRESS_JSON = _BL_OUT_BASE / "m2_progress.json"
_M2_LONG_CSV = _BL_OUT_BASE / "all_methods_long.csv"


def _resolve_executable(env_name, candidates):
    env_value = os.environ.get(env_name)
    ordered = ([env_value] if env_value else []) + list(candidates)
    for cand in ordered:
        if not cand:
            continue
        cand = str(cand)
        found = shutil.which(cand) if os.path.sep not in cand else cand
        if found and Path(found).exists():
            return str(Path(found))
    return None


_BET_BIN = _resolve_executable("FSL_BET_BIN", [
    shutil.which("bet"),
    "/home/nabolitains/fsl/share/fsl/bin/bet",
])
_SYNTHSTRIP_PY = Path(os.environ.get("SYNTHSTRIP_PY", "resources/synthstrip_standalone.py"))
_SYNTHSTRIP_WT = Path(os.environ.get("SYNTHSTRIP_WT", "resources/models/synthstrip.1.pt"))
_HDBET_BIN = _resolve_executable("HDBET_BIN", [
    Path(".venv/bin/hd-bet"),
    Path(".venv/bin/hd_bet"),
    Path("venv_wsl/bin/hd-bet"),
    shutil.which("hd-bet"),
    shutil.which("hd_bet"),
    "/mnt/d/CHARKO-RESEARCH/FEDERATED-LEARNING/V2/venv_wsl/bin/hd-bet",
])

if HDBET_DEVICE is None:
    try:
        import torch
        HDBET_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        HDBET_DEVICE = "cuda"

print("═══ M2 — Préflight baselines ═══")
print(f"FSL BET    : {_BET_BIN}")
print(f"SynthStrip : script={_SYNTHSTRIP_PY} | weights={_SYNTHSTRIP_WT}")
print(f"HD-BET     : {_HDBET_BIN} | device={HDBET_DEVICE} | TTA={'off' if HDBET_DISABLE_TTA else 'on'}")
if _HDBET_BIN is None:
    raise FileNotFoundError("HD-BET introuvable. Définir HDBET_BIN ou installer hd-bet dans venv_wsl.")
if not _SYNTHSTRIP_PY.exists() or not _SYNTHSTRIP_WT.exists():
    raise FileNotFoundError("SynthStrip standalone ou poids introuvables.")
if _BET_BIN is None:
    raise FileNotFoundError("FSL BET introuvable. Définir FSL_BET_BIN ou corriger l'installation FSL.")


def _atomic_write_json(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    tmp.replace(path)


def _write_progress(stage, method=None, subject=None, idx=None, total=None):
    _atomic_write_json(_M2_PROGRESS_JSON, {
        "stage": stage,
        "method": method,
        "subject": subject,
        "index": idx,
        "total": total,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    })


def _clean_baseline_subject(out_dir, subj_id, label):
    if not FORCE_BASELINE_RERUN:
        return
    patterns = [
        f"{subj_id}_{label}*.nii.gz",
        f"{subj_id}_{label}*.json",
        "dataset.json",
        "plans.json",
        "predict_from_raw_data_args.json",
    ]
    for pattern in patterns:
        for _p in out_dir.glob(pattern):
            try:
                _p.unlink()
            except FileNotFoundError:
                pass


def _run_command(cmd, timeout_s, log_prefix):
    log_prefix = Path(log_prefix)
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)
        runtime = time.time() - t0
        stdout = proc.stdout or ""
        stderr = proc.stderr or ""
        returncode = int(proc.returncode)
    except subprocess.TimeoutExpired as e:
        runtime = time.time() - t0
        stdout = (e.stdout.decode() if isinstance(e.stdout, bytes) else (e.stdout or ""))
        stderr = (e.stderr.decode() if isinstance(e.stderr, bytes) else (e.stderr or ""))
        stderr += f"\nTIMEOUT after {timeout_s}s"
        returncode = 124
    log_prefix.with_suffix(".stdout.log").write_text(stdout)
    log_prefix.with_suffix(".stderr.log").write_text(stderr)
    return {
        "cmd": shlex.join([str(c) for c in cmd]),
        "returncode": returncode,
        "runtime_s": round(runtime, 2),
        "stdout_log": str(log_prefix.with_suffix(".stdout.log")),
        "stderr_log": str(log_prefix.with_suffix(".stderr.log")),
        "stderr_tail": "\n".join(stderr.splitlines()[-8:]),
    }


def _canonicalize_mask(candidate_paths, canonical_mask):
    for cand in candidate_paths:
        cand = Path(cand)
        if cand.exists():
            canonical_mask = Path(canonical_mask)
            if cand.resolve() != canonical_mask.resolve():
                if canonical_mask.exists():
                    canonical_mask.unlink()
                shutil.move(str(cand), str(canonical_mask))
            return canonical_mask
    return None


def run_bet(t1_path, out_dir, subj_id):
    _clean_baseline_subject(out_dir, subj_id, "bet")
    _out_brain = out_dir / f"{subj_id}_bet_brain.nii.gz"
    _out_mask = out_dir / f"{subj_id}_bet_mask.nii.gz"
    if _out_mask.exists() and not FORCE_BASELINE_RERUN:
        return _out_mask, {"runtime_s": 0.0, "cmd": "cache"}
    cmd = [_BET_BIN, str(t1_path), str(_out_brain), "-m", "-R"]
    meta = _run_command(cmd, 180, _BL_LOG_DIR / f"{subj_id}_fsl_bet")
    _mask = _canonicalize_mask([
        _out_mask,
        out_dir / f"{subj_id}_bet_brain_mask.nii.gz",
    ], _out_mask)
    if meta["returncode"] != 0 and _mask is None:
        meta["error"] = f"FSL BET failed rc={meta['returncode']}: {meta['stderr_tail']}"
    return _mask, meta


def run_synthstrip(t1_path, out_dir, subj_id):
    _clean_baseline_subject(out_dir, subj_id, "synthstrip")
    _out_brain = out_dir / f"{subj_id}_synthstrip_brain.nii.gz"
    _out_mask = out_dir / f"{subj_id}_synthstrip_mask.nii.gz"
    if _out_mask.exists() and not FORCE_BASELINE_RERUN:
        return _out_mask, {"runtime_s": 0.0, "cmd": "cache"}
    cmd = [
        sys.executable, str(_SYNTHSTRIP_PY),
        "-i", str(t1_path), "-o", str(_out_brain),
        "-m", str(_out_mask), "--model", str(_SYNTHSTRIP_WT),
    ]
    meta = _run_command(cmd, 300, _BL_LOG_DIR / f"{subj_id}_synthstrip")
    _mask = _out_mask if _out_mask.exists() else None
    if meta["returncode"] != 0 and _mask is None:
        meta["error"] = f"SynthStrip failed rc={meta['returncode']}: {meta['stderr_tail']}"
    return _mask, meta


def run_hdbet(t1_path, out_dir, subj_id):
    _clean_baseline_subject(out_dir, subj_id, "hdbet")
    # HD-BET 2.x writes mask as output[:-7] + '_bet.nii.gz'. This was the old M2 bug:
    # it expected '*_mask.nii.gz' and therefore marked valid HD-BET masks as missing.
    _out_stub = out_dir / f"{subj_id}_hdbet.nii.gz"
    _out_mask = out_dir / f"{subj_id}_hdbet_mask.nii.gz"
    if _out_mask.exists() and not FORCE_BASELINE_RERUN:
        return _out_mask, {"runtime_s": 0.0, "cmd": "cache"}
    cmd = [
        _HDBET_BIN, "-i", str(t1_path), "-o", str(_out_stub),
        "-device", HDBET_DEVICE, "--save_bet_mask", "--no_bet_image",
    ]
    if HDBET_DISABLE_TTA:
        cmd.append("--disable_tta")
    meta = _run_command(cmd, HDBET_TIMEOUT_S, _BL_LOG_DIR / f"{subj_id}_hdbet")
    _auto_mask = out_dir / f"{subj_id}_hdbet_bet.nii.gz"
    _mask = _canonicalize_mask([
        _out_mask,
        _auto_mask,
        out_dir / f"{subj_id}_hdbet_brain_bet.nii.gz",  # legacy output name from older M2
        *sorted(out_dir.glob(f"{subj_id}_hdbet*_bet.nii.gz")),
    ], _out_mask)
    if meta["returncode"] != 0 and _mask is None:
        meta["error"] = f"HD-BET failed rc={meta['returncode']}: {meta['stderr_tail']}"
    elif _mask is None:
        meta["error"] = "HD-BET finished but mask not found; expected suffix *_bet.nii.gz"
    return _mask, meta


BASELINES = {
    "FSL BET": run_bet,
    "SynthStrip": run_synthstrip,
    "HD-BET": run_hdbet,
}

if FORCE_BASELINE_RERUN:
    for _p in [
        _BL_OUT_BASE / "comparison_table.csv",
        _BL_OUT_BASE / "all_methods_long.csv",
        _BL_OUT_BASE / "FSL_BET_results.csv",
        _BL_OUT_BASE / "FSL_BET_results_live.csv",
        _BL_OUT_BASE / "SynthStrip_results.csv",
        _BL_OUT_BASE / "SynthStrip_results_live.csv",
        _BL_OUT_BASE / "HD-BET_results.csv",
        _BL_OUT_BASE / "HD-BET_results_live.csv",
        _M2_PROGRESS_JSON,
    ]:
        if _p.exists():
            _p.unlink()

# ── Boucle baselines ───────────────────────────────────────────────────────
print("\n═══ M2 — Baselines sur 20 sujets test ═══\n")
_write_progress("started", total=len(TEST_SUBJECTS))
_bl_results = {bl: [] for bl in BASELINES}
_long_rows = []

for _idx, _sid in enumerate(TEST_SUBJECTS, 1):
    _t1_path = _GS_BASE / _sid / f"sub-{_sid}_ses-NFB3_T1w.nii.gz"
    _gs_mask = _GS_BASE / _sid / f"sub-{_sid}_ses-NFB3_T1w_brainmask.nii.gz"
    _bl_dir = _BL_OUT_BASE / _sid
    _bl_dir.mkdir(parents=True, exist_ok=True)

    if not _t1_path.exists() or not _gs_mask.exists():
        raise FileNotFoundError(f"T1 ou gold-standard manquant pour {_sid}")

    for _bl_name, _bl_fn in BASELINES.items():
        _write_progress("running", _bl_name, _sid, _idx, len(TEST_SUBJECTS))
        try:
            _mask, _meta = _bl_fn(_t1_path, _bl_dir, _sid)
            if _mask and Path(str(_mask)).exists():
                _m = compute_metrics(_mask, _gs_mask)
                _m.update({
                    "subject": _sid,
                    "method": _bl_name,
                    "mask_path": str(_mask),
                    "error": "",
                })
            else:
                _m = {
                    "subject": _sid, "method": _bl_name,
                    "dice": float("nan"), "hd95": float("nan"), "assd": float("nan"),
                    "vol_pred_cm3": float("nan"), "vol_gs_cm3": float("nan"),
                    "mask_path": "", "error": _meta.get("error", "mask missing"),
                }
            _m.update({k: _meta.get(k) for k in ["cmd", "returncode", "runtime_s", "stdout_log", "stderr_log"]})
        except Exception as _e:
            _m = {
                "subject": _sid, "method": _bl_name,
                "dice": float("nan"), "hd95": float("nan"), "assd": float("nan"),
                "vol_pred_cm3": float("nan"), "vol_gs_cm3": float("nan"),
                "mask_path": "", "error": str(_e),
                "cmd": "", "returncode": None, "runtime_s": None,
                "stdout_log": "", "stderr_log": "",
            }
        _bl_results[_bl_name].append(_m)
        _long_rows.append(_m)
        pd.DataFrame(_bl_results[_bl_name]).to_csv(
            _BL_OUT_BASE / f"{_bl_name.replace(' ','_')}_results_live.csv", index=False
        )
        pd.DataFrame(_long_rows).to_csv(_M2_LONG_CSV, index=False)
        _d = _m.get("dice", float("nan"))
        _st = f"{_d:.4f}" if not (isinstance(_d, float) and (_d != _d)) else "ERREUR"
        print(f"  [{_bl_name}] {_sid} → Dice={_st}  t={_m.get('runtime_s')}s  err={_m.get('error','')}")

# ── Tableau récapitulatif ──────────────────────────────────────────────────
print("\n═══ Tableau comparatif ═══\n")
_summary = []

_df_sam_ok = _df_m1.copy()
_df_sam_ok["dice"] = pd.to_numeric(_df_sam_ok["dice"], errors="coerce")
_df_sam_ok = _df_sam_ok[_df_sam_ok["dice"].notna()].sort_values("subject")


def _mean_std(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().values
    if len(values) == 0:
        return "N/D"
    if len(values) == 1:
        return f"{values[0]:.4f} ± 0.0000"
    return f"{np.mean(values):.4f} ± {np.std(values, ddof=1):.4f}"


def _mean_std2(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().values
    if len(values) == 0:
        return "N/D"
    if len(values) == 1:
        return f"{values[0]:.2f} ± 0.00"
    return f"{np.mean(values):.2f} ± {np.std(values, ddof=1):.2f}"

_summary.append({
    "Méthode": "SAM-NeuroAdapt",
    "Dice": _mean_std(_df_sam_ok["dice"]),
    "HD95 (mm)": _mean_std2(_df_sam_ok["hd95"]),
    "ASSD (mm)": _mean_std2(_df_sam_ok["assd"]),
    "N": len(_df_sam_ok),
    "N paired": len(_df_sam_ok),
    "ΔDice vs SAM": "—",
    "p (Dice)": "—",
})

_required_failures = []
for _bl_name, _bl_data in _bl_results.items():
    _df_bl = pd.DataFrame(_bl_data)
    _df_bl.to_csv(_BL_OUT_BASE / f"{_bl_name.replace(' ','_')}_results.csv", index=False)
    _df_bl["dice"] = pd.to_numeric(_df_bl["dice"], errors="coerce")
    _df_bl_ok = _df_bl[_df_bl["dice"].notna()].sort_values("subject")
    if REQUIRE_ALL_BASELINES_COMPLETE and len(_df_bl_ok) != len(TEST_SUBJECTS):
        _required_failures.append(f"{_bl_name}: {len(_df_bl_ok)}/{len(TEST_SUBJECTS)} valid")

    _paired = _df_sam_ok[["subject", "dice"]].merge(
        _df_bl_ok[["subject", "dice"]], on="subject", suffixes=("_sam", "_bl")
    ).sort_values("subject")
    if len(_paired) >= 5:
        try:
            _, _p_val = stats.wilcoxon(_paired["dice_sam"].values, _paired["dice_bl"].values)
            _p_str = f"{_p_val:.4f}"
        except Exception as _e:
            _p_str = f"N/D ({_e})"
        _delta = float(np.mean(_paired["dice_sam"].values - _paired["dice_bl"].values))
        _delta_str = f"{_delta:+.4f}"
    else:
        _p_str = "N/D"
        _delta_str = "N/D"

    _summary.append({
        "Méthode": _bl_name,
        "Dice": _mean_std(_df_bl_ok["dice"]),
        "HD95 (mm)": _mean_std2(_df_bl_ok["hd95"]),
        "ASSD (mm)": _mean_std2(_df_bl_ok["assd"]),
        "N": len(_df_bl_ok),
        "N paired": len(_paired),
        "ΔDice vs SAM": _delta_str,
        "p (Dice)": _p_str,
    })

_df_summary = pd.DataFrame(_summary)
_df_summary.to_csv(_BL_OUT_BASE / "comparison_table.csv", index=False)
_write_progress("finished", total=len(TEST_SUBJECTS))
print(_df_summary.to_string(index=False))
print("\n(p = Wilcoxon signed-rank vs SAM-NeuroAdapt, α=0.05; appariement par subject)")
print(f"\nTableau sauvegardé: {_BL_OUT_BASE / 'comparison_table.csv'}")
print(f"Résultats longs: {_M2_LONG_CSV}")
print(f"Logs commandes: {_BL_LOG_DIR}")

if _required_failures:
    raise RuntimeError("M2 incomplet: " + "; ".join(_required_failures))


### M1-EXTERNAL-ATLAS-GOLD - moteur leger
Moteur M1 reutilisable pour les datasets gold-standard standardises. Il utilise les atlas externes prepares dans `dataset/atlas_library` et evite le LOO-SyN lourd.


In [ ]:
# M1-EXTERNAL-ATLAS-GOLD - moteur reutilisable atlas externe
# Prerequis: executer les cellules de base du notebook jusqu'a SAM2BrainExtractor et compute_metrics.

from pathlib import Path
import json
import math
import os
import re
import shutil
import time

import nibabel as nib
import numpy as np
import pandas as pd
from scipy import ndimage as _m1ea_ndi
from IPython.display import display

from scripts.adult_external_refinement import apply_adult_external_refinement
from scripts.external_atlas_config import (
    config_for_profile,
    default_output_slug,
    external_atlas_files_for_row,
    profile_for_row,
)
from scripts.neuro_utils import mask_stats


_M1EA_REQUIRED = [
    "run_preprocessing_pipeline",
    "advanced_atlas_registration",
    "compute_metrics",
    "SegmentationParams",
    "SAM2BrainExtractor",
]
_m1ea_missing = [name for name in _M1EA_REQUIRED if name not in globals()]
if _m1ea_missing:
    raise RuntimeError(
        "Executer d'abord les cellules de base avant M1-EXTERNAL-ATLAS-GOLD. "
        + "Symboles manquants: "
        + ", ".join(_m1ea_missing)
    )


def _m1ea_bool(value, default=False):
    if value is None:
        return default
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"1", "true", "yes", "y", "oui"}


def _m1ea_cell(row, key, default=""):
    value = row.get(key, default)
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass
    return str(value)


def _m1ea_existing_path(row, keys):
    for key in keys:
        value = _m1ea_cell(row, key, "")
        if value and value.lower() != "nan":
            path = Path(value)
            if path.exists():
                return path
    return None


def _m1ea_safe_subject(subject):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(subject)).strip("_")


def _m1ea_prefix(subject):
    subject = str(subject)
    return subject if subject.startswith("sub-") else f"sub-{subject}"


def _m1ea_registration_id(subject):
    subject = str(subject)
    return subject[4:] if subject.startswith("sub-") else subject


def _m1ea_json_default(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        value = float(obj)
        return None if not math.isfinite(value) else value
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)


def _m1ea_write_progress(path, stage, current=None, index=None, total=None, rows=None, start_time=None):
    rows = rows or []
    payload = {
        "updated": time.strftime("%Y-%m-%d %H:%M:%S"),
        "stage": stage,
        "current": current,
        "index": index,
        "total": total,
        "completed_rows": len(rows),
        "errors": int(sum(1 for row in rows if str(row.get("error", "")).strip())),
    }
    if start_time is not None:
        payload["age_min"] = round((time.time() - start_time) / 60.0, 2)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(payload, indent=2, default=_m1ea_json_default))


def _m1ea_load_records(dataset, manifest=None, profile_filter=None, subjects=None, n_subjects=None, require_atlas=True):
    manifest_path = Path(manifest) if manifest else Path("dataset/manifests") / f"{dataset}.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest introuvable: {manifest_path}")

    df = pd.read_csv(manifest_path)
    subjects_set = set(str(s) for s in subjects) if subjects else None
    records = []
    skipped = []

    for _, row in df.iterrows():
        rd = row.to_dict()
        subject = _m1ea_cell(rd, "subject", "")
        if not subject:
            continue
        if subjects_set and subject not in subjects_set:
            continue
        if profile_filter and _m1ea_cell(rd, "profile", "") != profile_filter:
            continue
        if not _m1ea_bool(rd.get("has_gold", True), True) or _m1ea_bool(rd.get("qc_only", False), False):
            skipped.append((subject, "not_gold_or_qc_only"))
            continue

        t1_path = _m1ea_existing_path(rd, ["input_path", "t1_path", "source_t1_path"])
        gold_path = _m1ea_existing_path(rd, ["brain_mask_path", "gold_mask_path", "mask_path"])
        if t1_path is None or gold_path is None:
            skipped.append((subject, "missing_t1_or_gold"))
            continue

        profile = profile_filter or profile_for_row(dataset, rd)
        atlas_paths = external_atlas_files_for_row(dataset, rd, profile_filter=profile)
        if require_atlas and not atlas_paths:
            skipped.append((subject, "missing_external_atlas"))
            continue

        records.append(
            {
                "dataset": dataset,
                "subject": subject,
                "session": _m1ea_cell(rd, "session", ""),
                "split": _m1ea_cell(rd, "split", ""),
                "profile": profile,
                "row": rd,
                "t1_path": t1_path,
                "gold_path": gold_path,
                "atlas_paths": atlas_paths,
            }
        )

    if subjects_set:
        found = {record["subject"] for record in records}
        missing = sorted(subjects_set - found)
        if missing:
            raise RuntimeError(f"Sujets demandes absents ou non exploitables dans {manifest_path}: {missing}")

    if n_subjects is not None:
        records = records[: int(n_subjects)]
    if not records:
        reason = f" Aucun sujet exploitable. Skips={skipped[:10]}" if skipped else ""
        raise RuntimeError(f"Aucun sujet gold-standard trouve pour {dataset}/{profile_filter}.{reason}")
    return records


def _m1ea_subject_paths(anat_dir, subject):
    prefix = _m1ea_prefix(subject)
    return {
        "n4": Path(anat_dir) / f"{prefix}_desc-n4_T1w.nii.gz",
        "brain": Path(anat_dir) / f"{prefix}_desc-brain_T1w.nii.gz",
        "mask": Path(anat_dir) / f"{prefix}_desc-brain_mask.nii.gz",
    }


def _m1ea_cached_registered_atlases(atlas_dir, subject):
    reg_id = _m1ea_registration_id(subject)
    registered = {}
    for path in sorted(Path(atlas_dir).glob(f"sub-{reg_id}_*.nii.gz")):
        name = path.name
        prefix = f"sub-{reg_id}_"
        if name.startswith(prefix):
            key = name[len(prefix) :].replace(".nii.gz", "")
            registered[key] = path
    return registered


def _m1ea_select_registered_atlas(registered_paths, cfg):
    scored = []
    min_cm3 = float(cfg.get("atlas_qc_min_cm3", cfg.get("min_volume_cm3", 0.0)))
    max_cm3 = float(cfg.get("atlas_qc_max_cm3", cfg.get("max_volume_cm3", 1e9)))
    mid_cm3 = 0.5 * (min_cm3 + max_cm3)
    span_cm3 = max(max_cm3 - min_cm3, 1.0)
    max_components = int(cfg.get("max_components", 3))

    for name, path in registered_paths.items():
        path = Path(path)
        if not path.exists():
            continue
        try:
            img = nib.load(str(path))
            mask = img.get_fdata(dtype=np.float32) > 0.5
            stats = mask_stats(mask, img)
            volume = float(stats["volume_cm3"])
            components = int(stats["components"])
            score = 1.0 - abs(volume - mid_cm3) / span_cm3
            if min_cm3 <= volume <= max_cm3:
                score += 1.0
            else:
                score -= 0.75
            score -= 0.03 * max(0, components - 1)
            score -= 0.20 * max(0, components - max_components)
            if "nomask" in name:
                score -= 0.02
            if "oasis30ants" in name or "mniinfant" in name:
                score += 0.10
            if "unc_" in name:
                score -= 0.04
            scored.append(
                {
                    "name": name,
                    "path": path,
                    "score": float(score),
                    "atlas_volume_cm3": volume,
                    "atlas_components": components,
                    "atlas_extent_ratio": float(stats.get("extent_ratio", 0.0)),
                }
            )
        except Exception as exc:
            scored.append({"name": name, "path": path, "score": -1e9, "error": str(exc)})

    if not scored:
        return None, []
    scored = sorted(scored, key=lambda item: item["score"], reverse=True)
    return scored[0], scored


def _m1ea_quality(metrics):
    dice = float(metrics.get("dice", np.nan))
    hd95 = float(metrics.get("hd95", np.inf))
    if np.isfinite(dice) and dice >= 0.97 and (not np.isfinite(hd95) or hd95 <= 8.0):
        return "good"
    if np.isfinite(dice) and dice >= 0.94:
        return "review"
    return "fail"


def run_m1_external_atlas_gold(
    dataset,
    profile_filter=None,
    manifest=None,
    out_base=None,
    n_subjects=None,
    subjects=None,
    device=None,
    force_preprocessing=False,
    force_atlas_registration=True,
    force_extraction=True,
    force_clean_outputs=True,
    require_atlas=True,
):
    """Run SAM-NeuroAdapt with lightweight external atlas guidance on a gold manifest."""

    slug = default_output_slug(dataset, profile_filter)
    out_base = Path(out_base) if out_base else Path("outputs") / f"{slug}_m1_external_atlas_eval"
    out_base.mkdir(parents=True, exist_ok=True)

    live_csv = out_base / f"sam_neuroadapt_{slug}_results_live.csv"
    final_csv = out_base / f"sam_neuroadapt_{slug}_results.csv"
    progress_json = out_base / f"m1_{slug}_progress.json"
    atlas_scores_json = out_base / f"m1_{slug}_atlas_scores.json"

    if force_clean_outputs:
        for path in [live_csv, final_csv, progress_json, atlas_scores_json]:
            if path.exists():
                path.unlink()

    device = device or ("cuda" if globals().get("torch", None) is not None and torch.cuda.is_available() else "cpu")
    records = _m1ea_load_records(
        dataset,
        manifest=manifest,
        profile_filter=profile_filter,
        subjects=subjects,
        n_subjects=n_subjects,
        require_atlas=require_atlas,
    )

    print(f"M1-EXTERNAL-ATLAS {dataset}/{profile_filter or 'all'}: {len(records)} sujets")
    print(f"Manifest: {manifest or Path('dataset/manifests') / (dataset + '.csv')}")
    print(f"Sortie  : {out_base}")
    print(f"Device  : {device}")

    rows = []
    atlas_scores = {}
    start_time = time.time()

    for index, record in enumerate(records, 1):
        subject = record["subject"]
        safe_subject = _m1ea_safe_subject(subject)
        anat_dir = out_base / safe_subject / "anat"
        atlas_dir = anat_dir / "atlases_advanced"
        paths = _m1ea_subject_paths(anat_dir, subject)

        if force_clean_outputs and anat_dir.exists():
            shutil.rmtree(anat_dir)
        anat_dir.mkdir(parents=True, exist_ok=True)
        atlas_dir.mkdir(parents=True, exist_ok=True)

        cfg = config_for_profile(record["profile"])
        row_out = {
            "dataset": dataset,
            "profile": record["profile"],
            "subject": subject,
            "session": record["session"],
            "split": record["split"],
            "source_t1_path": str(record["t1_path"]),
            "gold_mask_path": str(record["gold_path"]),
            "atlas_candidates": "|".join(record["atlas_paths"].keys()),
            "output_brain_path": str(paths["brain"]),
            "output_mask_path": str(paths["mask"]),
            "error": "",
        }

        print(f"\n[{index}/{len(records)}] {subject} | profile={record['profile']}")
        print(f"Atlas candidats: {', '.join(record['atlas_paths'].keys())}")
        _m1ea_write_progress(progress_json, "preprocessing", subject, index, len(records), rows, start_time)

        try:
            t0 = time.time()
            if paths["n4"].exists() and not force_preprocessing:
                n4_path = paths["n4"]
                row_out["preprocessing_stage"] = "cached"
                row_out["preprocessing_sec"] = 0.0
            else:
                prep = run_preprocessing_pipeline(
                    record["t1_path"],
                    anat_dir,
                    subject_id=subject,
                    mask_path=None,
                )
                if not prep or not prep.get("success"):
                    raise RuntimeError(f"preprocessing_failed stage={prep.get('stage') if prep else 'unknown'}")
                n4_path = Path(prep.get("n4_path") or prep.get("n4_corrected_path") or paths["n4"])
                row_out["preprocessing_stage"] = str(prep.get("stage", "complete"))
                row_out["preprocessing_sec"] = round(time.time() - t0, 2)
            if not n4_path.exists():
                raise FileNotFoundError(f"N4 introuvable: {n4_path}")

            _m1ea_write_progress(progress_json, "atlas_registration", subject, index, len(records), rows, start_time)
            t0 = time.time()
            if force_atlas_registration:
                reg = advanced_atlas_registration(
                    subject_n4_path=n4_path,
                    atlas_paths=record["atlas_paths"],
                    output_dir=atlas_dir,
                    subject_id=_m1ea_registration_id(subject),
                    create_brain_mask=True,
                    verbose=False,
                )
                registered = {k: Path(v) for k, v in reg.get("registered_atlas_paths", {}).items()}
            else:
                registered = _m1ea_cached_registered_atlases(atlas_dir, subject)
                if not registered:
                    reg = advanced_atlas_registration(
                        subject_n4_path=n4_path,
                        atlas_paths=record["atlas_paths"],
                        output_dir=atlas_dir,
                        subject_id=_m1ea_registration_id(subject),
                        create_brain_mask=True,
                        verbose=False,
                    )
                    registered = {k: Path(v) for k, v in reg.get("registered_atlas_paths", {}).items()}
            row_out["atlas_registration_sec"] = round(time.time() - t0, 2)

            selected, scored = _m1ea_select_registered_atlas(registered, cfg)
            atlas_scores[subject] = [
                {**item, "path": str(item["path"])} for item in scored
            ]
            atlas_scores_json.write_text(json.dumps(atlas_scores, indent=2, default=_m1ea_json_default))
            if selected is None:
                raise RuntimeError("aucun atlas recale exploitable")
            selected_path = Path(selected["path"])
            row_out.update(
                {
                    "selected_atlas": selected["name"],
                    "selected_atlas_path": str(selected_path),
                    "selected_atlas_score": round(float(selected["score"]), 4),
                    "atlas_volume_cm3": round(float(selected.get("atlas_volume_cm3", np.nan)), 3),
                    "atlas_components": int(selected.get("atlas_components", -1)),
                }
            )
            print(
                f"Atlas retenu: {selected['name']} | "
                f"vol={row_out['atlas_volume_cm3']} cm3 | components={row_out['atlas_components']}"
            )

            _m1ea_write_progress(progress_json, "sam2_extraction", subject, index, len(records), rows, start_time)
            if paths["mask"].exists() and not force_extraction:
                row_out["extraction_sec"] = 0.0
                row_out["extraction_stage"] = "cached"
            else:
                t0 = time.time()
                params = SegmentationParams(
                    device=device,
                    initial_confidence_threshold=float(cfg.get("confidence", 0.70)),
                    propagation_confidence_threshold=max(0.45, float(cfg.get("confidence", 0.70)) - 0.10),
                    use_atlas_guidance=True,
                    atlas_paths={"brain_mask": selected_path},
                    atlas_weight=float(cfg.get("atlas_weight", 0.72)),
                    post_process=True,
                    min_brain_volume_cm3=float(cfg.get("min_volume_cm3", 650.0)),
                    max_brain_volume_cm3=float(cfg.get("max_volume_cm3", 2100.0)),
                    atlas_fallback_min_cm3=float(cfg.get("min_volume_cm3", 650.0)),
                    atlas_fallback_max_cm3=float(cfg.get("max_volume_cm3", 2100.0)),
                    enable_atlas_fallback=True,
                    debug_dir=anat_dir / "debug_sam2",
                )
                extractor = SAM2BrainExtractor(params=params)
                brain_path, mask_path = extractor.extract_brain(
                    input_path=n4_path,
                    output_path=paths["brain"],
                    save_metrics=True,
                    save_uncertainty_map=True,
                )
                if not mask_path or not Path(mask_path).exists():
                    raise RuntimeError("SAM2BrainExtractor n'a pas produit de masque")
                row_out["extraction_sec"] = round(time.time() - t0, 2)
                row_out["extraction_stage"] = "sam2"

            _m1ea_write_progress(progress_json, "atlas_refinement", subject, index, len(records), rows, start_time)
            refine_info = apply_adult_external_refinement(
                mask_path=paths["mask"],
                brain_path=paths["brain"],
                input_image_path=n4_path,
                atlas_path=selected_path,
                metrics_path=anat_dir / "external_atlas_refinement_metrics.json",
                config=cfg.get("refine", {}),
            )
            row_out.update(refine_info)

            metrics = compute_metrics(paths["mask"], record["gold_path"])
            row_out.update(metrics)
            row_out["quality"] = _m1ea_quality(metrics)
            print(
                f"Dice={row_out['dice']:.4f} | HD95={row_out['hd95']:.3f} | "
                f"quality={row_out['quality']}"
            )
        except Exception as exc:
            row_out["error"] = str(exc)
            row_out["quality"] = "error"
            print(f"ERREUR {subject}: {exc}")

        rows.append(row_out)
        pd.DataFrame(rows).to_csv(live_csv, index=False)
        _m1ea_write_progress(progress_json, "subject_done", subject, index, len(records), rows, start_time)

    df = pd.DataFrame(rows)
    df.to_csv(final_csv, index=False)
    _m1ea_write_progress(progress_json, "done", None, len(records), len(records), rows, start_time)

    print(f"\nM1-EXTERNAL-ATLAS termine: {final_csv}")
    ok = df[(df["error"].fillna("") == "") & pd.to_numeric(df.get("dice"), errors="coerce").notna()]
    if len(ok):
        display(
            ok[["subject", "profile", "dice", "hd95", "selected_atlas", "quality"]]
            .sort_values("dice", ascending=False)
            .reset_index(drop=True)
        )
        display(
            ok.groupby("profile")
            .agg(n=("dice", "count"), dice_mean=("dice", "mean"), dice_min=("dice", "min"), hd95_mean=("hd95", "mean"))
            .reset_index()
        )
    return {
        "dataframe": df,
        "csv": final_csv,
        "live_csv": live_csv,
        "progress_json": progress_json,
        "atlas_scores_json": atlas_scores_json,
        "out_base": out_base,
    }


In [ ]:
IBSR18_EXTERNAL_RESULTS = run_m1_external_atlas_gold(
    dataset="IBSR18",
    profile_filter="adult_t1",
    out_base="outputs/ibsr18_m1_external_atlas_eval",
    n_subjects=2,  # smoke test d'abord
    force_clean_outputs=True,
    force_atlas_registration=True,
    force_extraction=True,
)


### M1-BOBs-EXTERNAL-ATLAS
Execution BOBs infant T1/T2 avec atlas MNIInfant age-matche et fallback UNC infant.


In [ ]:
RUN_M1_BOBS_EXTERNAL = False
M1_BOBS_EXTERNAL_N_SUBJECTS = None  # None = tous les sujets gold; mettre 2 pour un smoke test

if RUN_M1_BOBS_EXTERNAL:
    M1_BOBS_EXTERNAL_RESULTS = run_m1_external_atlas_gold(
        dataset="BOBs",
        profile_filter="infant_t1t2",
        out_base="outputs/bobs_m1_external_atlas_eval",
        n_subjects=M1_BOBS_EXTERNAL_N_SUBJECTS,
        force_clean_outputs=True,
        force_atlas_registration=True,
        force_extraction=True,
    )
else:
    print("M1-BOBs-EXTERNAL-ATLAS pret. Mettre RUN_M1_BOBS_EXTERNAL=True pour lancer.")


### M1-SynthStripT1-infant-EXTERNAL-ATLAS
Execution du sous-ensemble SynthStripT1 infant avec MNIInfant et fallback UNC infant.


In [ ]:
RUN_M1_SYNTHSTRIPT1_INFANT_EXTERNAL = False
M1_SYNTHSTRIPT1_INFANT_EXTERNAL_N_SUBJECTS = None

if RUN_M1_SYNTHSTRIPT1_INFANT_EXTERNAL:
    M1_SYNTHSTRIPT1_INFANT_EXTERNAL_RESULTS = run_m1_external_atlas_gold(
        dataset="SynthStripT1",
        profile_filter="infant_t1",
        out_base="outputs/synthstript1_infant_m1_external_atlas_eval",
        n_subjects=M1_SYNTHSTRIPT1_INFANT_EXTERNAL_N_SUBJECTS,
        force_clean_outputs=True,
        force_atlas_registration=True,
        force_extraction=True,
    )
else:
    print("M1-SynthStripT1-infant-EXTERNAL-ATLAS pret. Mettre RUN_M1_SYNTHSTRIPT1_INFANT_EXTERNAL=True pour lancer.")


### M1-SynthStripT1-adult-v4
Évaluation du sous-ensemble SynthStripT1 adulte avec le moteur externe adaptatif v4 no-LOO figé.


In [ ]:
# ============================================================================
# M1-SynthStripT1-v4 - Evaluation externe adulte SAM-NeuroAdapt sans LOO
# ============================================================================
import importlib
import os
from pathlib import Path

import scripts.adult_candidate_adjudicator as _m1_adult_candidate_adjudicator
import scripts.adult_external_refinement as _m1_adult_external_refinement
import scripts.adult_t1_adaptive_atlas as _m1_adult_t1_adaptive_atlas
import scripts.adult_t1_boundary_refinement as _m1_adult_t1_boundary_refinement
import scripts.adult_t1_strong_refinement as _m1_adult_t1_strong_refinement
import scripts.adaptive_mask_fusion as _m1_adaptive_mask_fusion
import scripts.external_atlas_config as _m1_external_atlas_config
import scripts.m1_external_adaptive_runner as _m1_external_adaptive_runner

importlib.reload(_m1_adult_candidate_adjudicator)
importlib.reload(_m1_adult_external_refinement)
importlib.reload(_m1_adult_t1_adaptive_atlas)
importlib.reload(_m1_adult_t1_boundary_refinement)
importlib.reload(_m1_adult_t1_strong_refinement)
importlib.reload(_m1_adaptive_mask_fusion)
importlib.reload(_m1_external_atlas_config)
_m1_external_adaptive_runner = importlib.reload(_m1_external_adaptive_runner)
run_m1_external_adaptive_gold = _m1_external_adaptive_runner.run_m1_external_adaptive_gold

RUN_M1_SYNTHSTRIPT1_V4 = os.environ.get("RUN_M1_SYNTHSTRIPT1_V4", "0").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}
M1_EXT_DATASET = "SynthStripT1"
M1_EXT_PROFILE = "adult_t1"
M1_EXT_MANIFEST = Path("datasets/manifests/SynthStripT1.csv")
M1_EXT_OUT_BASE = Path("outputs/synthstript1_m1_eval_v4_external_templates_frozen")

_synthstript1_n_subjects = os.environ.get("SYNTHSTRIPT1_N_SUBJECTS", "5").strip()
SYNTHSTRIPT1_N_SUBJECTS = None if _synthstript1_n_subjects.lower() in {"", "none", "all", "full"} else int(_synthstript1_n_subjects)
_synthstript1_subjects = os.environ.get("SYNTHSTRIPT1_SUBJECTS", "").strip()
SYNTHSTRIPT1_SUBJECTS = [s.strip() for s in _synthstript1_subjects.split(",") if s.strip()] or None
_synthstript1_min_smoke = os.environ.get("SYNTHSTRIPT1_MIN_SMOKE_MEAN_DICE", "").strip()
SYNTHSTRIPT1_MIN_SMOKE_MEAN_DICE = None if _synthstript1_min_smoke.lower() in {"", "none", "off"} else float(_synthstript1_min_smoke)
SYNTHSTRIPT1_REPRO_SEED = int(os.environ.get("SAM_NEUROADAPT_SEED", "20260517"))
SYNTHSTRIPT1_ARCHIVE_PREVIOUS = os.environ.get("M1_EXT_ARCHIVE_PREVIOUS", "1").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}

# Garde-fous explicites: atlas externes uniquement; aucun LOO, aucun SOTA-prior.
M1_EXT_ATLAS_MODE = "external_adaptive_no_loo"
M1_EXT_USE_LOO_ATLAS_LIBRARY = False
M1_EXT_ENABLE_ATLAS_FALLBACK = False

print(f"SynthStripT1 manifest: {M1_EXT_MANIFEST} | exists={M1_EXT_MANIFEST.exists()}")
print(f"SynthStripT1 run: n_subjects={SYNTHSTRIPT1_N_SUBJECTS} | subjects={SYNTHSTRIPT1_SUBJECTS} | min_smoke_mean_dice={SYNTHSTRIPT1_MIN_SMOKE_MEAN_DICE} | seed={SYNTHSTRIPT1_REPRO_SEED}")

if RUN_M1_SYNTHSTRIPT1_V4:
    SYNTHSTRIPT1_M1_RESULTS = run_m1_external_adaptive_gold(
        globals(),
        dataset=M1_EXT_DATASET,
        profile_filter=M1_EXT_PROFILE,
        manifest=M1_EXT_MANIFEST,
        out_base=M1_EXT_OUT_BASE,
        n_subjects=SYNTHSTRIPT1_N_SUBJECTS,
        subjects=SYNTHSTRIPT1_SUBJECTS,
        force_clean_outputs=True,
        force_atlas_registration=True,
        force_extraction=True,
        reproducibility_seed=SYNTHSTRIPT1_REPRO_SEED,
        archive_previous_outputs=SYNTHSTRIPT1_ARCHIVE_PREVIOUS,
        deterministic_threads=True,
        min_smoke_mean_dice=SYNTHSTRIPT1_MIN_SMOKE_MEAN_DICE,
        target_mean_dice=0.98,
    )
else:
    print("M1-SynthStripT1-v4 pret. Mettre RUN_M1_SYNTHSTRIPT1_V4=1 puis reexecuter cette cellule.")
    print("Watcher: python3 scripts/watch_m1_progress.py --dataset SynthStripT1 --version v4")


### M1-MRBrainS13-v4
Évaluation MRBrainS13 adulte T1 avec le moteur externe adaptatif v4 no-LOO figé.


In [ ]:
# ============================================================================
# M1-MRBrainS13-v4 - Evaluation externe adulte SAM-NeuroAdapt sans LOO
# ============================================================================
import importlib
import os
from pathlib import Path

import scripts.adult_candidate_adjudicator as _m1_adult_candidate_adjudicator
import scripts.adult_external_refinement as _m1_adult_external_refinement
import scripts.adult_t1_adaptive_atlas as _m1_adult_t1_adaptive_atlas
import scripts.adult_t1_boundary_refinement as _m1_adult_t1_boundary_refinement
import scripts.adult_t1_strong_refinement as _m1_adult_t1_strong_refinement
import scripts.adaptive_mask_fusion as _m1_adaptive_mask_fusion
import scripts.external_atlas_config as _m1_external_atlas_config
import scripts.m1_external_adaptive_runner as _m1_external_adaptive_runner

importlib.reload(_m1_adult_candidate_adjudicator)
importlib.reload(_m1_adult_external_refinement)
importlib.reload(_m1_adult_t1_adaptive_atlas)
importlib.reload(_m1_adult_t1_boundary_refinement)
importlib.reload(_m1_adult_t1_strong_refinement)
importlib.reload(_m1_adaptive_mask_fusion)
importlib.reload(_m1_external_atlas_config)
_m1_external_adaptive_runner = importlib.reload(_m1_external_adaptive_runner)
run_m1_external_adaptive_gold = _m1_external_adaptive_runner.run_m1_external_adaptive_gold

RUN_M1_MRBRAINS13_V4 = os.environ.get("RUN_M1_MRBRAINS13_V4", "0").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}
M1_EXT_DATASET = "MRBrainS13"
M1_EXT_PROFILE = "adult_t1"
M1_EXT_MANIFEST = Path("datasets/manifests/MRBrainS13.csv")
M1_EXT_OUT_BASE = Path("outputs/mrbrains13_m1_eval_v4_external_templates_frozen")

_mrbrains13_n_subjects = os.environ.get("MRBRAINS13_N_SUBJECTS", "5").strip()
MRBRAINS13_N_SUBJECTS = None if _mrbrains13_n_subjects.lower() in {"", "none", "all", "full"} else int(_mrbrains13_n_subjects)
_mrbrains13_subjects = os.environ.get("MRBRAINS13_SUBJECTS", "").strip()
MRBRAINS13_SUBJECTS = [s.strip() for s in _mrbrains13_subjects.split(",") if s.strip()] or None
_mrbrains13_min_smoke = os.environ.get("MRBRAINS13_MIN_SMOKE_MEAN_DICE", "").strip()
MRBRAINS13_MIN_SMOKE_MEAN_DICE = None if _mrbrains13_min_smoke.lower() in {"", "none", "off"} else float(_mrbrains13_min_smoke)
MRBRAINS13_REPRO_SEED = int(os.environ.get("SAM_NEUROADAPT_SEED", "20260517"))
MRBRAINS13_ARCHIVE_PREVIOUS = os.environ.get("M1_EXT_ARCHIVE_PREVIOUS", "1").strip().lower() in {"1", "true", "yes", "y", "on", "oui"}

# Garde-fous explicites: atlas externes uniquement; aucun LOO, aucun SOTA-prior.
M1_EXT_ATLAS_MODE = "external_adaptive_no_loo"
M1_EXT_USE_LOO_ATLAS_LIBRARY = False
M1_EXT_ENABLE_ATLAS_FALLBACK = False

print(f"MRBrainS13 manifest: {M1_EXT_MANIFEST} | exists={M1_EXT_MANIFEST.exists()}")
print(f"MRBrainS13 run: n_subjects={MRBRAINS13_N_SUBJECTS} | subjects={MRBRAINS13_SUBJECTS} | min_smoke_mean_dice={MRBRAINS13_MIN_SMOKE_MEAN_DICE} | seed={MRBRAINS13_REPRO_SEED}")

if RUN_M1_MRBRAINS13_V4:
    MRBRAINS13_M1_RESULTS = run_m1_external_adaptive_gold(
        globals(),
        dataset=M1_EXT_DATASET,
        profile_filter=M1_EXT_PROFILE,
        manifest=M1_EXT_MANIFEST,
        out_base=M1_EXT_OUT_BASE,
        n_subjects=MRBRAINS13_N_SUBJECTS,
        subjects=MRBRAINS13_SUBJECTS,
        force_clean_outputs=True,
        force_atlas_registration=True,
        force_extraction=True,
        reproducibility_seed=MRBRAINS13_REPRO_SEED,
        archive_previous_outputs=MRBRAINS13_ARCHIVE_PREVIOUS,
        deterministic_threads=True,
        min_smoke_mean_dice=MRBRAINS13_MIN_SMOKE_MEAN_DICE,
        target_mean_dice=0.98,
    )
else:
    print("M1-MRBrainS13-v4 pret. Mettre RUN_M1_MRBRAINS13_V4=1 puis reexecuter cette cellule.")
    print("Watcher: python3 scripts/watch_m1_progress.py --dataset MRBrainS13 --version v4")


### M2-EXTERNAL-SOTA - moteur generique
Moteur de comparaison SOTA reutilisable: FSL BET, SynthStrip et HD-BET sur les memes sujets que les sorties M1 externes.


In [ ]:
# M2-EXTERNAL-SOTA - moteur reutilisable v4
# Compare FSL BET, SynthStrip et HD-BET sur exactement les sujets termines par M1.

import importlib

from IPython.display import display

import scripts.m2_external_sota_runner as _m2_external_sota_runner

_m2_external_sota_runner = importlib.reload(_m2_external_sota_runner)
run_m2_external_sota = _m2_external_sota_runner.run_m2_external_sota

print("M2-EXTERNAL-SOTA v4 pret: runner synchronise sur le CSV M1 courant, avec delta_dice_vs_sam et p_wilcoxon_dice.")


### M2-NFBS-v4-SOTA
Comparaison SOTA sur les sujets termines par M1-NFBS-v4, sans utiliser les sorties SOTA dans M1.


In [ ]:
RUN_M2_NFBS_V4_SOTA = False
M2_NFBS_V4_SOTA_N_SUBJECTS = None

if RUN_M2_NFBS_V4_SOTA:
    M2_NFBS_V4_SOTA_RESULTS = run_m2_external_sota(
        dataset="NFBS",
        m1_csv="outputs/nfbs_m1_eval_v4_external_templates_frozen/sam_neuroadapt_nfbs_adultt1_results.csv",
        out_base="outputs/nfbs_sota_eval_v4_external_templates_frozen",
        n_subjects=M2_NFBS_V4_SOTA_N_SUBJECTS,
        force_rerun=True,
        require_all_tools=True,
        require_all_complete=True,
    )
    display(M2_NFBS_V4_SOTA_RESULTS["summary"])
else:
    print("M2-NFBS-v4-SOTA pret. Mettre RUN_M2_NFBS_V4_SOTA=True apres M1-NFBS-v4.")


### M2-IBSR18-IBSR_16_3-SOTA
Comparaison cible sur le sujet IBSR_16_3 pour diagnostiquer le cas penaliseur de SAM-NeuroAdapt.


In [ ]:
RUN_M2_IBSR18_IBSR_16_3_V4_SOTA = False

if RUN_M2_IBSR18_IBSR_16_3_V4_SOTA:
    M2_IBSR18_IBSR_16_3_V4_SOTA_RESULTS = run_m2_external_sota(
        dataset="IBSR18_IBSR_16_3",
        m1_csv="outputs/ibsr18_m1_eval_v4_external_templates_frozen/sam_neuroadapt_ibsr18_adultt1_results.csv",
        out_base="outputs/ibsr18_ibsr_16_3_sota_eval_v4_external_templates_frozen",
        subjects=["IBSR_16_3"],
        force_rerun=True,
        require_all_tools=True,
        require_all_complete=True,
    )
    display(M2_IBSR18_IBSR_16_3_V4_SOTA_RESULTS["summary"])
else:
    print("M2-IBSR18-IBSR_16_3-SOTA pret. Mettre RUN_M2_IBSR18_IBSR_16_3_V4_SOTA=True apres M1-IBSR18-v4.")


### M2-BOBs-SOTA
Comparaison FSL BET, SynthStrip et HD-BET sur les sujets termines par M1-BOBs-EXTERNAL-ATLAS.


In [ ]:
RUN_M2_BOBS_EXTERNAL_SOTA = False
M2_BOBS_EXTERNAL_SOTA_N_SUBJECTS = None

if RUN_M2_BOBS_EXTERNAL_SOTA:
    M2_BOBS_EXTERNAL_SOTA_RESULTS = run_m2_external_sota(
        dataset="BOBs",
        m1_csv="outputs/bobs_m1_external_atlas_eval/sam_neuroadapt_bobs_infantt1t2_results.csv",
        out_base="outputs/bobs_m2_external_sota_eval",
        n_subjects=M2_BOBS_EXTERNAL_SOTA_N_SUBJECTS,
        force_rerun=True,
        require_all_tools=True,
        require_all_complete=True,
    )
else:
    print("M2-BOBs-SOTA pret. Mettre RUN_M2_BOBS_EXTERNAL_SOTA=True apres M1-BOBs.")


### M2-SynthStripT1-infant-SOTA
Comparaison SOTA sur les sujets termines par M1-SynthStripT1-infant-EXTERNAL-ATLAS.


In [ ]:
RUN_M2_SYNTHSTRIPT1_INFANT_EXTERNAL_SOTA = False
M2_SYNTHSTRIPT1_INFANT_EXTERNAL_SOTA_N_SUBJECTS = None

if RUN_M2_SYNTHSTRIPT1_INFANT_EXTERNAL_SOTA:
    M2_SYNTHSTRIPT1_INFANT_EXTERNAL_SOTA_RESULTS = run_m2_external_sota(
        dataset="SynthStripT1_infant",
        m1_csv="outputs/synthstript1_infant_m1_external_atlas_eval/sam_neuroadapt_synthstript1_infantt1_results.csv",
        out_base="outputs/synthstript1_infant_m2_external_sota_eval",
        n_subjects=M2_SYNTHSTRIPT1_INFANT_EXTERNAL_SOTA_N_SUBJECTS,
        force_rerun=True,
        require_all_tools=True,
        require_all_complete=True,
    )
else:
    print("M2-SynthStripT1-infant-SOTA pret. Mettre RUN_M2_SYNTHSTRIPT1_INFANT_EXTERNAL_SOTA=True apres M1 infant.")


### M2-SynthStripT1-adult-v4-SOTA
Comparaison SOTA sur les sujets termines par M1-SynthStripT1-adult-v4.


In [ ]:
RUN_M2_SYNTHSTRIPT1_ADULT_V4_SOTA = False
M2_SYNTHSTRIPT1_ADULT_V4_SOTA_N_SUBJECTS = None

if RUN_M2_SYNTHSTRIPT1_ADULT_V4_SOTA:
    M2_SYNTHSTRIPT1_ADULT_V4_SOTA_RESULTS = run_m2_external_sota(
        dataset="SynthStripT1_adult",
        m1_csv="outputs/synthstript1_m1_eval_v4_external_templates_frozen/sam_neuroadapt_synthstript1_adultt1_results.csv",
        out_base="outputs/synthstript1_sota_eval_v4_external_templates_frozen",
        n_subjects=M2_SYNTHSTRIPT1_ADULT_V4_SOTA_N_SUBJECTS,
        force_rerun=True,
        require_all_tools=True,
        require_all_complete=True,
    )
    display(M2_SYNTHSTRIPT1_ADULT_V4_SOTA_RESULTS["summary"])
else:
    print("M2-SynthStripT1-adult-v4-SOTA pret. Mettre RUN_M2_SYNTHSTRIPT1_ADULT_V4_SOTA=True apres M1-SynthStripT1-adult-v4.")


### M2-MRBrainS13-v4-SOTA
Comparaison SOTA sur les sujets termines par M1-MRBrainS13-v4.


In [ ]:
RUN_M2_MRBRAINS13_V4_SOTA = False
M2_MRBRAINS13_V4_SOTA_N_SUBJECTS = None

if RUN_M2_MRBRAINS13_V4_SOTA:
    M2_MRBRAINS13_V4_SOTA_RESULTS = run_m2_external_sota(
        dataset="MRBrainS13",
        m1_csv="outputs/mrbrains13_m1_eval_v4_external_templates_frozen/sam_neuroadapt_mrbrains13_adultt1_results.csv",
        out_base="outputs/mrbrains13_sota_eval_v4_external_templates_frozen",
        n_subjects=M2_MRBRAINS13_V4_SOTA_N_SUBJECTS,
        force_rerun=True,
        require_all_tools=True,
        require_all_complete=True,
    )
    display(M2_MRBRAINS13_V4_SOTA_RESULTS["summary"])
else:
    print("M2-MRBrainS13-v4-SOTA pret. Mettre RUN_M2_MRBRAINS13_V4_SOTA=True apres M1-MRBrainS13-v4.")


### M2-ALL-DATASETS-SOTA — Benchmark SOTA complet (tous datasets)

Évalue **HD-BET / SynthStrip / FSL BET** sur l'ensemble des datasets disponibles avec masques gold standard.  
Métriques : **Dice** (↑) et **HD95 mm** (↓) par sujet, agrégées par dataset × méthode.  
Prérequis : Cell 16 (`compute_metrics`) doit avoir été exécutée.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  M2-ALL-DATASETS-SOTA — Benchmark SOTA complet tous datasets         ║
# ║  HD-BET / SynthStrip / FSL BET  ·  Dice + HD95 par sujet            ║
# ║  Prérequis : Cell 16 (compute_metrics) doit être exécutée avant     ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm

try:
    import torch as _torch
    _M2ALL_CUDA = _torch.cuda.is_available()
    _M2ALL_GPU_NAME = _torch.cuda.get_device_name(0) if _M2ALL_CUDA else "CPU"
except ImportError:
    _M2ALL_CUDA = False
    _M2ALL_GPU_NAME = "CPU"

# ── Répertoires ────────────────────────────────────────────────────────
_M2ALL_BASE = Path(".")
_M2ALL_OUT  = _M2ALL_BASE / "outputs" / "m2_all_datasets_sota"
_M2ALL_OUT.mkdir(parents=True, exist_ok=True)

# ── 1. Datasets à évaluer ─────────────────────────────────────────────
# BrainMetShare exclus: has_gold=False pour tous (masque=métastases, pas de masque cerveau)
# SynthStrip/SynthStripT1 exclus: données d'entraînement SynthStrip → biais
_M2ALL_DATASETS = [
    ("LPBA40",     "datasets/manifests/LPBA40.csv",     "T1w"),
    ("NFBS",       "datasets/manifests/NFBS.csv",       "T1w"),
    ("IBSR18",     "datasets/manifests/IBSR18.csv",     "T1w"),
    ("MRBrainS13", "datasets/manifests/MRBrainS13.csv", "T1w"),
    ("iSeg2017",   "datasets/manifests/iSeg2017.csv",   "T1w"),
    ("BOBs",       "datasets/manifests/BOBs.csv",       "T2w"),
]

# ── 2. Découverte des binaires SOTA ───────────────────────────────────
def _m2all_find_bin(*candidates):
    import sys
    venv_bin = Path(sys.executable).parent
    for c in candidates:
        p = shutil.which(c)
        if p:
            return p
        for extra in [venv_bin, venv_bin.parent / "bin"]:
            ep = extra / c
            if ep.exists():
                return str(ep)
    return None

# SynthStrip binaire local dans resources/tools/
_SS_LOCAL = _M2ALL_BASE / "resources/tools/mri_synthstrip"
_M2ALL_BET_BIN  = _m2all_find_bin("bet")
_M2ALL_SS_BIN   = (os.environ.get("SYNTHSTRIP_BIN") or
                   (str(_SS_LOCAL) if _SS_LOCAL.exists() else None) or
                   _m2all_find_bin("mri_synthstrip", "synthstrip"))
_M2ALL_SS_MODEL = Path(os.environ.get("SYNTHSTRIP_WT",
                       str(_M2ALL_BASE / "resources/models/synthstrip.1.pt")))
_M2ALL_HB_BIN   = (os.environ.get("HDBET_BIN") or
                   _m2all_find_bin("hd-bet", "hd_bet", "hdbet"))
_M2ALL_HB_DEV   = "cuda" if _M2ALL_CUDA else "cpu"

print("─── Outils SOTA détectés ────────────────────────────────────")
print(f"  FSL BET    : {'✓  ' + str(_M2ALL_BET_BIN) if _M2ALL_BET_BIN else '✗  introuvable'}")
print(f"  SynthStrip : {'✓  ' + str(_M2ALL_SS_BIN)  if _M2ALL_SS_BIN  else '✗  introuvable'}")
print(f"  HD-BET     : {'✓  ' + str(_M2ALL_HB_BIN)  if _M2ALL_HB_BIN  else '✗  introuvable'}")
print(f"  Device     : {_M2ALL_GPU_NAME}")

# ── 3. Reorientation canonique RAS ────────────────────────────────────
def _m2all_reorient(src, dst):
    """Réoriente en RAS canonique. Squeeze les 4D×1 (iSeg2017) → 3D.
    Détecte et re-génère si le cache est encore 4D (run précédent sans squeeze)."""
    dst = Path(dst)
    if dst.exists():
        try:
            if nib.load(str(dst)).ndim == 4:
                dst.unlink()   # cache 4D invalide → forcer la re-génération
            else:
                return
        except Exception:
            return
    img = nib.load(str(src))
    if img.ndim == 4 and img.shape[3] == 1:
        img = nib.Nifti1Image(img.get_fdata()[..., 0], img.affine, img.header)
    nib.save(nib.as_closest_canonical(img), str(dst))

# ── 4. Runners SOTA ───────────────────────────────────────────────────

def _m2all_run_bet(img, out_dir, sid, timeout=180):
    """bet input stub -m -R → {stub}_mask.nii.gz  (stub sans extension)"""
    stub     = out_dir / f"{sid}_bet_brain"
    out_mask = Path(str(stub) + "_mask.nii.gz")
    if out_mask.exists():
        return out_mask
    subprocess.run(
        [_M2ALL_BET_BIN, str(img), str(stub), "-m", "-R"],
        check=True, capture_output=True, timeout=timeout,
    )
    return out_mask if out_mask.exists() else None

def _m2all_run_synthstrip(img, out_dir, sid, timeout=600):
    """mri_synthstrip -i input -o brain -m mask"""
    out_mask  = out_dir / f"{sid}_ss_mask.nii.gz"
    out_brain = out_dir / f"{sid}_ss_brain.nii.gz"
    if out_mask.exists():
        return out_mask
    cmd = [_M2ALL_SS_BIN, "-i", str(img), "-o", str(out_brain), "-m", str(out_mask)]
    if _M2ALL_SS_MODEL.exists() and _M2ALL_SS_MODEL.stat().st_size > 10_000:
        cmd += ["--model", str(_M2ALL_SS_MODEL)]
    subprocess.run(cmd, check=True, capture_output=True, timeout=timeout)
    return out_mask if out_mask.exists() else None

def _m2all_run_hdbet(img, out_dir, sid, timeout=900):
    """hd-bet -i input -o output.nii.gz → output_mask.nii.gz
    HD-BET exige que -o se termine par .nii.gz."""
    out_brain = out_dir / f"{sid}_hdbet.nii.gz"     # doit avoir l'extension
    out_mask  = out_dir / f"{sid}_hdbet_mask.nii.gz"
    if out_mask.exists():
        return out_mask
    subprocess.run(
        [_M2ALL_HB_BIN, "-i", str(img), "-o", str(out_brain),
         "-device", _M2ALL_HB_DEV,
         "--save_bet_mask", "--no_bet_image", "--disable_tta"],
        check=True, capture_output=True, timeout=timeout,
    )
    return out_mask if out_mask.exists() else None

_M2ALL_RUNNERS = {}
if _M2ALL_BET_BIN: _M2ALL_RUNNERS["FSL-BET"]   = _m2all_run_bet
if _M2ALL_SS_BIN:  _M2ALL_RUNNERS["SynthStrip"] = _m2all_run_synthstrip
if _M2ALL_HB_BIN:  _M2ALL_RUNNERS["HD-BET"]     = _m2all_run_hdbet

if not _M2ALL_RUNNERS:
    raise RuntimeError(
        "Aucun outil SOTA disponible.\n"
        "  FSL BET    → sudo apt install fsl\n"
        "  SynthStrip → resources/tools/mri_synthstrip (déjà dans le projet)\n"
        "  HD-BET     → pip install hd-bet"
    )
print(f"\nOutils actifs : {list(_M2ALL_RUNNERS.keys())}\n")

# ── 5. Boucle principale ──────────────────────────────────────────────
_m2all_rows = []

for ds_name, manifest_rel, modality in _M2ALL_DATASETS:
    mpath = _M2ALL_BASE / manifest_rel
    if not mpath.exists():
        print(f"[SKIP] {ds_name}: manifest introuvable → {mpath}")
        continue

    df_m = pd.read_csv(mpath)
    if "has_gold" in df_m.columns:
        df_m = df_m[df_m["has_gold"].astype(str).str.strip().str.lower() == "true"]
    n = len(df_m)
    print(f"{'─'*62}\n{ds_name}: {n} sujets ({modality})")

    for _, row in tqdm(df_m.iterrows(), total=n, desc=ds_name, leave=True):
        sid = str(row.get("subject", row.get("subject_id", f"row{_}")))

        if modality == "T2w":
            img_col = next((c for c in ["t2_path", "input_path"]
                            if c in row.index and pd.notna(row.get(c))), None)
        else:
            img_col = next((c for c in ["t1_path", "input_path"]
                            if c in row.index and pd.notna(row.get(c))), None)

        mask_col = next((c for c in ["brain_mask_path", "mask_path"]
                         if c in row.index and pd.notna(row.get(c))), None)

        def _err(msg):
            _m2all_rows.append({"dataset": ds_name, "subject": sid,
                                 "method": "—", "dice": np.nan,
                                 "hd95": np.nan, "error": msg})

        if img_col is None:  _err("no_input_col"); continue
        if mask_col is None: _err("no_mask_col");  continue

        img_path = _M2ALL_BASE / str(row[img_col])
        gs_path  = _M2ALL_BASE / str(row[mask_col])

        if not img_path.exists(): _err(f"img_missing:{img_path.name}");  continue
        if not gs_path.exists():  _err(f"mask_missing:{gs_path.name}"); continue

        sub_out = _M2ALL_OUT / ds_name / sid
        sub_out.mkdir(parents=True, exist_ok=True)

        img_can = sub_out / f"{sid}_input_canonical.nii.gz"
        gs_can  = sub_out / f"{sid}_gs_canonical.nii.gz"
        try:
            _m2all_reorient(img_path, img_can)
            _m2all_reorient(gs_path,  gs_can)
        except Exception as e:
            _err(f"reorient:{e}"); continue

        for method, runner in _M2ALL_RUNNERS.items():
            rec = {"dataset": ds_name, "subject": sid, "method": method,
                   "dice": np.nan, "hd95": np.nan, "assd": np.nan,
                   "vol_pred_cm3": np.nan, "vol_gs_cm3": np.nan, "error": None}
            try:
                pred = runner(img_can, sub_out, sid)
                if pred is None or not pred.exists():
                    raise FileNotFoundError("masque SOTA absent après exécution")
                m = compute_metrics(str(pred), str(gs_can))
                rec.update({
                    "dice":         m.get("dice",         np.nan),
                    "hd95":         m.get("hd95",         np.nan),
                    "assd":         m.get("assd",         np.nan),
                    "vol_pred_cm3": m.get("vol_pred_cm3", np.nan),
                    "vol_gs_cm3":   m.get("vol_gs_cm3",   np.nan),
                })
            except subprocess.TimeoutExpired:
                rec["error"] = "timeout"
            except subprocess.CalledProcessError as e:
                stderr = e.stderr.decode(errors="replace")[-200:] if e.stderr else ""
                rec["error"] = f"tool_rc{e.returncode}:{stderr}"
            except Exception as e:
                rec["error"] = str(e)[:150]
            _m2all_rows.append(rec)

# ── 6. Sauvegarde CSV ─────────────────────────────────────────────────
df_all = pd.DataFrame(_m2all_rows)
csv_out = _M2ALL_OUT / "sota_all_datasets_results.csv"
df_all.to_csv(csv_out, index=False)
print(f"\nRésultats bruts sauvegardés → {csv_out}")

# ── 7. Tableau récapitulatif ──────────────────────────────────────────
df_ok = df_all[df_all["error"].isna() & df_all["dice"].notna()]
summary = (
    df_ok.groupby(["dataset", "method"])
    .agg(
        N         =("subject",  "count"),
        Dice_moy  =("dice",     "mean"),
        Dice_std  =("dice",     "std"),
        HD95_moy  =("hd95",     "mean"),
        HD95_std  =("hd95",     "std"),
    )
    .round(4)
    .reset_index()
)

print(f"\n{'='*72}")
print("BENCHMARK SOTA — TOUS DATASETS   (Dice ↑  |  HD95 mm ↓)")
print(f"{'='*72}")
print(summary.to_string(index=False))

df_err = df_all[df_all["error"].notna()]
if not df_err.empty:
    print(f"\n⚠  {len(df_err)} échecs détectés :")
    print(df_err[["dataset", "subject", "method", "error"]].to_string(index=False))

try:
    display(
        summary.style
        .background_gradient(subset=["Dice_moy"], cmap="RdYlGn", vmin=0, vmax=1)
        .background_gradient(subset=["HD95_moy"], cmap="RdYlGn_r")
        .format({
            "Dice_moy": "{:.4f}", "Dice_std": "±{:.4f}",
            "HD95_moy": "{:.2f}", "HD95_std": "±{:.2f} mm",
        })
        .set_caption("Benchmark SOTA — Tous datasets (Dice / HD95)")
    )
except Exception:
    pass


### M3-HD-BET — Évaluation HD-BET sur tous les datasets

Exécution en 2 phases :
- **`HDBET_MODE = "test"`** — 1 sujet par dataset (validation rapide, ~10 s)
- **`HDBET_MODE = "full"`** — tous les sujets + tableau comparatif FSL-BET / SynthStrip / HD-BET

Prérequis : Cell 16 (`compute_metrics`) et Cell 83 (`M2-ALL-DATASETS-SOTA`) exécutées au moins une fois.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  M3-HD-BET — Évaluation HD-BET  ·  Dice + HD95 par sujet            ║
# ║  Phase test : 1 sujet/dataset   |   Phase full : tous les sujets     ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── Contrôle d'exécution ────────────────────────────────────────────
# "test" → 1 sujet par dataset (validation rapide avant le run complet)
# "full" → tous les sujets + tableau comparatif SOTA final
HDBET_MODE = "full"   # ← changer en "full" après validation

import os, subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm

try:
    import torch as _torch
    _M3_CUDA = _torch.cuda.is_available()
    _M3_GPU  = _torch.cuda.get_device_name(0) if _M3_CUDA else "CPU"
except ImportError:
    _M3_CUDA = False
    _M3_GPU  = "CPU"

_M3_BASE    = Path(".")
_M3_OUT     = _M3_BASE / "outputs" / "m2_all_datasets_sota"  # même dossier que M2
_M3_CSV_ALL = _M3_OUT / "sota_all_datasets_results.csv"

# ── Datasets (mêmes que M2, sans BrainMetShare) ───────────────────────
_M3_DATASETS = [
    ("LPBA40",     "datasets/manifests/LPBA40.csv",     "T1w"),
    ("NFBS",       "datasets/manifests/NFBS.csv",       "T1w"),
    ("IBSR18",     "datasets/manifests/IBSR18.csv",     "T1w"),
    ("MRBrainS13", "datasets/manifests/MRBrainS13.csv", "T1w"),
    ("iSeg2017",   "datasets/manifests/iSeg2017.csv",   "T1w"),
    ("BOBs",       "datasets/manifests/BOBs.csv",       "T2w"),
]

# ── Découverte HD-BET ─────────────────────────────────────────────────
def _m3_find_bin(*candidates):
    import sys
    venv_bin = Path(sys.executable).parent
    for c in candidates:
        p = shutil.which(c)
        if p:
            return p
        for ep in [venv_bin, venv_bin.parent / "bin"]:
            candidate = ep / c
            if candidate.exists():
                return str(candidate)
    return None

_M3_HB_BIN = os.environ.get("HDBET_BIN") or _m3_find_bin("hd-bet", "hd_bet", "hdbet")
_M3_HB_DEV = "cuda" if _M3_CUDA else "cpu"

if not _M3_HB_BIN:
    raise RuntimeError("HD-BET introuvable. pip install hd-bet")
print(f"HD-BET : {_M3_HB_BIN}")
print(f"Device : {_M3_GPU}")
print(f"Mode   : {HDBET_MODE}\n")

# ── Reorientation canonique (identique M2) ────────────────────────────
def _m3_reorient(src, dst):
    dst = Path(dst)
    if dst.exists():
        try:
            if nib.load(str(dst)).ndim == 4:
                dst.unlink()
            else:
                return
        except Exception:
            return
    img = nib.load(str(src))
    if img.ndim == 4 and img.shape[3] == 1:
        img = nib.Nifti1Image(img.get_fdata()[..., 0], img.affine, img.header)
    nib.save(nib.as_closest_canonical(img), str(dst))

# ── Runner HD-BET (corrigé) ───────────────────────────────────────────
# HD-BET génère {base}_bet.nii.gz  (base = output sans .nii.gz)
# Exemple : -o sid_hdbet.nii.gz  →  masque : sid_hdbet_bet.nii.gz
def _m3_run_hdbet(img, out_dir, sid, timeout=900):
    out_arg  = out_dir / f"{sid}_hdbet.nii.gz"   # argument -o : doit finir par .nii.gz
    out_mask = out_dir / f"{sid}_hdbet_bet.nii.gz"  # masque réel produit par HD-BET
    if out_mask.exists():
        return out_mask                           # résumé : masque déjà présent
    subprocess.run(
        [_M3_HB_BIN, "-i", str(img), "-o", str(out_arg),
         "-device", _M3_HB_DEV,
         "--save_bet_mask", "--no_bet_image", "--disable_tta"],
        check=True, capture_output=True, timeout=timeout,
    )
    return out_mask if out_mask.exists() else None

# ── Boucle d'évaluation ───────────────────────────────────────────────
_m3_rows = []
_m3_ok   = True   # flag de validation (test mode)

for ds_name, manifest_rel, modality in _M3_DATASETS:
    mpath = _M3_BASE / manifest_rel
    if not mpath.exists():
        print(f"[SKIP] manifest introuvable: {mpath}")
        continue

    df_m = pd.read_csv(mpath)
    if "has_gold" in df_m.columns:
        df_m = df_m[df_m["has_gold"].astype(str).str.strip().str.lower() == "true"]
    
    # En mode test : 1 seul sujet par dataset
    subjects = df_m.head(1) if HDBET_MODE == "test" else df_m
    n = len(subjects)
    label = f"(test 1/{len(df_m)})" if HDBET_MODE == "test" else f"({n} sujets)"
    print(f"{'─'*60}\n{ds_name} {label}")

    for _, row in tqdm(subjects.iterrows(), total=n, desc=f"HD-BET {ds_name}", leave=True):
        sid = str(row.get("subject", row.get("subject_id", f"row{_}")))

        if modality == "T2w":
            img_col = next((c for c in ["t2_path", "input_path"]
                            if c in row.index and pd.notna(row.get(c))), None)
        else:
            img_col = next((c for c in ["t1_path", "input_path"]
                            if c in row.index and pd.notna(row.get(c))), None)
        mask_col = next((c for c in ["brain_mask_path", "mask_path"]
                         if c in row.index and pd.notna(row.get(c))), None)

        rec = {"dataset": ds_name, "subject": sid, "method": "HD-BET",
               "dice": np.nan, "hd95": np.nan, "assd": np.nan,
               "vol_pred_cm3": np.nan, "vol_gs_cm3": np.nan, "error": None}

        if img_col is None or mask_col is None:
            rec["error"] = "no_path_col"; _m3_rows.append(rec); continue

        img_path = _M3_BASE / str(row[img_col])
        gs_path  = _M3_BASE / str(row[mask_col])
        if not img_path.exists(): rec["error"] = f"img_missing"; _m3_rows.append(rec); continue
        if not gs_path.exists():  rec["error"] = f"mask_missing"; _m3_rows.append(rec); continue

        sub_out = _M3_OUT / ds_name / sid
        sub_out.mkdir(parents=True, exist_ok=True)

        img_can = sub_out / f"{sid}_input_canonical.nii.gz"
        gs_can  = sub_out / f"{sid}_gs_canonical.nii.gz"
        try:
            _m3_reorient(img_path, img_can)
            _m3_reorient(gs_path,  gs_can)
        except Exception as e:
            rec["error"] = f"reorient:{e}"; _m3_rows.append(rec); continue

        try:
            pred = _m3_run_hdbet(img_can, sub_out, sid)
            if pred is None or not pred.exists():
                raise FileNotFoundError("masque HD-BET absent")
            m = compute_metrics(str(pred), str(gs_can))
            rec.update({
                "dice":         m.get("dice",         np.nan),
                "hd95":         m.get("hd95",         np.nan),
                "assd":         m.get("assd",         np.nan),
                "vol_pred_cm3": m.get("vol_pred_cm3", np.nan),
                "vol_gs_cm3":   m.get("vol_gs_cm3",   np.nan),
            })
        except subprocess.TimeoutExpired:
            rec["error"] = "timeout"; _m3_ok = False
        except subprocess.CalledProcessError as e:
            stderr = e.stderr.decode(errors="replace")[-200:] if e.stderr else ""
            rec["error"] = f"tool_rc{e.returncode}:{stderr}"; _m3_ok = False
        except Exception as e:
            rec["error"] = str(e)[:150]; _m3_ok = False
        _m3_rows.append(rec)

df_m3 = pd.DataFrame(_m3_rows)

# ── Résultats ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Résultats HD-BET — mode '{HDBET_MODE}'")
print(f"{'='*60}")
df_ok  = df_m3[df_m3["error"].isna() & df_m3["dice"].notna()]
df_err = df_m3[df_m3["error"].notna()]

if not df_ok.empty:
    print(df_ok[["dataset","subject","dice","hd95"]].to_string(index=False))

if not df_err.empty:
    print(f"\n⚠  {len(df_err)} erreur(s) :")
    print(df_err[["dataset","subject","error"]].to_string(index=False))
    _m3_ok = False

# ── Guide selon le mode ───────────────────────────────────────────────
if HDBET_MODE == "test":
    if _m3_ok and df_err.empty:
        print(f"\n✓  Validation OK ({len(df_ok)}/6 datasets)")
        print('   → Changez HDBET_MODE = "full" et ré-exécutez la cellule.')
    else:
        print("\n✗  Validation échouée. Voir les erreurs ci-dessus.")

else:  # mode full → mise à jour CSV + tableau comparatif
    # Fusionner avec résultats existants (remplace les lignes HD-BET)
    if _M3_CSV_ALL.exists():
        df_prev = pd.read_csv(_M3_CSV_ALL)
        df_prev = df_prev[df_prev["method"] != "HD-BET"]   # supprimer anciens résultats HD-BET
        df_merged = pd.concat([df_prev, df_m3], ignore_index=True)
    else:
        df_merged = df_m3

    df_merged.to_csv(_M3_CSV_ALL, index=False)
    print(f"CSV mis à jour → {_M3_CSV_ALL}")

    # Tableau comparatif
    df_valid = df_merged[df_merged["error"].isna() & df_merged["dice"].notna()]
    comp = (
        df_valid.groupby(["dataset", "method"])
        .agg(N=("subject","count"), Dice=("dice","mean"),
             Dice_std=("dice","std"), HD95=("hd95","mean"), HD95_std=("hd95","std"))
        .round(4).reset_index()
    )
    print(f"\n{'='*72}")
    print("COMPARAISON SOTA — FSL-BET / SynthStrip / HD-BET")
    print(f"{'='*72}")
    print(comp.to_string(index=False))

    try:
        display(
            comp.style
            .background_gradient(subset=["Dice"], cmap="RdYlGn", vmin=0, vmax=1)
            .background_gradient(subset=["HD95"], cmap="RdYlGn_r")
            .format({"Dice": "{:.4f}", "Dice_std": "±{:.4f}",
                     "HD95": "{:.2f}",  "HD95_std": "±{:.2f} mm"})
            .set_caption("Comparaison SOTA — Dice ↑  |  HD95 mm ↓")
        )
    except Exception:
        pass


### M1-Pediatric-iSegStyle-Scalable

Cellule optionnelle scalable pour lancer l'architecture M1-iSeg2017 sur les datasets pediatriques standardises.
Elle reste inactive par defaut; le GT est utilise uniquement pour les metriques finales.


In [ ]:
# M1-Pediatric-iSegStyle-Scalable
import os
from pathlib import Path

from scripts.external_atlas_config import default_output_slug
from scripts.m1_pediatric_iseg_style_runner import run_m1_pediatric_iseg_style_gold

def _m1ped_env_bool(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

RUN_M1_PEDIATRIC_ISEGSTYLE = _m1ped_env_bool('RUN_M1_PEDIATRIC_ISEGSTYLE', False)
M1_PEDIATRIC_DATASETS = [
    item.strip()
    for item in os.environ.get('M1_PEDIATRIC_DATASETS', 'BOBs,MCRIB,MCRIB_WM').split(',')
    if item.strip()
]
_m1ped_n = os.environ.get('M1_PEDIATRIC_N_SUBJECTS', '').strip()
M1_PEDIATRIC_N_SUBJECTS = int(_m1ped_n) if _m1ped_n else None
M1_PEDIATRIC_TARGET_MEAN_DICE = os.environ.get('M1_PEDIATRIC_TARGET_MEAN_DICE', '').strip()
M1_PEDIATRIC_TARGET_MEAN_DICE = float(M1_PEDIATRIC_TARGET_MEAN_DICE) if M1_PEDIATRIC_TARGET_MEAN_DICE else None
M1_PEDIATRIC_RESULTS = {}

if RUN_M1_PEDIATRIC_ISEGSTYLE:
    for _dataset in M1_PEDIATRIC_DATASETS:
        _manifest = Path('datasets/manifests') / f'{_dataset}.csv'
        if not _manifest.exists() and _dataset == 'MCRIB2':
            _manifest = Path('datasets/manifests/MCRIB_WM.csv')
        if not _manifest.exists():
            print(f'[skip] {_dataset}: manifest absent {_manifest}')
            continue
        _slug = default_output_slug(_dataset, None)
        print(f'[M1 pediatric iSegStyle] dataset={_dataset} manifest={_manifest} n={M1_PEDIATRIC_N_SUBJECTS}')
        M1_PEDIATRIC_RESULTS[_dataset] = run_m1_pediatric_iseg_style_gold(
            globals(),
            dataset=_dataset,
            manifest=_manifest,
            out_base=Path('outputs') / f'{_slug}_m1_pediatric_isegstyle_eval',
            n_subjects=M1_PEDIATRIC_N_SUBJECTS,
            require_atlas=True,
            force_clean_outputs=True,
            target_mean_dice=M1_PEDIATRIC_TARGET_MEAN_DICE,
            reproducibility_seed=int(os.environ.get('SAM_NEUROADAPT_SEED', '20260517')),
        )
else:
    print('M1-Pediatric-iSegStyle-Scalable pret. Definir RUN_M1_PEDIATRIC_ISEGSTYLE=1 pour lancer.')


In [ ]:
!SAM_NEUROADAPT_SEED=20260517 .venv/bin/python scripts/run_m1_pediatric_isegstyle.py --dataset BOBs

### M2-Pediatric-SOTA-Scalable

Cellule scalable pour evaluer FSL BET, SynthStrip et HD-BET sur les datasets pediatriques/neonataux standardises. Ces sorties restent strictement des baselines M2: elles ne sont jamais reutilisees par M1.


In [ ]:
# M2-Pediatric-SOTA-Scalable
import os
import importlib
from pathlib import Path

import scripts.m2_pediatric_sota_runner as _m2_ped_sota
_m2_ped_sota = importlib.reload(_m2_ped_sota)

def _m2ped_env_bool(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

def _m2ped_env_list(name, default):
    raw = os.environ.get(name, default)
    return [item.strip() for item in raw.replace(';', ',').split(',') if item.strip()]

RUN_M2_PEDIATRIC_SOTA = _m2ped_env_bool('RUN_M2_PEDIATRIC_SOTA', False)
M2_PEDIATRIC_SOTA_DATASETS = _m2ped_env_list(
    'M2_PEDIATRIC_SOTA_DATASETS',
    'iSeg2017,BOBs,MCRIB,MCRIB_WM,MCRIB2,SynthStripT1',
)
M2_PEDIATRIC_SOTA_METHODS = _m2ped_env_list('M2_PEDIATRIC_SOTA_METHODS', 'fsl_bet,synthstrip,hdbet')
_m2ped_n = os.environ.get('M2_PEDIATRIC_SOTA_N_SUBJECTS', '').strip()
M2_PEDIATRIC_SOTA_N_SUBJECTS = int(_m2ped_n) if _m2ped_n else None
M2_PEDIATRIC_SOTA_OUTPUT_ROOT = Path(os.environ.get('M2_PEDIATRIC_SOTA_OUTPUT_ROOT', 'outputs/pediatric_sota_eval'))
M2_PEDIATRIC_SOTA_FORCE = _m2ped_env_bool('M2_PEDIATRIC_SOTA_FORCE', False)
M2_PEDIATRIC_SOTA_ALLOW_FAILURES = _m2ped_env_bool('M2_PEDIATRIC_SOTA_ALLOW_FAILURES', False)

if RUN_M2_PEDIATRIC_SOTA:
    print('[M2 pediatric SOTA] datasets=', ','.join(M2_PEDIATRIC_SOTA_DATASETS))
    print('[M2 pediatric SOTA] methods=', ','.join(M2_PEDIATRIC_SOTA_METHODS))
    print('[M2 pediatric SOTA] n_subjects=', M2_PEDIATRIC_SOTA_N_SUBJECTS)
    M2_PEDIATRIC_SOTA_RESULTS = _m2_ped_sota.run_m2_pediatric_sota_all(
        datasets=M2_PEDIATRIC_SOTA_DATASETS,
        manifest_root='datasets/manifests',
        output_root=M2_PEDIATRIC_SOTA_OUTPUT_ROOT,
        methods=M2_PEDIATRIC_SOTA_METHODS,
        n_subjects=M2_PEDIATRIC_SOTA_N_SUBJECTS,
        force=M2_PEDIATRIC_SOTA_FORCE,
        require_all_tools=True,
        require_all_complete=not M2_PEDIATRIC_SOTA_ALLOW_FAILURES,
    )
    print('Resultats longs:', M2_PEDIATRIC_SOTA_RESULTS['long_csv'])
    print('Resume:', M2_PEDIATRIC_SOTA_RESULTS['summary_csv'])
    try:
        display(M2_PEDIATRIC_SOTA_RESULTS['summary'])
    except Exception:
        print(M2_PEDIATRIC_SOTA_RESULTS['summary'])
else:
    print('M2-Pediatric-SOTA-Scalable pret. Definir RUN_M2_PEDIATRIC_SOTA=1 pour lancer.')
